### About this notebook

**Update (v3):** This version reruns the open-sourced submission (v1) with **4 passes on all 25 demo games**, using a runtime budget scaled to match the time available per game in the competition. I couldn’t complete this run before publishing, as I had run out of Kaggle compute. The notebook is otherwise identical.

**For details on the solution, see my [write-up](https://www.kaggle.com/writeups/dfranzen/arc-agi-3-milestone-2).**

This notebook contains my solution for **ARC Prize 2026 — ARC-AGI-3, Milestone 2**, built on [Tufa Labs’ Duck harness](https://www.kaggle.com/code/jeroencottaar/tufa-labs-duck-harness-june-30-milestone-winner). Full credit goes to Jeroen Cottaar and Tufa Labs for the original Duck solver, harness, and notebook.

The attached source bundle is an unchanged copy of Tufa Labs’ original bundle. **All my modifications to the harness are included in the patch at the beginning of this notebook**, which is applied automatically during setup. The patch is based on the original [Duck harness](https://github.com/Tufalabs/duck-harness) at commit `7652836`.

I also switched to [Qwen3.8-Flash-Next](https://huggingface.co/Qwen/Qwen3.8-Flash-Next), using [Intel’s AutoRound quantization](https://huggingface.co/Intel/Qwen3.8-Flash-Next-W4A16-AutoRound) and a separate quantized [MTP draft checkpoint](https://huggingface.co/albucino/Qwen3.8-Flash-Next-W4A16-FP8PLE) provided by Albucino for speculative decoding. Inference runs through [Pennyroyal, John Pezzulli’s SGLang fork](https://github.com/jpezzulli/sglang-rtxpro6000), v2.5.3, based on commit [`d00d88e`](https://github.com/jpezzulli/sglang-rtxpro6000/commit/d00d88efc8d6281b12be4f4073126aec95038c55), with contributions and patches from [Mamy Ratsimbazafy](https://github.com/mratsim/sglang-qwen38fn-sm120-turbo) and [Gabriel Olympie](https://github.com/gabrielolympie/sglang-flashnext-sm120), plus my own additional patches.

Third-party code and model weights remain subject to their respective licenses.

**Running your own copy:** After copying this notebook, manually select the **RTX PRO 6000** GPU before running it. Keep all attached source, wheelhouse, and model inputs enabled.

For more information, see my [GitHub repository](https://github.com/da-fr/arc-agi-3-solution).

## 0. Harness patch

Patch to add our modifications to the original tufa harness.

In [ ]:
%%writefile /kaggle/harness-changes.patch
diff --git a/ARC3-Inference/inference/agent/action_names.py b/ARC3-Inference/inference/agent/action_names.py
index 4b8dfa3..1ef0773 100644
--- a/ARC3-Inference/inference/agent/action_names.py
+++ b/ARC3-Inference/inference/agent/action_names.py
@@ -1,6 +1,7 @@
 """Shared action-name mapping between model-facing labels and engine actions."""
 from __future__ import annotations
 
+import os
 from typing import Iterable
 
 
@@ -17,8 +18,32 @@ ENGINE_TO_MODEL_ACTION = {
 MODEL_TO_ENGINE_ACTION = {value: key for key, value in ENGINE_TO_MODEL_ACTION.items()}
 
 
+def reset_exposed() -> bool:
+    """Opt-in RESET advertising and snippet guards; off preserves legacy use."""
+    return os.environ.get("EXPOSE_RESET", "off").strip().lower() in {
+        "on", "1", "true", "yes",
+    }
+
+
+def undo_exposure_mode() -> str:
+    """Read at call time; unset/invalid values preserve the original wiring.
+
+    tufa: advertise ACTION7 verbatim but reject execution (legacy behavior).
+    off: hide ACTION7/UNDO and reject execution.
+    on: advertise UNDO and accept either name for engine ACTION7.
+    """
+    mode = os.environ.get("EXPOSE_UNDO", "tufa").strip().lower()
+    return mode if mode in {"tufa", "off", "on"} else "tufa"
+
+
 def to_model_action(name: str | None) -> str:
     raw = str(name or "").strip().upper()
+    if raw in {"ACTION7", "UNDO"}:
+        mode = undo_exposure_mode()
+        if mode == "off":
+            return ""
+        if mode == "on":
+            return "UNDO"
     return ENGINE_TO_MODEL_ACTION.get(raw, raw)
 
 
@@ -26,6 +51,8 @@ def to_engine_action(name: str | None) -> str | None:
     raw = str(name or "").strip().upper()
     if not raw:
         return None
+    if raw in {"ACTION7", "UNDO"}:
+        return "ACTION7" if undo_exposure_mode() == "on" else None
     if raw in ENGINE_TO_MODEL_ACTION:
         return raw
     return MODEL_TO_ENGINE_ACTION.get(raw)
diff --git a/ARC3-Inference/inference/agent/noop_repeat_guard.py b/ARC3-Inference/inference/agent/noop_repeat_guard.py
new file mode 100644
index 0000000..314e47a
--- /dev/null
+++ b/ARC3-Inference/inference/agent/noop_repeat_guard.py
@@ -0,0 +1,178 @@
+"""Known-no-op repeat guard.
+
+Records ``(level, interior_state_hash, action_signature)`` triples that were
+observed to change nothing outside the board border, and refuses to execute
+them again in a bit-identical state.
+
+Two deliberate choices:
+
+* The state key uses ``_interior_state_hash`` - the same border-cropped hash
+  the death ledger uses - so "the same state" means the same thing everywhere
+  in the harness, and a ticking HUD timer does not make every state unique.
+
+* Blocking is overridable. A blocked action arms a single slot; if the model
+  immediately re-issues that same action, it executes. This avoids the
+  self-sealing failure of a hard block: once an action is on the list, a hard
+  block guarantees no contradicting evidence can ever arrive, so a mechanic
+  gated on hidden state (a counter, a cooldown, an off-screen timer) would be
+  foreclosed forever. Each insistence costs exactly one action and the entry
+  survives, so the model can always insist again.
+
+Known limitation: the harness does not receive per-action frame counts, so an
+action whose effect appears only in intermediate animation frames (a rejected
+click, a bounce) looks identical to one that did nothing, and will be
+recorded. The override path is the only recourse there, which is why this
+guard defaults to off.
+"""
+from __future__ import annotations
+
+import re
+from collections import OrderedDict
+from typing import Any, Callable
+
+
+_DISPLAY_COORDS_RE = re.compile(
+    r"^([A-Z0-9_]+)\(\s*(?:ROW\s*=\s*)?(-?\d+)\s*,\s*(?:COL\s*=\s*)?(-?\d+)\s*\)$"
+)
+
+
+def action_signature(action: Any) -> str:
+    """Stable signature for one action. Mouse coordinates are part of the
+    identity: a click is only "the same action" at the same cell.
+
+    Accepts both shapes the harness uses - the dict form the sandbox passes
+    (`{"action": "MOUSE", "row": 4, "col": 7}`) and the display form the death
+    ledger stores (`"MOUSE(row=4, col=7)"`) - and normalizes them to the same
+    string, so a verdict recorded through one path is found through the other.
+    Callers must pass MODEL-facing names (UP/DOWN/...), not engine names
+    (ACTION1/ACTION2/...).
+    """
+    if isinstance(action, dict):
+        name = str(action.get("action") or action.get("name") or "").strip().upper()
+        row = action.get("row")
+        col = action.get("col")
+        if row is not None or col is not None:
+            return f"{name}({row},{col})"
+        return name
+    text = " ".join(str(action or "").split()).upper()
+    match = _DISPLAY_COORDS_RE.match(text)
+    if match:
+        return f"{match.group(1)}({match.group(2)},{match.group(3)})"
+    return text
+
+
+class NoopRepeatGuard:
+    """Per-level store of state+action combinations known to do nothing."""
+
+    def __init__(self, *, max_states_per_level: int = 512, max_actions_per_state: int = 16) -> None:
+        self._max_states = max(1, int(max_states_per_level))
+        self._max_actions = max(1, int(max_actions_per_state))
+        self._states: "OrderedDict[str, OrderedDict[str, None]]" = OrderedDict()
+        self._level: Any = None
+        # One slot per guard kind. A shared slot let whichever guard ran first
+        # consume the other's override: the death check runs before the no-op
+        # check, cleared the arm on every non-match, and re-armed it, so the
+        # no-op guard's "issue it again and it will execute" never fired. A
+        # mouse-only game observed 291 replies and one executed action.
+        self._armed_noop: tuple[str, str] | None = None
+        self._armed_death: tuple[str, str] | None = None
+
+    # -- level scoping -------------------------------------------------------
+    def note_level(self, level: Any) -> None:
+        """Drop everything when the level changes: a new layout means old
+        state hashes describe boards that no longer exist. Deaths and resets
+        deliberately do NOT clear the store - the board returns to the level
+        start, so recorded no-ops still hold and carry across attempts."""
+        if level != self._level:
+            self._level = level
+            self._states.clear()
+            self._armed_noop = None
+            self._armed_death = None
+
+    # -- recording -----------------------------------------------------------
+    def observe(self, state_hash: str, action_sig: str, *, gameplay_changed: Any) -> None:
+        if not state_hash or not action_sig:
+            return
+        if gameplay_changed is False:
+            entry = self._states.get(state_hash)
+            if entry is None:
+                entry = OrderedDict()
+                self._states[state_hash] = entry
+                while len(self._states) > self._max_states:
+                    self._states.popitem(last=False)
+            entry[action_sig] = None
+            while len(entry) > self._max_actions:
+                entry.popitem(last=False)
+        elif gameplay_changed is True:
+            # Evidence contradicts a recorded no-op: drop it. Reachable via the
+            # override path, where a blocked action is executed after all.
+            entry = self._states.get(state_hash)
+            if entry is not None and action_sig in entry:
+                del entry[action_sig]
+                if not entry:
+                    del self._states[state_hash]
+
+    # -- blocking ------------------------------------------------------------
+    def should_block(self, state_hash: str, action_sig: str) -> bool:
+        """True when this combination is a known no-op AND the model has not
+        just been told so. Arms the override slot when it blocks."""
+        if not state_hash or not action_sig:
+            return False
+        key = (state_hash, action_sig)
+        if self._armed_noop == key:
+            # the model saw the refusal and insisted: let it through, and
+            # re-arm nothing so a third attempt is blocked again
+            self._armed_noop = None
+            return False
+        entry = self._states.get(state_hash)
+        if not entry or action_sig not in entry:
+            self._armed_noop = None
+            return False
+        self._armed_noop = key
+        return True
+
+    def note_executed(self, state_hash: str, action_sig: str) -> None:
+        """Any executed action that is not the armed one clears the slot, so
+        the override only applies to an immediate repeat."""
+        key = (state_hash, action_sig)
+        if self._armed_noop and self._armed_noop != key:
+            self._armed_noop = None
+        if self._armed_death and self._armed_death != key:
+            self._armed_death = None
+
+    # -- death verdicts ------------------------------------------------------
+    def should_block_death(
+        self,
+        state_hash: str,
+        action_sig: str,
+        is_known_fatal: "Callable[[str, str], bool]",
+    ) -> bool:
+        """Same block-then-insist contract as `should_block`, but the verdict
+        comes from the death ledger rather than this store.
+
+        The ledger already keeps, per recorded attempt, an `actions` list and
+        an index-aligned `hashes` list of interior state hashes taken BEFORE
+        each action - so "this action, from this exact board, killed us" is a
+        lookup, not a sequence match. Matching per position rather than by
+        sequence prefix is what makes an interleaved or differently-routed
+        approach to the same fatal move still get caught, while a batch whose
+        earlier actions changed the board correctly does NOT match.
+
+        Deaths are more likely than no-ops to hinge on state the board does
+        not show (a hazard's facing, a timer), so blocking must stay
+        overridable: under permadeath a hard block could foreclose the only
+        path through a level with no way to ever discover the error.
+        """
+        if not state_hash or not action_sig:
+            return False
+        key = (state_hash, action_sig)
+        if self._armed_death == key:
+            self._armed_death = None
+            return False
+        if not is_known_fatal(state_hash, action_sig):
+            # deliberately does NOT clear the slot: an action this guard has no
+            # verdict on is none of its business, and clearing here would drop
+            # an override armed for a different action
+            return False
+        self._armed_death = key
+        return True
diff --git a/ARC3-Inference/inference/agent/priority_pace_reference.json b/ARC3-Inference/inference/agent/priority_pace_reference.json
new file mode 100644
index 0000000..9c624f8
--- /dev/null
+++ b/ARC3-Inference/inference/agent/priority_pace_reference.json
@@ -0,0 +1,16 @@
+{
+  "fallback_tokens": 45924.98353694247,
+  "by_level": {
+    "1": 27011.05963663083,
+    "2": 51072.88442280241,
+    "3": 48932.24773612239,
+    "4": 56352.32702962253,
+    "5": 55796.95578535604,
+    "6": 67264.12383313512,
+    "7": 40699.37351375245,
+    "8": 22696.312661663567
+  },
+  "method": "Kaplan-Meier restricted mean token cost, capped at 120000 tokens",
+  "training": "1417 reached-level episodes from historical transcripts; pooled by level number, no game identity at inference",
+  "limitations": "Historical config differs from current config. Reference is a bounded cost scale, not an unbiased uncapped completion time. Last survival value carried to cap beyond observed support."
+}
diff --git a/ARC3-Inference/inference/agent/priority_scheduler.py b/ARC3-Inference/inference/agent/priority_scheduler.py
new file mode 100644
index 0000000..14bb787
--- /dev/null
+++ b/ARC3-Inference/inference/agent/priority_scheduler.py
@@ -0,0 +1,134 @@
+"""Pure scheduler inputs and scoring; never reads hidden game metadata."""
+from __future__ import annotations
+
+import math
+from dataclasses import dataclass
+
+
+# Expected future level-weight sum, conditional on completing the current level.
+# 80k tokens are shared by ALL future levels; efficiency is NOT baked in.
+# Source: arc3-levelup-chances, pooled_late_provisional, N6..10_H80000 / 0.8.
+# KM duration distributions: separate levels 1-4, pooled observed levels 5-8
+# for levels 5 onward. Levels 9-10 are extrapolated. Costs rounded up to 250
+# tokens. 336 historical passes / 1,417 reached levels; not a contest fit.
+# Tuples are indexed by current_level - 1. The final level has no future tail.
+TAIL_LOOKUP_80K: dict[int, tuple[float, ...]] = {
+    6: (3.9082328014, 5.0756939323, 5.3532912917, 5.6080111343,
+        4.0898867400, 0.0),
+    7: (3.9262240771, 5.1669653335, 5.6943514417, 6.6500041602,
+        6.6562876217, 4.7715345300, 0.0),
+    8: (3.9291260268, 5.1855629875, 5.7851444897, 7.0053702980,
+        7.8471367942, 7.7045641090, 5.4531823200, 0.0),
+    9: (3.9295002558, 5.1885168225, 5.8035897108, 7.0958580947,
+        8.2469236991, 9.0442694281, 8.7528405964, 6.1348301100, 0.0),
+    10: (3.9295397677, 5.1888936715, 5.8065473797, 7.1136691211,
+         8.3474656955, 9.4884771003, 10.2414020621, 9.8011170838,
+         6.8164779000, 0.0),
+}
+
+
+# Heuristic continuation value by levels remaining AFTER the current level:
+# 3 or more -> 8; 2 -> 7; 1 -> 5; 0 -> 0. These are final B values,
+# so neither score normalization nor the empirical efficiency discount applies.
+TAIL_LOOKUP_REMAINING: dict[int, tuple[float, ...]] = {
+    n: (8.0,) * (n - 3) + (7.0, 5.0, 0.0) for n in range(6, 11)
+}
+
+
+def parse_total_levels(value: object) -> int | None:
+    """Validate public level-count metadata, then clip to the table domain."""
+    if value is None or isinstance(value, bool):
+        return None
+    try:
+        number = float(value)
+    except (TypeError, ValueError, OverflowError):
+        return None
+    if not math.isfinite(number) or number <= 0 or not number.is_integer():
+        return None
+    return min(10, max(6, int(number)))
+
+
+@dataclass
+class ProgressPace:
+    completed: int = 0
+    log_cost_ratio: float = 0.0
+
+    def record_completion(self, tokens_used: float, reference_tokens: float) -> None:
+        observation = math.log(max(1.0, tokens_used) / max(1.0, reference_tokens))
+        self.log_cost_ratio = (
+            observation if self.completed == 0
+            else 0.6 * self.log_cost_ratio + 0.4 * observation
+        )
+        self.completed += 1
+
+    def cost_multiplier(self) -> float:
+        confidence = self.completed / (self.completed + 3.0)
+        return math.exp(min(math.log(2.0), max(math.log(0.5),
+                                             confidence * self.log_cost_ratio)))
+
+
+@dataclass(frozen=True)
+class PrioritySnapshot:
+    level: int
+    actions: int
+    tokens: float
+    cost_multiplier: float = 1.0
+    # Appended to preserve four-argument callers and older pickled snapshots.
+    total_levels: int | None = None
+
+
+def priority_value(
+    state: PrioritySnapshot, *, endgame: bool = False,
+    tail_fraction: float | None = None, human_actions: float = 25.0,
+    action_scale: float = 115.0, token_scale: float = 62000.0,
+    action_weight: float = 0.25, tail_base: float = 6.0, tail_cap: float = 6.0,
+    normalize_score: bool = False, tail_lookup: bool | str = False,
+    tail_efficiency: float = 0.8,
+) -> int:
+    """No phase offset: all candidates must be evaluated at the same time.
+
+    tail_fraction=None retains the endgame rule. Supplying a fraction
+    keeps the likelihood proxy active and fades the tail instead of switching.
+    The likelihood proxy is heuristic, not a calibrated completion probability.
+
+    tail_lookup=True uses the empirical 80k table; "remaining" uses the
+    fixed 8/7/5/0 table without an efficiency discount. False keeps the capped
+    tail. normalize_score scales only A, never B or the endgame's 0.25 floor.
+    """
+    level = max(1, state.level)
+    total_levels = parse_total_levels(getattr(state, "total_levels", None)) or 10
+    actions, tokens = max(0, state.actions), max(0.0, state.tokens)
+    pace = min(2.0, max(0.5, state.cost_multiplier))
+    efficiency = (human_actions / (human_actions + actions)) ** 2
+    current_value = level * efficiency
+    if normalize_score:
+        # Normalize immediate reward A to a ten-level reference (sum 1..10).
+        # B is a separately chosen continuation value, not normalized reward.
+        current_value *= 55.0 / (total_levels * (total_levels + 1) / 2)
+    if endgame and tail_fraction is None:
+        # Preserve the original arithmetic when normalization is disabled.
+        value = (
+            level * 0.25 + current_value if normalize_score
+            else level * (0.25 + efficiency)
+        ) / pace * 100
+    else:
+        if tail_lookup == "remaining":
+            index = min(total_levels, level) - 1
+            tail = TAIL_LOOKUP_REMAINING[total_levels][index]
+        elif tail_lookup:
+            future_efficiency = (
+                min(1.0, max(0.0, tail_efficiency))
+                if math.isfinite(tail_efficiency) else 0.8
+            )
+            index = min(total_levels, level) - 1
+            tail = TAIL_LOOKUP_80K[total_levels][index] * future_efficiency
+        else:
+            tail = min(tail_cap, tail_base + level - 1)
+        if tail_fraction is not None:
+            tail *= min(1.0, max(0.0, tail_fraction))
+        chance = (
+            action_weight * 0.5 ** ((actions / action_scale) ** 2)
+            + (1.0 - action_weight) * 0.5 ** ((tokens / (token_scale * pace)) ** 2)
+        )
+        value = (current_value + tail) * chance / pace * 100
+    return max(1, int(value))
diff --git a/ARC3-Inference/inference/agent/prompts.py b/ARC3-Inference/inference/agent/prompts.py
index 2e59574..52351db 100644
--- a/ARC3-Inference/inference/agent/prompts.py
+++ b/ARC3-Inference/inference/agent/prompts.py
@@ -2,6 +2,62 @@
 
 from inference.utils.grid_utils import ARC_COLOR_LEGEND
 
+LEVEL_TRANSFER_SYSTEM_GUIDANCE = (
+    "- Levels usually build on mechanics learned in earlier levels, especially the most "
+    "recent one. Carry forward supported knowledge as a starting hypothesis, while "
+    "re-checking anything contradicted by new evidence. New levels often introduce "
+    "additional mechanics, sometimes through unfamiliar board elements. These additions "
+    "are often important for solving the level. The goal may remain the same but require "
+    "new mechanics to reach it, or the goal itself may change.\n"
+)
+
+LEVEL_START_USER_PROMPT = (
+    "You have completed the previous level. `current_frame` now contains the starting "
+    "board of the next level; any accompanying current-grid image shows this new board. "
+    "Build a new plan for this layout rather than continuing the previous level's action "
+    "sequence.\n\n"
+    "Start from the mechanics you established on the previous level; do not rediscover "
+    "them without reason. Inspect the new board for unfamiliar elements, changed "
+    "arrangements, or interactions your previous understanding does not explain. New "
+    "elements often introduce mechanics needed to solve this level, so prioritize small, "
+    "informative tests when their behavior is unclear.\n\n"
+    "Reassess the goal: does the previous objective still apply, now requiring the new "
+    "mechanics, or does the evidence suggest a different objective? Combine retained "
+    "knowledge with new findings to plan for this board."
+)
+
+
+def apply_level_transfer_system_guidance(prompt: str) -> str:
+    """Opt-in system wording; the caller gates ARC3_LEVEL_TRANSFER_GUIDANCE.
+
+    Preserve the original constants for the default arm. The first user prompt
+    after completion gives the new-board instructions; the system retains flag
+    definitions and stop instructions for failure and run completion.
+    """
+    return (
+        prompt.replace(
+            "- Levels often build on earlier mechanics, but layouts and interactions can still change between levels.\n",
+            LEVEL_TRANSFER_SYSTEM_GUIDANCE,
+        )
+        .replace(
+            "- Strategies may transfer loosely across levels, but layouts and mechanics can change. Re-check the new board before repeating a plan.\n",
+            "",
+        )
+        .replace(
+            "- Re-ground on the newest frame after any score increase or abrupt scene change; the returned board may already be the next level.\n",
+            "",
+        )
+        .replace(
+            "- `WIN` means the whole game is solved. Mid-run level completion is more likely to appear as a score increase while play continues.\n",
+            "- `WIN` means the whole game is solved.\n",
+        )
+        .replace(
+            "- If an action result reports `game_over`, `run_complete`, `level_completed`, or `done`, stop acting immediately and re-ground on the next turn.\n",
+            "- If an action result reports `game_over`, `run_complete`, or `done`, stop acting immediately and re-ground on the next turn.\n",
+        )
+    )
+
+
 TOOL_CALL_FORMAT_GUIDANCE = (
     "When calling `python`, emit exactly the tool-call format shown elsewhere in this prompt for this model. "
     "Use only that format; do not add markdown fences, prose wrappers, or alternate tool-call syntax. "
@@ -34,13 +90,50 @@ VISUAL_GAME_ADDENDUM = (
     "- For `MOUSE`, pass `row` and `col` integer arguments. `row` is vertical position, `col` is horizontal position.\n"
 )
 
+ACTION_INFO_ADDENDUM = (
+    "\n\nAction meanings (use only actions currently available):\n"
+    "- `UP`, `DOWN`, `LEFT`, and `RIGHT` are directional controls; what they affect "
+    "depends on the game.\n"
+    "- When available, `SPACE` performs a game-specific action, such as interacting, "
+    "selecting, rotating, attaching/detaching, or executing. Test its effect rather "
+    "than assuming what it does.\n"
+    "- When available, `MOUSE` clicks a board location. Pass integer `row` and `col` "
+    "fields from 0 to 63. Coordinates are zero-based from the top-left: `row` "
+    "increases downward and `col` increases rightward.\n"
+)
+
+UNDO_INFO_ADDENDUM = (
+    "- When available, `UNDO` reverses a previous action, usually the last turn. "
+    "Check what it restores. It cannot recover a failed attempt after game over.\n"
+)
+
+RESET_INFO_ADDENDUM = (
+    "- When available, `RESET` usually restores the current level to its starting "
+    "state, including the remaining-action/time bar, while keeping completed levels. "
+    "Use it to recover from an unrecoverable position or start a different approach. "
+    "RESET itself counts as one action, and actions already spent still count toward "
+    "your score. RESET must be the first game action in a Python snippet; inspection "
+    "or computation may precede it.\n"
+)
+
+GAMEPLAY_CHANGED_ADDENDUM = (
+    # One region, named once and referred back to twice. "outside that margin"
+    # and "inside that border" both read as possibly including the border
+    # cells themselves; "ignores those edge cells" states the rule before any
+    # geometry, and "further in" cannot be read as the border.
+    "- `board_changed` is true when ANY cell differs, including cells within {border} of the "
+    "edge where a timer or remaining-steps bar usually sits; `gameplay_changed` ignores those "
+    "edge cells and is true only when a cell further in differs. An action that only shrinks "
+    "a timer bar in those edge cells sets `board_changed` but not `gameplay_changed`.\n"
+)
+
 STRUCTURED_RUNTIME_STATE_ADDENDUM = (
     "\n\nRuntime variables inside every `python` tool call:\n"
     "- `current_frame` is a lightweight frame view for the latest environment state.\n"
     "- `current_frame` exposes only `.ascii`, `.step`, `.level`, `.shape`, and `.segmentation`.\n"
     "- `current_frame.ascii` is a single newline-delimited string containing the latest board rendered with the letter-coded ARC color symbols.\n"
     "- `current_frame.segmentation` parses the board into objects. It returns `{'nodes': [...], 'adjacency_list': [...]}`.\n"
-    "- Each node in `segmentation['nodes']` is one 4-connected same-color object with: `id` (index, ordered top-most-left-most), `color` (ARC color character), `hash` (a signature of the object's color and shape that ignores its position -- equal hashes mean the same object regardless of where it is, so use it to track an object across frames or to spot multiple identical objects in one frame), `pixels` (cell count), `boundary` (clockwise outer-perimeter corner points as `[row, col]`), and `children` (ids of objects fully enclosed by this one).\n"
+    "- Each node in `segmentation['nodes']` is one 4-connected same-color object with: `id` (index, ordered top-most-left-most), `color` (ARC color character), `hash` (a signature of the object's color and shape that ignores position AND rotation -- equal hashes indicate matching color and shape up to rotation; they do not establish object identity), `rotation` (degrees clockwise from the object's canonical orientation: 0/90/180/270, reduced to 0/90 for 2-fold symmetric shapes, `None` for fully symmetric ones), `rotational_symmetry` (1, 2, or 4), `pose_hash` (the rotation-SENSITIVE variant, if you need to match exact orientation), `shape_hash` (color-independent pose shape signature), `pixels` (cell count), `boundary` (clockwise outer-perimeter corner points as `[row, col]` cell coordinates), and `children` (ids of objects fully enclosed by this one).\n"
     "- `segmentation['adjacency_list']` is a list of `[i, j]` node-id pairs whose objects share an edge.\n"
     
     "- `current_frame.step` is the current environment step count.\n"
@@ -50,20 +143,48 @@ STRUCTURED_RUNTIME_STATE_ADDENDUM = (
 
     "- `history` is a chronological list of action/frame snapshots.\n"
     "- `history` is a Python list of objects, not a dict.\n"
-    "- Each history entry exposes only `.action` and `.frame`; entries are not subscriptable like `entry['action']`.\n"
+    "- Each history entry exposes `.action`, `.frame`, and `.result`; entries are not subscriptable like `entry['action']`.\n"
     "- Each `history[i].frame` is the frame after `history[i].action`; each frame exposes only `.ascii`, `.step`, `.level`, `.shape`, and `.segmentation`.\n"
     "- Important history semantics: when `history` is non-empty, `history[-1].frame` is the same latest/post-action board as `current_frame`. It is not the previous board. To inspect the state before the latest action, use `previous_frame` or `history[-2].frame` when available.\n"
     "- `previous_frame` is the frame before the most recent real environment action, or `None` if no previous frame is available.\n"
     "- `last_action` is the most recent real environment action name/display, or `None` before any real action.\n"
     "- `last_action_frame` is the post-action frame for `last_action`; it matches `current_frame` after a real action.\n"
-    "- `transitions` is a chronological list of actual action transitions, excluding the initial seeded frame. Each transition exposes `.action`, `.before_frame`, `.after_frame`, `.frame` (alias of `.after_frame`), and `.result`.\n"
-    "- `last_transition` is `transitions[-1]` or `None`. Its `.result` mirrors `last_action_result`; older transitions may have an empty `.result`. For before/after diffs, compare `last_transition.before_frame` to `last_transition.after_frame`; do not compare `current_frame` to `history[-1].frame`.\n"
-    "- `last_action_result` is the persisted result dict from the most recent `action(...)` call. It remains available across later Python inspection calls that do not call `action(...)`, and is `{}` before any action result exists. Read transition metadata from fields/keys such as `last_action_result['board_changed']`, `last_action_result['done']`, `last_action_result['level_completed']`, `last_action_result['game_over']`, `last_action_result['run_complete']`, `last_action_result['reward']`, and `last_action_result['valid_actions']`.\n"
+    "- `transitions` is a chronological list of actual action transitions, excluding the initial seeded frame. Each transition exposes `.action`, `.before_frame`, `.after_frame`, `.frame` (alias of `.after_frame`), and `.result`. Each `.result` describes that individual action and is retained with its history entry.\n"
+    "- `last_transition` is `transitions[-1]` or `None`. A refused action creates no transition and does not overwrite existing transition results. For before/after diffs, compare `last_transition.before_frame` to `last_transition.after_frame`; do not compare `current_frame` to `history[-1].frame`.\n"
+    "- `last_action_call_result` contains the result of the latest model-issued `action(...)` call, including batch totals, executed/skipped actions, and stop reasons. `action(...)` returns this result. It remains available across later Python inspection calls and analyzer turns, and is `{}` before any action call result exists.\n"
+    "- On the next analyzer turn after automatic RESET, `current_frame` shows the restarted level. The fatal action and RESET are separate transitions. Each transition result has `automatic=True` for a harness-initiated action and `automatic=False` for a model-issued action. Automatic RESET does not overwrite `last_action_call_result`: it still describes the call that caused death.\n"
     "- `valid_actions` is the current list of valid action names.\n"
     "- Call `action(actions)` to execute one or more real environment actions from Python.\n"
     "- Pass `action(actions)` a list like `['LEFT']` or `[{'action': 'MOUSE', 'row': 4, 'col': 7}]`.\n"
     "- One action usually returns one frame, but a single action can result in a short multi-frame animation.\n"
-    "- After `action(actions)` returns, `current_frame`, `previous_frame`, `history`, `transitions`, `valid_actions`, and `last_action_result` are refreshed.\n"
+    "- After `action(actions)` returns, `current_frame`, `previous_frame`, `history`, `transitions`, `valid_actions`, and `last_action_call_result` are refreshed.\n"
+)
+
+ANIMATION_ADDENDUM_TIMELINE = (
+    "- `last_animation_timeline` is a diff timeline of those frames: it "
+    "shows which cells changed at each step. It is a dict with `action`, `frames` and "
+    "`steps`, one entry per step that changed anything. Each entry has `step`, "
+    "`changed`, `bbox`, and "
+    "either `changes` (the cells, as `old>new @ (row,col) ...`) or `transitions` (a "
+    "count per colour change, when there were too many cells to list).\n"
+)
+
+ANIMATION_ADDENDUM = (
+    "- One action can return a short animation. `current_frame` is its final frame.\n"
+    "- When the final executed action of the latest model-issued call animated, `last_action_call_result['animation']` "
+    "reports `frames`, `transient_pixels` and `transient_bbox`. Transient cells "
+    "changed and then changed BACK during the animation, so they appear in no frame "
+    "you can otherwise reach - not in `current_frame`, not in `previous_frame`, not "
+    "in `history`.\n"
+    "- `last_animation_frames` is the frames of the last executed action that "
+    "animated, as views with "
+    "`.ascii`, `.shape` and `.segmentation`, the same as `current_frame`. Index 0 is "
+    "the first frame of the animation and the last entry is the board it settled on. "
+    "After a death these are the frames of the FATAL action, not of the automatic "
+    "reset that followed it. Crop "
+    "their `.ascii` yourself; printing a whole 64x64 frame will not fit the tool "
+    "budget. Empty when the action did not animate, and reading them costs no "
+    "in-game action.\n"
 )
 
 MULTIMODAL_CONTEXT_ADDENDUM = (
@@ -73,7 +194,7 @@ MULTIMODAL_CONTEXT_ADDENDUM = (
     "- You can use images and other tools to understand the game state and guide your strategy, each may be useful depending on the current uncertainty.\n"
 )
 
-PYTHON_ADDENDUM = (
+PYTHON_ADDENDUM_HEAD = (
     "\n\nPython tool guidance:\n"
     "- Use `current_frame.segmentation` as your primary view of the board -- objects, colors, containment, adjacency, and cross-frame object hashes.\n"
     "- Use `current_frame.ascii` only to read a small, specific region of the board when `segmentation` is not enough; never use it to scan or summarize the whole board.\n"
@@ -82,9 +203,13 @@ PYTHON_ADDENDUM = (
     "- The only tool is `python`; call it with one ephemeral `code` string.\n"
     "- Always inspect `current_frame`, `history`, and `valid_actions` from Python instead of reasoning from the raw board by eye.\n"
     "- For the most recent change, compare `previous_frame` to `current_frame`, or `last_transition.before_frame` to `last_transition.after_frame`. `history[-1].frame` is the current frame, so comparing it to `current_frame` only compares the board to itself.\n"
-    "- Maintain a compact working world model: what entities or regions exist, what actions seem to do, what the goal likely is, what remains uncertain, and what plan best fits the evidence so far.\n"
+)
+
+# split so the world-model lines can be swapped in at their original
+# position - immediately before the IMPORTANT search guidance - rather than
+# appended after the whole addendum
+PYTHON_ADDENDUM_TAIL = (
     "- IMPORTANT: Especially when the game is about making an agent navigate to a target, it is usually safer to write an explicit search algorithm such as BFS. More generally, when the objective is understood but the best action order is unclear, pathfinding, flood fill, BFS, DFS, beam search, shortest-path search, limited action-sequence search, or custom heuristics are all valid.\n"
-    "- Optimize for the shortest reliable sequence that advances the current goal as described by your world model. If confidence is low, program a discriminating probe and revise the world model from the result.\n"
     "- Once the important state variables and action effects are sufficiently understood, stop probing and search in the inferred state space.\n"
     "- Inspect current and history frames from Python instead of describing frames freehand.\n"
     "- Never print or echo full board frames. Return only compact derived summaries such as object lists, diffs, coordinates, counts, or tiny local crops.\n"
@@ -98,17 +223,124 @@ PYTHON_ADDENDUM = (
     "- `action(...)` accepts an ordered list of one or more actions. Once your code has selected a reliable sequence, it is often useful to batch it.\n"
     "- You can also call `action(...)` multiple times in one Python snippet, including inside loops. Each call updates the preloaded variables before execution continues.\n"
     "- If an action result reports `game_over`, `run_complete`, `level_completed`, or `done`, stop acting immediately and re-ground on the next turn.\n"
+    "- Flag semantics: `game_over` = this attempt FAILED (death or a limit ran out) — the level auto-resets to its initial state and completed levels are kept; it never means the run is won. `level_completed` = advanced one level. `run_complete`/`done` = the whole game is won. Deaths come either from the action itself (e.g., a hazard cell) or from a step/time budget depleting — watch for a bar at the grid border that shrinks each action, and count remaining budget into your plans.\n"
+)
+
+FRAME_DIFF_HINT_ADDENDUM = (
+    "- `frame_diff(before=None, after=None)` compares two frames. Omitted arguments default to `previous_frame` and `current_frame`, respectively.\n"
+    "- It returns a dictionary containing `changed_cell_count` (an integer) and lists named `moved`, `rotated`, `appeared`, `disappeared`, `changed_color`, and `resized`. Unchanged objects are omitted.\n"
+    "- Object-level changes are inferred by matching same-color, 4-connected components between frames: cells connect through shared edges, not diagonally. A game object may therefore be split into several components, or touching same-color objects may form one component.\n"
+    "- `moved` describes components matched at different positions. Matching does not establish persistent object identity: with multiple similar components, the reported correspondence may be ambiguous or incorrect. Movement combined with rotation includes `rotated_by` in degrees clockwise; `rotated` describes rotation in place.\n"
+    "- `changed_color` requires identical occupied cells with a different color. Entries contain `at`, `pixels`, `before_color`, `after_color`, `before_hash`, and `after_hash`.\n"
+    "- `resized` describes same-color components with overlapping footprints whose size changed. This can include backgrounds and budget bars. `at_before` and `at_after` can differ.\n"
+    "- `appeared` and `disappeared` entries include `bbox` in `[r0, c0, r1, c1]` order. Object hashes use the same rotation-invariant shape-and-color representation as segmentation; they are not unique object identities, and multiple objects can share a hash.\n"
+    "- Examples: `frame_diff()` compares the latest action’s before/after frames; `frame_diff(history[-3].frame, history[-2].frame)` compares two explicitly selected historical frames, when available.\n"
+)
+
+STEP_VERIFICATION_ADDENDUM = (
+    "- During a multi-action plan, verify important effects by comparing the actual game "
+    "state with your predictions—for example, changed cells, object properties, positions, "
+    "or counters. Make these comparisons in code so the batch can stop when a mismatch "
+    "undermines the remaining plan. Test uncertain interactions before committing to long "
+    "batches.\n"
+    "- If your search finds no solution under your current model of the game, remember "
+    "that the game is solvable. Reconsider your mechanics, goal, search implementation, "
+    "or search limits, including interactions with new elements. Take a targeted action "
+    "to test an uncertain rule or overlooked interaction, then update your model from "
+    "the result.\n"
+)
+
+WORLD_MODEL_ADDENDUM = (
+    "- Maintain a compact working world model: what entities or regions exist, what actions "
+    "seem to do, what the goal likely is, what remains uncertain, and what plan best fits the "
+    "evidence so far.\n"
+    "- Optimize for the shortest reliable sequence that advances the current goal as described "
+    "by your world model. If confidence is low, program a discriminating probe and revise the "
+    "world model from the result.\n"
+)
+
+WORLD_MODEL_FREE_ADDENDUM = (
+    "- Optimize for the shortest reliable sequence that advances the current goal. If "
+    "confidence is low, program a discriminating probe and revise your approach from the "
+    "result.\n"
+)
+
+DEATH_GUARD_ADDENDUM = (
+    "- An action that ended a previous attempt from this EXACT board state is refused before "
+    "execution: it raises `KnownDeathActionError`, nothing runs, and no attempt is lost. "
+    "`stop_reason` is `known_death`. The guard checks every position of a batch, not just the "
+    "first, so a fatal move is caught whichever route reached it. To repeat such an action "
+    "deliberately, issue it again as your very next action.\n"
 )
 
+NOOP_GUARD_ADDENDUM = (
+    "- An action already proven to change nothing in this EXACT board state is refused before "
+    "execution: it raises `KnownNoOpActionError`, which is NOT catchable and ends the snippet - "
+    "nothing runs, "
+    "and no action budget is spent. `stop_reason` is `known_noop`. If you have a specific reason "
+    "to believe the action will behave differently now, issue that same action again as your very "
+    "next action and it will execute.\n"
+)
+
+PREFER_TOOL_CALLS_LINE = (
+    # The closing clause used to warn that a long chain "wastes the whole turn".
+    # That overstated it - a truncated reply keeps its reasoning in history, and
+    # what is lost is only that nothing executed - and it aimed at the tail,
+    # since truncation touched about 4% of replies. The speed claim is the
+    # stronger one and it is true: decode runs at tens of tokens per second per
+    # stream, while a tool call is a short generation plus a mostly-cached
+    # prefill, so 2,000 tokens of further reasoning costs far more wall-clock
+    # than running the code that would settle the question.
+    "- Reading and computing cost nothing; only `action(...)` spends the level budget. When you "
+    "are unsure, prefer another tool call over more reasoning: the code answers what the "
+    "reasoning would only guess at, and running it is faster than thinking your way to the "
+    "same answer.\n"
+)
+
+EPHEMERAL_FUNCTIONS_LINE = (
+    "- The `python` tool code is not saved between calls, so rewrite any custom utility logic you still need.\n"
+)
+
+PERSISTENT_FUNCTIONS_LINE = (
+    "- Python variables reset between tool calls. Eligible functions you define are "
+    "retained automatically {lifetime}. Call your retained functions directly without "
+    "repeating their definitions. Redefine a function when you need to change it.\n"
+    "- For a function to be retained, its dependencies must also be available next call. "
+    "Pass snippet-specific data as arguments. You may use Python builtins, provided "
+    "globals such as `current_frame`, and other retained functions. Provided globals "
+    "are refreshed for each call. {import_guidance} Use plain "
+    "top-level definitions without decorators or annotations, and literal defaults. "
+    "Tool results report which functions were retained and explain any rejection.\n"
+)
+
+
 COMPACT_TOOL_SESSION_ADDENDUM = (
     "\n\nTool session rules:\n"
     "- You have exactly one tool: `python`.\n"
     f"- {TOOL_CALL_FORMAT_GUIDANCE}\n"
-    "- The `python` tool code is not saved between calls, so rewrite any custom utility logic you still need.\n"
+    "{persistent_functions}"
     "- You can call the `python` tool as many times as you want per step. Investigate until your code has a clear probe or plan.\n"
     "- Do not ration tool calls when the state is unclear. Spend extra tool calls to confirm what changed between frames and whether the last action affected gameplay state or only HUD elements such as countdown bars.\n"
-    "- After `action(...)` returns, the structured runtime state is refreshed before the next Python statement and before the next tool call. Inspection-only Python calls do not clear `last_action_result`.\n"
+    "{prefer_tool_calls}"
+    "- After `action(...)` returns, the structured runtime state is refreshed before the next Python statement and before the next tool call. Inspection-only Python calls do not clear `last_action_call_result`.\n"
     "- Each `python` tool call has a hard time limit of 30 seconds.\n"
     "- Tool responses are capped to about {tool_output_tokens} tokens. If a response is cut off, the tool result will tell you that.\n"
     "- Keep code snippets short and purpose-built rather than dumping large frameworks into one call.\n"
 )
+
+
+SUMMARY_REQUEST_PROMPT = (
+    "Summarize everything above: what you have established about this game, what you have "
+    "ruled out, what you were in the middle of, and anything else a later turn would need. "
+    "Keep uncertain ideas tentative, and preserve unresolved alternatives. "
+    "Do not act and do not call any tool - write the summary only.\n"
+    # No "the turns after this one stay visible": the summary is appended at
+    # the end of history, so nothing follows it when it is written. That
+    # sentence belonged to an earlier design where the summary was inserted
+    # ahead of a retained tail, and here it points at messages that do not
+    # exist yet.
+    "Write it to stand completely on its own. Everything above this point will eventually be "
+    "dropped and replaced by this summary, and earlier summaries go with it, so anything you "
+    "leave out is gone.\n"
+    "Length is not a problem here; leaving something out is."
+)
diff --git a/ARC3-Inference/inference/agent/python_tool_sandbox.py b/ARC3-Inference/inference/agent/python_tool_sandbox.py
index ac24747..58a94b3 100644
--- a/ARC3-Inference/inference/agent/python_tool_sandbox.py
+++ b/ARC3-Inference/inference/agent/python_tool_sandbox.py
@@ -3,6 +3,7 @@ from __future__ import annotations
 
 import inspect
 import json
+import logging
 import os
 import queue
 import signal
@@ -14,12 +15,15 @@ import textwrap
 import time
 from typing import Any, Callable
 
+from inference.utils import frame_diff as _frame_diff_module
 from inference.utils import segmentation as _segmentation
+from inference.utils import retained_functions as _retained_functions
 from inference.utils.grid_utils import ARC_COLOR_CHARS
 
 
 _SANDBOX_BOOTSTRAP = textwrap.dedent(
     r"""
+    import ast
     import builtins
     import contextlib
     import io
@@ -37,12 +41,22 @@ _SANDBOX_BOOTSTRAP = textwrap.dedent(
 
     __SEGMENTATION_SOURCE__
 
+    __FRAME_DIFF_SOURCE__
+
+    __RETAINED_FUNCTIONS_SOURCE__
+
     HOST_STDOUT = sys.stdout
 
     SAFE_MODULES = {
         "bisect",
         "collections",
         "copy",
+        # pure-Python sequence comparison, no I/O. Measured across 32
+        # transcripts it was imported 9 times and USED none of them - the model
+        # reaches for it on "compare two things" and then writes the comparison
+        # by hand. Allowing it therefore buys robustness, not capability: a
+        # stray import line no longer costs the whole snippet.
+        "difflib",
         "fractions",
         "functools",
         "heapq",
@@ -56,6 +70,25 @@ _SANDBOX_BOOTSTRAP = textwrap.dedent(
         "string",
     }
     SAFE_BUILTINS = {
+        # Exception types. Only four were present, so anything narrower than
+        # `except Exception` failed at the except clause and took the snippet
+        # with it. These are inert classes and expose nothing; SystemExit and
+        # KeyboardInterrupt stay out because the sandbox uses them for its own
+        # timeout and shutdown handling.
+        "ArithmeticError",
+        "AssertionError",
+        "AttributeError",
+        "EOFError",
+        "IndexError",
+        "KeyError",
+        "LookupError",
+        "NameError",
+        "NotImplementedError",
+        "OverflowError",
+        "RecursionError",
+        "StopIteration",
+        "UnboundLocalError",
+        "ZeroDivisionError",
         "abs",
         "all",
         "any",
@@ -147,9 +180,10 @@ _SANDBOX_BOOTSTRAP = textwrap.dedent(
 
 
     class HistoryEntryView:
-        def __init__(self, *, action, frame):
+        def __init__(self, *, action, frame, result=None):
             self.action = action
             self.frame = frame
+            self.result = dict(result) if isinstance(result, dict) else {}
 
         def __str__(self):
             return f"AsciiHistoryEntryView(action={self.action!r}, frame={self.frame})"
@@ -197,12 +231,13 @@ _SANDBOX_BOOTSTRAP = textwrap.dedent(
                 HistoryEntryView(
                     action=str(entry.get("action", "")),
                     frame=_frame_from_payload(entry.get("frame")),
+                    result=entry.get("result"),
                 )
             )
         return items
 
 
-    def _transitions_from_history(history, last_action_result):
+    def _transitions_from_history(history):
         transitions = []
         for index, entry in enumerate(history):
             action = str(getattr(entry, "action", "") or "").strip()
@@ -214,11 +249,9 @@ _SANDBOX_BOOTSTRAP = textwrap.dedent(
                     action=action,
                     before_frame=before_frame,
                     after_frame=entry.frame,
-                    result={},
+                    result=entry.result,
                 )
             )
-        if transitions and isinstance(last_action_result, dict):
-            transitions[-1].result = dict(last_action_result)
         return transitions
 
 
@@ -327,15 +360,38 @@ _SANDBOX_BOOTSTRAP = textwrap.dedent(
             "result": None,
         }
         runtime_globals["__builtins__"]["__import__"] = _safe_import
+        _animation_views = []
+
+        def frame_diff(before=None, after=None):
+            # Object-level diff between two frames. Defaults: previous_frame -> current_frame.
+            # Returns only what changed; unchanged objects are omitted.
+            if before is None:
+                before = runtime_globals.get("previous_frame")
+            if after is None:
+                after = runtime_globals.get("current_frame")
+            if before is None or after is None:
+                return {"error": "need two frames (no previous_frame available?)"}
+            if before.shape != after.shape:
+                return {"changed_cell_count": -1, "note": "frame shapes differ; full redraw",
+                        "before_shape": before.shape, "after_shape": after.shape}
+            return compute_frame_diff(
+                before._grid,
+                after._grid,
+                before.segmentation.get("nodes", []),
+                after.segmentation.get("nodes", []),
+            )
 
         def _refresh_state(state_payload):
             current_frame = _frame_from_payload(state_payload.get("current_frame"))
             history = _history_from_payload(state_payload.get("history"))
-            last_action_result = state_payload.get("last_action_result")
+            last_action_call_result = state_payload.get("last_action_call_result")
             action_result = (
-                dict(last_action_result) if isinstance(last_action_result, dict) else {}
+                dict(last_action_call_result) if isinstance(last_action_call_result, dict) else {}
             )
-            transitions = _transitions_from_history(history, action_result)
+            # action_echo is printed by action(), then removed from its return value.
+            # Keep the exposed call result identical across subsequent inspections.
+            action_result.pop("action_echo", None)
+            transitions = _transitions_from_history(history)
             last_transition = transitions[-1] if transitions else None
 
             runtime_globals["current_frame"] = current_frame
@@ -350,12 +406,98 @@ _SANDBOX_BOOTSTRAP = textwrap.dedent(
                 last_transition.after_frame if last_transition is not None else None
             )
             runtime_globals["last_action"] = last_transition.action if last_transition is not None else None
+            runtime_globals["frame_diff"] = frame_diff
             runtime_globals["valid_actions"] = [str(item) for item in state_payload.get("valid_actions", [])]
-            runtime_globals["last_action_result"] = action_result
+            runtime_globals["death_ledger"] = state_payload.get("death_ledger") or {"attempts": []}
+            runtime_globals["last_action_call_result"] = action_result
+
+        # Both refusals derive from BaseException, not Exception, so a
+        # defensive `except Exception:` around action(...) - which models write
+        # routinely - cannot swallow them. A refusal means the action did not
+        # run at all, so the snippet's assumed state is already wrong and every
+        # line after it rests on a false premise. One run treated 1500
+        # consecutive refusals as proof that every cell on the board was inert.
+        # StaleStateActionError below is deliberately catchable: it reports a
+        # belief that MAY be stale rather than a verdict on the action.
+        class KnownNoOpActionError(BaseException):
+            pass  # raised INSTEAD of executing an action already proven inert here
+
+        class KnownDeathActionError(BaseException):
+            pass  # raised INSTEAD of executing an action that ended a prior attempt
+
+        # Raised INSTEAD of executing an action this snippet has already run
+        # from this exact board. Buggy loop code can spend a whole life bar
+        # getting nowhere - a cursor that wraps turns RIGHT into a short cycle,
+        # a mistaken `while pos < j: LEFT` walks away from its target forever -
+        # and the terminal latch only stops that once the game actually ends,
+        # measured at 20 to 30 wasted actions. BaseException like the two above:
+        # the shape that produces these loops is `try: action(...) except
+        # Exception:` inside a while, which would otherwise swallow the verdict
+        # and keep going. No override slot: the snippet is not reading the
+        # refusal, its code is running, and writing a new snippet IS the
+        # response - the map is rebuilt per execution.
+        class RepeatedActionInStateError(BaseException):
+            pass
+
+        # Raised when a PREVIOUS action in this snippet changed nothing, and the
+        # snippet then tries to act again. Deliberately a plain Exception, not
+        # BaseException: unlike the two refusals above this is a warning about
+        # the snippet's beliefs rather than a verdict on the action, and a model
+        # that has re-read current_frame may legitimately want to proceed.
+        class StaleStateActionError(Exception):
+            pass
+
+        runtime_globals["KnownNoOpActionError"] = KnownNoOpActionError
+        runtime_globals["KnownDeathActionError"] = KnownDeathActionError
+        runtime_globals["StaleStateActionError"] = StaleStateActionError
+        runtime_globals["RepeatedActionInStateError"] = RepeatedActionInStateError
+        # armed when an action changes nothing; scoped to THIS snippet because
+        # the closure is rebuilt per execution, so no reset path can be missed
+        _stale_state: list[str] = []
+
+        # Raised when a snippet tries to act after the attempt or level ended.
+        # Derives from BaseException, not Exception, so a defensive
+        # `try: action(...) except Exception:` around the call - which models
+        # write routinely - cannot swallow it. Continuing to execute a path
+        # computed against a board that no longer exists is never correct.
+        class TerminalStateActionError(BaseException):
+            pass
+
+        runtime_globals["TerminalStateActionError"] = TerminalStateActionError
+        # latched when an action reports a terminal result; scoped to THIS
+        # snippet because these closures are rebuilt for every execution, so
+        # no reset path can be missed and the next snippet always starts clear
+        _terminal_state = []
+
+        _TERMINAL_DETAIL = {
+            "game_over": (
+                "the previous action reported game_over. The level has been reset to its "
+                "start, so the board you planned against is gone and this action was not "
+                "executed. The next turn begins from the new state."
+            ),
+            "level_completed": (
+                "the previous action reported level_completed. A new level has been loaded, "
+                "so the board you planned against is gone and this action was not executed. "
+                "The next turn begins from the new state."
+            ),
+            "run_complete": (
+                "the previous action reported run_complete. The run is finished and this "
+                "action was not executed."
+            ),
+        }
 
         def action(actions):
+            if _terminal_state:
+                raise TerminalStateActionError(_TERMINAL_DETAIL[_terminal_state[0]])
             normalized_actions = _normalize_actions(actions)
-            _send({"type": "action", "actions": normalized_actions})
+            _send({
+                "type": "action",
+                "actions": normalized_actions,
+                # the solver decides precedence: a known death or a known no-op
+                # is a sharper message than "your state may be stale", and it
+                # carries a concrete override
+                "stale_after": _stale_state[0] if _stale_state else None,
+            })
             reply = _recv()
             if reply.get("type") == "action_error":
                 raise RuntimeError(str(reply.get("error", "action failed")))
@@ -363,28 +505,225 @@ _SANDBOX_BOOTSTRAP = textwrap.dedent(
                 raise RuntimeError("Invalid action response from sandbox host.")
             action_result = reply.get("action_result") or {}
             action_results.append(action_result)
+            # Cached animations describe the previous action. Invalidate the
+            # existing views so aliases also refresh, even if a guard raises.
+            for view in _animation_views:
+                view.invalidate()
             _refresh_state(reply.get("state") or {})
+            if action_result.get("stop_reason") == "reset_not_first":
+                raise RuntimeError(str(action_result.get("stop_detail")))
+            if action_result.get("stop_reason") == "stale_state":
+                raise StaleStateActionError(
+                    str(action_result.get("stop_detail") or "stale state")
+                )
+            if action_result.get("stop_reason") == "known_death":
+                raise KnownDeathActionError(
+                    str(action_result.get("stop_detail") or "known fatal action was not executed.")
+                )
+            if action_result.get("stop_reason") == "repeated_action_in_state":
+                raise RepeatedActionInStateError(
+                    str(action_result.get("stop_detail") or "")
+                    or "this action was already run from this board in this snippet"
+                )
+            if action_result.get("stop_reason") == "known_noop":
+                # nothing executed and no action was spent; halt the snippet so
+                # the model re-plans instead of continuing on a false premise
+                raise KnownNoOpActionError(
+                    str(action_result.get("stop_detail") or "known no-op action was not executed.")
+                )
+            for _flag in ("run_complete", "done", "level_completed", "game_over"):
+                if action_result.get(_flag):
+                    # `done` means the run was won; report it as run_complete
+                    _terminal_state.append(
+                        "run_complete" if _flag == "done" else _flag
+                    )
+                    break
+            _echo = action_result.pop("action_echo", None)
+            if _echo:
+                # into the snippet's own stdout, so a loop can see what its
+                # call did without waiting for the next turn's opener
+                print(_echo)
+            gameplay_changed = action_result.get("gameplay_changed")
+            if gameplay_changed is True:
+                # a real change means the snippet's beliefs are current again
+                del _stale_state[:]
+            if gameplay_changed is False and action_result.get("executed"):
+                _executed = action_result.get("executed_actions")
+                _name = (
+                    _executed[-1]
+                    if isinstance(_executed, list) and _executed
+                    else action_result.get("action_display")
+                    or action_result.get("action_name")
+                    or "?"
+                )
+                # Arm rather than raise. The action executed and changed
+                # nothing, which is often exactly what the snippet went looking
+                # for - it printed the board before, and wants to print it
+                # after. Killing it here throws away the second half of the
+                # measurement AND the batch stop's own trace, which is computed
+                # and then discarded. So let the snippet finish observing, and
+                # stop it only if it tries to ACT on the stale belief.
+                if not _stale_state:
+                    _stale_state.append(str(_name))
             return action_result
 
+        def _animation_fetch(kind):
+            _send({"type": "animation", "request": {"kind": kind}})
+            reply = _recv()
+            if reply.get("type") == "animation_error":
+                raise RuntimeError(str(reply.get("error", "animation failed")))
+            if reply.get("type") != "animation_result":
+                raise RuntimeError("Invalid animation response from sandbox host.")
+            payload = reply.get("animation")
+            if kind != "frames":
+                return payload or {}
+            return [
+                FrameView(
+                    ascii=item.get("ascii", ""),
+                    step=item.get("step", 0),
+                    level=item.get("level", 0),
+                    shape=item.get("shape", (0, 0)),
+                    grid=item.get("grid", ()),
+                )
+                for item in (payload or [])
+            ]
+
         runtime_globals["action"] = action
+        if initial.get("animation_enabled"):
+            # Globals rather than attributes on last_transition: after a death
+            # the harness appends a RESET entry, so last_transition is the
+            # reset and not the action that animated. The record is simply
+            # "the last executed action that animated", and naming it that way
+            # removes the mismatch. Empty when the last executed action
+            # returned a single frame.
+            class _LazyAnimation:
+                def __init__(self, kind, empty):
+                    self._kind = kind
+                    self._empty = empty
+                    self._value = None
+
+                def _get(self):
+                    if self._value is None:
+                        self._value = _animation_fetch(self._kind) or self._empty
+                    return self._value
+
+                def invalidate(self):
+                    self._value = None
+
+                def __len__(self):
+                    return len(self._get())
+
+                def __iter__(self):
+                    return iter(self._get())
+
+                def __getitem__(self, key):
+                    return self._get()[key]
+
+                def __bool__(self):
+                    return bool(self._get())
+
+                def __contains__(self, key):
+                    return key in self._get()
+
+                def keys(self):
+                    return self._get().keys()
+
+                def get(self, key, default=None):
+                    return self._get().get(key, default)
+
+                def __repr__(self):
+                    return repr(self._get())
+
+            runtime_globals["last_animation_frames"] = _LazyAnimation("frames", [])
+            runtime_globals["last_animation_timeline"] = _LazyAnimation("timeline", {})
+            _animation_views.extend((runtime_globals["last_animation_frames"],
+                                     runtime_globals["last_animation_timeline"]))
+
         _refresh_state(initial.get("state") or {})
 
+        _persistence_enabled = bool(initial.get("persistence_enabled"))
+        _retention_options = {
+            "retain_imports": bool(initial.get("retain_imports")),
+            "repair_hints": bool(initial.get("repair_hints")),
+        }
+        _kept_pool = {}
+        _retention_rejected = []
+        _available_names = set(runtime_globals) | set(runtime_globals["__builtins__"])
+        if _persistence_enabled:
+            _before_restore = dict(runtime_globals)
+            try:
+                _kept_pool, _retention_rejected = restore_functions(
+                    initial.get("kept_functions") or [], runtime_globals, _available_names,
+                    **_retention_options)
+            except Exception:
+                # Optional cache maintenance must never prevent this snippet running.
+                runtime_globals.clear()
+                runtime_globals.update(_before_restore)
+                _kept_pool = {}
+                _retention_rejected = [{"name": "all", "reason": "restoration failed; redefine needed functions"}]
+
+        def _collect_retention_payload():
+            if not _persistence_enabled:
+                return {}
+            try:
+                kept, rejected = collect_functions(
+                    _code_text, _kept_pool, runtime_globals, _available_names,
+                    **_retention_options)
+                return {
+                    "keepable_functions": [{"name": n, "source": src} for n, src in kept.items()],
+                    "retention_rejected": _retention_rejected + rejected,
+                }
+            except Exception:
+                # Bookkeeping failures may drop helpers, but must never replace
+                # the snippet's original error/output or replay its actions.
+                return {"keepable_functions": [], "retention_rejected": [
+                    {"name": "all", "reason": "retention failed; redefine needed functions"}]}
+
         try:
-            compiled = compile(str(initial.get("code", "")), "<python_tool>", "exec")
+            _code_text = str(initial.get("code", ""))
+            _tree = ast.parse(_code_text, "<python_tool>")
+            _last_expr = None
+            if _tree.body and isinstance(_tree.body[-1], ast.Expr):
+                _last_expr = ast.Expression(_tree.body[-1].value)
+                ast.copy_location(_last_expr, _tree.body[-1])
+                _tree.body = _tree.body[:-1]
             with contextlib.redirect_stdout(stdout):
-                exec(compiled, runtime_globals, runtime_globals)
+                if _tree.body:
+                    exec(compile(_tree, "<python_tool>", "exec"), runtime_globals, runtime_globals)
+                if _last_expr is not None:
+                    _value = eval(compile(_last_expr, "<python_tool>", "eval"), runtime_globals, runtime_globals)
+                    if _value is not None:
+                        print(repr(_value))
+                        if runtime_globals.get("result") is None:
+                            runtime_globals["result"] = _value
             _send(
                 {
                     "type": "final",
+                    **_collect_retention_payload(),
                     "stdout": stdout.getvalue(),
                     "result": _json_safe(runtime_globals.get("result")),
                     "action_results": _json_safe(action_results),
                 }
             )
-        except Exception as exc:
+        except (
+            KnownNoOpActionError,
+            RepeatedActionInStateError,
+            KnownDeathActionError,
+            TerminalStateActionError,
+            Exception,
+        ) as exc:
+            # The three refusals derive from BaseException so a model's own
+            # `except Exception:` cannot swallow them - but THIS handler is the
+            # same construct, so without naming them explicitly they escaped it
+            # too, killed the child process, and the host reported a bare
+            # "Sandbox process exited unexpectedly". The refusal messages - the
+            # action name, that nothing was spent, how to override - were built
+            # and then never delivered. They stop the snippet either way; the
+            # point is that the model is told why.
             _send(
                 {
                     "type": "error",
+                    **_collect_retention_payload(),
                     "error": _sanitize_exception(exc),
                     "stdout": stdout.getvalue(),
                     "action_results": _json_safe(action_results),
@@ -395,12 +734,31 @@ _SANDBOX_BOOTSTRAP = textwrap.dedent(
     if __name__ == "__main__":
         main()
     """
-).replace("__SEGMENTATION_SOURCE__\n", inspect.getsource(_segmentation))
+).replace("__RETAINED_FUNCTIONS_SOURCE__\n", inspect.getsource(_retained_functions)).replace(
+    "__SEGMENTATION_SOURCE__\n", inspect.getsource(_segmentation)
+).replace(
+    "__FRAME_DIFF_SOURCE__\n", inspect.getsource(_frame_diff_module)
+)
+
+
+log = logging.getLogger(__name__)
 
 
 def _sanitize_host_error_text(text: str) -> str:
-    if not str(text or "").strip():
-        return "Sandbox process exited unexpectedly."
+    """What the MODEL is told when the sandbox process dies.
+
+    Deliberately fixed: the child's traceback carries host paths and internals
+    that have no business in the model's context. But the text was previously
+    read and thrown away entirely, so a crash left no trace anywhere - the one
+    thing that knows what went wrong said nothing. It now goes to the harness
+    log instead, where it is useful and nobody is prompted by it.
+    """
+    detail = str(text or "").strip()
+    if detail:
+        log.warning(
+            "sandbox process died; stderr follows:\n%s",
+            detail[-4000:],
+        )
     return "Sandbox process exited unexpectedly."
 
 
@@ -450,7 +808,11 @@ def run_sandboxed_python(
     code: str,
     timeout_seconds: int,
     initial_state: dict[str, Any],
-    action_handler: Callable[[list[dict[str, Any]]], dict[str, Any]],
+    action_handler: Callable[..., dict[str, Any]],
+    animation_handler: Callable[[dict[str, Any]], dict[str, Any]] | None = None,
+    kept_functions: list[str] | None = None,
+    retain_imports: bool = False,
+    repair_hints: bool = False,
 ) -> dict[str, Any]:
     with tempfile.TemporaryDirectory(prefix="rgb_python_tool_") as sandbox_dir:
         host_action_results: list[dict[str, Any]] = []
@@ -489,6 +851,11 @@ def run_sandboxed_python(
             process.stdin,
             {
                 "code": code,
+                "kept_functions": list(kept_functions or []),
+                "persistence_enabled": kept_functions is not None,
+                "retain_imports": retain_imports,
+                "repair_hints": repair_hints,
+                "animation_enabled": animation_handler is not None,
                 "timeout_seconds": timeout_seconds,
                 "sandbox_cwd": sandbox_dir,
                 "state": initial_state,
@@ -534,9 +901,37 @@ def run_sandboxed_python(
                 }
 
             msg_type = str(message.get("type", "")).strip()
+            if msg_type == "animation":
+                if animation_handler is None:
+                    _send_json_line(
+                        process.stdin,
+                        {"type": "animation_error",
+                         "error": "Animation frames are not available in this session."},
+                    )
+                    continue
+                try:
+                    animation_payload = animation_handler(dict(message.get("request") or {}))
+                except Exception:  # noqa: BLE001
+                    _send_json_line(
+                        process.stdin,
+                        {"type": "animation_error",
+                         "error": "animation failed in sandbox host."},
+                    )
+                    continue
+                _send_json_line(
+                    process.stdin,
+                    {"type": "animation_result", "animation": animation_payload},
+                )
+                continue
             if msg_type == "action":
                 try:
-                    action_result_payload = action_handler(list(message.get("actions") or []))
+                    # the message carries more than the action list: stale_after
+                    # tells the solver a previous action in this snippet changed
+                    # nothing, which it weighs after its own guards
+                    action_result_payload = action_handler(
+                        list(message.get("actions") or []),
+                        stale_after=message.get("stale_after"),
+                    )
                 except Exception:  # noqa: BLE001
                     _send_json_line(
                         process.stdin,
@@ -566,6 +961,9 @@ def run_sandboxed_python(
                     "result": message.get("result"),
                     "error": str(message.get("error", "") or ""),
                     "action_results": list(message.get("action_results") or host_action_results),
+                    "keepable_functions": list(message.get("keepable_functions") or []),
+                    **({"retention_rejected": message.get("retention_rejected", [])}
+                       if kept_functions is not None else {}),
                 }
 
             _wait_for_process_exit(process)
diff --git a/ARC3-Inference/inference/agent/runtime_state.py b/ARC3-Inference/inference/agent/runtime_state.py
index 03edebb..5a00139 100644
--- a/ARC3-Inference/inference/agent/runtime_state.py
+++ b/ARC3-Inference/inference/agent/runtime_state.py
@@ -2,7 +2,7 @@
 from __future__ import annotations
 
 import json
-from dataclasses import dataclass
+from dataclasses import dataclass, field
 from pathlib import Path
 from typing import Any
 
@@ -42,6 +42,7 @@ class Frame:
 class HistoryEntry:
     action: str
     frame: Frame
+    result: dict[str, Any] = field(default_factory=dict)
 
 
 def normalize_grid(raw: Any) -> tuple[tuple[int, ...], ...]:
@@ -95,13 +96,17 @@ def history_entry_from_payload(payload: Any) -> HistoryEntry | None:
     frame = frame_from_payload(payload.get("frame"))
     if frame is None:
         return None
-    return HistoryEntry(action=str(payload.get("action", "")).strip(), frame=frame)
+    return HistoryEntry(
+        action=str(payload.get("action", "")).strip(), frame=frame,
+        result=dict(payload["result"]) if isinstance(payload.get("result"), dict) else {},
+    )
 
 
 def history_entry_to_payload(entry: HistoryEntry) -> dict[str, Any]:
     return {
         "action": entry.action,
         "frame": frame_to_payload(entry.frame),
+        "result": dict(getattr(entry, "result", {}) or {}),
     }
 
 
diff --git a/ARC3-Inference/inference/agent/tool_agent.py b/ARC3-Inference/inference/agent/tool_agent.py
index 59655e9..84a9cdd 100644
--- a/ARC3-Inference/inference/agent/tool_agent.py
+++ b/ARC3-Inference/inference/agent/tool_agent.py
@@ -1,11 +1,16 @@
 """Direct OpenAI-compatible tool-calling analyzer for ARC puzzle runs."""
 from __future__ import annotations
 
+import base64
+import heapq
+import threading
 import json
 import logging
 import os
+import random
 import re
 import time
+from functools import lru_cache
 from dataclasses import dataclass
 from pathlib import Path
 from typing import Any, Callable, Optional
@@ -13,25 +18,61 @@ from urllib.parse import urlparse, urlunparse
 
 import requests
 
-from inference.agent.action_names import to_engine_action, to_model_action
+from inference.agent.noop_repeat_guard import NoopRepeatGuard, action_signature
+from inference.agent.priority_scheduler import (
+    ProgressPace, PrioritySnapshot, parse_total_levels, priority_value,
+)
+from inference.utils.animation import (
+    _transient_cells as _animation_transient_cells,
+    build_animation_frames,
+    build_animation_view,
+    describe_animation,
+)
+from inference.agent.action_names import reset_exposed, to_engine_action, to_model_action, undo_exposure_mode
+from inference.utils.retained_functions import function_signature
 from inference.agent.prompts import (
+    apply_level_transfer_system_guidance,
+    LEVEL_START_USER_PROMPT,
     COMPACT_TOOL_SESSION_ADDENDUM,
+    PERSISTENT_FUNCTIONS_LINE,
+    EPHEMERAL_FUNCTIONS_LINE,
+    SUMMARY_REQUEST_PROMPT,
+    PREFER_TOOL_CALLS_LINE,
     GAME_OVERVIEW_ADDENDUM,
-    PYTHON_ADDENDUM,
+    WORLD_MODEL_ADDENDUM,
+    WORLD_MODEL_FREE_ADDENDUM,
+    FRAME_DIFF_HINT_ADDENDUM,
+    STEP_VERIFICATION_ADDENDUM,
+    PYTHON_ADDENDUM_HEAD,
+    PYTHON_ADDENDUM_TAIL,
+    GAMEPLAY_CHANGED_ADDENDUM,
     STRUCTURED_RUNTIME_STATE_ADDENDUM,
+    ANIMATION_ADDENDUM,
+    ANIMATION_ADDENDUM_TIMELINE,
     MULTIMODAL_CONTEXT_ADDENDUM,
     TOOL_CALL_FORMAT_GUIDANCE,
     VISUAL_GAME_ADDENDUM,
+    ACTION_INFO_ADDENDUM,
+    UNDO_INFO_ADDENDUM,
+    RESET_INFO_ADDENDUM,
 )
 
 from inference.agent.vision_context import (
+    animation_composite_part,
+    animation_death_image_mode,
+    animation_image_mode,
+    animation_peak_part,
     current_grid_image_enabled,
     current_grid_image_part,
 )
 
 from inference.agent.python_tool_sandbox import run_sandboxed_python
 from inference.agent.runtime_state import Frame, HistoryEntry, RUNTIME_STATE_FILENAME, load_runtime_state
-from inference.utils.openai_compat import build_chat_payload, build_headers
+from inference.utils.openai_compat import (
+    assemble_streamed_chat_response,
+    build_chat_payload,
+    build_headers,
+)
 
 log = logging.getLogger(__name__)
 
@@ -73,6 +114,14 @@ def _get_env_bool(name: str, default: bool) -> bool:
     return default
 
 
+def _level_transfer_guidance_enabled() -> bool:
+    """Opt-in level-transfer prompts; set before constructing the agent.
+
+    ARC3_LEVEL_TRANSFER_GUIDANCE defaults off, preserving the original prompts.
+    """
+    return _get_env_bool("ARC3_LEVEL_TRANSFER_GUIDANCE", False)
+
+
 def _contains_tool_call_markup(*chunks: str) -> bool:
     for chunk in chunks:
         lowered = chunk.lower()
@@ -141,6 +190,13 @@ _LOCAL_ANALYZER_TOOL_STEPS = _get_env_int("LOCAL_ANALYZER_TOOL_STEPS", 12)
 _LOCAL_ANALYZER_TOOL_TIMEOUT = _get_env_int("LOCAL_ANALYZER_TOOL_TIMEOUT", 30)
 _LOCAL_ANALYZER_TOOL_OUTPUT_TOKENS = _get_env_int("LOCAL_ANALYZER_TOOL_OUTPUT_TOKENS", 1024)
 _LOCAL_ANALYZER_YIELD_SECONDS = _get_env_float("LOCAL_ANALYZER_YIELD_SECONDS", 0.0)
+# A wall-clock budget is really a token budget with the throughput baked in:
+# the same 45 seconds bought 2,140 generated tokens on one backend and 3,829 on
+# a faster one, so every serving change silently retuned it. Measured in
+# tokens the setting means the same thing whatever the hardware does. Zero
+# disables, as with the seconds; if both are set, whichever comes first ends
+# the turn.
+_LOCAL_ANALYZER_YIELD_TOKENS = _get_env_int("LOCAL_ANALYZER_YIELD_TOKENS", 0)
 _LOCAL_ANALYZER_ENABLE_THINKING = _get_env_bool("LOCAL_ANALYZER_ENABLE_THINKING", True)
 _LOCAL_ANALYZER_TEMPERATURE = _get_env_float("LOCAL_ANALYZER_TEMPERATURE", 0.6)
 _LOCAL_ANALYZER_TOP_P = _get_env_float("LOCAL_ANALYZER_TOP_P", 0.95)
@@ -148,15 +204,102 @@ _LOCAL_ANALYZER_TOP_K = _get_env_int("LOCAL_ANALYZER_TOP_K", 20)
 _LOCAL_ANALYZER_SEED = _get_env_int("LOCAL_ANALYZER_SEED", -1)
 _REQUEST_SAFETY_MARGIN_TOKENS = 512
 _CONTEXT_OVERFLOW_RETRY_TRIM_TOKENS = 512
-_PERSISTENT_HISTORY_ASSISTANT_TURNS = 30
+def _context_drain_floor_mode() -> bool:
+    """Whether the drain stops ABOVE the target rather than below it.
+
+    History is dropped in whole blocks, so trimming "until under the target"
+    normally overshoots by most of a block. With a 40k budget and a 12k drain
+    the band is nominally 28k-40k, but simulation with ~6k blocks gives an
+    effective 14k-38k: the average sits ~3k below the nominal midpoint and the
+    minimum is far below the target.
+
+    Floor mode drops the mandatory blocks to get under budget, then stops
+    before a drop would cross below the target - so the retained content keeps
+    a real floor and the band means what it says. The cost is trim frequency:
+    the overshoot is what lengthens the interval between trims, so removing it
+    trims about half again as often, which shortens the stable prefix and cuts
+    the prefix-cache hit rate. Off by default for that reason; the same average
+    depth is available by raising the budget, which does not touch trim
+    frequency."""
+    return _get_env_bool("ARC3_CONTEXT_DRAIN_FLOOR", False)
+
+
+def _context_drain_tokens() -> int:
+    """Hysteresis for context trimming, in tokens.
+
+    Without it, trimming drops the oldest exchange the moment the estimate
+    exceeds the budget and stops the instant it fits - so once history reaches
+    the ceiling, nearly every turn evicts a little. Each eviction is cheap in
+    tokens but total in cache cost: prefix caching keys on the token sequence
+    from position zero, so removing anything from the FRONT shifts every
+    surviving token and invalidates the whole cached prefix, not just the part
+    removed. Frequent small evictions are therefore the worst case.
+
+    With a drain set, trimming still TRIGGERS at the budget but keeps dropping
+    until it is `drain` tokens below it, so evictions become rare and large:
+    the same tokens discarded overall, a fraction of the invalidations. The
+    cost is average history depth - the window sits below the budget most of
+    the time rather than pressed against it.
+
+    Size it against tokens per turn: at ~4-5k per turn including the opener,
+    8-12k buys two or three clean turns between evictions. 0 (default) keeps
+    the original drop-until-it-fits behavior."""
+    return max(0, _get_env_int("ARC3_CONTEXT_DRAIN_TOKENS", 0))
+
+
+def _history_turn_drain() -> int:
+    """Hysteresis for the assistant-turn cap, in assistant messages.
+
+    The token trimmer got hysteresis in ARC3_CONTEXT_DRAIN_TOKENS, but the
+    turn cap runs AFTER it and had none: once history sits at the cap, every
+    commit shaves back to exactly N, so the prefix is invalidated on every
+    single turn. With a 46k window and a cap of 30 assistant messages the cap
+    is the binding limit, which is why a token drain alone changed nothing.
+
+    Set this and the cap triggers at N but drains to N - drain, so evictions
+    become periodic instead of continuous."""
+    return max(0, _get_env_int("ARC3_HISTORY_TURN_DRAIN", 0))
+
+
+def _history_drain_coalesce() -> bool:
+    """Take the deep turn cut whenever the token trimmer already dropped.
+
+    Invalidation is all-or-nothing: removing anything from the front of
+    history shifts every surviving token, so a commit that has already
+    dropped blocks for token reasons has already paid the full cache cost.
+    Dropping turns in that same commit is therefore free, whereas letting the
+    cap fire on its own a turn later costs a second invalidation. This makes
+    the two mechanisms evict together rather than alternately."""
+    return _get_env_bool("ARC3_HISTORY_DRAIN_COALESCE", False)
+
+
+def _persistent_history_assistant_turns() -> int:
+    """Cap on assistant messages kept in persistent history.
+
+    Counts ASSISTANT MESSAGES, not analyzer turns: a turn that inspects, then
+    computes, then acts spends three of them, so 30 is closer to ten or
+    fifteen turns of real history than to thirty. The cap is independent of
+    the token budget, so with a large context window it can be the binding
+    constraint while thousands of tokens sit unused - which is what the
+    default 30 does at 98k, having been chosen when the window was 32k.
+
+    0 or negative disables the cap entirely, leaving the token budget as the
+    only limit."""
+    return _get_env_int("ARC3_HISTORY_ASSISTANT_TURNS", 30)
+_WM_NUDGE_TURNS = _get_env_int("ARC3_WM_NUDGE_TURNS", 0)
+_WM_WIPE_ON_GAME_OVER = _get_env_int("ARC3_WM_WIPE_ON_GAME_OVER", 1)
+_AUTO_FRAME_DIFF = _get_env_bool("ARC3_AUTO_FRAME_DIFF", False)
+_AUTO_FRAME_DIFF_BUDGET = _get_env_int("ARC3_AUTO_FRAME_DIFF_BUDGET", 300)
+_AUTO_FRAME_DIFF_MAX_GROUP = 40
+_DIFF_IMAGE = _get_env_bool("ARC3_DIFF_IMAGE", False)
 _RESPONSE_META_MAX_CHARS = 4000
 
 _PYTHON_TOOL_DESCRIPTION = (
     "Run one ephemeral Python snippet against preloaded ASCII game state. Available globals: "
     "`current_frame`, `previous_frame`, `history`, `transitions`, `last_transition`, "
-    "`valid_actions`, `last_action_result`, "
+    "`valid_actions`, `last_action_call_result`, "
     "and `action(actions)` for executing one or more real environment actions. "
-    "`current_frame` and each `history[*].frame` expose only `.ascii`, `.segmentation`, `.step`, and `.level`; "
+    "`current_frame` and each `history[*].frame` expose only `.ascii`, `.segmentation`, `.step`, `.level`, and `.shape` (a `(rows, cols)` tuple); "
     "`history[-1].frame` is the current post-action frame, not the previous frame. "
     "For before/after diffs, compare `previous_frame` to `current_frame` or use `last_transition.before_frame` and `.after_frame`. "
     "For MOUSE, pass `row` and `col` integer fields; legacy x/y fields are rejected. "
@@ -198,8 +341,10 @@ def _terminal_action_stop_detail(reason: str | None) -> str:
         return "No further actions were executed because the run is already complete."
     if reason == "game_over":
         return (
-            "No further actions were executed because the previous action reached GAME_OVER; "
-            "the runner will auto-reset before the next analyzer turn."
+            "No further actions were executed because the previous action reached GAME_OVER "
+            "(attempt failed; the run is NOT finished). The runner will auto-reset the level "
+            "before your next turn. Cause is either the action itself (hazard) or a depleted "
+            "step/time budget (usually a shrinking bar at the grid border)."
         )
     if reason == "level_completed":
         return (
@@ -215,6 +360,1324 @@ def _display_action_number(action_num: int) -> int:
     return max(1, int(action_num) + 1)
 
 
+def _stale_summary_lines(summary: dict[str, Any]) -> list[str]:
+    """Rendering for a step summary carried over from an earlier exchange of
+    the same turn. States that explicitly instead of re-narrating the old
+    outcome as if it had just happened. Two ways a turn can reach here:
+    a snippet ran and executed nothing (default wording), or the exchange
+    produced no tool call at all (stale_reason='no_tool_call') - in which case
+    claiming a snippet ran would be false."""
+    if summary.get("stale_reason") == "no_tool_call":
+        lines = [
+            "Your previous exchange this turn produced no tool call and "
+            "executed no actions."
+        ]
+    else:
+        lines = [
+            "Your last python snippet executed NO actions (it only computed or "
+            "inspected state - that is fine)."
+        ]
+    executed_actions = summary.get("executed_actions")
+    rendered = ""
+    if isinstance(executed_actions, list):
+        rendered = ", ".join(
+            str(name).strip() for name in executed_actions[:10] if str(name).strip()
+        )
+    if summary.get("game_over"):
+        fatal = str(summary.get("fatal_action") or "")
+        lines.append(
+            "Reminder of the still-pending outcome from earlier this turn: the "
+            f"executed sequence ({rendered or 'see earlier message'}) ended in "
+            "GAME OVER"
+            + (f" after '{fatal}'" if fatal else "")
+            + "; the level was auto-reset and you are planning the fresh attempt "
+            "now. Do NOT re-submit that sequence - the full diagnosis "
+            "instructions were given in an earlier message this turn."
+        )
+    elif summary.get("level_transition"):
+        lines.append(
+            "Reminder: the earlier sequence this turn completed the level; you "
+            "are now on the new level."
+        )
+    elif rendered:
+        lines.append(
+            "For reference, the most recent executed sequence earlier this turn "
+            f"was: {rendered}."
+        )
+    return lines
+
+
+def _state_identity_border() -> int:
+    """Border cropped when deciding whether two boards are THE SAME STATE.
+
+    Distinct from ARC3_NOOP_GUARD_BORDER, which asks a different question. The
+    batch stop asks "did this action accomplish anything meaningful", and there
+    cropping the HUD is right: a timer ticking down is not progress. The repeat
+    guards and the death ledger ask "have I been here before", and there
+    cropping is a liability - two boards with identical interiors but different
+    HUD are treated as one state, so a mechanic gated on anything the HUD
+    encodes (a countdown, a phase, a collected count) can be mis-recorded. The
+    resulting block is permanent and self-sealing: the action never executes,
+    so no evidence can arrive to correct it.
+
+    Defaults to 0 - exact whole-frame matching, which is safe by construction
+    and matches the reference implementation. Raise it to trade that guarantee
+    for more matches."""
+    return max(0, _get_env_int("ARC3_STATE_IDENTITY_BORDER", 0))
+
+
+def _interior_state_hash(grid: list[list[int]]) -> str:
+    """Hash identifying whether two boards are the same state, for the repeat
+    guards and the death ledger. Crops ARC3_STATE_IDENTITY_BORDER cells, which
+    defaults to 0 - the whole frame, HUD included. Small grids hash whole."""
+    import hashlib
+
+    border = _state_identity_border()
+    rows = len(grid)
+    cols = max((len(row) for row in grid), default=0)
+    if rows > 2 * border and cols > 2 * border:
+        core = [tuple(row[border:cols - border]) for row in grid[border:rows - border]]
+    else:
+        core = [tuple(row) for row in grid]
+    return hashlib.sha1(repr(core).encode("utf-8")).hexdigest()
+
+
+_LEAKED_TOOLCALL_RE = None
+
+
+def _extract_leaked_tool_calls(message: dict[str, Any]) -> dict[str, Any]:
+    """Fallback for servers without a tool-call parser for the model's
+    dialect: when a response carries no structured tool_calls but the content
+    contains Nemotron-style <TOOLCALL>[{...}]</TOOLCALL> markup, parse it and
+    synthesize the OpenAI-shaped tool_calls the harness expects. Malformed
+    payloads leave the message untouched. Disable with
+    ARC3_TOOLCALL_TEXT_FALLBACK=0."""
+    global _LEAKED_TOOLCALL_RE
+    if not _get_env_bool("ARC3_TOOLCALL_TEXT_FALLBACK", True):
+        return message
+    if not isinstance(message, dict) or message.get("tool_calls"):
+        return message
+    content = message.get("content")
+    if not isinstance(content, str) or "<TOOLCALL>" not in content:
+        return message
+    import json as _json
+    import re as _re
+    import uuid as _uuid
+
+    if _LEAKED_TOOLCALL_RE is None:
+        _LEAKED_TOOLCALL_RE = _re.compile(
+            r"<TOOLCALL>\s*(\[.*?\])\s*</TOOLCALL>", _re.DOTALL
+        )
+    match = _LEAKED_TOOLCALL_RE.search(content)
+    if match is None:
+        return message
+    try:
+        parsed = _json.loads(match.group(1))
+    except _json.JSONDecodeError:
+        return message
+    if not isinstance(parsed, list):
+        parsed = [parsed]
+    calls: list[dict[str, Any]] = []
+    for item in parsed:
+        if not isinstance(item, dict) or not item.get("name"):
+            continue
+        arguments = item.get("arguments", {})
+        if not isinstance(arguments, str):
+            arguments = _json.dumps(arguments, ensure_ascii=False)
+        calls.append(
+            {
+                "id": f"call_{_uuid.uuid4().hex}",
+                "type": "function",
+                "function": {
+                    "name": str(item["name"]),
+                    "arguments": arguments,
+                },
+            }
+        )
+    if not calls:
+        return message
+    remainder = (content[: match.start()] + content[match.end():]).strip()
+    updated = dict(message)
+    updated["tool_calls"] = calls
+    updated["content"] = remainder or None
+    return updated
+
+
+def _env_int(name: str, default: int) -> int:
+    """A numeric override that cannot take the run down on a typo."""
+    try:
+        return int(os.environ.get(name, "") or default)
+    except (TypeError, ValueError):
+        return default
+
+
+def _env_float(name: str, default: float) -> float:
+    try:
+        return float(os.environ.get(name, "") or default)
+    except (TypeError, ValueError):
+        return default
+
+
+_RETRYABLE_HTTP_STATUSES = {429, 500, 502, 503, 504, 520, 522, 524}
+# Establishing a TCP connection is fast or it is not happening: a server that
+# has exited refuses instantly, and one that is unreachable will not become
+# reachable inside the read budget. Separate from the read timeout so a dead
+# endpoint fails in seconds rather than waiting out the generation budget.
+_CONNECT_TIMEOUT_SECONDS = 10.0
+
+
+def _http_retry_delay_seconds(
+    attempt: int,
+    retry_after: str | None,
+    base_seconds: float,
+    max_seconds: float = 60.0,
+) -> float:
+    """Exponential backoff with jitter, or the server's Retry-After when it
+    parses as a number and asks for longer. Capped at 60s per wait.
+
+    Retry-After raises the wait, never lowers it. A provider under load can
+    answer `Retry-After: 1` to every request - observed at 100 consecutive
+    retries roughly 1.3s apart - and honouring that literally means hammering
+    an endpoint that has already said no, which neither clears the rate limit
+    nor helps the run. The configured base is the floor; the server can still
+    ask for more.
+    """
+    backoff = min(max_seconds, base_seconds * (2 ** attempt)) + random.uniform(0.0, 1.0)
+    if retry_after:
+        try:
+            # max_seconds caps the exponential, not an explicit request from the
+            # server: a Retry-After of 30 against a 5s ceiling still waits 30,
+            # because the endpoint said so and hammering it earlier helps no one
+            return max(backoff, min(60.0, float(retry_after)))
+        except ValueError:
+            pass
+    return backoff
+
+
+def _post_with_retries(
+    post_fn,
+    *,
+    retries: int,
+    base_seconds: float,
+    max_seconds: float = 60.0,
+    initial_seconds: float = 0.0,
+    sleep=time.sleep,
+    monotonic=time.monotonic,
+):
+    """Retry transient HTTP failures (rate limits, gateway errors) in place,
+    BEFORE they escalate to turn abort + history rollback. Returns the final
+    response either way; non-retryable statuses return immediately and flow
+    into the existing error handling. Rate-limit retries here preserve the
+    turn's partial tool work, which the abort path deliberately discards."""
+    # A negative count means keep trying. Zero keeps its obvious meaning - do
+    # not retry - so a typo disables retries rather than hanging the run.
+    unlimited = retries < 0
+    budget = max(0, retries)
+    # A floor on total retry time, used once. On Kaggle the benchmark is
+    # released before the server has finished loading, so the first request of
+    # every game meets a refused connection and has to wait it out - minutes,
+    # not the seconds the ordinary budget allows. Expressed as time rather than
+    # a count because the count that covers a 15-minute load would be absurd at
+    # any sensible delay, and would then apply to every later failure too.
+    started_at = monotonic()
+
+    def _budget_label(attempt_index: int) -> str:
+        """What the log should call the limit. Only reached outside a grace
+        period now - inside one the retry lines are suppressed entirely, since
+        "1/3" describes a budget that is not the one in force."""
+        if unlimited:
+            return "unlimited"
+        if attempt_index >= budget:
+            return f"{initial_seconds:.0f}s grace"
+        return str(budget)
+
+    def _exhausted(attempt_index: int) -> bool:
+        if unlimited:
+            return False
+        if attempt_index < budget:
+            return False
+        return (monotonic() - started_at) >= initial_seconds
+
+    def _announce_grace(reason: str) -> None:
+        """Said once, on the first failure, so the grace is visible from the
+        start. Without it the log shows three ordinary-looking retries before
+        the "Ns grace" label appears, and a reader watching a cold start cannot
+        tell a long wait from a broken one."""
+        nonlocal announced
+        if announced or initial_seconds <= 0:
+            return
+        announced = True
+        log.warning(
+            "first analyzer request failed (%s); retrying for up to %.0fs while "
+            "the endpoint comes up, then falling back to %d retries per request",
+            reason, initial_seconds, budget,
+        )
+
+    announced = False
+    response = None
+    attempt = -1
+    while True:
+        attempt += 1
+        try:
+            response = post_fn()
+        except requests.exceptions.RequestException as exc:
+            # The server was never reached - refused, DNS, a socket error, or a
+            # connect timeout. Without this the exception leaves the turn
+            # entirely and the analyzer-level retry starts it over, discarding
+            # whatever tool work it had already done; retrying the request in
+            # place keeps it. A ReadTimeout is deliberately not caught: the
+            # response body stalled, which has its own path that can yield with
+            # history preserved.
+            if _is_read_timeout(exc) or _exhausted(attempt):
+                raise
+            _announce_grace(type(exc).__name__)
+            delay = _http_retry_delay_seconds(attempt, None, base_seconds, max_seconds)
+            # Silent while the grace is carrying the loop: at 14 games polling a
+            # cold endpoint every 5s for 15 minutes this line alone would be
+            # ~2,500 entries, burying everything else. The announcement above
+            # already said a wait is in progress, and the outcome is logged when
+            # the loop ends either way.
+            if initial_seconds <= 0:
+                log.warning(
+                    "analyzer endpoint unreachable (%s); retry %d/%s in %.1fs",
+                    type(exc).__name__, attempt + 1, _budget_label(attempt), delay,
+                )
+            sleep(delay)
+            continue
+        status = getattr(response, "status_code", 200)
+        if status not in _RETRYABLE_HTTP_STATUSES or _exhausted(attempt):
+            return response
+        _announce_grace(f"HTTP {status}")
+        retry_after = None
+        headers = getattr(response, "headers", None)
+        if headers is not None:
+            retry_after = headers.get("Retry-After")
+        delay = _http_retry_delay_seconds(attempt, retry_after, base_seconds, max_seconds)
+        if initial_seconds <= 0:
+            log.warning(
+                "analyzer endpoint returned HTTP %s; retry %d/%s in %.1fs",
+                status, attempt + 1, _budget_label(attempt), delay,
+            )
+        sleep(delay)
+    return response
+
+
+_CONTROL_MESSAGE_KEY = "_arc3_control"
+
+_YIELD_RESUME_PROMPT_WITH_TOOLS = (
+    "You yielded control on the turn time budget. Your tool results above are "
+    "still valid - do NOT restart your analysis or re-inspect the board from "
+    "scratch. Continue from them and call `action(actions)` with your best "
+    "next action or short batch."
+)
+
+_YIELD_RESUME_PROMPT_WITH_TOOLS_NEUTRAL = (
+    "You yielded control on the turn time budget. This is a scheduling "
+    "interruption, not a signal that your analysis is complete. Your tool "
+    "results above are still valid - do NOT restart or re-inspect the board "
+    "from scratch. Pick up where you left off: keep investigating if you still "
+    "need to, and call `action(actions)` once your evidence supports a choice."
+)
+
+
+def _yield_resume_tone() -> str:
+    """Wording for the commonest resumption message.
+
+    'commit' (default) is the original text, which reads as though the model
+    has finished: it asks for the best next action. At a 60s budget that fires
+    after very little deliberation, so it can push a commitment the evidence
+    does not yet support.
+
+    'neutral' keeps the load-bearing part - do not restart, your results are
+    valid - while naming the interruption for what it is and leaving the
+    decision to continue investigating open. The anti-stall backstops are
+    untouched: the no-tool-call nudge, the tool-step-exhaustion prompt and the
+    timeout prompt all still push to act, and they fire on exhausted work
+    rather than on elapsed time.
+
+    'minimal' goes further and does not mention the yield at all. A yield that
+    executed nothing and a snippet that only inspected are the same situation
+    from the model's side - nothing happened, carry on - and the scheduling
+    reason is harness detail it cannot act on. The wording echoes the mid-turn
+    stale opener so a resumption reads as a continuation rather than as its own
+    kind of event, keeping only the part that is load-bearing: the earlier
+    results are still good.
+
+    Read the arm as actions performed against tokens generated: the change
+    should raise tokens per action if it is working as intended, and the
+    question is whether the actions get better.
+    """
+    tone = os.environ.get("ARC3_YIELD_RESUME_TONE", "").strip().lower()
+    return tone if tone in ("neutral", "minimal") else "commit"
+
+_YIELD_RESUME_PROMPT_NO_TOOLS = (
+    "You spent the entire turn budget reasoning without calling a tool. Stop "
+    "deliberating and call the `python` tool NOW: inspect "
+    "`current_frame.segmentation`, then call `action(actions)` with the best "
+    "available action. Acting on a partial model beats another turn of "
+    "analysis."
+)
+
+_YIELD_RESUME_PROMPT_WITH_TOOLS_MINIMAL = (
+    # Says nothing about yielding. From the model's side a yield with nothing
+    # executed and an inspect-only snippet are the same situation - nothing
+    # happened, carry on - and the scheduling reason is harness detail it
+    # cannot act on. Deliberately echoes the mid-turn stale opener it already
+    # sees often, so a resumption reads as a continuation rather than as a
+    # distinct kind of message. Kept: the earlier results are still good, so
+    # do not start over.
+    "Your last python snippet executed no actions (it only computed or "
+    "inspected state - that is fine). Your tool results above are still "
+    "valid: continue from them rather than re-inspecting the board from "
+    "scratch."
+)
+
+_YIELD_RESUME_PROMPT_NO_TOOLS_MINIMAL = (
+    # The with-tools wording would be false here: there was no snippet at all,
+    # so there are no results to continue from and the useful instruction is
+    # where to start instead.
+    "Your previous exchange produced no tool call and executed no actions. "
+    "Start with the `python` tool: inspect `current_frame.segmentation`, then "
+    "call `action(actions)` when your evidence supports a choice."
+)
+
+_YIELD_RESUME_PROMPT_NO_TOOLS_NEUTRAL = (
+    # The same edit as the with-tools pair: keep what is load-bearing - that
+    # nothing executed, and the concrete first step - and drop the pressure to
+    # commit. This fires when the yield budget expired on a turn that never
+    # called a tool, so the model really did spend the turn reasoning; what is
+    # unwarranted is telling it that acting on a partial model is better, when
+    # the budget expiring says nothing about whether its analysis was close.
+    "You yielded control on the turn time budget without calling a tool. This "
+    "is a scheduling interruption, not a signal that your analysis is complete "
+    "- but nothing has been executed and the board is unchanged. Start with "
+    "the `python` tool: inspect `current_frame.segmentation`, then call "
+    "`action(actions)` once your evidence supports a choice."
+)
+
+_DEAD_REASONING_STUB = "(previous attempt produced no tool call)"
+
+_DEGENERATE_CONTENT_STUB = (
+    "(previous reply was highly repetitive and has been omitted; do not "
+    "continue that pattern - write concise plain prose)"
+)
+
+_STATE_ONLY_CAPTION = (
+    # Leads with the assurance rather than with "Current board state":
+    # _build_user_message appends its own "Current grid image:" immediately
+    # below, so opening with a second caption put two labels for one picture
+    # next to each other. The load-bearing part is that the opener has not
+    # gone stale, not that a board follows.
+    "Nothing has been executed since the turn opener above, so the step, "
+    "level and valid actions stated there still hold."
+)
+
+
+def _yield_resume_prompt_mode() -> str:
+    """How a resumed turn re-opens.
+
+    'full'       (default) rebuild the complete opener - 2-3k tokens restating
+                 state the retained opener already carries.
+    'short'      a brief continuation sentence and nothing else.
+    'state_only' the continuation sentence plus a one-line caption and the
+                 CURRENT board image(s). Measurements showed 'short' holding up
+                 while images were attached and degrading without them, which
+                 is consistent with the omitted text (batch summary, state
+                 line, diffs, ledger) being what the image was covering for.
+                 This mode makes that explicit: drop the boilerplate, keep the
+                 state, in the cheapest form available.
+    """
+    mode = os.environ.get("ARC3_YIELD_RESUME_PROMPT", "").strip().lower()
+    return mode if mode in ("short", "state_only") else "full"
+
+
+_RESUME_PROMPT_TIMEOUT = (
+    "The previous request to the model timed out and was abandoned. Your tool "
+    "results above are still valid - do NOT restart your analysis. Continue "
+    "from them and call `action(actions)` with your best next action, keeping "
+    "your reasoning short so the next request completes."
+)
+
+_RESUME_PROMPT_TOOL_STEPS_EXHAUSTED = (
+    "You used every available tool step this turn without calling "
+    "`action(actions)`. Your tool results above are still valid - do NOT begin "
+    "a new investigation. Act NOW with the best option you already have."
+)
+
+
+def _yield_resume_prompt(reason: str) -> str:
+    """Continuation text for a resumed turn, chosen by why it ended.
+
+    'timeout'      - the request exceeded the analyzer read timeout: the work
+                     already done this turn is intact, the model just needs to
+                     wrap up quickly.
+    'tool_steps'   - the step budget ran out mid-investigation: results exist,
+                     the missing thing is the action.
+    'yield_tools'  - the time budget ran out with tool work done: same premise,
+                     softer framing.
+    'yield_bare'   - the turn produced only reasoning, so there is nothing
+                     above to continue from and the useful instruction is to
+                     stop deliberating.
+    """
+    if reason == "timeout":
+        return _RESUME_PROMPT_TIMEOUT
+    if reason == "tool_steps":
+        return _RESUME_PROMPT_TOOL_STEPS_EXHAUSTED
+    tone = _yield_resume_tone()
+    if reason == "yield_tools":
+        if tone == "minimal":
+            return _YIELD_RESUME_PROMPT_WITH_TOOLS_MINIMAL
+        return (
+            _YIELD_RESUME_PROMPT_WITH_TOOLS_NEUTRAL
+            if tone == "neutral"
+            else _YIELD_RESUME_PROMPT_WITH_TOOLS
+        )
+    if tone == "minimal":
+        return _YIELD_RESUME_PROMPT_NO_TOOLS_MINIMAL
+    return (
+        _YIELD_RESUME_PROMPT_NO_TOOLS_NEUTRAL
+        if tone == "neutral"
+        else _YIELD_RESUME_PROMPT_NO_TOOLS
+    )
+
+
+def _fatal_lookahead_enabled() -> bool:
+    """Block a batch that replays a recorded fatal path BEFORE spending any of
+    its actions, rather than running the harmless prefix and refusing only the
+    final move.
+
+    Default off keeps the per-position behaviour: the prefix executes and the
+    model ends up at the brink, which costs those actions but leaves it
+    somewhere it can still choose differently. With lookahead on, nothing is
+    spent and the model must resubmit a shorter batch to reach that point."""
+    return _get_env_bool("ARC3_DEATH_GUARD_LOOKAHEAD", False)
+
+
+def _yield_on_timeout_enabled() -> bool:
+    """Treat an analyzer read timeout as a yield instead of a failed turn.
+
+    The default path clears preserve_history, so the whole invocation's tool
+    calls and results are rolled back and the retry re-derives them from
+    scratch - on Kaggle that is a ~900s round trip discarded. Nothing can have
+    executed when a timeout fires (an executing dispatch breaks the loop before
+    another request is issued), so preserving the exchanges is safe: history
+    ends on a tool message and the resumed invocation appends a user message,
+    exactly the shape every yielded turn already produces.
+
+    Scoped to read timeouts only. Connection errors and HTTP failures keep the
+    rollback path, and context-overflow handling is untouched - preserving an
+    over-long context would just reproduce the failure."""
+    return _get_env_bool("ARC3_YIELD_ON_TIMEOUT", True)
+
+
+def _is_read_timeout(exc: BaseException) -> bool:
+    """requests raises ReadTimeout for a stalled response body; ConnectTimeout
+    (a sibling of the same base) means the server was never reached, so it is
+    deliberately excluded."""
+    return isinstance(exc, requests.exceptions.ReadTimeout)
+
+
+def _stub_dead_reasoning_enabled() -> bool:
+    """Replace reasoning-only assistant replies (no tool calls, no content)
+    with a short stub IN-TURN. Without this the dead trace is re-sent on every
+    later request of the same turn - observed at ~35k characters, truncated
+    mid-sentence by finish_reason=length, i.e. a broken thought replayed as
+    context."""
+    return _get_env_bool("ARC3_STUB_DEAD_REASONING", False)
+
+
+_HISTORY_IMAGE_PLACEHOLDER = "[grid image omitted from history]"
+
+
+def _history_image_keep() -> int | None:
+    """How many of the most recent image-bearing history MESSAGES keep their
+    images. None (unset) preserves today's behavior - images accumulate for as
+    long as their message is retained, ~400 tokens each at
+    MULTIMODAL_UPSCALE=10. 0 strips every image from history, so only the
+    current turn is visual. Counting per message rather than per image keeps a
+    multi-image turn (game-over visuals) together in one slot."""
+    raw = os.environ.get("ARC3_HISTORY_IMAGE_KEEP", "").strip()
+    if not raw:
+        return None
+    try:
+        return max(0, int(raw))
+    except ValueError:
+        return None
+
+
+def _resume_prompt_images_enabled() -> bool:
+    """Whether a resumption message re-attaches the grid image. A resumption
+    only happens when nothing executed, so the board is identical to the one
+    the retained opener already showed; skipping it saves an image per
+    resumption and stops resumptions from occupying history retention slots.
+    Default True = today's behavior."""
+    return _get_env_bool("ARC3_RESUME_PROMPT_IMAGES", True)
+
+
+def _message_has_image(message: dict[str, Any]) -> bool:
+    content = message.get("content")
+    if not isinstance(content, list):
+        return False
+    return any(
+        isinstance(part, dict) and part.get("type") == "image_url"
+        for part in content
+    )
+
+
+def _strip_message_images(message: dict[str, Any]) -> dict[str, Any]:
+    """Replace image parts with a short text marker. The marker keeps the
+    message well formed (a message whose only part was an image would
+    otherwise be empty) and tells the model a visual was there rather than
+    silently showing a turn that never had one."""
+    content = message.get("content")
+    if not isinstance(content, list):
+        return message
+    parts: list[Any] = []
+    for part in content:
+        if isinstance(part, dict) and part.get("type") == "image_url":
+            parts.append({"type": "text", "text": _HISTORY_IMAGE_PLACEHOLDER})
+        else:
+            parts.append(part)
+    return {**message, "content": parts}
+
+
+def _apply_history_image_window(
+    messages: list[dict[str, Any]], keep: int
+) -> list[dict[str, Any]]:
+    """Keep images on the `keep` most recent image-bearing messages, strip the
+    rest. Walks backwards so recency wins; originals are never mutated."""
+    seen = 0
+    result: list[dict[str, Any]] = []
+    for message in reversed(messages):
+        if _message_has_image(message):
+            seen += 1
+            if seen > keep:
+                message = _strip_message_images(message)
+        result.append(message)
+    result.reverse()
+    return result
+
+
+# Tag for the pair of messages a rolling summary is made of: the request that
+# asks for it and the reply that is it. Deliberately absent from
+# _CONTROL_KIND_ENV, so no pruning knob can remove them - the trimmer's
+# summary-aware drain navigates by these, and losing one would silently return
+# it to ordinary block dropping.
+_SUMMARY_CONTROL_KIND = "summary"
+# Below this a reply that also called a tool is a preamble rather than a
+# summary. Real ones measured 4,385 to 6,248 characters; an introduction to a
+# tool call measured nine.
+_SUMMARY_MIN_USABLE_CHARS = 500
+
+
+def _summary_interval_tokens() -> int:
+    """New context tokens between summary attempts, successful or not. 0 disables.
+
+    Count newly appended prompts, assistant messages (including reasoning), tool
+    results and images, excluding the summary exchange itself. Re-sending or
+    evicting existing history does not change this counter.
+
+    The model is asked for a summary on a schedule rather than when it feels
+    like one. Measured across a run where the prompt asked for a note every
+    turn: one game in fourteen complied consistently, and the rest wrote
+    something on roughly half their replies - so a mechanism that depends on
+    the model choosing to summarise does not get summaries.
+
+    Each summary is generated from live context, which keeps both the KV prefix
+    and the recurrent state intact - a summary built from a truncated prefix
+    would force a full prefill, and recurrent state cannot be rewound the way a
+    KV prefix can be reused.
+
+    The summary is appended when it is made, so every later turn is generated
+    with it already present. Nothing is ever inserted underneath a message that
+    was produced without it.
+    """
+    return max(0, _get_env_int("ARC3_SUMMARY_INTERVAL_TOKENS", 0))
+
+
+def _drain_stop_at_summaries() -> bool:
+    """Whether the drain may stop early on a summary.
+
+    The drain normally evicts from the budget down to budget-minus-drain in one
+    go. With this on it stops sooner when the head is already a summary request
+    and the request is back under budget - so the retained history starts with a
+    digest of everything dropped, without evicting a token more than the
+    ordinary drain would.
+
+    The earlier design made summaries the ONLY boundary, which could not
+    overshoot in the good case but removed two intervals at once whenever a
+    summary had failed - 38% of a 42k window at an 8k interval. Here the drain
+    target still bounds the drop, so the worst case is exactly today's
+    behaviour and the summary is an early exit rather than the sole exit.
+
+    The trade is coverage: when no summary falls inside the drain range the drop
+    stops mid-span with the previous summary already gone, and that stretch has
+    nothing standing in for it. So ARC3_SUMMARY_INTERVAL_TOKENS wants to be
+    comfortably SMALLER than ARC3_CONTEXT_DRAIN_TOKENS - several candidates per
+    drop - which is the opposite of sizing the interval to replace the drain.
+    """
+    return _get_env_bool("ARC3_DRAIN_STOP_AT_SUMMARIES", False)
+
+
+def _hide_followup_summaries() -> bool:
+    """Send only the first summary in history, hiding the rest.
+
+    Each summary covers everything before it, so a later one restates an
+    earlier one's material - measured at 1 or 2 carried per request, 4,496
+    characters each, so roughly 6% of a 42k window at worst.
+
+    Applied to the wire and to the estimate, never to the stored history: the
+    hidden ones stay so that when eviction removes the first, the next becomes
+    the first and appears. Dropping them at commit would delete what it meant
+    to hide.
+
+    The cost is the thing the rolling design otherwise avoids. A hidden summary
+    is absent while the turns after it are generated and becomes their head
+    later, so those messages end up under something that was not there when
+    they were written - which is exactly the rearrangement the append-only
+    scheme was built to prevent.
+    """
+    return _get_env_bool("ARC3_HIDE_FOLLOWUP_SUMMARIES", False)
+
+
+def _apply_summary_visibility(
+    messages: list[dict[str, Any]], *, evicted: bool
+) -> list[dict[str, Any]]:
+    """Hide completed summaries, preserving a pending final summary request.
+
+    Before the first eviction no completed summaries are shown. What they
+    cover is still in context verbatim, so a summary there is pure duplication.
+    Hiding them all means appending one does not change the prompt, leaving the fill
+    phase a perfect append with a prefix that never moves. The first summary
+    appears when eviction starts, which invalidates the prefix anyway.
+    """
+    if not _hide_followup_summaries():
+        return messages
+
+    # The request currently asking for a summary has no reply yet. It must
+    # reach the model even when completed summaries are hidden. Split it off
+    # without mutating the input; only stored pairs participate in visibility.
+    pending_request: list[dict[str, Any]] = []
+    history = messages
+    if (
+        messages
+        and _message_is_summary(messages[-1])
+        and str(messages[-1].get("role", "")).strip() == "user"
+    ):
+        pending_request = messages[-1:]
+        history = messages[:-1]
+    if not evicted:
+        return [m for m in history if not _message_is_summary(m)] + pending_request
+    kept: list[dict[str, Any]] = []
+    seen_pair = False
+    in_first_pair = False
+    for message in history:
+        if not _message_is_summary(message):
+            kept.append(message)
+            continue
+        is_request = str(message.get("role", "")).strip() == "user"
+        if is_request:
+            # the request opens a pair; the first one and its reply stay
+            in_first_pair = not seen_pair
+            seen_pair = True
+        if in_first_pair:
+            kept.append(message)
+    return kept + pending_request
+
+
+def _summary_replaces_history() -> bool:
+    """Whether a successful summary discards the history it summarised.
+
+    The opposite end of the design from drain sizing: instead of evicting a
+    fixed span and hoping a summary falls inside it, the summary IS the
+    history. Everything above it goes the moment it lands, leaving system
+    prompt, request, summary, and whatever the turn appends after.
+
+    Safe in the sense that matters - nothing is dropped unless a usable summary
+    exists, so a failed or rejected one leaves ordinary eviction in charge and
+    the next turn tries again. But it is the more aggressive option, not the
+    safer one: a large drain still leaves a turn or two of verbatim history,
+    where this leaves none.
+
+    Worth trying because the summaries turn out to be good. One measured at
+    7,107 characters carried the board layout, the movement lattice, which
+    actions did nothing and why, an unresolved two-branch interpretation of the
+    last result, and the next probes to run - which is most of what the evicted
+    turns were holding. And the raw frames are not in the history anyway:
+    current_frame, previous_frame, history and frame_diff are runtime globals
+    rebuilt every turn, so what eviction destroys is the interpretation, which
+    is precisely what the summary keeps.
+
+    Costs: the prefix is invalidated on every summary rather than on every
+    eviction, and the turn immediately after has no verbatim record of the turn
+    before it beyond what the opener carries.
+    """
+    return _get_env_bool("ARC3_SUMMARY_REPLACES_HISTORY", False)
+
+
+def _summary_turn_context() -> bool:
+    """Whether the summary request is told what just happened.
+
+    The request goes out at turn start, BEFORE this turn's opener exists, so
+    the newest thing in context is the previous turn's tool results. A level-up
+    is visible there only as a `level_completed` flag inside a JSON payload -
+    so a summary written at that moment can describe the level just finished as
+    if it were still current, which is exactly the material a later turn needs
+    to be right about.
+
+    What goes in: what executed, whether the board changed, why a sequence
+    stopped early, whether the level changed, and the step and level now.
+
+    What stays out: the valid actions, which are state for the next decision
+    and go stale the moment the summary outlives the turn; the tool inventory
+    and the world-model demands, which are instructions that compete with
+    "write the summary"; and the animation line, which is a pointer to go and
+    read an accessor that will describe a different action by the time anyone
+    reads the summary.
+    """
+    return _get_env_bool("ARC3_SUMMARY_TURN_CONTEXT", False)
+
+
+def _summary_enable_thinking() -> bool:
+    """Whether the summary request lets the model think first.
+
+    OFF by default. Thinking about a summary is not obviously worth it - the
+    material is all in front of the model - and it is expensive twice over:
+    observed at 18,303 characters of reasoning that reached the output limit
+    with no summary at all, and it doubles the generation for a request that
+    already runs once per interval.
+
+    The template must be able to suppress it without moving the prefix. In the
+    Qwen3-family templates the `enable_thinking` conditional wraps BOTH the
+    empty think block at the tail AND the "Reasoning effort is set to ..."
+    sentence at the top of the system message, so the stock template diverges
+    from the first tokens and every summary prefills the whole context -
+    measured as prefix reuse falling from 75% to 45%. Forcing that conditional
+    true (`{%- if true %}`) keeps the sentence unconditional and leaves only
+    the tail block varying, which restores a byte-identical prefix. Check that
+    before assuming this default is free.
+    """
+    return _get_env_bool("ARC3_SUMMARY_ENABLE_THINKING", False)
+
+
+def _summary_max_gen_tokens(default: int | None) -> int:
+    """Output limit: the normal cap, or 8192 when normal output is unlimited.
+
+    The summary is asked for everything worth keeping, so it legitimately wants
+    more room than a turn reply - and it is one request per interval, so a
+    larger ceiling costs little. Raising this keeps the thinking;
+    ARC3_SUMMARY_ENABLE_THINKING=0 removes it.
+    """
+    fallback = default if default is not None and default > 0 else 8192
+    return max(1, _get_env_int("ARC3_SUMMARY_MAX_GEN", fallback))
+
+
+def _message_is_summary(message: dict[str, Any]) -> bool:
+    return message.get(_CONTROL_MESSAGE_KEY) == _SUMMARY_CONTROL_KIND
+
+
+_CONTROL_KIND_ENV = {
+    "nudge": "ARC3_PRUNE_NUDGES",
+    "resume": "ARC3_PRUNE_RESUME_PROMPTS",
+    "stub": "ARC3_PRUNE_DEAD_STUBS",
+}
+
+
+def _prune_control_context_enabled() -> bool:
+    """Master switch: drop every kind of tagged control message when
+    committing turn messages to persistent history. The per-kind knobs
+    (ARC3_PRUNE_NUDGES / _RESUME_PROMPTS / _DEAD_STUBS) turn individual kinds
+    on without it, so an arm can attribute each effect separately."""
+    return _get_env_bool("ARC3_PRUNE_CONTROL_CONTEXT", False)
+
+
+def _prune_control_kind_enabled(kind: str) -> bool:
+    """A kind is pruned if the master switch is on OR its own knob is set.
+
+    - 'nudge'  : the "you have not acted yet" corrective (a few hundred tokens)
+    - 'resume' : the message appended when re-entering a non-executing turn
+                 (a full opener in `full` mode, the short continuation in
+                 `short` mode) - a restatement of unchanged state
+    - 'stub'   : the placeholder that ARC3_STUB_DEAD_REASONING leaves behind
+    """
+    if _prune_control_context_enabled():
+        return True
+    env_name = _CONTROL_KIND_ENV.get(kind)
+    return bool(env_name) and _get_env_bool(env_name, False)
+
+
+def _any_control_pruning_enabled() -> bool:
+    return _prune_control_context_enabled() or any(
+        _get_env_bool(name, False) for name in _CONTROL_KIND_ENV.values()
+    )
+
+
+def _mark_control_message(message: dict[str, Any], kind: str = "resume") -> dict[str, Any]:
+    message[_CONTROL_MESSAGE_KEY] = kind
+    return message
+
+
+def _strip_control_keys(messages: list[dict[str, Any]]) -> list[dict[str, Any]]:
+    """Wire-safe copy: the private tag never leaves the process."""
+    cleaned: list[dict[str, Any]] = []
+    for message in messages:
+        if _CONTROL_MESSAGE_KEY in message:
+            copy = dict(message)
+            copy.pop(_CONTROL_MESSAGE_KEY, None)
+            cleaned.append(copy)
+        else:
+            cleaned.append(message)
+    return cleaned
+
+
+def _prune_control_messages(messages: list[dict[str, Any]]) -> list[dict[str, Any]]:
+    """Remove tagged control messages, then strip trailing user messages.
+
+    The trailing-user strip is alternation repair, not a preference: pruning a
+    spin-out turn (opener + dead reasoning, no tool calls) would otherwise
+    leave the opener dangling while the next invocation appends another user
+    message. A turn with no tool calls executed nothing, so the next opener
+    regenerates identical state."""
+    kept = [
+        m
+        for m in messages
+        if not (
+            m.get(_CONTROL_MESSAGE_KEY)
+            and _prune_control_kind_enabled(str(m.get(_CONTROL_MESSAGE_KEY)))
+        )
+    ]
+    while kept and str(kept[-1].get("role", "")).strip() == "user":
+        kept.pop()
+    return kept
+
+
+def _reasoning_history_keys() -> tuple[str, ...]:
+    """Message key(s) under which prior reasoning is sent back to the server.
+
+    Backends disagree: OpenRouter returns and reads `reasoning`, while
+    llama.cpp returns `reasoning_content` and its chat template reads ONLY
+    that - so reasoning stored under `reasoning` is silently discarded there
+    (measured: 69 prompt tokens vs 3670 for the same text under the two keys).
+    A dropped reasoning block is invisible except as prompt growth that falls
+    short of the tokens generated.
+
+    Default `reasoning` preserves existing behaviour. Set to
+    `reasoning_content` for llama.cpp, or to a comma list to send both - which
+    is backend-agnostic but risks a template that renders both keys duplicating
+    the text."""
+    raw = os.environ.get("ARC3_REASONING_HISTORY_KEY", "").strip()
+    keys = tuple(k.strip() for k in raw.split(",") if k.strip())
+    return keys or ("reasoning",)
+
+
+def _level_inventory_enabled() -> bool:
+    """The START-vs-START object inventory shown on the first turn of a new
+    level.
+
+    It has its own switch because it answers a different question from the
+    turn-to-turn diff: not "what did my last actions do" but "what is new
+    about this level", with `appeared` types flagged as likely new mechanics.
+    Previously it rode on the auto-diff machinery and was suppressed only when
+    BOTH ARC3_AUTO_FRAME_DIFF and ARC3_DIFF_IMAGE were off - a coupling that
+    was invisible from the knob names."""
+    return _get_env_bool("ARC3_LEVEL_INVENTORY", False)
+
+
+def _resume_diff_image_enabled() -> bool:
+    """Re-attach the current turn's diff image to resumption messages.
+
+    The diff image is produced only while building a full turn opener, so a
+    resumption (short/state_only) carries the current grid but no diff. That
+    is normally harmless - a resumption happens when nothing executed, so the
+    diff would be identical to the one the turn's original opener already
+    showed. It stops being harmless when ARC3_HISTORY_IMAGE_KEEP has stripped
+    that opener's images: the model has then lost the diff entirely, and the
+    resumption is its only chance to see what changed.
+
+    Costs roughly one image per resumption. Off by default."""
+    return _get_env_bool("ARC3_RESUME_DIFF_IMAGE", False)
+
+
+def _reasoning_effort_ladder() -> list[str]:
+    """Reduced reasoning-effort levels to step down through after a truncated
+    generation, e.g. "medium,low".
+
+    A reply cut off at the output ceiling is wasted twice over: the tokens are
+    spent, and if the cut lands inside a tool call the turn produces nothing
+    callable. Rather than accommodating long generations with a bigger ceiling,
+    this asks the model for less deliberation until it completes a turn -
+    stepping one rung down per truncation and resetting once an action
+    executes, which is the evidence that the turn recovered.
+
+    Empty (default) disables the behaviour entirely."""
+    raw = os.environ.get("ARC3_REASONING_EFFORT_LADDER", "").strip()
+    return [level.strip() for level in raw.split(",") if level.strip()]
+
+
+def _preserve_thinking_kwarg() -> bool | None:
+    """Value for the `preserve_thinking` chat-template kwarg, or None to omit.
+
+    The payload builder sets chat_template_kwargs wholesale for vLLM, which
+    replaces rather than merges - so a server-side
+    --default-chat-template-kwargs carrying preserve_thinking can be silently
+    dropped by a request. Defaults to on so the request asserts it rather than
+    depending on server-side merge semantics; set to an empty string to omit
+    the key entirely and leave the server default in charge."""
+    raw = os.environ.get("ARC3_PRESERVE_THINKING", "1").strip().lower()
+    if not raw:
+        return None
+    return raw in ("1", "true", "yes", "on")
+
+
+def _openrouter_provider_prefs(provider: str) -> dict[str, Any] | None:
+    """OpenRouter request-level provider pinning. ARC3_OPENROUTER_PROVIDER
+    holds a provider name (or a comma-separated preference order); routing is
+    then restricted to that list with fallbacks disabled, so every request in
+    a run is served by a known engine. Under :nitro-style routing, providers
+    differ in media handling (some silently drop video/image parts),
+    quantization, and sampling behavior - a silent confound for any controlled
+    comparison. Unset or non-OpenRouter: no change to the request."""
+    if str(provider or "").strip().lower() != "openrouter":
+        return None
+    raw = os.environ.get("ARC3_OPENROUTER_PROVIDER", "").strip()
+    if not raw:
+        return None
+    order = [name.strip() for name in raw.split(",") if name.strip()]
+    if not order:
+        return None
+    return {"order": order, "allow_fallbacks": False}
+
+
+_SLIM_MEMORY_SECTIONS = ("World model", "Plan")
+_MEDIUM_MEMORY_SECTIONS = ("World model", "Plan", "Action model")
+
+_MEMORY_SECTION_GUIDANCE_FULL = (
+                "If you include assistant text before a tool call, use it to update your persistent memory sections. Section meanings: `World model:` = durable RULES of the environment that will still be true many actions from now (mechanics, what objects/colors mean, movement rules, win conditions) — never current positions, events, or what just happened. `Goal model:` = what winning requires. `Action model:` = what each action does mechanically. `Recent findings:` = this turn's observations and events (positions, action results, surprises). `Open questions:` = unresolved hypotheses. `Plan:` = intended next steps. `Cross-level notes:` = knowledge that carries across levels. Example — GOOD: `World model: Red cells are walls; each move shifts the player 2 cells; yellow keys open doors of the same color.` BAD: `World model: Batch stopped after 2 DOWNs, player now at (4,2).` (that belongs in `Recent findings:`). IMPORTANT: each labeled section you write REPLACES the stored version shown under 'Working world model carried from earlier turns' — write the COMPLETE updated section; any fact you omit is permanently lost."
+)
+
+_MEMORY_SECTION_MEANINGS = {
+    "World model": (
+        "durable RULES of the environment that will still be true many actions from now "
+        "(mechanics, what objects/colors mean, movement rules, win conditions) - never "
+        "current positions or what just happened"
+    ),
+    "Goal model": "what winning requires",
+    "Action model": "what each action does mechanically",
+    "Recent findings": "this turn's observations",
+    "Open questions": "unresolved hypotheses",
+    "Plan": "intended next steps",
+    "Cross-level notes": "knowledge that carries across levels",
+}
+
+
+def _advertised_section_labels() -> tuple[str, ...]:
+    """The sections the model should be TOLD about: exactly those the current
+    mode carries.
+
+    Advertising all seven while a mode drops five invites the model to write
+    into sections it will never see again - the write is parsed, stored, and
+    silently discarded from the prompt. It also confounds a slim/medium arm,
+    because a poor result could mean either that fewer sections hurt or that
+    output was wasted on sections that went nowhere."""
+    labels = _memory_section_labels()
+    if labels is None:
+        return tuple(_MEMORY_SECTION_MEANINGS)
+    # canonical order, so the advertisement matches the order the carried block
+    # renders them in
+    chosen = set(labels)
+    return tuple(name for name in _MEMORY_SECTION_MEANINGS if name in chosen)
+
+
+def _section_list_phrase(labels: tuple[str, ...], conjunction: str = "and") -> str:
+    """Render the labels as prose. The conjunction is a parameter because the
+    two call sites word it differently: the opener lists what is available
+    ("and"), the no-tool-call followup offers alternatives ("or")."""
+    quoted = [f"`{name}:`" for name in labels]
+    if len(quoted) == 1:
+        return quoted[0]
+    if len(quoted) == 2:
+        # no serial comma before a two-item conjunction
+        return f" {conjunction} ".join(quoted)
+    return ", ".join(quoted[:-1]) + f", {conjunction} " + quoted[-1]
+
+
+def _memory_section_guidance_upstream() -> str:
+    return (
+        "If you include assistant text before a tool call, keep it short and use it to "
+        "update the world model. Helpful optional prefixes are "
+        f"{_section_list_phrase(_advertised_section_labels())}."
+    )
+
+
+def _memory_section_guidance_empty() -> str:
+    labels = _advertised_section_labels()
+    meanings = " ".join(
+        f"`{name}:` = {_MEMORY_SECTION_MEANINGS[name]}." for name in labels
+    )
+    return (
+        "You have no stored memory sections yet. If you include assistant text before a "
+        f"tool call, use it to start them: {meanings}"
+    )
+
+
+_MEMORY_SECTION_GUIDANCE_UPSTREAM = (
+    "If you include assistant text before a tool call, keep it short and use it to update "
+    "the world model. Helpful optional prefixes are `World model:`, `Goal model:`, "
+    "`Action model:`, `Recent findings:`, `Open questions:`, `Plan:`, and "
+    "`Cross-level notes:`."
+)
+
+
+def _wm_section_explanation_mode() -> str:
+    """Which wording explains the memory sections in the turn opener.
+
+    'tufa' (also spelled 'upstream', the default) is the terse original: one
+    sentence naming the prefixes, framed
+    as optional. 'full' (default) is ours - a glossary of what each section
+    means, a GOOD/BAD example, and the warning that a written section REPLACES
+    the stored one so an omitted fact is lost. Roughly 1100 characters against
+    250, every turn, and it asks for complete rewrites rather than short
+    updates - so it is a candidate for longer generations as well as better
+    memory."""
+    mode = os.environ.get("ARC3_WM_SECTION_EXPLANATION", "tufa").strip().lower()
+    return "full" if mode == "full" else "upstream"
+
+
+_MEMORY_SECTION_GUIDANCE_EMPTY = (
+    "You have no stored memory sections yet. If you include assistant text before a tool "
+    "call, use it to start them: `World model:` = durable RULES of the environment that will "
+    "still be true many actions from now (mechanics, what objects/colors mean, movement "
+    "rules, win conditions) - never current positions or what just happened. `Goal model:` = "
+    "what winning requires. `Action model:` = what each action does mechanically. `Recent "
+    "findings:` = this turn's observations. `Open questions:` = unresolved hypotheses. "
+    "`Plan:` = intended next steps. `Cross-level notes:` = knowledge that carries across "
+    "levels."
+)
+
+_MEMORY_SECTION_GUIDANCE_SLIM = (
+                "If you include assistant text before a tool call, use it to update your persistent memory sections. Section meanings: `World model:` = durable RULES of the environment that will still be true many actions from now (mechanics, what objects/colors mean, movement rules, win conditions) \u2014 never current positions, events, or what just happened. `Plan:` = intended next steps. Example \u2014 GOOD: `World model: Red cells are walls; each move shifts the player 2 cells; yellow keys open doors of the same color.` BAD: `World model: Batch stopped after 2 DOWNs, player now at (4,2).` (that is a passing event, not a rule). IMPORTANT: each labeled section you write REPLACES the stored version shown under 'Working world model carried from earlier turns' \u2014 write the COMPLETE updated section; any fact you omit is permanently lost."
+)
+
+
+def _wm_revise_nudges_enabled() -> bool:
+    """The sticky 'LEVEL CHANGED / ATTEMPT RESET - revise your sections' hint.
+
+    Built for an observed failure (the champion drifting without revising),
+    but a later model maintained its sections diligently with no prompting at
+    all, and the mechanism has needed two corrections since. Turning it off
+    changes ONLY the pressure: sections are still parsed, stored and
+    re-injected, so a null result means the nagging was not earning its
+    prompt weight."""
+    return not _memory_sections_disabled() and _get_env_bool(
+        "ARC3_WM_REVISE_NUDGES", False
+    )
+
+
+def _wm_rebuild_nudge_enabled() -> bool:
+    """The post-wipe "Rebuild them NOW: write fresh World model:, Goal model:,
+    Action model:" demand, which repeats every turn until the model complies.
+
+    An unmodified run wipes the sections silently - the carried block simply
+    vanishes and reappears when the model next writes something. This nudge
+    applies its strongest pressure exactly when the model has least to say: it
+    has just lost its notes and has not yet explored the new level. Default off
+    for that reason; the wipe itself is unaffected."""
+    return not _memory_sections_disabled() and _get_env_bool(
+        "ARC3_WM_REBUILD_NUDGE", False
+    )
+
+
+def _memory_sections_disabled() -> bool:
+    """ARC3_MEMORY_SECTIONS=off removes the persistent-memory mechanic entirely:
+    no carried block, no section guidance, no nudges, no parsing, and the
+    world-model lines drop out of the system prompt too. The model is left to
+    reason from history alone, which is what an unmodified run does before any
+    of this machinery existed."""
+    return os.environ.get("ARC3_MEMORY_SECTIONS", "").strip().lower() == "off"
+
+
+def _memory_section_labels() -> tuple[str, ...] | None:
+    """Which memory sections are advertised and carried, or None for all.
+
+    'slim' keeps World model and Plan - the two the model actually maintains.
+    Measured over 25 games: World model 351 writes, Plan 255, Open questions
+    25, Recent findings 11, Action model 10, Goal model 9, Cross-level notes
+    ZERO. The dropped sections are also the stale ones: a section written once
+    and never revised keeps being re-injected as current, which was observed
+    holding an answered question open for a whole game.
+
+    'medium' additionally keeps Action model. It is written in a third of
+    games and describes MECHANICS rather than transient state, so it ages far
+    better than `Recent findings:` or `Open questions:` - on a game whose
+    difficulty is working out what each control does, it is the section least
+    worth dropping."""
+    mode = os.environ.get("ARC3_MEMORY_SECTIONS", "").strip().lower()
+    if mode == "slim":
+        return _SLIM_MEMORY_SECTIONS
+    if mode == "medium":
+        return _MEDIUM_MEMORY_SECTIONS
+    return None
+
+
+def _message_display_text(message: dict[str, Any]) -> str:
+    """Transcript rendering of a built user message. Content-parts messages
+    carry their text in parts; images are noted rather than dumped."""
+    content = message.get("content")
+    if isinstance(content, str):
+        return content
+    if not isinstance(content, list):
+        return ""
+    pieces: list[str] = []
+    for part in content:
+        if not isinstance(part, dict):
+            continue
+        if part.get("type") == "image_url":
+            pieces.append("[image attached]")
+        else:
+            text = str(part.get("text") or "").strip()
+            if text:
+                pieces.append(text)
+    return "\n\n".join(pieces)
+
+
+def _wm_revise_hint(reason: str, already_shown: bool) -> str:
+    """Text for the sticky revise-nudge. Full event text on first showing;
+    accurate reminder wording on repeats (the event is no longer 'just now',
+    and the adjacent step summary may correctly say the level is unchanged)."""
+    if reason == "level":
+        if not already_shown:
+            return (
+                "LEVEL CHANGED - your memory sections were KEPT, not wiped. Learned "
+                "mechanics, action semantics, and the general goal usually carry over "
+                "to the new level; the map/layout is new, and levels often introduce "
+                "new elements or mechanics. On this first turn, scan the grid for "
+                "unfamiliar elements (compare objects against your World/Action "
+                "model), then revise your `World model:` and `Plan:` sections - "
+                "update the map, keep what still holds, and note anything new."
+            )
+        return (
+            "Reminder: a level change occurred earlier and you have STILL not "
+            "revised your `World model:` or `Plan:` sections since. Do it now - "
+            "update the map for this level, keep what still holds, and note "
+            "anything new."
+        )
+    if reason == "death":
+        if not already_shown:
+            return (
+                "ATTEMPT RESET - your memory sections were KEPT, not wiped. The "
+                "level is unchanged, so your map and mechanics knowledge still "
+                "hold. Amend your sections NOW with what this death taught you "
+                "(the fatal action or sequence and its cause), and revise your "
+                "`Plan:` to avoid repeating it."
+            )
+        return (
+            "Reminder: this attempt was reset by an earlier death and you have "
+            "STILL not amended your memory sections with what it taught you. Do "
+            "it now - record the fatal action or sequence and its cause, and "
+            "revise your `Plan:` to avoid repeating it."
+        )
+    return ""
+
+
+def _suppress_board_dumps(text: str) -> tuple[str, int]:
+    """Collapse runs of full-width board rows in tool output. Enforces the
+    existing 'never print full boards' policy mechanically for models that
+    ignore it: keeps the first rows of each run as orientation, replaces the
+    rest with a stub, and appends a compute-don't-count nudge. Small crops
+    (fewer than 6 rows) pass through untouched. ARC3_BOARD_DUMP_GUARD=0
+    disables."""
+    if not _get_env_bool("ARC3_BOARD_DUMP_GUARD", False):
+        return text, 0
+
+    def _board_like(line: str) -> bool:
+        stripped = line.strip()
+        return (
+            len(stripped) >= 24
+            and stripped.isalnum()
+            and len(set(stripped)) <= 10
+        )
+
+    lines = text.split("\n")
+    out: list[str] = []
+    suppressed = 0
+    i = 0
+    while i < len(lines):
+        if _board_like(lines[i]):
+            j = i
+            while j < len(lines) and _board_like(lines[j]):
+                j += 1
+            run = j - i
+            if run >= 6:
+                out.extend(lines[i:i + 2])
+                out.append(f"[... {run - 2} full-width board rows suppressed ...]")
+                suppressed += run - 2
+            else:
+                out.extend(lines[i:j])
+            i = j
+        else:
+            out.append(lines[i])
+            i += 1
+    if suppressed:
+        out.append(
+            "[BOARD PRINT GUARD] Printing full boards is disallowed and "
+            "character counting is unreliable for you. Compute over the grid "
+            "in python (segmentation nodes, slicing, run-length checks, BFS) "
+            "and print only coordinates, counts, and small crops."
+        )
+    return "\n".join(out), suppressed
+
+
+def _memory_section_max_chars() -> int:
+    """Cap on a single stored memory section, in characters.
+
+    These sections are re-injected into EVERY later turn opener, so an
+    oversized one is paid for repeatedly and grows the prompt without bound.
+    The extraction path used to store them uncapped, which let a degenerate
+    75k-character `World model:` into every subsequent prompt until the
+    request exceeded the model's context entirely. Legitimate sections run a
+    few hundred characters, so the default leaves ample room while making the
+    runaway case impossible. 0 disables the cap (the old behaviour)."""
+    return max(0, _get_env_int("ARC3_MEMORY_SECTION_MAX_CHARS", 8000))
+
+
+def _degeneracy_ratio_threshold() -> float:
+    """Compression ratio below which text is treated as degenerate.
+
+    Measured on representative text: varied prose 0.20-0.28, a coordinate
+    list 0.28, an ASCII board map 0.016, one sentence repeated 300 times
+    0.008, the field mojibake spiral 0.0016. The default sits below the board
+    map so a pasted grid is not falsely rejected, while still catching genuine
+    repetition loops by a wide margin. Rejecting on compressibility catches
+    those generally rather than one encoding artifact, and does not penalise
+    legitimate unicode. 0 disables the check."""
+    try:
+        return max(0.0, float(os.environ.get("ARC3_DEGENERACY_RATIO", "0.01")))
+    except ValueError:
+        return 0.01
+
+
+def _degeneracy_min_chars() -> int:
+    """Length floor for the degeneracy check: short strings compress badly for
+    uninteresting reasons, so testing them would produce false positives."""
+    return max(1, _get_env_int("ARC3_DEGENERACY_MIN_CHARS", 500))
+
+
+def _is_degenerate_text(value: str) -> bool:
+    threshold = _degeneracy_ratio_threshold()
+    if threshold <= 0:
+        return False
+    text = str(value or "")
+    if len(text) < _degeneracy_min_chars():
+        return False
+    try:
+        import zlib
+
+        encoded = text.encode("utf-8", "replace")
+        return len(zlib.compress(encoded, 6)) / max(1, len(encoded)) < threshold
+    except Exception:
+        return False
+
+
 def _normalize_summary_text(value: Any, *, max_chars: int | None = 280) -> str:
     text = " ".join(str(value or "").split())
     if max_chars is None or max_chars <= 0 or len(text) <= max_chars:
@@ -223,9 +1686,100 @@ def _normalize_summary_text(value: Any, *, max_chars: int | None = 280) -> str:
     return f"{text[:max_chars].rstrip()}... [{omitted} chars omitted]"
 
 
+_HEADER_PARENTHETICAL_RE = re.compile(r"\([^)]*\)")
+# Only SHORT decorations are stripped. A long parenthetical or emphasised aside
+# means the line is a sentence rather than a header, and removing it could turn
+# prose into a spurious match - so the header is left intact and simply fails to
+# match, which loses the section rather than mis-capturing it.
+_HEADER_DECORATION_MAX_CHARS = 30
+
+
+def _normalize_section_header(head: str) -> str:
+    """Reduce a candidate section header to its bare label.
+
+    Models decorate these constantly. Observed in a single game: `Revised
+    world model`, `World model update`, `World model updates`, `World model
+    revision`, `Updated world model`, `World model (revised)`, `World model
+    (L4)`, `Action model (from run 1)`, `Plan (19 moves)`, and `Revised world
+    model - **important finding**`. Under exact matching, thirteen of
+    twenty-seven memory writes in that game were silently discarded, including
+    the one that correctly derived the level's kill rule.
+    """
+    def drop_short(match: "re.Match[str]") -> str:
+        return " " if len(match.group(0)) <= _HEADER_DECORATION_MAX_CHARS else match.group(0)
+
+    text = _HEADER_PARENTHETICAL_RE.sub(drop_short, head)
+    # an emphasised trailing aside is dropped on the same terms; the em-dash is
+    # deliberately NOT a cut point - it would discard text AFTER the label, which
+    # can change what the header means, and it earned nothing in practice
+    if "**" in text:
+        head_part, _, rest = text.partition("**")
+        if len(rest) <= _HEADER_DECORATION_MAX_CHARS:
+            text = head_part
+    text = text.replace("*", " ").replace("_", " ").replace("#", " ")
+    # dangling dashes left behind by a removed decoration are punctuation, not
+    # content: stripping them discards nothing, unlike cutting AT a dash
+    return " ".join(text.split()).strip(" \u2014\u2013-")
+
+
+def _tolerant_section_headers_enabled() -> bool:
+    """Accept decorated section headers (`Updated world model:`, `Plan (19
+    moves):`) rather than only the exact label.
+
+    Doubled memory capture on a measured game (12 sections -> 24), which also
+    doubles the carried world-model block the model is asked to rewrite in
+    full each turn - so it is a candidate for longer generations, not only for
+    better memory. Off restores exact matching."""
+    return _get_env_bool("ARC3_TOLERANT_SECTION_HEADERS", False)
+
+
+def _match_section_label(head: str, labels: list[str]) -> str | None:
+    """The canonical label a header refers to, or None.
+
+    Accepts the bare label, or the label with at most ONE extra word before or
+    after it. One word rather than more is deliberate: it covers every
+    decoration seen in practice while keeping prose from matching. Labels are
+    tried longest-first so `Cross-level notes` is not shadowed by `Plan`."""
+    tolerant = _tolerant_section_headers_enabled()
+    # with the knob off, compare the RAW head: normalising would still strip
+    # parentheticals and emphasis, which is itself part of the tolerance
+    normalized = (
+        _normalize_section_header(head).lower() if tolerant else " ".join(head.split()).lower()
+    )
+    if not normalized:
+        return None
+    if tolerant:
+        # A trailing parenthetical is decoration, not part of the label. The
+        # model versions and annotates its own sections - "World model
+        # (revised)", "(major revision)", "World model v34 (level 3)" - and
+        # those are the same section wearing a hat. Measured on one run: the
+        # one-extra-word rule alone caught 89% of section writes, stripping a
+        # trailing parenthetical first took it to 98%.
+        #
+        # Stripping rather than widening the word allowance to two: a
+        # parenthetical is a bounded construct, whereas two free words start
+        # matching prose. The 60-character cap in _extract_labeled_blocks is
+        # applied to the UNDECORATED head, so a long decoration is rejected
+        # before it can be stripped down to a match.
+        without_paren = re.sub(r"\s*\([^)]*\)\s*$", "", normalized).strip()
+        if without_paren:
+            normalized = without_paren
+    for label in sorted(labels, key=len, reverse=True):
+        lowered = label.lower()
+        if normalized == lowered:
+            return label
+        if not tolerant:
+            continue
+        prefix = lowered + " "
+        if normalized.startswith(prefix) and len(normalized[len(prefix):].split()) == 1:
+            return label
+        suffix = " " + lowered
+        if normalized.endswith(suffix) and len(normalized[: -len(suffix)].split()) == 1:
+            return label
+    return None
+
+
 def _extract_labeled_blocks(content: str, labels: list[str]) -> dict[str, str]:
-    normalized_labels = {label.lower(): label for label in labels}
-    targets = tuple(f"{label.lower()}:" for label in labels)
     extracted: dict[str, list[str]] = {label: [] for label in labels}
     current_label: str | None = None
 
@@ -234,15 +1788,17 @@ def _extract_labeled_blocks(content: str, labels: list[str]) -> dict[str, str]:
         candidate = stripped
         while candidate.startswith(("-", "*")):
             candidate = candidate[1:].lstrip()
-        lowered = candidate.lower()
 
         matched_label: str | None = None
         inline_value = ""
-        for target in targets:
-            if lowered.startswith(target):
-                matched_label = normalized_labels[target[:-1]]
-                inline_value = candidate[len(target):].strip()
-                break
+        if ":" in candidate:
+            head, _, rest = candidate.partition(":")
+            # a header is a short line-initial phrase, not a sentence that
+            # happens to contain a colon
+            if len(head) <= 60:
+                matched_label = _match_section_label(head, labels)
+                if matched_label is not None:
+                    inline_value = rest.strip()
 
         if matched_label is not None:
             current_label = matched_label
@@ -254,13 +1810,641 @@ def _extract_labeled_blocks(content: str, labels: list[str]) -> dict[str, str]:
             extracted[current_label].append(stripped)
 
     return {
-        label: _normalize_summary_text("\n".join(lines).strip(), max_chars=None)
+        label: _normalize_summary_text(
+            "\n".join(lines).strip(), max_chars=_memory_section_max_chars() or None
+        )
         for label, lines in extracted.items()
         if "\n".join(lines).strip()
     }
 
 
-def _extract_scientist_note(content: str) -> dict[str, str]:
+def _opener_commit_hint() -> bool:
+    """Ask the turn opener for a commitment, the way a resumption already does.
+
+    Every piece of commitment pressure in the harness lives in the resumption
+    path - "call action(actions) with your best next action or short batch",
+    "Stop deliberating", "Act NOW". The fresh opener says nothing about acting
+    at all. Measured across two runs, sequences following a resumption batch
+    6.73 and 6.91 actions; those following a fresh opener batch 5.43 and 3.91,
+    and the opener figure is what collapsed when resumptions became rarer.
+
+    Deliberately asymmetric: probing stays available but has to be justified,
+    while batching is the default. An earlier prompt that told the model to
+    "re-issue actions one at a time" produced a 200-iteration single-step loop,
+    so the wording avoids prescribing a size while naming the batch.
+    """
+    return _get_env_bool("ARC3_OPENER_COMMIT_HINT", False)
+
+
+def _new_noop_wording() -> bool:
+    """Whether the patch-110 rewordings are used.
+
+    They replaced "changed nothing" with "no change inside the board area"
+    across every guard message. Off restores the wording of the run that scored
+    4.05, so the rewordings can be judged apart from the outcome line they
+    shipped with."""
+    return _get_env_bool("ARC3_NEW_CHANGED_PROMPTS", False)
+
+
+def _animation_timeline_enabled() -> bool:
+    """Whether the diff timeline is offered alongside the raw frames.
+
+    The timeline is a cheap first look, but a summary that reads coherently
+    invites stopping at it: on one level it produced a confident and wrong
+    mechanic that the model only corrected after going to the frames. Off, the
+    model computes its own comparison from `animation_frames` - a few lines it
+    writes readily - at the cost of a snippet before it learns anything, which
+    is wasted on animations that turn out to be irrelevant.
+
+    Only an experiment settles which costs more.
+    """
+    return _get_env_bool("ARC3_ANIMATION_TIMELINE", True)
+
+
+def _dedupe_multicall_line() -> bool:
+    """Drop the duplicated multi-call permission and its stale caveat.
+
+    Upstream states "you may call action() more than once" twice per opener,
+    about thirty lines apart, the second time without the caveat - so the last
+    word the model gets is the permission stripped of its qualifier.
+
+    The caveat itself is now enforced three ways: the batch loop breaks on a
+    terminal result and skips the rest, the per-snippet latch refuses the next
+    action() call, and an executed action ends the turn so the next snippet
+    always opens on a fresh board. On is the change; off keeps upstream.
+    """
+    return _get_env_bool("ARC3_DEDUPE_MULTICALL_LINE", False)
+
+
+def _prefer_tool_calls() -> bool:
+    """State the cost asymmetry: tool calls are free, actions are not.
+
+    The prompt already says not to ration tool calls, but only ever compares
+    them with other tool calls - never with reasoning, which is the trade the
+    model actually makes when it writes twelve thousand characters of
+    hypothesis before its first snippet. Observed: a turn spent 18,012
+    characters reasoning, hit the output cap, and executed nothing.
+    """
+    return _get_env_bool("ARC3_PREFER_TOOL_CALLS", False)
+
+
+# Three bands, a million apart, so no calculated value can cross between them.
+# The maximum _game_priority is about 1,325.
+#
+#   ~2,000,000   never trimmed: the original prefix is whole and this game has
+#                never paid a prefill, so it is the cheapest of all to continue
+#   ~1,000,000   warm again: trimmed once, prefix rebuilt. Ordered within the
+#                band by the estimate at that trim, so a game that was doing
+#                well sits above one that was stalling
+#      0..1,325  awaiting prefill: the trim invalidated the prefix and the next
+#                request has to pay for it anyway, which is the one moment
+#                yielding the slot is free
+#
+# The old scheme sent a flat 999999 while the prefix held, so every game tied at
+# startup and the server broke ties by arrival. Subtracting the queue position
+# gives a deterministic order instead.
+# ---------------------------------------------------------------- TEMPORARY
+# Diagnostic for "the backend never shows more than 13 concurrent requests with
+# 14 games". Records where each game thread is and prints a census once a
+# second, so the missing one can be located rather than guessed at.
+# Remove once the cause is known.
+_GATE_STATS = {"enqueued": 0, "blocked": 0, "blocked_ms": 0, "handover_ms": 0}
+_DIAG_LOCK = threading.Lock()
+_DIAG_STATE: dict[str, str] = {}
+_DIAG_LAST = [0.0]
+
+
+def _diag(agent: Any, where: str) -> None:
+    if not _get_env_bool("ARC3_DIAG_CONCURRENCY", False):
+        return
+    name = getattr(agent, "_diag_name", "?")
+    now = time.monotonic()
+    with _DIAG_LOCK:
+        _DIAG_STATE[name] = where
+        if now - _DIAG_LAST[0] < 1.0:
+            return
+        _DIAG_LAST[0] = now
+        counts: dict[str, int] = {}
+        for value in _DIAG_STATE.values():
+            counts[value] = counts.get(value, 0) + 1
+        total = len(_DIAG_STATE)
+        inflight = counts.get("request", 0)
+        summary = "  ".join(
+            f"{k}={v}" for k, v in sorted(counts.items(), key=lambda kv: -kv[1])
+        )
+        log.warning(
+            "DIAG %d games, %d in flight | %s | gate: %d handovers, %d blocked, "
+            "%dms blocked, %dms total",
+            total, inflight, summary,
+            _GATE_STATS["enqueued"], _GATE_STATS["blocked"],
+            _GATE_STATS["blocked_ms"], _GATE_STATS["handover_ms"],
+        )
+        if inflight < total:
+            others = sorted(
+                f"{g}:{w}" for g, w in _DIAG_STATE.items() if w != "request"
+            )
+            log.warning("DIAG   elsewhere: %s", ", ".join(others))
+
+
+def _priority_refresh_enabled() -> bool:
+    return any(_get_env_bool(name, False) for name in (
+        "ARC3_PRIORITY_REFRESH_QUEUE", "ARC3_PRIORITY_PACE", "ARC3_PRIORITY_TAIL_FADE",
+    ))
+
+
+def _priority_action_count(summary: dict[str, Any]) -> int:
+    return int(summary.get("end_action_num") or 0)
+
+
+@lru_cache(maxsize=8)
+def _load_pace_reference(path: str) -> dict[str, Any]:
+    data = json.loads(Path(path).read_text())
+    values = [data["fallback_tokens"], *data["by_level"].values()]
+    if not all(isinstance(v, (int, float)) and 0 < v < float("inf") for v in values):
+        raise ValueError("Pace reference costs must be finite positive numbers")
+    return data
+
+
+def _pace_reference_tokens(level: int) -> float:
+    path = os.environ.get("ARC3_PRIORITY_PACE_REFERENCE", "").strip()
+    if not path:
+        path = str(Path(__file__).with_name("priority_pace_reference.json"))
+    reference = _load_pace_reference(path)
+    return float(reference["by_level"].get(str(level), reference["fallback_tokens"]))
+
+
+class _PriorityGate:
+    """Harness-side admission control, for when the server ignores `priority`.
+
+    The request field is a hint a backend may drop; this enforces the same
+    ordering locally by limiting how many games hold a slot at once and
+    choosing who gets a freed one.
+
+    A game keeps its slot while its prefix is intact, because interrupting it
+    there throws away a cache entry about to be reused. When the trimmer drops
+    a message the prefix is invalid anyway and the next request must be
+    prefilled from the divergence point - the one moment when yielding is free -
+    so that is where the handover happens.
+
+    Release and re-enqueue are one critical section. Releasing first would leave
+    a window in which a lower-priority waiter takes the slot the caller was
+    about to compete for.
+    """
+
+    def __init__(self, slots: int) -> None:
+        self._cond = threading.Condition()
+        self._free = max(1, int(slots))
+        self._waiting: list[tuple[int, int]] = []
+        self._seq = 0
+        self._admitted: set[int] = set()
+        self._snapshots: dict[int, PrioritySnapshot] = {}
+        self._clock_start: float | None = None
+        self._clock_deadline: float | None = None
+        self._tail_fade_announced = False
+
+    def configure_clock(self, started_at: float, deadline: float | None) -> None:
+        """One shared monotonic clock, configured before initial admission.
+
+        Earliest session start/deadline wins; appropriate for simultaneously
+        launched sessions. No clock is inferred from the last action payload.
+        """
+        if _get_env_bool("ARC3_PRIORITY_TAIL_FADE", False) and deadline is None:
+            raise ValueError("ARC3_PRIORITY_TAIL_FADE requires a finite runtime or soft deadline")
+        with self._cond:
+            self._clock_start = min(self._clock_start, started_at) if self._clock_start is not None else started_at
+            if deadline is not None:
+                self._clock_deadline = min(self._clock_deadline, deadline) if self._clock_deadline is not None else deadline
+
+    def _snapshot_priority(self, snapshot: PrioritySnapshot, now: float) -> int:
+        elapsed = max(0.0, now - self._clock_start) if self._clock_start is not None else 0.0
+        endgame = _endgame_start_minutes() > 0 and elapsed >= 60 * _endgame_start_minutes()
+        tail_fraction = None
+        if _get_env_bool("ARC3_PRIORITY_TAIL_FADE", False):
+            if self._clock_deadline is None or self._clock_start is None:
+                raise ValueError("ARC3_PRIORITY_TAIL_FADE requires a finite runtime or soft deadline")
+            fraction_setting = os.environ.get("ARC3_PRIORITY_TAIL_FADE_FRACTION", "").strip()
+            if fraction_setting:
+                try:
+                    fraction = float(fraction_setting)
+                except ValueError:
+                    raise ValueError("ARC3_PRIORITY_TAIL_FADE_FRACTION must be > 0 and <= 1") from None
+                if not 0.0 < fraction <= 1.0:
+                    raise ValueError("ARC3_PRIORITY_TAIL_FADE_FRACTION must be > 0 and <= 1")
+                # Use the full shared run duration, not the shrinking time left.
+                window = fraction * max(0.0, self._clock_deadline - self._clock_start)
+            else:
+                window = max(1.0, 60 * _get_env_float("ARC3_PRIORITY_TAIL_FADE_MINUTES", 90.0))
+            remaining = max(0.0, self._clock_deadline - now)
+            tail_fraction = remaining / window if window > 0.0 else 0.0
+            if remaining <= window:
+                # Snapshot scoring also happens outside _pump's lock. Announce
+                # once per shared gate, even when several games trim together.
+                with self._cond:
+                    if not self._tail_fade_announced:
+                        self._tail_fade_announced = True
+                        log.warning(
+                            "tail fade phase after %.1f minutes: %.1f minutes "
+                            "remaining; fading tail B to zero over the final "
+                            "%.1f minutes, retaining hazard C",
+                            elapsed / 60.0, remaining / 60.0, window / 60.0,
+                        )
+        return priority_value(
+            snapshot, endgame=endgame, tail_fraction=tail_fraction,
+            human_actions=_priority_human_actions(), action_scale=_priority_action_scale(),
+            token_scale=_priority_token_scale(), action_weight=_priority_action_weight(),
+            tail_base=_priority_tail_base(), tail_cap=_priority_tail_cap(),
+            **_priority_level_options(),
+        )
+
+    def acquire(self, priority: int) -> None:
+        with self._cond:
+            self._enqueue_and_wait(priority)
+
+    def release(self) -> None:
+        with self._cond:
+            self._free += 1
+            self._pump()
+
+    def handover(self, priority: int, snapshot: PrioritySnapshot | None = None) -> None:  # noqa: D401
+        """Give up the slot and immediately compete for it again.
+
+        The caller may win it straight back, which is the right outcome when
+        nothing else is waiting or everything else is further along.
+        """
+        _t0 = time.monotonic()
+        with self._cond:
+            self._free += 1
+            self._enqueue_and_wait(priority, snapshot)
+        _GATE_STATS["handover_ms"] += int(1000 * (time.monotonic() - _t0))
+
+    def _enqueue_and_wait(self, priority: int, snapshot: PrioritySnapshot | None = None) -> None:
+        # TEMPORARY: with slots == games nobody should ever block here
+        _GATE_STATS["enqueued"] += 1
+        self._seq += 1
+        token = self._seq
+        if snapshot is not None:
+            self._snapshots[token] = snapshot
+        # negated so the heap pops the HIGHEST priority; the sequence number
+        # breaks ties by arrival
+        heapq.heappush(self._waiting, (-int(priority), token))
+        self._pump()
+        if token not in self._admitted:
+            _GATE_STATS["blocked"] += 1
+            t0 = time.monotonic()
+            while token not in self._admitted:
+                self._cond.wait()
+            _GATE_STATS["blocked_ms"] += int(1000 * (time.monotonic() - t0))
+        self._admitted.discard(token)
+
+    def _pump(self) -> None:
+        if _priority_refresh_enabled() and self._free > 0:
+            now = time.monotonic()
+            self._waiting = [
+                (-self._snapshot_priority(self._snapshots[token], now), token)
+                if token in self._snapshots else (priority, token)
+                for priority, token in self._waiting
+            ]
+            heapq.heapify(self._waiting)
+        woke = False
+        while self._free > 0 and self._waiting:
+            _, token = heapq.heappop(self._waiting)
+            self._snapshots.pop(token, None)
+            self._admitted.add(token)
+            self._free -= 1
+            woke = True
+        if woke:
+            self._cond.notify_all()
+
+
+def _max_active_streams() -> int:
+    """Games allowed to hold a slot at once. 0 disables the gate entirely.
+
+    Below the game count this staggers the start - a slot only frees when a
+    game's context is first trimmed, which measurements put 30 to 50 turns in -
+    so early games run alone for a while and late ones begin much later. That is
+    the point: a prefix is reused for as long as it lasts, instead of fourteen
+    games each progressing at a fourteenth of the rate.
+    """
+    return max(0, _get_env_int("ARC3_MAX_ACTIVE_STREAMS", 0))
+
+
+_PRIORITY_GATE: _PriorityGate | None = None
+_PRIORITY_GATE_LOCK = threading.Lock()
+
+
+def _priority_gate() -> _PriorityGate | None:
+    """Process-wide, built on first use. Games run as threads in one process,
+    so a module-level gate needs no plumbing and nothing to pickle."""
+    global _PRIORITY_GATE
+    slots = _max_active_streams()
+    if slots <= 0:
+        return None
+    with _PRIORITY_GATE_LOCK:
+        if _PRIORITY_GATE is None:
+            _PRIORITY_GATE = _PriorityGate(slots)
+            log.warning("priority gate active: %d concurrent streams", slots)
+        return _PRIORITY_GATE
+
+
+_PRIORITY_BAND = 1_000_000
+_PRIORITY_UNTRIMMED_BASE = 2_000_000
+
+
+def _priority_level_options() -> dict[str, Any]:
+    # Keep boolean lookup settings backward compatible. The explicit
+    # "remaining" mode selects the 8/7/5/0 tail, already at its final scale.
+    lookup = os.environ.get("ARC3_PRIORITY_TAIL_LOOKUP", "").strip().lower()
+    return {
+        "normalize_score": _get_env_bool("ARC3_PRIORITY_SCORE_NORMALIZATION", False),
+        "tail_lookup": (
+            "remaining" if lookup == "remaining"
+            else _get_env_bool("ARC3_PRIORITY_TAIL_LOOKUP", False)
+        ),
+        "tail_efficiency": _get_env_float("ARC3_PRIORITY_TAIL_EFFICIENCY", 0.8),
+    }
+
+
+def _priority_action_scale() -> float:
+    return max(1.0, _get_env_float("ARC3_PRIORITY_ACTION_SCALE", 115.0))
+
+
+def _priority_token_scale() -> float:
+    return max(1.0, _get_env_float("ARC3_PRIORITY_TOKEN_SCALE", 62000.0))
+
+
+def _priority_action_weight() -> float:
+    """How much of `c` comes from the action curve rather than the token one.
+
+    A blend rather than max(actions, tokens): max sits below both curves, so it
+    demotes harder than either measure alone warrants. Replaying four runs
+    against their own level-up costs, max was the weakest of every combination
+    tried, while the weight itself barely mattered - 0.25 to 1.0 all landed
+    within a few percent of each other.
+    """
+    return min(1.0, max(0.0, _get_env_float("ARC3_PRIORITY_ACTION_WEIGHT", 0.25)))
+
+
+def _priority_tail_cap() -> float:
+    """Ceiling on `b`, the value of every level after the current one.
+
+    min(cap, level + 1) rather than a constant. Chaining the measured
+    completion rates makes `b` nearly flat, which is why a constant looked
+    right - but that chain prices expected future score, and what the scheduler
+    needs is the term that lets depth outrank freshness. Replayed, a flat tail
+    was the worst option tested at every constrained budget; the cap matters
+    more than the slope, and a lower cap is better because it leaves more of
+    the ordering to `a`.
+    """
+    return max(0.0, _get_env_float("ARC3_PRIORITY_TAIL_CAP", 4.0))
+
+
+def _priority_tail_base() -> float:
+    """Value of `b` at level 1, before the per-level rise.
+
+    b = min(cap, base + level - 1). Base 2 with cap 4 gives the fitted form,
+    2 3 4 4 4...; base 6 with cap 6 gives a flat tail, which is what this
+    replaced and is worth being able to get back for a comparison.
+    """
+    return max(0.0, _get_env_float("ARC3_PRIORITY_TAIL_BASE", 2.0))
+
+
+def _priority_human_actions() -> float:
+    """Denominator offset in the efficiency term.
+
+    The benchmark scores a level by min(1, (human/used)^2). The level is not
+    finished yet, so the term belongs at the expected completion point rather
+    than the current count - hence an offset instead of a bare division.
+    """
+    return max(1.0, _get_env_float("ARC3_PRIORITY_HUMAN_ACTIONS", 25.0))
+
+
+def _endgame_start_minutes() -> float:
+    """Wall-clock minutes from a game's start before it switches phase.
+
+    0 disables the endgame phase entirely.
+    """
+    return max(0.0, _get_env_float("ARC3_ENDGAME_START_MINUTES", 0.0))
+
+
+# Endgame priorities sit in the upper half of whichever warmth band the game is
+# in: the cache argument for the bands is orthogonal to the phase, so a game
+# about to pay a prefill still yields to one that has a warm prefix. The widest
+# value either phase can produce is under 2,000 even at level 15, so half a
+# band is ample.
+_PRIORITY_ENDGAME_OFFSET = 500_000
+
+
+def _game_priority(
+    level: int,
+    actions_on_level: int,
+    tokens_on_level: int = 0,
+    endgame: bool = False,
+    total_levels: int | None = None,
+) -> int:
+    """Expected remaining score for a game on `level` after this much spend.
+
+    Two phases. In the main one, (a + b) * c:
+
+      a  what finishing THIS level banks - the score becomes `level` levels
+         completed, discounted by the benchmark's efficiency term
+      b  every level after it, min(cap, base + level - 1)
+      c  whether this level still yields anything, as a blend of two measured
+         decay curves
+
+    In the endgame there is no time to reach a further level, so the tail and
+    the hazard stop meaning anything and what remains is simply what completing
+    this level would bank. The 0.25 floor lets efficiency separate games at the
+    same depth without letting a shallower one overtake a deeper one.
+
+    Replaying four runs against their own level-up costs, the endgame switch was
+    worth 8 to 12% of total score at constrained budgets and nothing at all once
+    the budget covered what the games actually spent - which is where real runs
+    sit, so this matters only if the competition budget is tighter.
+    """
+    level = max(1, int(level))
+    actions = max(0, int(actions_on_level))
+    tokens = max(0, int(tokens_on_level))
+    options = _priority_level_options()
+    if options["normalize_score"] or options["tail_lookup"]:
+        # Same scoring as the refreshed queue, retaining this legacy path's
+        # phase offset. With both flags off, execute the original path below.
+        value = priority_value(
+            PrioritySnapshot(level, actions, tokens, total_levels=total_levels),
+            endgame=endgame, human_actions=_priority_human_actions(),
+            action_scale=_priority_action_scale(), token_scale=_priority_token_scale(),
+            action_weight=_priority_action_weight(), tail_base=_priority_tail_base(),
+            tail_cap=_priority_tail_cap(), **options,
+        )
+        return value + (_PRIORITY_ENDGAME_OFFSET if endgame else 0)
+    human = _priority_human_actions()
+    efficiency = min(1.0, (human / (human + actions)) ** 2)
+
+    if endgame:
+        value = level * (0.25 + efficiency)
+        return _PRIORITY_ENDGAME_OFFSET + max(1, int(value * 100))
+
+    a = level * efficiency
+    b = min(_priority_tail_cap(), _priority_tail_base() + level - 1.0)
+    weight = _priority_action_weight()
+    by_actions = actions / _priority_action_scale()
+    by_tokens = tokens / _priority_token_scale()
+    c = (
+        weight * 0.5 ** (by_actions * by_actions)
+        + (1.0 - weight) * 0.5 ** (by_tokens * by_tokens)
+    )
+    return max(1, int((a + b) * c * 100))
+
+
+def _persistent_functions_scope() -> str:
+    """How long a kept helper lives: 'levelup' (default), 'turn', or 'game'.
+
+    'turn' clears at every analyzer invocation that is not a resumption, so a
+    turn that inspects, acts and yields loses its toolkit even though the
+    investigation continues. That is conservative but costly: the helpers the
+    keepability check lets through are pure transforms taking the board as an
+    argument, and those do not care that an action happened.
+
+    'levelup' keeps them until the level changes, which is where a helper can
+    genuinely go wrong - not by holding stale data, which the check already
+    prevents, but by encoding an assumption in its code, like a glyph parser
+    with row offsets from the previous layout.
+
+    'game' keeps helpers across turns and level transitions. The model is
+    reminded to recheck level-specific assumptions. New game sessions still
+    start with an empty pool; snippet variables remain ephemeral in every scope.
+    """
+    scope = os.environ.get("ARC3_PERSISTENT_FUNCTIONS_SCOPE", "").strip().lower()
+    return scope if scope in ("turn", "levelup", "game") else "levelup"
+
+
+def _repeat_state_guard_enabled() -> bool:
+    """Refuse an action this snippet has already run from this exact board.
+
+    Buggy loop code can spend a whole life bar getting nowhere: a cursor that
+    wraps turns RIGHT into a five-state cycle, and `while pos < j: LEFT` walks
+    away from its target forever. The terminal latch does stop these, but only
+    once the game ends - 20 to 30 actions later.
+
+    Keyed on (board, action) rather than on repetition counts, because the
+    loops mostly alternate: LEFT, RIGHT, LEFT, RIGHT defeats a same-action rule
+    but revisits its first state on the third action.
+
+    Per snippet, and no override slot. A repeat ACROSS snippets is the model
+    choosing, having seen the board and the results; a repeat inside one is
+    code executing a plan fixed before any of it ran, which is the only case
+    that can be a bug the model has not had a chance to notice. It is also not
+    reading the refusal - its code is running - so writing a new snippet is the
+    response, and that clears the map.
+    """
+    return _get_env_bool("ARC3_REPEAT_STATE_GUARD", False)
+
+
+def _guards_active(level: Any) -> bool:
+    """Whether the per-action guards apply at this level.
+
+    Level 1 pays the smallest share of the score - one part in N - and it is
+    where the model knows least, so it probes hardest: repeating an action to
+    see whether the repeat differs, retrying what killed it to learn what
+    killed it. The guards read that as waste. ARC3_GUARDS_FROM_LEVEL=2 lets
+    level 1 run unguarded.
+
+    Covers the stale block and both repeat guards. NOT the batch no-op stop,
+    which costs one action of a planned sequence rather than refusing a
+    deliberate probe, and not the terminal latch, which stops a snippet acting
+    against a board that no longer exists.
+    """
+    try:
+        floor = max(1, int(os.environ.get("ARC3_GUARDS_FROM_LEVEL", "1") or 1))
+    except ValueError:
+        floor = 1
+    try:
+        return int(level) >= floor
+    except (TypeError, ValueError):
+        return True  # unknown level: guard rather than not
+
+
+def _persistent_functions() -> bool:
+    """Retain eligible function sources across calls; off by default.
+
+    Scope is controlled by ARC3_PERSISTENT_FUNCTIONS_SCOPE. State and variables
+    are never serialized. Definitions are validated and recreated in each fresh
+    sandbox, against its current harness globals.
+    """
+    return _get_env_bool("ARC3_PERSISTENT_FUNCTIONS", False)
+
+
+def _animation_enabled() -> bool:
+    """Whether intermediate animation frames are captured and offered.
+
+    Off by default: this is new information for the model rather than a fix to
+    something broken, so it needs to be measurable on its own."""
+    return _get_env_bool("ARC3_ANIMATION", False)
+
+
+def _report_gameplay_changed() -> bool:
+    """Tell the model WHICH kind of change an action produced.
+
+    The harness computes two flags - `board_changed` over the whole frame and
+    `gameplay_changed` over the interior only - but the turn opener reports the
+    first and the system prompt names only the first. On a game with a
+    remaining-steps bar the whole frame differs after every action, so the
+    opener says a change occurred every time and hedges: "verify that it
+    affected gameplay objects rather than only HUD elements". Meanwhile the
+    per-action trace, built from the second flag, can say the same action
+    changed nothing - so the two lines of the same message contradict each
+    other, and on a single-action turn only the misleading one appears.
+
+    On (default) the opener reports the three states that actually occur and
+    the system prompt explains both flags. The wording stays geometric -
+    "cells inside the board area" rather than "gameplay objects" - because the
+    test is a border crop, not a semantic one."""
+    return _get_env_bool("ARC3_REPORT_GAMEPLAY_CHANGED", False)
+
+
+def _explain_gameplay_changed() -> bool:
+    """Whether the system prompt explains the two change flags.
+
+    Split from the per-turn line because the two halves have very different
+    reach and very different arguments. The sentence is paid once and states
+    something the model cannot work out for itself: `last_action_call_result`
+    exposes both flags and nothing says what distinguishes them, so a game with
+    a timer bar makes `board_changed` true after every action.
+
+    The opener line is paid on every executing turn - on one measured run,
+    thousands of times against thirteen batch-guard firings - and only
+    summarises a comparison the model can make with frame_diff. It is also the
+    half with evidence against it: the one run that carried it scored 2.30 on
+    competition where the run without it scored 4.05, though that run changed
+    guard wording at the same time.
+
+    Defaults to following ARC3_REPORT_GAMEPLAY_CHANGED so the pair behaves as
+    before unless it is set explicitly.
+    """
+    raw = os.environ.get("ARC3_EXPLAIN_GAMEPLAY_CHANGED", "").strip()
+    if not raw:
+        return _report_gameplay_changed()
+    return raw.lower() not in ("0", "false", "no", "off")
+
+
+def _alias_respects_stored() -> bool:
+    """Whether `Hypothesis:` may replace a world model that already exists.
+
+    The alias condition checks only THIS reply's extraction: a reply carrying a
+    hypothesis and no `World model:` has its hypothesis stored as the world
+    model, overwriting whatever was there. That is right on a first turn and
+    destructive later - `Hypothesis` connotes untested, so a speculation
+    silently replaces tested content. Measured on 25 games: 7 such overwrites
+    against 351 world-model writes with tolerant headers on, 16 against 311 in
+    an unpatched run.
+
+    On (default) the aliases apply only when the STORED value is also empty.
+    Off restores the original behaviour."""
+    return _get_env_bool("ARC3_ALIAS_RESPECTS_STORED", True)
+
+
+def _extract_scientist_note(
+    content: str, stored: dict[str, Any] | None = None
+) -> dict[str, str]:
     if not content.strip():
         return {}
     extracted = _extract_labeled_blocks(
@@ -287,12 +2471,19 @@ def _extract_scientist_note(content: str) -> dict[str, str]:
         "current_plan": extracted.get("Plan", ""),
         "cross_level_notes": extracted.get("Cross-level notes", ""),
     }
-    if not result["world_model"]:
-        result["world_model"] = extracted.get("Hypothesis", "")
-    if not result["recent_findings"]:
-        result["recent_findings"] = extracted.get("History check", "")
-    if not result["current_plan"]:
-        result["current_plan"] = extracted.get("Next test", "")
+    respect_stored = _alias_respects_stored()
+    existing = stored or {}
+
+    def apply_alias(key: str, label: str) -> None:
+        if result[key]:
+            return
+        if respect_stored and str(existing.get(key) or "").strip():
+            return
+        result[key] = extracted.get(label, "")
+
+    apply_alias("world_model", "Hypothesis")
+    apply_alias("recent_findings", "History check")
+    apply_alias("current_plan", "Next test")
     return result
 
 
@@ -309,7 +2500,17 @@ def _empty_world_model() -> dict[str, str]:
 
 
 def _request_tool_choice(tools: list[dict[str, Any]] | None) -> str | None:
-    return "auto" if tools else None
+    """Tool-choice sent with tool-bearing requests. LOCAL_ANALYZER_TOOL_CHOICE
+    overrides the default "auto": set "omit"/"none" to drop the field for
+    servers whose validation rejects "auto" without a tool-call parser (tools
+    are still sent, so chat templates still render tool instructions); any
+    other value (e.g. "required") passes through verbatim."""
+    if not tools:
+        return None
+    choice = os.environ.get("LOCAL_ANALYZER_TOOL_CHOICE", "auto").strip().lower()
+    if choice in ("", "omit", "none"):
+        return None
+    return choice
 
 
 def _trim_log_text(text: str, *, max_chars: int = _RESPONSE_META_MAX_CHARS) -> str:
@@ -323,6 +2524,7 @@ def _trim_log_text(text: str, *, max_chars: int = _RESPONSE_META_MAX_CHARS) -> s
 def _format_model_response_meta(
     *,
     finish_reason: str,
+    served_by: str,
     reasoning: str,
     content: str,
     tool_calls: list[dict[str, Any]],
@@ -332,6 +2534,7 @@ def _format_model_response_meta(
 ) -> str:
     lines = [
         f"finish_reason: {finish_reason or '(empty)'}",
+        f"served_by: {served_by or '(unknown)'}",
         f"tool_call_count: {len(tool_calls)}",
         f"content_chars: {len(content)}",
         f"reasoning_chars: {len(reasoning)}",
@@ -347,15 +2550,203 @@ def _format_model_response_meta(
     return "\n".join(lines)
 
 
+def _estimate_diff_tokens(text: str) -> int:
+    return len(text) // 3
+
+
+def _diff_entry_line(bucket: str, entry: dict[str, Any]) -> str:
+    color = entry.get("color")
+    pixels = entry.get("pixels")
+    if bucket == "moved":
+        line = f"moved: color {color} {pixels}px {entry.get('from')}->{entry.get('to')}"
+        if entry.get("rotated_by"):
+            line += f" rotated_by {entry['rotated_by']}"
+        return line
+    if bucket == "rotated":
+        return (
+            f"rotated: color {color} {pixels}px by {entry.get('rotated_by')} "
+            f"at {entry.get('at')}"
+        )
+    if bucket == "changed_color":
+        return (
+            f"recolored at {entry.get('at')}: color "
+            f"{entry.get('before_color')}->{entry.get('after_color')} "
+            f"({entry.get('pixels')}px)"
+        )
+    if bucket == "resized":
+        return (
+            f"resized: color {entry.get('color')} "
+            f"{entry.get('before_pixels')}->{entry.get('after_pixels')}px "
+            f"at {entry.get('at_before')}"
+        )
+    return f"{bucket}: color {color} {pixels}px at {entry.get('at')}"
+
+
+_DIFF_BUCKETS = ("moved", "rotated", "changed_color", "resized", "appeared", "disappeared")
+
+
+def _render_auto_frame_diff(
+    diff: dict[str, Any], budget_tokens: int, total_cells: int | None = None
+) -> list[str]:
+    """Render a frame diff as prompt lines, degrading to stay under budget:
+    full listing -> aggregated by (bucket, color, pixels) -> counts only.
+    Resized entries covering most of the grid (the background reshaping as
+    collateral of other objects moving) are elided as noise; the callable
+    frame_diff() keeps them."""
+    if total_cells:
+        resized = diff.get("resized") or []
+        kept = [e for e in resized if (e.get("before_pixels") or 0) * 2 < total_cells]
+        if len(kept) != len(resized):
+            diff = dict(diff)
+            diff["resized"] = kept
+    count = diff.get("changed_cell_count")
+    if count == -1:
+        return [f"grid shape changed ({diff.get('before_shape')} -> {diff.get('after_shape')}); per-object diff skipped."]
+    header = f"{count} cells changed."
+    # rung 1: full listing
+    full = [header]
+    for bucket in _DIFF_BUCKETS:
+        for entry in diff.get(bucket) or []:
+            full.append("- " + _diff_entry_line(bucket, entry))
+    if _estimate_diff_tokens("\n".join(full)) <= budget_tokens:
+        return full
+    # rung 2: aggregate identical entries, largest objects first, capped
+    lines = [header]
+    for bucket in _DIFF_BUCKETS:
+        entries = list(diff.get(bucket) or [])
+        if not entries:
+            continue
+        entries.sort(key=lambda e: -(e.get("pixels") or e.get("after_pixels") or 0))
+        groups: dict[tuple, int] = {}
+        for entry in entries:
+            key = (entry.get("color"), entry.get("pixels"))
+            groups[key] = groups.get(key, 0) + 1
+        shown = 0
+        for entry in entries:
+            if shown >= 3:
+                break
+            key = (entry.get("color"), entry.get("pixels"))
+            if groups.get(key, 0) > 3:
+                continue
+            lines.append("- " + _diff_entry_line(bucket, entry))
+            shown += 1
+        swarm = {k: n for k, n in groups.items() if n > 3}
+        for (color, pixels), n in sorted(swarm.items(), key=lambda kv: -kv[1]):
+            lines.append(f"- {bucket}: {n}x color {color} {pixels}px (positions elided)")
+        remaining = len(entries) - shown - sum(swarm.values())
+        if remaining > 0:
+            lines.append(f"- {bucket}: +{remaining} more")
+    lines.append("(call frame_diff() in python for the full object list)")
+    if _estimate_diff_tokens("\n".join(lines)) <= budget_tokens:
+        return lines
+    # rung 3: counts only, cannot be large
+    parts = [
+        f"{len(diff.get(bucket) or [])} {bucket}"
+        for bucket in _DIFF_BUCKETS
+        if diff.get(bucket)
+    ]
+    return [
+        header + " " + ", ".join(parts) + " (details omitted; call frame_diff() in python)."
+    ]
+
+
 def _build_system_prompt(*, tool_output_tokens: int) -> str:
     prompt = "You are a coding agent solving a grid-based puzzle game."
+    # ARC3_SYSTEM_PROMPT_PREFIX (read at call time): prepended verbatim as the
+    # first line(s) of the system prompt. Intended for model-specific control
+    # directives that live in the system prompt, e.g. Nemotron's "/think"
+    # reasoning toggle. Empty/unset leaves the prompt unchanged.
+    _prefix = os.environ.get("ARC3_SYSTEM_PROMPT_PREFIX", "").strip()
+    if _prefix:
+        prompt = f"{_prefix}\n\n{prompt}"
     prompt += GAME_OVERVIEW_ADDENDUM
     prompt += STRUCTURED_RUNTIME_STATE_ADDENDUM
+    if _explain_gameplay_changed():
+        prompt += GAMEPLAY_CHANGED_ADDENDUM.format(
+            border=f"{_get_env_int('ARC3_NOOP_GUARD_BORDER', 4)} cells"
+        )
+    if _animation_enabled():
+        prompt += ANIMATION_ADDENDUM
+        if _animation_timeline_enabled():
+            prompt += ANIMATION_ADDENDUM_TIMELINE
     if current_grid_image_enabled():
         prompt += MULTIMODAL_CONTEXT_ADDENDUM
+    if _AUTO_FRAME_DIFF:
+        prompt += (
+            " Each turn's message includes an 'Auto frame diff' section summarizing "
+            "object-level changes since your previous analyzer turn (it spans all "
+            "actions executed in between, not just the last one). Treat it as ground "
+            "truth for what changed, consult it before acting, and call `frame_diff()` "
+            "in python when you need per-action or full object detail."
+        )
+    if _DIFF_IMAGE and current_grid_image_enabled():
+        prompt += (
+            " Alongside the current grid image you receive a diff image: cells that "
+            "changed since your previous analyzer turn are shown in their new true "
+            "color; unchanged cells are dimmed to a dark navy that is not a game "
+            "color. Use it to locate change at a glance."
+        )
     prompt += VISUAL_GAME_ADDENDUM
-    prompt += PYTHON_ADDENDUM
-    prompt += COMPACT_TOOL_SESSION_ADDENDUM.format(tool_output_tokens=tool_output_tokens)
+    if _get_env_bool("ARC3_ACTION_INFO", False):
+        # The action-info section replaces the earlier coordinate-only guidance.
+        prompt = prompt.replace(
+            "- For `MOUSE`, pass `row` and `col` integer arguments. `row` is vertical position, `col` is horizontal position.\n",
+            "",
+        )
+        prompt += ACTION_INFO_ADDENDUM
+        if undo_exposure_mode() == "on":
+            prompt += UNDO_INFO_ADDENDUM
+        if reset_exposed():
+            prompt += RESET_INFO_ADDENDUM
+    prompt += (
+        PYTHON_ADDENDUM_HEAD.replace(
+            "- Every `python` tool call starts fresh. Re-import modules or re-define any custom utility logic you need.\n",
+            "- Python variables reset between tool calls. Re-import modules as needed; "
+            "eligible functions are retained as described in the tool session rules below.\n",
+        ) if _persistent_functions() else PYTHON_ADDENDUM_HEAD
+    )
+    prompt += (
+        WORLD_MODEL_FREE_ADDENDUM if _memory_sections_disabled() else WORLD_MODEL_ADDENDUM
+    )
+    prompt += PYTHON_ADDENDUM_TAIL
+    if _get_env_bool("ARC3_FRAME_DIFF_HINT", False):
+        # documents `frame_diff(before, after)`. The function stays callable
+        # either way - this only controls whether the model is told about it,
+        # so the arm measures whether knowing costs more than it returns.
+        prompt += FRAME_DIFF_HINT_ADDENDUM
+    if _get_env_bool("ARC3_STEP_VERIFICATION_HINT", False):
+        # General checks during action sequences and targeted probes when search fails.
+        prompt += STEP_VERIFICATION_ADDENDUM
+    # only describe the guards that are actually armed: prompt weight spent on a
+    # mechanism that cannot fire is weight the model reads and cannot use, and an
+    # exception it is told about but never sees is worse than silence
+    # The guards are no longer described up front. Every claim the addenda made -
+    # not executed, nothing spent, the batch is checked throughout, how to
+    # override - is repeated in the refusal message itself, which arrives with
+    # the concrete action attached and only when it is relevant. Describing a
+    # mechanism the model may never meet is prompt weight spent on nothing.
+    prompt += COMPACT_TOOL_SESSION_ADDENDUM.format(
+        tool_output_tokens=tool_output_tokens,
+        prefer_tool_calls=PREFER_TOOL_CALLS_LINE if _prefer_tool_calls() else "",
+        persistent_functions=(
+            PERSISTENT_FUNCTIONS_LINE.format(lifetime=(
+                "for later calls throughout this game, including across level changes; "
+                "they are cleared when a new game starts"
+                if _persistent_functions_scope() == "game" else
+                "for later calls on the same level; they are cleared when the level changes"
+                if _persistent_functions_scope() == "levelup" else
+                "for later calls in this analyzer turn, including yield resumptions; "
+                "they are cleared at the start of the next turn"
+            ), import_guidance=(
+                "Supported explicit top-level imports used by retained functions are restored "
+                "with them; imports inside a function also work."
+                if _get_env_bool("ARC3_PERSISTENT_FUNCTIONS_IMPORTS", False) else
+                "Put needed imports inside the function."
+            )) if _persistent_functions() else EPHEMERAL_FUNCTIONS_LINE
+        ),
+    )
+    if _level_transfer_guidance_enabled():
+        prompt = apply_level_transfer_system_guidance(prompt)
     return prompt
 
 
@@ -378,6 +2769,12 @@ class AnalyzerTurnResult:
 class _ToolDispatchResult:
     content: str
     step_executed: bool = False
+    # True when an executed action changed the board outside the HUD border, or
+    # ended the level or the attempt. "Executed" alone is too weak a signal for
+    # the reasoning-effort ladder: bumping a wall executes and achieves nothing,
+    # and resetting on it lets the model oscillate between truncating at full
+    # effort and acting inertly at reduced effort, never settling.
+    made_progress: bool = False
 
 
 @dataclass(frozen=True)
@@ -398,6 +2795,7 @@ class _AsciiFrameView:
 class _AsciiHistoryEntryView:
     action: str
     frame: _AsciiFrameView
+    result: dict[str, Any]
 
     def __str__(self) -> str:
         return f"AsciiHistoryEntryView(action={self.action!r}, frame={self.frame})"
@@ -422,7 +2820,10 @@ def _to_ascii_history_views(history_entries: list[HistoryEntry]) -> list[_AsciiH
         frame_view = _to_ascii_frame_view(entry.frame)
         if frame_view is None:
             continue
-        views.append(_AsciiHistoryEntryView(action=entry.action, frame=frame_view))
+        views.append(_AsciiHistoryEntryView(
+            action=entry.action, frame=frame_view,
+            result=dict(getattr(entry, "result", {}) or {}),
+        ))
     return views
 
 
@@ -445,26 +2846,167 @@ def _ascii_history_view_payload(history_entries: list[HistoryEntry]) -> list[dic
         frame_payload = _ascii_frame_view_payload(entry.frame)
         if frame_payload is None:
             continue
-        payload.append({"action": entry.action, "frame": frame_payload})
+        payload.append({
+            "action": entry.action, "frame": frame_payload,
+            "result": dict(getattr(entry, "result", {}) or {}),
+        })
     return payload
 
 
-def _format_action_span(start_action_num: int | None, end_action_num: int | None) -> str | None:
-    if start_action_num is None or end_action_num is None:
+_VISION_PATCH_PIXELS = 32          # Qwen3.6: 16px patches, 2x2 merge
+_VISION_SENTINEL_TOKENS = 2        # vision start/end markers
+_IMAGE_TOKENS_FALLBACK = 402       # 64-cell board at MULTIMODAL_UPSCALE=10
+_PNG_SIGNATURE = b"\x89PNG\r\n\x1a\n"
+
+
+def _text_token_chars() -> int:
+    """Characters per token for the text heuristic - the SEED value only.
+
+    Measured against a server-reported prompt_tokens on a real request: 25,189
+    characters of system prompt plus opener tokenised to 5,892, i.e. about 4.3
+    characters per token. The 3 default therefore over-counts text by roughly
+    40%, so the trimmer evicts history the budget could have held. This value
+    is used until the first response supplies a measurement; after that
+    ARC3_CALIBRATE_TEXT_TOKENS drives the divisor from the server's own
+    figures."""
+    return max(1, _get_env_int("ARC3_TEXT_TOKEN_CHARS", 3))
+
+
+def _calibrate_text_tokens() -> bool:
+    """Derive the characters-per-token divisor from the server's reported
+    prompt_tokens rather than a fixed guess.
+
+    Every response carries prompt_tokens for the request that produced it - an
+    exact count including the chat template. Subtracting the vision cost of the
+    request's images leaves the text, and dividing by the text characters gives
+    the divisor for the next estimate. No extra requests, no tokenizer, and it
+    tracks content: code-heavy turns tokenise differently from prose.
+
+    The last measurement is used rather than an average: history carries
+    forward, so the prompt just measured is mostly the prompt about to be
+    built, and averaging would blend in content already evicted."""
+    return _get_env_bool("ARC3_CALIBRATE_TEXT_TOKENS", True)
+
+
+# Asymmetric on purpose. Over-counting trims history and degrades one game;
+# under-counting overflows the context, and an overflowing request holds a
+# slot and can pressure the others. Measured across 22 games: 2.75-3.69,
+# mean 3.12. The 3.3 ceiling therefore binds on the sparsest games by
+# design - a deliberate haircut - while the floor is set low enough that
+# only an arithmetic failure reaches it, and such a game is likely lost
+# anyway.
+def _text_token_chars_min() -> float:
+    """Floor for the calibrated divisor. Low enough that only an arithmetic
+    failure reaches it, and a game in that state is likely lost anyway."""
+    # a divisor of zero would make every estimate infinite and trim the whole
+    # history on the first request
+    return max(0.1, _get_env_float("ARC3_TEXT_TOKEN_CHARS_MIN", 1.0))
+
+
+def _text_token_chars_max() -> float:
+    """Ceiling for the calibrated divisor.
+
+    Exposed because 3.3 is a fitted constant, not a property of the tokeniser:
+    it came from 22 games measuring 2.75-3.69, so it binds on the sparsest
+    boards by design. A run whose prompts are denser than that sample - a
+    different game set, a longer system prompt, another model - is being
+    trimmed harder than its own measurements justify, and raising this is how
+    that is tested.
+    """
+    # never below the floor: an inverted pair would clamp every measurement to
+    # the ceiling and silently invert the guard's meaning
+    return max(_text_token_chars_min(),
+               _get_env_float("ARC3_TEXT_TOKEN_CHARS_MAX", 3.3))
+
+
+def _image_token_estimate_enabled() -> bool:
+    """Count image parts by their real vision-token cost instead of by the
+    length of their base64 payload. Without this an image is charged roughly
+    one token per three base64 characters - thousands for a board render whose
+    true cost is a few hundred - so enabling images silently evicts real
+    history."""
+    return _get_env_bool("ARC3_IMAGE_TOKEN_ESTIMATE", True)
+
+
+def _png_dimensions(data_url: str) -> tuple[int, int] | None:
+    """Width/height from a base64 PNG data URL, read from the IHDR header
+    without decoding the image. Only the first 24 bytes are needed."""
+    marker = "base64,"
+    index = data_url.find(marker)
+    if index < 0:
+        return None
+    head = data_url[index + len(marker):index + len(marker) + 32]
+    if len(head) < 32:
+        return None
+    try:
+        raw = base64.b64decode(head, validate=True)
+    except Exception:
         return None
-    if start_action_num <= 0 or end_action_num <= 0:
+    if len(raw) < 24 or not raw.startswith(_PNG_SIGNATURE) or raw[12:16] != b"IHDR":
         return None
-    if start_action_num == end_action_num:
-        return f"{start_action_num}"
-    return f"{start_action_num}-{end_action_num}"
+    width = int.from_bytes(raw[16:20], "big")
+    height = int.from_bytes(raw[20:24], "big")
+    if width <= 0 or height <= 0:
+        return None
+    return width, height
+
+
+def _image_part_tokens(part: dict[str, Any]) -> int:
+    """Vision tokens for one image content part: one token per merged patch
+    plus the two sentinel tokens. ARC3_IMAGE_TOKENS_FLAT overrides the
+    computation (useful for non-Qwen backends) and is also the fallback when
+    the header cannot be read."""
+    flat = _get_env_int("ARC3_IMAGE_TOKENS_FLAT", 0)
+    if flat > 0:
+        return flat
+    url = ""
+    image_url = part.get("image_url")
+    if isinstance(image_url, dict):
+        url = str(image_url.get("url") or "")
+    dims = _png_dimensions(url) if url else None
+    if dims is None:
+        return _IMAGE_TOKENS_FALLBACK
+    width, height = dims
+    cells = -(-width // _VISION_PATCH_PIXELS) * -(-height // _VISION_PATCH_PIXELS)
+    return cells + _VISION_SENTINEL_TOKENS
+
+
+def _split_messages_for_estimate(
+    messages: list[dict[str, Any]],
+) -> tuple[list[dict[str, Any]], int]:
+    """Return (messages with image payloads replaced by a short placeholder,
+    total vision tokens). Originals are never mutated."""
+    image_tokens = 0
+    rewritten: list[dict[str, Any]] = []
+    for message in messages:
+        content = message.get("content")
+        if not isinstance(content, list):
+            rewritten.append(message)
+            continue
+        parts: list[Any] = []
+        for part in content:
+            if isinstance(part, dict) and part.get("type") == "image_url":
+                image_tokens += _image_part_tokens(part)
+                parts.append({"type": "image_url", "image_url": {"url": "<image>"}})
+            else:
+                parts.append(part)
+        rewritten.append({**message, "content": parts})
+    return rewritten, image_tokens
 
 
-def _estimate_tokens(value: Any) -> int:
+def _estimate_tokens(value: Any, chars_per_token: float | None = None) -> int:
     try:
-        rendered = json.dumps(value, ensure_ascii=True, sort_keys=True, default=str)
+        # ensure_ascii=False is deliberate: with escaping ON, every non-ASCII
+        # character renders as a six-character \uXXXX sequence, so any prompt
+        # containing arrows, box-drawing glyphs or a multiplication sign is
+        # counted at up to 6x its real size. The trimmer decides when to evict
+        # in these units, so the inflation makes it discard history it did not
+        # need to - and the error grows with the share of non-ASCII text.
+        rendered = json.dumps(value, ensure_ascii=False, sort_keys=True, default=str)
     except TypeError:
         rendered = str(value)
-    return max(1, (len(rendered) + 2) // 3)
+    chars = chars_per_token or _text_token_chars()
+    return max(1, int(len(rendered) / chars) + (1 if len(rendered) % chars else 0))
 
 
 def _host_accessible_base_url(base_url: str) -> str:
@@ -785,20 +3327,29 @@ def _append_request_snapshot(
     event: str | None = None,
     tool_choice: str | None = None,
     finish_reason: str | None = None,
+    served_by: str | None = None,
     analysis_step: int | None = None,
     action: int | None = None,
     request_index_within_turn: int | None = None,
+    usage: dict[str, Any] | None = None,
+    chat_template_kwargs: dict[str, Any] | None = None,
 ) -> None:
     payload = {
         "messages": messages,
         "tools": tools or [],
     }
+    if isinstance(chat_template_kwargs, dict) and chat_template_kwargs:
+        payload["chat_template_kwargs"] = dict(chat_template_kwargs)
+    if isinstance(usage, dict) and usage:
+        payload["usage"] = usage
     if event:
         payload["event"] = event
     if tool_choice:
         payload["tool_choice"] = tool_choice
     if finish_reason is not None:
         payload["finish_reason"] = str(finish_reason)
+    if served_by:
+        payload["served_by"] = str(served_by)
     if analysis_step is not None:
         payload["analysis_step"] = analysis_step
     if action is not None:
@@ -885,9 +3436,13 @@ def _extract_reasoning_text(message: dict[str, Any]) -> str:
 
 
 def _is_context_length_error(exc: BaseException) -> bool:
-    message = str(exc).lower()
+    message = str(exc).lower().replace("\u2019", "'")
     return (
         "maximum context length" in message
+        # SGLang reports an oversized prompt with this wording instead of
+        # "maximum context length". Route it through the same drain/retry path.
+        or "is longer than the model's context length" in message
+        or "context_length_exceeded" in message
         or "reduce the length of the input prompt" in message
         or "parameter=input_tokens" in message
         or '"param":"input_tokens"' in message
@@ -899,6 +3454,7 @@ class _ChatCompletionResult:
     message: dict[str, Any]
     finish_reason: str = ""
     usage: dict[str, Any] | None = None
+    served_by: str = ""
 
 
 class ToolAgent:
@@ -910,6 +3466,7 @@ class ToolAgent:
         model: str = _DEFAULT_ANALYZER_MODEL,
         timeout: Optional[float] = None,
         save_request_logs: bool = False,
+        dispatch_index: int = 0,
         api_key: str | None = None,
         base_url: str | None = None,
         provider: str | None = None,
@@ -932,6 +3489,10 @@ class ToolAgent:
         self._tool_steps = None if _LOCAL_ANALYZER_TOOL_STEPS <= 0 else max(1, _LOCAL_ANALYZER_TOOL_STEPS)
         self._python_timeout = min(30, max(1, _LOCAL_ANALYZER_TOOL_TIMEOUT))
         self._yield_seconds = None if _LOCAL_ANALYZER_YIELD_SECONDS <= 0 else float(_LOCAL_ANALYZER_YIELD_SECONDS)
+        self._yield_tokens = (
+            None if _LOCAL_ANALYZER_YIELD_TOKENS <= 0 else int(_LOCAL_ANALYZER_YIELD_TOKENS)
+        )
+        self._turn_generated_tokens = 0
         configured_max_output = _LOCAL_ANALYZER_MAX_OUTPUT
         self._max_output_tokens = None if configured_max_output <= 0 else max(1, configured_max_output)
         self._reply_reserve_tokens = self._max_output_tokens or 512
@@ -946,6 +3507,75 @@ class ToolAgent:
             1024,
             _LOCAL_ANALYZER_CONTEXT_WINDOW - self._reply_reserve_tokens - self._request_safety_margin_tokens,
         )
+        self._turns_without_wm_update = 0
+        self._pending_wm_rebuild_reason = ""
+        self._wm_absorbed_in_turn = False
+        self._auto_diff_prev_frame: Frame | None = None
+        self._pending_diff_image: dict[str, Any] | None = None
+        self._context_was_trimmed = False
+        self._has_evicted = False
+        self._actions_at_level_start = 0
+        self._tokens_at_level_start = 0
+        self._progress_pace = ProgressPace()
+        self._pace_last_completed_level = 0
+        # queue position spreads the untrimmed band so games do not tie
+        self._diag_name = f"g{int(dispatch_index):02d}"
+        self._game_elapsed_seconds: float | None = None
+        self._endgame_announced = False
+        self._priority_current = _PRIORITY_UNTRIMMED_BASE - max(0, int(dispatch_index))
+        self._priority_next = self._priority_current
+        self._kept_functions: dict[str, str] = {}
+        self._last_animation_chain: list[Any] = []
+        self._last_action_gameplay_changed: bool | None = None
+        self._pending_animation_image: dict[str, Any] | None = None
+        self._pending_animation_image_label: str = ""
+        self._pending_diff_image_label: str = ""
+        self._turn_diff_image: dict[str, Any] | None = None
+        self._turn_diff_image_label: str = ""
+        self._pending_gameover_images: list[tuple[str, dict[str, Any]]] = []
+        self._noop_guard_pre_hash: str = ""
+        # (interior board hash, action signature) already run in THIS snippet.
+        # Written in _record_guard_observations, read in _action_guard_hook,
+        # cleared before each sandbox call - three sites, one lifetime.
+        self._summary_context_tokens_since_attempt = 0
+        self._http_initial_grace_used = False
+        self._repeat_state_seen: set[tuple[str, str]] = set()
+        # (state_hash, action_sig) for each action the guard hook waved through,
+        # in execution order - the only place the intermediate board states of a
+        # batch are visible to the agent, since the solver reports outcomes per
+        # position but not the boards they ran against.
+        self._guard_hook_trace: list[tuple[str, str]] = []
+        self._rejected_section_updates: list[str] = []
+        # characters per token, measured from the last response; None until one
+        # arrives, and reported in the transcript whenever it changes
+        self._text_chars_per_token: float | None = None
+        # index into the reasoning-effort ladder; -1 means "server default"
+        self._reasoning_effort_rung: int = -1
+        self._guard_death_override_batch: bool = False
+        self._noop_guard_deaths_enabled: bool = _get_env_bool(
+            "ARC3_DEATH_REPEAT_GUARD", False
+        )
+        self._noop_repeat_guard: NoopRepeatGuard | None = (
+            NoopRepeatGuard()
+            if (
+                _get_env_bool("ARC3_NOOP_REPEAT_GUARD", False)
+                or self._noop_guard_deaths_enabled
+            )
+            else None
+        )
+        # only the no-op half is gated separately; the shared store is created
+        # when either half is enabled
+        self._noop_guard_noops_enabled: bool = _get_env_bool(
+            "ARC3_NOOP_REPEAT_GUARD", False
+        )
+        self._resume_after_yield: bool = False
+        self._resume_reason: str = ""
+        self._prev_level_start_frame: Frame | None = None
+        self._cur_level_start_frame: Frame | None = None
+        self._death_ledger_level: Any = None
+        self._death_ledger_attempts: list[dict[str, Any]] = []
+        self._death_ledger_index: dict[str, list[tuple[int, int]]] = {}
+        self._death_ledger_total: int = 0
         self._history_messages: list[dict[str, Any]] = []
         self._session_runtime_dir: Path | None = None
         self._session_total_tokens = 0
@@ -953,7 +3583,7 @@ class ToolAgent:
         self._step_env_callback: Callable[[dict[str, Any]], dict[str, Any]] | None = None
         self._current_valid_actions: list[str] = []
         self._last_step_summary: dict[str, Any] | None = None
-        self._last_action_result: dict[str, Any] | None = None
+        self._last_action_call_result: dict[str, Any] | None = None
         self._summarized_knowledge = _empty_world_model()
 
     def _headers(self) -> dict[str, str]:
@@ -976,12 +3606,20 @@ class ToolAgent:
         runtime_dir = state_path.parent
         if self._session_runtime_dir != runtime_dir:
             self._session_runtime_dir = runtime_dir
+            self._priority_level_count_warned = False
+            if _persistent_functions():
+                self._kept_functions = {}
+                self._retained_clear_notice = ""
             self._history_messages = []
             self._session_total_tokens = 0
             self._session_generated_tokens = 0
+            self._summary_context_tokens_since_attempt = 0
             self._last_step_summary = None
-            self._last_action_result = None
+            self._last_action_call_result = None
             self._summarized_knowledge = _empty_world_model()
+            self._turns_without_wm_update = 0
+            self._pending_wm_rebuild_reason = ""
+            self._pending_wm_revise_reason = ""
 
     @property
     def total_tokens(self) -> int:
@@ -991,7 +3629,19 @@ class ToolAgent:
     def generated_tokens(self) -> int:
         return max(0, int(self._session_generated_tokens))
 
-    def _accumulate_usage_tokens(self, usage: dict[str, Any] | None) -> None:
+    def _accumulate_usage_tokens(
+        self, usage: dict[str, Any] | None, *, count_toward_turn: bool = True
+    ) -> None:
+        """Add a response's usage to the counters.
+
+        count_toward_turn=False for generation that is not part of the turn's
+        deliberation - the rolling summary. Its tokens are real GPU time and
+        belong in the session and per-game totals, or a summary-heavy run looks
+        cheaper than it is and gets more compute than
+        max_generated_tokens_per_game intends. But the per-turn yield budget
+        measures how long the model has been working on THIS turn, and charging
+        it for a summary would shorten whichever turn happened to follow one.
+        """
         if not isinstance(usage, dict):
             return
         generated_token_count = 0
@@ -1003,6 +3653,8 @@ class ToolAgent:
             except (TypeError, ValueError):
                 continue
         self._session_generated_tokens += generated_token_count
+        if count_toward_turn:
+            self._turn_generated_tokens += generated_token_count
 
         total_tokens = usage.get("total_tokens")
         try:
@@ -1021,6 +3673,41 @@ class ToolAgent:
                 continue
         self._session_total_tokens += token_count
 
+    @staticmethod
+    def _turn_action_trace(executed_results: list[dict[str, Any]]) -> list[str]:
+        """Stitch per-call action traces into one sequentially numbered
+        turn-level trace; synthesize entries for calls without a trace."""
+        entries: list[str] = []
+        for item in executed_results:
+            item_trace = item.get("action_trace")
+            if isinstance(item_trace, list) and item_trace:
+                entries.extend(
+                    re.sub(r"^\d+:\s*", "", str(line)) for line in item_trace
+                )
+                continue
+            names = item.get("executed_actions")
+            if not isinstance(names, list) or not names:
+                display = str(item.get("action_display") or "").strip()
+                names = [display] if display else []
+            if item.get("game_over"):
+                verdict = "GAME OVER (attempt ended immediately after this action)"
+            elif item.get("run_complete") or item.get("done"):
+                verdict = "RUN COMPLETE"
+            elif item.get("level_completed"):
+                verdict = "LEVEL COMPLETED"
+            elif item.get("gameplay_changed") is False:
+                verdict = (
+                    "NO-OP (executed, no change inside the board area)"
+                    if _new_noop_wording()
+                    else "NO-OP (executed, changed nothing)"
+                )
+            elif item.get("gameplay_changed") is True:
+                verdict = "effect"
+            else:
+                verdict = "executed"
+            entries.extend(f"{name} -> {verdict}" for name in names)
+        return [f"{position}: {entry}" for position, entry in enumerate(entries, 1)]
+
     def _summarize_step_sequence(self, action_results: list[dict[str, Any]]) -> dict[str, Any] | None:
         if not action_results:
             return None
@@ -1063,58 +3750,161 @@ class ToolAgent:
             "level_transition": any(bool(item.get("level_completed")) for item in executed_results),
             "run_complete": any(bool(item.get("run_complete")) for item in executed_results),
             "game_over": any(bool(item.get("game_over")) for item in executed_results),
+            "game_over_trace": (
+                ToolAgent._turn_action_trace(executed_results)
+                if any(item.get("game_over") for item in executed_results)
+                else []
+            ),
+            "fatal_action": next(
+                (
+                    str(item.get("fatal_action") or item.get("action_display") or "")
+                    for item in executed_results
+                    if item.get("game_over")
+                ),
+                "",
+            ),
             "board_changed": any(bool(item.get("board_changed")) for item in executed_results),
+            # the interior-only flag, so the opener can say WHICH kind of change
+            # happened instead of hedging over the broader one
+            "gameplay_changed": any(
+                bool(item.get("gameplay_changed")) for item in executed_results
+            ),
             "stop_reason": last.get("stop_reason"),
+            "no_op_stops": [
+                {
+                    "action": item.get("no_op_action"),
+                    "index": item.get("no_op_action_index"),
+                    "skipped": item.get("skipped_actions") or [],
+                    "trace": item.get("action_trace") or [],
+                }
+                for item in executed_results
+                if item.get("stop_reason") == "no_op_action"
+            ],
+            # the animation of the last executed action, which is what
+            # last_animation_frames / last_animation_timeline will return when asked
+            "animation": (executed_results[-1].get("animation") if executed_results else None),
+            "animation_action": (
+                executed_results[-1].get("action_display") if executed_results else None
+            ),
+            "guard_stops": [
+                {
+                    "kind": item.get("stop_reason"),
+                    "action": item.get("blocked_action"),
+                    "detail": item.get("stop_detail"),
+                    "skipped": item.get("skipped_actions") or [],
+                    "trace": item.get("action_trace") or [],
+                }
+                # scans ALL results, not just executed ones: a refusal has
+                # executed=False by construction and is filtered out of
+                # executed_results before this point
+                for item in action_results
+                if item.get("stop_reason") in ("known_noop", "known_death", "stale_state")
+            ],
         }
 
-    def _describe_last_outcome(self, summary: dict[str, Any] | None) -> str:
-        if not summary:
-            return ""
-        span = _format_action_span(
-            summary.get("start_action_num"),
-            summary.get("end_action_num"),
-        )
-        count = summary.get("executed_count")
-        prefix = "Last executed sequence"
-        if span and count:
-            prefix = f"Actions {span} ({count} total)"
-        elif span:
-            prefix = f"Action span {span}"
-        elif count:
-            prefix = f"Last executed sequence ({count} total)"
+    # _describe_last_outcome removed. It built an outcome sentence and was
+    # called from nowhere - in this harness AND upstream - so neither its
+    # wording nor the fallback it carried ever reached a transcript. Two
+    # patches were spent revising text no model ever read. What it uniquely
+    # provided is now rendered inline in _build_user_prompt.
 
-        level = summary.get("level")
-        if summary.get("level_transition"):
-            level_text = f" to level {level}" if level is not None else ""
-            return f"{prefix} triggered a level transition{level_text}; re-ground on the new scene."
-        if summary.get("run_complete"):
-            return f"{prefix} completed the run."
-        if summary.get("game_over"):
-            return f"{prefix} reached GAME_OVER."
-
-        pieces = [prefix]
-        if summary.get("board_changed"):
-            pieces.append("produced a board change; verify that it affected gameplay objects rather than only HUD elements.")
-        else:
-            pieces.append("did not show a confirmed board change; treat this as weak evidence until verified.")
-        stop_reason = _normalize_summary_text(summary.get("stop_reason"))
-        if stop_reason:
-            pieces.append(f"stop_reason={stop_reason}.")
-        return " ".join(pieces)
-
-    def _update_summarized_knowledge_from_assistant(self, content: str) -> None:
-        note = _extract_scientist_note(content)
+    def _update_summarized_knowledge_from_assistant(self, content: str) -> list[str]:
+        if _memory_sections_disabled():
+            # nothing is carried or shown, so parsing would only build state
+            # that never reaches the model
+            return []
+        note = _extract_scientist_note(content, self._summarized_knowledge)
         if not note:
-            return
+            return []
+        updated: list[str] = []
+        rejected: list[str] = []
         for key, value in note.items():
-            if value:
-                self._summarized_knowledge[key] = value
+            if not value:
+                continue
+            if _is_degenerate_text(value):
+                # keep the previous version rather than truncating: a truncated
+                # degenerate section is still degenerate, and it would be
+                # re-injected into every later prompt
+                rejected.append(key)
+                continue
+            self._summarized_knowledge[key] = value
+            updated.append(key)
+        if rejected:
+            self._rejected_section_updates = sorted(set(rejected))
+            log.warning(
+                "rejected degenerate memory section update(s): %s",
+                ", ".join(self._rejected_section_updates),
+            )
+        if updated:
+            self._turns_without_wm_update = 0
+            self._wm_absorbed_in_turn = True
+            if "world_model" in updated:
+                self._pending_wm_rebuild_reason = ""
+            if "world_model" in updated or "current_plan" in updated:
+                self._pending_wm_revise_reason = ""
+        return updated
 
     def _update_summarized_knowledge_from_step_summary(self) -> None:
         summary = self._last_step_summary
         if not summary:
             return
-        if summary.get("level_transition") or summary.get("run_complete") or summary.get("game_over"):
+        wipe_reason = ""
+        revise_reason = ""
+        if summary.get("level_transition"):
+            # the priority estimate is per level, so both counts restart here
+            if _get_env_bool("ARC3_PRIORITY_PACE", False):
+                completed_level = max(1, int(summary.get("level") or 2) - 1)
+                if completed_level > self._pace_last_completed_level:
+                    self._progress_pace.record_completion(
+                        self._session_generated_tokens - self._tokens_at_level_start,
+                        _pace_reference_tokens(completed_level),
+                    )
+                    self._pace_last_completed_level = completed_level
+            self._actions_at_level_start = _priority_action_count(summary)
+            self._tokens_at_level_start = self._session_generated_tokens
+        if summary.get("level_transition") and _persistent_functions_scope() == "levelup":
+            # a helper can encode the old layout in its code even when it holds
+            # no stale data, so the level boundary is where it stops being safe
+            self._kept_functions = {}
+            if _persistent_functions():
+                self._retained_clear_notice = (
+                    "Your retained functions were cleared at this level transition. "
+                    "Redefine any helpers you still need after checking their assumptions "
+                    "against the new board."
+                )
+        elif (
+            summary.get("level_transition")
+            and _persistent_functions_scope() == "game"
+            and _persistent_functions()
+            and getattr(self, "_kept_functions", None)
+        ):
+            self._retained_clear_notice = (
+                "Your retained functions remain available from previous levels. "
+                "Check level-specific assumptions against the new board before reusing them."
+            )
+        if summary.get("level_transition"):
+            # read at call time (not a module constant) so this patch's hunks
+            # stay clear of the env-knob block that other patches extend
+            if _get_env_int("ARC3_WM_WIPE_ON_LEVEL", 1):
+                wipe_reason = "a level transition"
+            else:
+                revise_reason = "level"
+        elif summary.get("run_complete"):
+            wipe_reason = "run completion"
+        elif summary.get("game_over"):
+            if _WM_WIPE_ON_GAME_OVER:
+                wipe_reason = "a game over"
+            else:
+                revise_reason = "death"
+        if revise_reason:
+            self._pending_wm_revise_reason = revise_reason
+            self._wm_revise_hint_shown = False
+            # a newer keep-event supersedes any stale rebuild nag (and vice
+            # versa below): at most one pending nudge, always for the latest event
+            self._pending_wm_rebuild_reason = ""
+        if wipe_reason:
+            self._pending_wm_rebuild_reason = wipe_reason
+            self._pending_wm_revise_reason = ""
             for key in (
                 "world_model",
                 "goal_model",
@@ -1135,7 +3925,24 @@ class ToolAgent:
             ("Plan", self._summarized_knowledge.get("current_plan", "")),
             ("Cross-level notes", self._summarized_knowledge.get("cross_level_notes", "")),
         ]
+        if _memory_sections_disabled():
+            return []
+        rejected = list(getattr(self, "_rejected_section_updates", []))
+        self._rejected_section_updates = []
+        allowed = _memory_section_labels()
+        if allowed is not None:
+            entries = [(label, value) for label, value in entries if label in allowed]
         lines = [f"- {label}: {value}" for label, value in entries if value]
+        if rejected:
+            # without this the model sees its section unchanged and is likely to
+            # emit the same degenerate text again
+            lines.append(
+                "- NOTE: your last update to "
+                + ", ".join(f"`{name.replace('_', ' ')}`" for name in rejected)
+                + " was REJECTED as degenerate (highly repetitive or corrupted "
+                "text) and the previous version above was kept. Rewrite it "
+                "concisely in plain prose."
+            )
         if not lines:
             return []
         return [
@@ -1144,19 +3951,857 @@ class ToolAgent:
             "- Revise any item above immediately if `current_frame` or `history` contradicts it.",
         ]
 
+    def _prepare_auto_diff(
+        self,
+        current_frame: Frame | None,
+        previous_step_summary: dict[str, Any] | None,
+    ) -> list[str]:
+        """Track the frame shown at the previous analyzer turn and, when
+        enabled, produce the auto-injected frame-diff lines and/or the diff
+        image for the outgoing user message. Suppressed on the first turn and
+        on the turn after a game over (the auto-RESET makes that span
+        meaningless)."""
+        if not (_AUTO_FRAME_DIFF or _DIFF_IMAGE or _level_inventory_enabled()):
+            return []
+        # Imported lazily so this patch's hunks stay clear of the module
+        # import header, which other optional patches also modify.
+        from inference.agent.vision_context import diff_image_part
+        from inference.utils.frame_diff import compute_frame_diff
+        from inference.utils.grid_utils import ARC_COLOR_CHARS
+        from inference.utils.segmentation import segment_layer
+        self._pending_diff_image = None
+        self._pending_diff_image_label = ""
+        # a new opener supersedes the previous turn's diff whether or not it
+        # produces one of its own
+        self._turn_diff_image = None
+        self._turn_diff_image_label = ""
+        if current_frame is None:
+            return []
+        previous_frame = self._auto_diff_prev_frame
+        self._auto_diff_prev_frame = current_frame
+        if previous_frame is None:
+            self._cur_level_start_frame = current_frame
+            return []
+        prev_level = getattr(previous_frame, "level", None)
+        cur_level = getattr(current_frame, "level", None)
+        level_changed = (
+            prev_level is not None
+            and cur_level is not None
+            and prev_level != cur_level
+        )
+        if level_changed:
+            # shift the per-level start frames even when a game over follows:
+            # the current frame is this level's (near-)initial board
+            self._prev_level_start_frame = self._cur_level_start_frame
+            self._cur_level_start_frame = current_frame
+        if previous_step_summary and previous_step_summary.get("game_over"):
+            return []
+        if level_changed:
+            if not _level_inventory_enabled():
+                # still clear the diff image: a cross-level diff has no meaning
+                self._pending_diff_image = None
+                self._pending_diff_image_label = ""
+                return []
+            return self._level_transition_diff_lines(
+                current_frame, previous_step_summary
+            )
+        if not (_AUTO_FRAME_DIFF or _DIFF_IMAGE):
+            # reached only when the inventory knob alone kept us here
+            return []
+        prev_step = getattr(previous_frame, "step", None)
+        cur_step = getattr(current_frame, "step", None)
+        span = f"step {prev_step} -> {cur_step}"
+        if _DIFF_IMAGE:
+            part = diff_image_part(previous_frame, current_frame)
+            if part is not None and previous_frame.grid != current_frame.grid:
+                label = (
+                    f"Diff image ({span}): cells that changed since your previous "
+                    "analyzer turn in their new true color; unchanged cells dimmed "
+                    "to dark navy (an off-palette color)."
+                )
+                self._pending_diff_image = part
+                self._pending_diff_image_label = label
+                # kept beyond the single use above so a resumption can re-show
+                # it; replaced whenever a new opener produces a fresh diff, so
+                # a stale one from an earlier turn can never leak in
+                self._turn_diff_image = part
+                self._turn_diff_image_label = label
+        if not _AUTO_FRAME_DIFF:
+            return []
+        executed = 0
+        if previous_step_summary and not previous_step_summary.get("stale"):
+            # a stale summary describes an earlier exchange whose changes were
+            # already diffed then; nothing has executed since, so don't claim
+            # "no cell changed despite N executed actions"
+            try:
+                executed = int(previous_step_summary.get("executed_count") or 0)
+            except (TypeError, ValueError):
+                executed = 0
+        if previous_frame.grid == current_frame.grid:
+            if executed > 0:
+                return [
+                    f"Auto frame diff ({span}): no cell changed since your previous "
+                    f"analyzer turn despite {executed} executed action(s)."
+                ]
+            return []
+        prev_rows = len(previous_frame.grid)
+        cur_rows = len(current_frame.grid)
+        prev_cols = max((len(r) for r in previous_frame.grid), default=0)
+        cur_cols = max((len(r) for r in current_frame.grid), default=0)
+        if (prev_rows, prev_cols) != (cur_rows, cur_cols):
+            return [
+                f"Auto frame diff ({span}): grid shape changed "
+                f"({prev_rows}x{prev_cols} -> {cur_rows}x{cur_cols}); diff skipped."
+            ]
+        diff = compute_frame_diff(
+            previous_frame.grid,
+            current_frame.grid,
+            segment_layer(previous_frame.grid, ARC_COLOR_CHARS).get("nodes", []),
+            segment_layer(current_frame.grid, ARC_COLOR_CHARS).get("nodes", []),
+            max_group_match=_AUTO_FRAME_DIFF_MAX_GROUP,
+        )
+        rendered = _render_auto_frame_diff(
+            diff, _AUTO_FRAME_DIFF_BUDGET, total_cells=cur_rows * cur_cols
+        )
+        lines = [
+            f"Auto frame diff ({span}; covers everything since your previous "
+            "analyzer turn, not just the last action):"
+        ]
+        lines.extend(rendered)
+        return lines
+
+    def _death_ledger_record(self, history_entries: list["HistoryEntry"]) -> None:
+        entries = list(history_entries or [])
+        if len(entries) < 2 or str(entries[-1].action or "").upper() != "RESET":
+            return
+        death_idx = len(entries) - 2
+        death = entries[death_idx]
+        if death.frame is None or str(death.action or "").upper() == "RESET":
+            return
+        death_level = getattr(death.frame, "level", None)
+        if (
+            self._death_ledger_level is not None
+            and death_level is not None
+            and death_level != self._death_ledger_level
+        ):
+            self._death_ledger_attempts = []
+            self._death_ledger_index = {}
+        if death_level is not None:
+            self._death_ledger_level = death_level
+        start_idx = None
+        for idx in range(death_idx - 1, -1, -1):
+            entry = entries[idx]
+            if str(entry.action or "").upper() == "RESET":
+                start_idx = idx
+                break
+            level = (
+                getattr(entry.frame, "level", None)
+                if entry.frame is not None
+                else None
+            )
+            if (
+                death_level is not None
+                and level is not None
+                and level != death_level
+            ):
+                start_idx = idx + 1
+                break
+        partial = start_idx is None
+        if partial:
+            start_idx = 0
+        actions = [
+            str(entries[i].action) for i in range(start_idx + 1, death_idx + 1)
+        ]
+        frames = [entries[i].frame for i in range(start_idx, death_idx)]
+        trim = 0
+        for i in range(len(frames) - 1, -1, -1):
+            if frames[i] is None:
+                trim = i + 1
+                break
+        if trim:
+            frames = frames[trim:]
+            actions = actions[trim:]
+            partial = True
+        if not actions or len(frames) != len(actions):
+            return
+        for item in self._death_ledger_attempts:
+            if item["actions"] == actions:
+                self._death_ledger_total += 1
+                item["ids"].append(self._death_ledger_total)
+                return
+        self._death_ledger_total += 1
+        uidx = len(self._death_ledger_attempts)
+        hashes = [_interior_state_hash(f.grid) for f in frames]
+        self._death_ledger_attempts.append(
+            {
+                "actions": actions,
+                "hashes": hashes,
+                "ids": [self._death_ledger_total],
+                "partial": partial,
+            }
+        )
+        for pos, state_hash in enumerate(hashes):
+            self._death_ledger_index.setdefault(state_hash, []).append((uidx, pos))
+
+    def _death_ledger_payload(self) -> dict[str, Any]:
+        return {
+            "level": self._death_ledger_level,
+            "attempts": [
+                {
+                    "actions": list(item["actions"]),
+                    "state_hashes": list(item["hashes"]),
+                    "attempt_ids": list(item["ids"]),
+                    "partial": bool(item.get("partial")),
+                }
+                for item in self._death_ledger_attempts
+            ],
+        }
+
+    def _death_ledger_observe(
+        self,
+        history_entries: list["HistoryEntry"],
+        previous_step_summary: dict[str, Any] | None,
+    ) -> None:
+        """Record a death, if the previous sequence ended in one.
+
+        Called unconditionally rather than from the rendering path. The ledger
+        has TWO consumers - the advisory lines under ARC3_DEATH_LEDGER, and the
+        death guard's fatal lookup under ARC3_DEATH_REPEAT_GUARD - and the
+        recorder used to live inside the first. So enabling only the guard left
+        it consulting an index nothing ever wrote to: armed, checked, always
+        empty, every repeat of a fatal route allowed through. Observed as three
+        identical deaths in a row with the guard on.
+        """
+        if not (
+            previous_step_summary
+            and previous_step_summary.get("game_over")
+            and not previous_step_summary.get("stale")
+        ):
+            return
+        if not (
+            _get_env_bool("ARC3_DEATH_LEDGER", False)
+            or _get_env_bool("ARC3_DEATH_REPEAT_GUARD", False)
+        ):
+            # nothing reads it; skip the work rather than accumulate state
+            return
+        self._death_ledger_record(history_entries)
+
+    def _death_ledger_lines(
+        self,
+        current_frame: Frame | None,
+        history_entries: list["HistoryEntry"],
+        previous_step_summary: dict[str, Any] | None,
+    ) -> list[str]:
+        """Advisory display of recorded fatal continuations matching the
+        CURRENT interior state. Broad by design (bar/items may not be visible
+        in the interior), hence advisory phrasing; exact-sequence enforcement
+        is a separate concern."""
+        if current_frame is None:
+            return []
+        level = getattr(current_frame, "level", None)
+        if (
+            self._death_ledger_level is not None
+            and level is not None
+            and level != self._death_ledger_level
+        ):
+            self._death_ledger_attempts = []
+            self._death_ledger_index = {}
+            self._death_ledger_level = level
+        if not self._death_ledger_attempts:
+            return []
+        state_hash = _interior_state_hash(current_frame.grid)
+        matches: dict[int, int] = {}
+        for uidx, pos in self._death_ledger_index.get(state_hash, []):
+            matches[uidx] = max(matches.get(uidx, -1), pos)
+        if not matches:
+            return []
+        by_suffix: dict[tuple, list[int]] = {}
+        for uidx, pos in matches.items():
+            item = self._death_ledger_attempts[uidx]
+            suffix = tuple(item["actions"][pos:])
+            by_suffix.setdefault(suffix, []).extend(item["ids"])
+        ordered = sorted(by_suffix.items(), key=lambda kv: len(kv[0]))
+        total_deaths = sum(
+            len(item["ids"]) for item in self._death_ledger_attempts
+        )
+        lines = [
+            "Known fatal continuations FROM THE CURRENT BOARD STATE (this "
+            f"level has {total_deaths} failed attempt(s) recorded; the game "
+            "is deterministic):"
+        ]
+        for suffix, ids in ordered[:3]:
+            shown = ", ".join(suffix[:12]) + (", ..." if len(suffix) > 12 else "")
+            id_text = ", ".join(str(i) for i in sorted(set(ids)))
+            lines.append(
+                f"- {len(suffix)} action(s): {shown} -> GAME OVER "
+                f"(attempt(s) {id_text})"
+            )
+        if len(ordered) > 3:
+            lines.append(
+                f"- +{len(ordered) - 3} more matching fatal continuation(s)"
+            )
+        lines.append(
+            "Do NOT replay a listed continuation exactly - vary the route or "
+            "action at or before its first step. (Advisory: fatality can "
+            "depend on state not visible in the board interior, e.g. bar "
+            "level or held items; if you are certain the situation differs, "
+            "state why before proceeding.) The full ledger is available in "
+            "python as `death_ledger`."
+        )
+        return lines
+
+    def _level_transition_diff_lines(
+        self,
+        current_frame: Frame | None,
+        previous_step_summary: dict[str, Any] | None,
+    ) -> list[str]:
+        """First prompt of a new level: acknowledge the completing action and
+        render a START-vs-START inventory against the previous level instead
+        of the meaningless end-of-old-board vs new-board diff. The diff image
+        is suppressed for this prompt (no cross-level semantics)."""
+        from inference.utils.frame_diff import compute_frame_diff
+        from inference.utils.grid_utils import ARC_COLOR_CHARS
+        from inference.utils.segmentation import segment_layer
+
+        self._pending_diff_image = None
+        self._pending_diff_image_label = ""
+        lines: list[str] = []
+        final_action = ""
+        if previous_step_summary and not previous_step_summary.get("stale"):
+            actions = previous_step_summary.get("executed_actions")
+            if isinstance(actions, list) and actions:
+                final_action = str(actions[-1]).strip()
+        lines.append(
+            "Level completed - the board was replaced, so no object diff is "
+            "shown across the boundary"
+            + (f" (the final action {final_action!r} finished the level)" if final_action else "")
+            + "."
+        )
+        prev_start = self._prev_level_start_frame
+        cur_start = self._cur_level_start_frame
+        if prev_start is None or cur_start is None:
+            return lines
+        grid_a, grid_b = prev_start.grid, cur_start.grid
+        rows = len(grid_b)
+        cols = max((len(r) for r in grid_b), default=0)
+        if len(grid_a) != rows or max(
+            (len(r) for r in grid_a), default=0
+        ) != cols:
+            lines.append(
+                "New level inventory skipped (boards have different shapes)."
+            )
+            return lines
+        diff = compute_frame_diff(
+            grid_a,
+            grid_b,
+            segment_layer(grid_a, ARC_COLOR_CHARS).get("nodes", []),
+            segment_layer(grid_b, ARC_COLOR_CHARS).get("nodes", []),
+            max_group_match=_AUTO_FRAME_DIFF_MAX_GROUP,
+        )
+        lines.append(
+            "New level inventory (previous level START vs this level START; "
+            "read it as a TYPE comparison, not motion: 'moved' = object types "
+            "recurring from the previous level at a new spot, 'appeared' = "
+            "types NOT present on the previous level - likely new mechanics, "
+            "investigate these first, 'disappeared' = previous types absent "
+            "here; identical objects at identical positions are not listed):"
+        )
+        lines.extend(
+            _render_auto_frame_diff(
+                diff, _AUTO_FRAME_DIFF_BUDGET, total_cells=rows * cols
+            )
+        )
+        return lines
+
+    def _prepare_gameover_images(
+        self,
+        history_entries: list["HistoryEntry"],
+        previous_step_summary: dict[str, Any] | None = None,
+    ) -> None:
+        """Queue the death-turn visuals (ARC3_GAMEOVER_DIFF_IMAGE).
+
+        The auto-diff - and with it the diff image - is suppressed after a game
+        over because the auto-RESET makes that span meaningless, so the visual
+        slot is idle on exactly the turn where the death is invisible: the
+        board the model sees has already been reset. Two parts fill it:
+
+          1. the death frame itself (the budget bar in it is what the BAR RULE
+             asks about, and it exists nowhere else in the message), and
+          2. a change-highlight diff of the killing action (last alive frame ->
+             death frame), the spatial form of the textual 'Fatal step diff'.
+
+        Anchoring matches _game_over_diff_lines: history ends with the RESET
+        marker, entries[-2] is the death frame, entries[-3] the last alive one.
+        """
+        self._pending_gameover_images = []
+        if not _get_env_bool("ARC3_GAMEOVER_DIFF_IMAGE", False):
+            return
+        from inference.agent.vision_context import (
+            current_grid_image_part,
+            diff_image_part,
+        )
+
+        entries = list(history_entries or [])
+        if len(entries) < 3 or str(entries[-1].action or "").upper() != "RESET":
+            return
+        death = entries[-2]
+        last_alive = entries[-3]
+        if death.frame is None or str(death.action or "").upper() == "RESET":
+            return
+        death_part = current_grid_image_part(death.frame)
+        if death_part is not None:
+            self._pending_gameover_images.append(
+                (
+                    "Death frame image (the board at the moment the attempt "
+                    "ended, before the automatic reset): inspect the budget bar "
+                    "here for the BAR RULE.",
+                    death_part,
+                )
+            )
+        self._append_fatal_animation_image(previous_step_summary)
+        if last_alive.frame is None:
+            return
+        fatal_part = diff_image_part(last_alive.frame, death.frame)
+        if fatal_part is not None and last_alive.frame.grid != death.frame.grid:
+            self._pending_gameover_images.append(
+                (
+                    "Fatal step diff image (last alive frame -> death frame): "
+                    "cells changed by the killing action in their new true "
+                    "color; unchanged cells dimmed to dark navy (an off-palette "
+                    "color).",
+                    fatal_part,
+                )
+            )
+
+    def _game_over_diff_lines(
+        self,
+        history_entries: list["HistoryEntry"],
+        previous_step_summary: dict[str, Any] | None = None,
+    ) -> list[str]:
+        """After a game over (fresh rendering only), push two diffs over the
+        fatal sequence itself: sequence progress (frame before the sequence's
+        first action -> last alive frame, omitted when fewer than two actions
+        executed) and the fatal step (last alive -> death frame). Anchored by
+        the summary's executed_count - the turns before the fatal sequence
+        already delivered their own diffs while the model was alive."""
+        from inference.utils.frame_diff import compute_frame_diff
+        from inference.utils.grid_utils import ARC_COLOR_CHARS
+        from inference.utils.segmentation import segment_layer
+
+        entries = list(history_entries or [])
+        if len(entries) < 2 or str(entries[-1].action or "").upper() != "RESET":
+            return []
+        death_idx = len(entries) - 2
+        death = entries[death_idx]
+        if death.frame is None or str(death.action or "").upper() == "RESET":
+            return []
+        executed = 0
+        if previous_step_summary:
+            try:
+                executed = int(previous_step_summary.get("executed_count") or 0)
+            except (TypeError, ValueError):
+                executed = 0
+        if executed < 1:
+            executed = 1
+        start_idx = death_idx - executed
+        partial = start_idx < 0
+        if partial:
+            start_idx = 0
+        origin = "earliest retained frame" if partial else "sequence start"
+        death_level = getattr(death.frame, "level", None)
+        while start_idx < death_idx - 1:
+            frame = entries[start_idx].frame
+            level = getattr(frame, "level", None) if frame is not None else None
+            if (
+                frame is None
+                or (
+                    death_level is not None
+                    and level is not None
+                    and level != death_level
+                )
+            ):
+                # the sequence itself crossed a level-up (or a frame is
+                # missing); anchor at the first same-level frame within it
+                start_idx += 1
+                origin = "level start within the sequence"
+                continue
+            break
+        start = entries[start_idx]
+        last_alive = entries[death_idx - 1]
+        if start.frame is None or last_alive.frame is None:
+            return []
+
+        def _diff_lines(frame_a, frame_b, label: str) -> list[str]:
+            grid_a, grid_b = frame_a.grid, frame_b.grid
+            rows = len(grid_b)
+            cols = max((len(r) for r in grid_b), default=0)
+            if len(grid_a) != rows or max(
+                (len(r) for r in grid_a), default=0
+            ) != cols:
+                return [f"{label} skipped (grid shape changed)."]
+            diff = compute_frame_diff(
+                grid_a,
+                grid_b,
+                segment_layer(grid_a, ARC_COLOR_CHARS).get("nodes", []),
+                segment_layer(grid_b, ARC_COLOR_CHARS).get("nodes", []),
+                max_group_match=_AUTO_FRAME_DIFF_MAX_GROUP,
+            )
+            rendered = _render_auto_frame_diff(
+                diff, _AUTO_FRAME_DIFF_BUDGET, total_cells=rows * cols
+            )
+            return [label, *rendered]
+
+        out: list[str] = []
+        pre_death_actions = (death_idx - 1) - start_idx
+        if pre_death_actions >= 1:
+            out.extend(
+                _diff_lines(
+                    start.frame,
+                    last_alive.frame,
+                    f"Sequence progress diff ({origin} step "
+                    f"{getattr(start.frame, 'step', '?')} -> step "
+                    f"{getattr(last_alive.frame, 'step', '?')}, what the first "
+                    f"{pre_death_actions} action(s) of the fatal sequence "
+                    "changed - the reset reverted all of it):",
+                )
+            )
+        out.extend(
+            _diff_lines(
+                last_alive.frame,
+                death.frame,
+                f"Fatal step diff (step {getattr(last_alive.frame, 'step', '?')} "
+                f"-> step {getattr(death.frame, 'step', '?')}, what the killing "
+                f"action {str(death.action)!r} changed):",
+            )
+        )
+        return out
+
+    def _append_fatal_animation_image(self, summary: dict[str, Any] | None) -> None:
+        """An image of what the fatal action's animation hid, on a game over.
+
+        Gated on transient pixels alone, NOT on gameplay_changed: after a death
+        the board has already reset, so gameplay_changed compares the
+        pre-action board against the restarted level and is essentially always
+        true. The justification differs from the ordinary case too - there the
+        board shows nothing, here it shows a different board entirely, and
+        either way the fatal moment is unreachable.
+        """
+        if not _animation_enabled():
+            return
+        mode = animation_death_image_mode()
+        if mode == "off" or not summary:
+            return
+        animation = summary.get("animation") or {}
+        if not int(animation.get("transient_pixels") or 0):
+            return
+        chain = self._last_animation_chain or []
+        cells = _animation_transient_cells(chain)
+        if not cells:
+            return
+        if mode == "composite":
+            part = animation_composite_part(
+                chain, cells, border=_get_env_int("ARC3_NOOP_GUARD_BORDER", 4)
+            )
+            label = (
+                "Fatal action animation image: the cells that changed during the fatal "
+                "action's animation, each in the last colour it held before reverting; "
+                "everything else, including the border, is dimmed to dark navy (an "
+                "off-palette colour). This is a composite of all cells that changed "
+                "during the animation, not a single board state that the game "
+                "displayed. A cell is drawn nearer its true colour the longer it held "
+                "it, so a bright trail marks where something stayed and a uniformly "
+                "dim one marks something that passed through. Neither the death frame "
+                "nor the last alive frame contains these cells - they existed only "
+                "between the two."
+            )
+        else:
+            part, index = animation_peak_part(chain, cells)
+            label = (
+                f"Fatal action animation image (frame {index} of {len(chain) - 1}): the "
+                "board partway through the fatal action's animation, at the point where "
+                "the most cells were showing something they later reverted. This is a "
+                "real state the game displayed, between the last alive frame and the "
+                "death frame."
+            )
+        if part is not None:
+            self._pending_gameover_images.append((label, part))
+
+    def _prepare_animation_image(self, summary: dict[str, Any] | None) -> None:
+        """Attach an image of what the animation hid, when the board did not move.
+
+        Gated on gameplay_changed being false: that is exactly when the board
+        image the model receives is uninformative, because the action did
+        something and no frame it can reach shows what. On any other turn the
+        current board already carries the outcome.
+        """
+        if not _animation_enabled():
+            # ARC3_ANIMATION_IMAGE needs ARC3_ANIMATION: without it no chain is
+            # captured, so this would silently do nothing. Stated rather than
+            # left implicit - a knob that quietly has no effect is the failure
+            # mode that costs the most time to diagnose.
+            return
+        mode = animation_image_mode()
+        if mode == "off" or not summary:
+            return
+        if self._last_action_gameplay_changed is not False:
+            return
+        animation = summary.get("animation") or {}
+        if not int(animation.get("transient_pixels") or 0):
+            return
+        chain = self._last_animation_chain or []
+        cells = _animation_transient_cells(chain)
+        if not cells:
+            return
+        if mode == "composite":
+            part = animation_composite_part(
+                chain, cells, border=_get_env_int("ARC3_NOOP_GUARD_BORDER", 4)
+            )
+            label = (
+                "Animation image: the cells that changed during the animation, each in "
+                "the last colour it held before reverting; everything else, including "
+                "the border, is dimmed to dark navy (an off-palette colour). This is a "
+                "composite of all cells that changed during the animation, not a single "
+                "board state that the game displayed. A cell is drawn nearer its true "
+                "colour the longer it held it, so a bright trail marks where something "
+                "stayed and a uniformly dim one marks something that passed through. "
+                "The action left the board area unchanged, so none of these cells "
+                "appear in `current_frame`."
+            )
+        else:
+            part, index = animation_peak_part(chain, cells)
+            label = (
+                f"Animation image (frame {index} of {len(chain) - 1}): the board partway "
+                "through the animation, at the point where the most cells were showing "
+                "something they later reverted. This is a real state the game displayed, "
+                "but not one that survives into `current_frame`."
+            )
+        if part is not None:
+            self._pending_animation_image = part
+            self._pending_animation_image_label = label
+
+    def _retained_function_context(self) -> str:
+        """Optional prompt metadata must never interrupt game execution."""
+        if not _persistent_functions():
+            return ""
+        try:
+            notice = getattr(self, "_retained_clear_notice", "")
+            signatures = [function_signature(src) for src in self._kept_functions.values()]
+            parts = [notice] if notice else []
+            if signatures:
+                scope = {"levelup": "on this level", "turn": "in this turn", "game": "in this game"}[_persistent_functions_scope()]
+                parts.append(
+                    "Your retained functions from earlier Python calls " + scope + ": "
+                    + ", ".join(signatures) + ". These are your previous definitions, "
+                    "available to call directly. Their presence does not mean they are "
+                    "correct; revise them if new evidence contradicts their assumptions."
+                )
+            self._retained_clear_notice = ""
+            return "\n\n" + "\n".join(parts) if parts else ""
+        except Exception:
+            self._kept_functions = {}
+            self._retained_clear_notice = ""
+            return "\n\nYour retained functions were cleared after a retention error; redefine any you need."
+
+    def _retained_sources(self):
+        if not _persistent_functions():
+            return None
+        try:
+            return list(self._kept_functions.values())
+        except Exception:
+            self._kept_functions = {}
+            return []
+
+    def _record_retained_functions(self, sandbox_result, payload) -> None:
+        if not _persistent_functions():
+            return
+        try:
+            previous = self._kept_functions
+            kept = {}
+            signatures = {}
+            for item in sandbox_result.get("keepable_functions") or []:
+                name, source = item["name"], item["source"]
+                signature = function_signature(source)
+                if signature.split("(", 1)[0] != name:
+                    raise ValueError("invalid retained function metadata")
+                kept[name] = source
+                signatures[name] = signature
+            notes = []
+            scope = {"levelup": "on this level", "turn": "in this turn", "game": "in this game"}[_persistent_functions_scope()]
+            for name, source in kept.items():
+                if previous.get(name) != source:
+                    notes.append("Retained your function " + signatures[name] + " for later Python calls " + scope + ".")
+            rejected = sandbox_result.get("retention_rejected") or []
+            repair_hints = _get_env_bool("ARC3_PERSISTENT_FUNCTIONS_REPAIR_HINTS", False)
+            rejected_names = set()
+            seen_rejections = set()
+            for item in rejected:
+                name, reason = str(item["name"]), str(item["reason"])
+                hint = (str(item.get("hint") or "").strip() if repair_hints else
+                        "Pass snippet-specific data as arguments or redefine it if needed.")
+                rejection_key = (name, reason, hint)
+                if rejection_key in seen_rejections:
+                    continue
+                seen_rejections.add(rejection_key)
+                rejected_names.add(name)
+                if name == "all":
+                    notes.append("Your retained functions were cleared: " + reason + ".")
+                    continue
+                notes.append("Your function " + name + " was not retained: " + reason + "."
+                             + (" " + hint if hint else ""))
+            for name in previous.keys() - kept.keys() - rejected_names:
+                notes.append("Your previous function " + name + " is no longer retained; redefine it if needed.")
+            self._kept_functions = kept
+            if notes:
+                payload["function_retention"] = "\n".join(notes)
+        except Exception:
+            self._kept_functions = {}
+            payload["function_retention"] = "Your retained functions were cleared after a retention error; redefine any you need."
+
     def _build_user_message(self, user_prompt: str, current_frame: Frame | None) -> dict[str, Any]:
+        user_prompt += self._retained_function_context()
         image_part = current_grid_image_part(current_frame)
+        diff_part = self._pending_diff_image
+        diff_label = self._pending_diff_image_label
+        animation_part = self._pending_animation_image
+        animation_label = self._pending_animation_image_label
+        gameover_images = list(self._pending_gameover_images)
+        self._pending_diff_image = None
+        self._pending_diff_image_label = ""
+        self._pending_animation_image = None
+        self._pending_animation_image_label = ""
+        self._pending_gameover_images = []
+        if animation_part is not None:
+            # the animation image only ever fires when nothing inside the board
+            # moved, which is precisely when the diff image shows the ticking
+            # budget bar and nothing else - two images, one of them noise
+            diff_part = None
         if image_part is None:
             return {"role": "user", "content": user_prompt}
 
-        return {
-            "role": "user",
-            "content": [
-                {"type": "text", "text": f"{user_prompt}\n\nCurrent grid image:"},
-                image_part,
-            ],
-        }
+        content: list[dict[str, Any]] = [
+            {"type": "text", "text": f"{user_prompt}\n\nCurrent grid image (the board AFTER the automatic reset - this is what you act on now):" if gameover_images else f"{user_prompt}\n\nCurrent grid image:"},
+            image_part,
+        ]
+        if diff_part is not None:
+            content.append({"type": "text", "text": diff_label})
+            content.append(diff_part)
+        if animation_part is not None:
+            content.append({"type": "text", "text": animation_label})
+            content.append(animation_part)
+        for label, part in gameover_images:
+            content.append({"type": "text", "text": label})
+            content.append(part)
+        return {"role": "user", "content": content}
+
+
+    def _memory_section_guidance(self, block_shown: bool) -> list[str]:
+        """The section explanation, in upstream's slot just before the tool-call
+        format guidance.
+
+        Returns nothing when the mechanic is off. In 'full' mode the wording
+        depends on whether a stored block was shown: the REPLACES warning is
+        meaningless when there is nothing stored to replace, so a first turn or
+        a post-wipe turn gets the format glossary without it."""
+        if _memory_sections_disabled():
+            return []
+        if _wm_section_explanation_mode() == "upstream":
+            return [_memory_section_guidance_upstream()]
+        if not block_shown:
+            return [_memory_section_guidance_empty()]
+        if _memory_section_labels() is None:
+            return [_MEMORY_SECTION_GUIDANCE_FULL]
+        # a mode carrying a subset needs the subset described, not the fixed
+        # two-section text: `medium` carries three
+        return [
+            _memory_section_guidance_empty()
+            + " IMPORTANT: each labeled section you write REPLACES the stored version shown "
+            "under 'Working world model carried from earlier turns' - write the COMPLETE "
+            "updated section; any fact you omit is permanently lost."
+        ]
+
+    @staticmethod
+    def _manual_reset_context_line(summary: dict[str, Any]) -> str | None:
+        if not reset_exposed() or any(summary.get(flag) for flag in (
+            "stale", "game_over", "run_complete", "done", "level_transition", "level_completed",
+        )):
+            return None
+        actions = [str(name).strip().upper() for name in summary.get("executed_actions") or []]
+        if "RESET" not in actions:
+            return None
+        if actions == ["RESET"]:
+            return (
+                "You deliberately reset the current level. The current frame shows "
+                "the state returned by RESET."
+            )
+        return (
+            "You deliberately reset the current level, then executed the subsequent "
+            "actions listed above. The current frame shows their resulting state."
+        )
+
+    def _summary_turn_context_lines(self) -> list[str]:
+        """What just happened, for the summary request.
 
+        Deliberately not _build_user_prompt: that consumes _pending_diff_image,
+        _pending_animation_image and the wipe-nudge state, so calling it twice
+        would swallow images the real opener should carry. This reads
+        _last_step_summary and mutates nothing - the stale flag is set by tool
+        dispatch, never by building a prompt, so an extra read is safe.
+        """
+        summary = self._last_step_summary
+        if not summary or summary.get("stale"):
+            # nothing executed since the last opener; the block would say
+            # "Executed actions: none" and repeat an unchanged state line
+            return []
+        lines: list[str] = []
+        count = summary.get("executed_count")
+        try:
+            normalized = int(count) if count is not None else None
+        except (TypeError, ValueError):
+            normalized = None
+        if not normalized:
+            return []
+        label = "action" if normalized == 1 else "actions"
+        lines.append(f"The code executed {normalized} {label} in the previous sequence.")
+        rendered = summary.get("executed_actions") or []
+        if rendered:
+            prefix = (
+                "Executed actions:" if len(rendered) <= 10
+                else "Executed actions (first 10):"
+            )
+            lines.append(f"{prefix} {', '.join(str(a) for a in rendered[:10])}.")
+        changed = summary.get("gameplay_changed")
+        if _report_gameplay_changed():
+            if changed is True:
+                lines.append("That sequence changed cells inside the board area.")
+            elif changed is False and summary.get("board_changed"):
+                lines.append(
+                    "That sequence changed only cells near the edge - nothing inside "
+                    "the board area moved."
+                )
+            elif changed is False:
+                lines.append("That sequence produced no change anywhere on the board.")
+        for stop in (summary.get("no_op_stops") or [])[:3]:
+            lines.append(
+                f"A batched sequence was stopped by the no-op guard: "
+                f"{stop.get('action')!r} executed but changed nothing, and "
+                f"{stop.get('skipped', 0)} later action(s) were skipped."
+            )
+        for stop in (summary.get("guard_stops") or [])[:3]:
+            lines.append(
+                f"The sequence was stopped before {stop.get('action')!r}: "
+                f"{stop.get('reason', 'a guard refused it')}."
+            )
+        if summary.get("level_completed"):
+            lines.append("You have progressed to a new level!")
+        elif summary.get("game_over"):
+            lines.append("The attempt ended and the level was reset.")
+        else:
+            lines.append(self._manual_reset_context_line(summary) or "You are still on the same level.")
+        step = summary.get("action_num")
+        level = summary.get("level")
+        if step is not None and level is not None:
+            lines.append(f"Current state: step {step}, level {level}.")
+        return lines
 
     def _build_user_prompt(
         self,
@@ -1183,7 +4828,9 @@ class ToolAgent:
             default=current_level,
         )
         lines: list[str] = []
-        if previous_step_summary:
+        if previous_step_summary and previous_step_summary.get("stale"):
+            lines.extend(_stale_summary_lines(previous_step_summary))
+        elif previous_step_summary:
             count = previous_step_summary.get("executed_count")
             try:
                 normalized_count = int(count) if count is not None else None
@@ -1200,18 +4847,236 @@ class ToolAgent:
                 lines.append(f"{action_prefix} {', '.join(rendered_actions[:10])}.")
             else:
                 lines.append("Executed actions: none.")
+            if _report_gameplay_changed() and normalized_count:
+                # Whether the sequence MOVED anything. The harness computes this
+                # for every action - it drives the batch stop - and until now
+                # said nothing about it: `_describe_last_outcome` produces the
+                # sentence but is called nowhere, so neither its wording nor the
+                # older one it replaced has ever reached a transcript. On a
+                # single-action turn no per-action trace is built either, so the
+                # model had no account of the outcome from any source.
+                border = _get_env_int("ARC3_NOOP_GUARD_BORDER", 4)
+                if previous_step_summary.get("gameplay_changed"):
+                    lines.append("That sequence changed cells inside the board area.")
+                elif previous_step_summary.get("board_changed"):
+                    lines.append(
+                        f"That sequence changed only cells within {border} of the edge "
+                        "(usually a timer or remaining-steps bar) - nothing inside the "
+                        "board area moved."
+                    )
+                else:
+                    lines.append("That sequence produced no change anywhere on the board.")
+            if _animation_enabled() and not previous_step_summary.get("game_over"):
+                animation_line = describe_animation(
+                    previous_step_summary.get("animation"),
+                    action_display=previous_step_summary.get("animation_action"),
+                    # the animated action's own flag, not the sequence's: an
+                    # inert SPACE at the end of a batch that moved other things
+                    # still hid everything it did
+                    gameplay_changed=self._last_action_gameplay_changed,
+                    timeline=_animation_timeline_enabled(),
+                )
+                if animation_line:
+                    lines.append(animation_line)
+                self._prepare_animation_image(previous_step_summary)
+            no_op_stops = previous_step_summary.get("no_op_stops") or []
+            for stop in no_op_stops[:3]:
+                skipped = stop.get("skipped") or []
+                skipped_text = (
+                    f" The remaining {len(skipped)} action(s) {skipped!r} were skipped."
+                    if skipped
+                    else ""
+                )
+                lines.append(
+                    f"Note: a batched sequence was stopped by the no-op guard: action "
+                    f"{stop.get('action')!r} (position {stop.get('index')} in that batch) "
+                    + (f"executed but produced no change inside the board area.{skipped_text} "
+                       if _new_noop_wording() else
+                       f"executed but changed nothing outside the border.{skipped_text} ")
+                    +
+                    "That action IS included in the executed list above, so it is the "
+                    f"LAST of them - only the first {max(0, int(stop.get('index') or 1) - 1)} "
+                    "executed action(s) moved anything. Do not count submitted or executed "
+                    "actions to update your position; re-locate from current_frame."
+                )
+                trace = stop.get("trace") or []
+                if trace:
+                    lines.append("Per-action trace: " + "; ".join(str(t) for t in trace) + ".")
+            if len(no_op_stops) > 3:
+                lines.append(
+                    f"({len(no_op_stops) - 3} further no-op guard stop(s) occurred this sequence.)"
+                )
+            guard_stops = previous_step_summary.get("guard_stops") or []
+            for stop in guard_stops[:3]:
+                skipped = stop.get("skipped") or []
+                skipped_text = (
+                    f" The remaining {len(skipped)} action(s) ({skipped!r}) were never executed."
+                    if skipped
+                    else ""
+                )
+                if stop.get("kind") == "stale_state":
+                    # the exception said this in the turn where it happened;
+                    # restated here because the plan being void is worth
+                    # carrying into the turn that has to replace it
+                    lines.append(
+                        f"Note: the sequence was stopped before {stop.get('action')!r} - an "
+                        + ("earlier action in that snippet produced no change inside the "
+                           "board area, so the positions and "
+                           if _new_noop_wording() else
+                           "earlier action in that snippet changed nothing, so the positions and ")
+                        +
+                        f"paths it had computed may have been wrong.{skipped_text} Re-ground on "
+                        "the current board before planning again."
+                    )
+                elif stop.get("kind") == "known_death":
+                    lines.append(
+                        f"Note: the sequence was stopped before {stop.get('action')!r} - that "
+                        "action ended a previous attempt from that exact board state, so it was "
+                        f"NOT executed and the attempt is intact.{skipped_text} Plan a different "
+                        "move, or repeat it deliberately if you now believe it is safe."
+                    )
+                else:
+                    lines.append(
+                        f"Note: the sequence was stopped before {stop.get('action')!r} - that "
+                        + ("action already produced no change inside the board area in that "
+                           "exact board state, so it was "
+                           if _new_noop_wording() else
+                           "action already changed nothing in that exact board state, so it was ")
+                        +
+                        f"NOT executed and no action was spent.{skipped_text}"
+                    )
+                trace = stop.get("trace") or []
+                if trace:
+                    lines.append("Per-action trace: " + "; ".join(str(t) for t in trace) + ".")
+            if len(guard_stops) > 3:
+                lines.append(
+                    f"({len(guard_stops) - 3} further guard stop(s) occurred this sequence.)"
+                )
             if previous_step_summary.get("run_complete"):
                 lines.append("You have completed the run!")
             elif previous_step_summary.get("level_transition"):
-                lines.append("You have progressed to a new level!")
+                lines.append(
+                    LEVEL_START_USER_PROMPT
+                    if _level_transfer_guidance_enabled()
+                    else "You have progressed to a new level!"
+                )
             else:
-                lines.append("You are still on the same level.")
+                lines.append(
+                    self._manual_reset_context_line(previous_step_summary)
+                    or "You are still on the same level."
+                )
             if previous_step_summary.get("game_over"):
-                lines.append("The game is over.")
+                lines.append(
+                    "GAME OVER occurred during the previous sequence. This means the "
+                    "attempt FAILED (it does NOT mean the run is finished). The game has "
+                    "already been automatically RESET: the current level restarted from "
+                    "its initial state and previously completed levels are kept. A game "
+                    "over is caused either by the specific action taken (e.g., moving "
+                    "onto a hazard cell) or by a step/time budget running out, which is "
+                    "usually indicated by a bar at the grid border that shrinks with each "
+                    "action. Before acting again, diagnose the cause by inspecting the "
+                    "board right before the death. `history[-1].action` is 'RESET', "
+                    "`history[-2].action` is the fatal action, `history[-2].frame` is the "
+                    "death frame (the board after the fatal action), and "
+                    "`history[-3].frame` is the board immediately before "
+                    "the fatal action — check what cell the fatal move targeted there, and "
+                    "inspect the budget bar in the death frame. BAR RULE: the bar "
+                    "shrinking with each action is NORMAL — a slightly shorter bar is "
+                    "NOT evidence of the cause of death. The budget caused the death "
+                    "ONLY if the bar is fully (or almost fully) depleted in the death "
+                    "frame (`history[-2].frame`). If it IS depleted: you must reach the "
+                    "goal in fewer actions, or look for a way to restore the bar (e.g., "
+                    "collecting an item that refills it). If the bar clearly had budget "
+                    "remaining: the fatal action itself caused the death (hazard or "
+                    "forbidden move) — do NOT retry the same path; a DIFFERENT approach "
+                    "must be tried. Record the diagnosed cause in your world model."
+                )
+                if _animation_enabled():
+                    # after the BAR RULE, which tells the model how to read
+                    # history[-2] and history[-3]; this says what those two
+                    # frames cannot contain
+                    death_animation_line = describe_animation(
+                        previous_step_summary.get("animation"),
+                        action_display=previous_step_summary.get("animation_action"),
+                        # the animated action's own flag, not the sequence's: an
+                    # inert SPACE at the end of a batch that moved other things
+                    # still hid everything it did
+                    gameplay_changed=self._last_action_gameplay_changed,
+                        game_over=True,
+                        timeline=_animation_timeline_enabled(),
+                    )
+                    if death_animation_line:
+                        lines.append(death_animation_line)
+                fatal_action = str(previous_step_summary.get("fatal_action") or "")
+                if fatal_action:
+                    lines.append(
+                        f"The game over occurred immediately after action {fatal_action!r}. "
+                        "Apply the BAR RULE above to decide: bar fully depleted in the "
+                        "death frame means budget death (shorter route or bar restore "
+                        "needed); bar with budget remaining means this action itself was "
+                        "fatal and a DIFFERENT approach is required."
+                    )
+                game_over_trace = previous_step_summary.get("game_over_trace") or []
+                if game_over_trace:
+                    lines.append(
+                        "Per-action trace of the previous sequence: "
+                        + "; ".join(str(t) for t in game_over_trace)
+                        + "."
+                    )
+                self._prepare_gameover_images(
+                    history_entries, previous_step_summary
+                )
+                if _get_env_bool("ARC3_GAMEOVER_DIFF", False):
+                    # read at call time so this patch's hunks stay clear of the
+                    # env-knob block that other patches extend
+                    lines.extend(
+                        self._game_over_diff_lines(
+                            history_entries, previous_step_summary
+                        )
+                    )
+                lines.append(
+                    "Do NOT re-submit this same series of moves: the game is "
+                    "deterministic, so replaying the sequence that just killed you "
+                    "will kill you again. Change the plan BEFORE the fatal point - a "
+                    "different route, action, or timing. If you believe the same "
+                    "moves 'should' work, that belief is exactly what this death "
+                    "falsified; record the correction in your world model instead "
+                    "of retesting it."
+                )
         elif (current_frame is not None and current_frame.step > 0) or action_num > 0:
             lines.append("No previous action sequence was captured.")
         else:
             lines.append("No previous sequence has been executed yet.")
+        if self._pending_wm_rebuild_reason and _wm_rebuild_nudge_enabled():
+            lines.append(
+                f"IMPORTANT: your memory sections were reset by {self._pending_wm_rebuild_reason} "
+                "(cross-level notes were kept). Rebuild them NOW: write fresh "
+                # naming a section a restricted mode drops would ask for a
+                # rebuild that is parsed and then thrown away
+                f"{_section_list_phrase(_advertised_section_labels())} sections for this "
+                "level in your next message, before or alongside your tool call."
+            )
+        elif (
+            _wm_revise_nudges_enabled()
+            and getattr(self, "_pending_wm_revise_reason", "") in ("level", "death")
+        ):
+            _revise_reason = getattr(self, "_pending_wm_revise_reason", "")
+            lines.append(
+                _wm_revise_hint(
+                    _revise_reason,
+                    getattr(self, "_wm_revise_hint_shown", False),
+                )
+            )
+            self._wm_revise_hint_shown = True
+        elif _WM_NUDGE_TURNS > 0 and self._turns_without_wm_update >= _WM_NUDGE_TURNS:
+            lines.append(
+                f"You have not updated any memory section for "
+                f"{self._turns_without_wm_update} analysis steps. Review what recent "
+                "action traces, stops, and observations taught you, and write updated "
+                "sections now - knowledge that is not written down is eventually lost "
+                "to context eviction."
+            )
+        lines.extend(self._prepare_auto_diff(current_frame, previous_step_summary))
         state_line = f"Current state: step {current_step}, level {current_level}"
         if observed_max_level > current_level:
             state_line += f" out of observed max level {observed_max_level} so far"
@@ -1220,21 +5085,65 @@ class ToolAgent:
             [
                 state_line,
                 f"Valid actions right now: {_format_valid_action_line(valid_actions)}.",
-                "Only tool: `python`. It receives `current_frame`, `previous_frame`, `history`, `transitions`, `last_transition`, `valid_actions`, `last_action_result`, and `action(actions)`.",
+            ]
+        )
+        # recording is independent of display: the guard needs the ledger even
+        # when the advisory lines are switched off
+        self._death_ledger_observe(history_entries, previous_step_summary)
+        if _get_env_bool("ARC3_DEATH_LEDGER", False):
+            # read at call time so this patch's hunks stay clear of the
+            # env-knob block that other patches extend; placed directly after
+            # the valid-actions line so the fatal continuations sit adjacent
+            # to the action choice they constrain
+            lines.extend(
+                self._death_ledger_lines(
+                    current_frame, history_entries, previous_step_summary
+                )
+            )
+        lines.extend(
+            [
+                "Only tool: `python`. It receives `current_frame`, `previous_frame`, `history`, `transitions`, `last_transition`, `valid_actions`, `last_action_call_result`, `frame_diff(before, after)`, and `action(actions)`.",
                 "Only letter-coded board views and lightweight metadata are exposed; raw numeric color IDs are not available.",
                 "Keep tool output compact: use `current_frame.segmentation` as the primary view, and `current_frame.ascii` only for a small specific region; never print full boards.",
                 "For the most recent change, compare `previous_frame` to `current_frame`, or `last_transition.before_frame` to `last_transition.after_frame`; `history[-1].frame` is the current frame, not the previous one.",
-                "Use Python to inspect the evidence, refine that world model from the newest history, and search or score candidate actions or short sequences against the current goal as you currently understand it.",
-                "Maintain a compact working world model of what the current level seems to contain, what actions appear to do, what the goal seems to be, what is still uncertain, and what plan currently looks best.",
-                "Below you are provided with the current world model from the previous turn. The default behavior is to copy it and add or remove things based on the evidence that you gathered. BEFORE EXECUTING NEW ACTIONS YOU MUST ALWAYS GIVE THE REVISED VERSION OF THE WORLD MODEL.",
             ]
         )
-        lines.append(
-            "You may call `action(actions)` more than once in one Python snippet if your search or control loop needs it, "
-            "but stop immediately if a result reports `game_over`, `run_complete`, `level_completed`, or `done`."
-        )
-        lines.extend(self._summarized_knowledge_lines())
-        lines.append("end of world model. ")
+        # Upstream states four things about the world model here, and with
+        # ARC3_MEMORY_SECTIONS=off three of them are false or unanswerable: the
+        # block they refer to is suppressed, so "Below you are provided with the
+        # current world model" is followed by nothing, and the model is told in
+        # capitals to always give a revised version with nowhere to put it.
+        # Observed as a drop in replies carrying any assistant text at all,
+        # 76% to 43% - a broken instruction, not an absent one, which is why
+        # `off` was never a clean control.
+        #
+        # The first sentence loses only its middle clause: "from the newest
+        # history" attaches to inspecting the evidence just as well, so the
+        # remainder still says use Python, inspect, and score against the goal.
+        if _memory_sections_disabled():
+            lines.append(
+                "Use Python to inspect the evidence from the newest history, and search or score candidate actions or short sequences against the current goal as you currently understand it."
+            )
+        else:
+            lines.extend(
+                [
+                    "Use Python to inspect the evidence, refine that world model from the newest history, and search or score candidate actions or short sequences against the current goal as you currently understand it.",
+                    "Maintain a compact working world model of what the current level seems to contain, what actions appear to do, what the goal seems to be, what is still uncertain, and what plan currently looks best.",
+                    "Below you are provided with the current world model from the previous turn. The default behavior is to copy it and add or remove things based on the evidence that you gathered. BEFORE EXECUTING NEW ACTIONS YOU MUST ALWAYS GIVE THE REVISED VERSION OF THE WORLD MODEL.",
+                ]
+            )
+        if not _dedupe_multicall_line():
+            # kept mid-opener only for the upstream wording; the surviving copy
+            # in the closing block is where a note about calling action() twice
+            # belongs, next to the instruction to call it at all
+            lines.append(
+                "You may call `action(actions)` more than once in one Python snippet if your search or control loop needs it, "
+                "but stop immediately if a result reports `game_over`, `run_complete`, `level_completed`, or `done`."
+            )
+        knowledge_lines = self._summarized_knowledge_lines()
+        if knowledge_lines:
+            lines.extend(knowledge_lines)
+            lines.append("end of world model. ")
         if action_num == 0:
             lines.append(
                 "Ground yourself in `current_frame` before acting, but start with a compact structural summary rather than restating the full frame."
@@ -1247,12 +5156,21 @@ class ToolAgent:
             [
                 "When ready, call `action(actions)` from inside the `python` tool with the best valid action or ordered batch selected by your code. If your code has found a reliable short sequence, prefer batching it in one call.",
                 "You may call `action(actions)` more than once in one Python snippet if your search or control loop needs it.",
-                "If you include assistant text before a tool call, keep it short and use it to update the world model. Helpful optional prefixes are `World model:`, `Goal model:`, `Action model:`, `Recent findings:`, `Open questions:`, `Plan:`, and `Cross-level notes:`.",
+                *self._memory_section_guidance(bool(knowledge_lines)),
                 TOOL_CALL_FORMAT_GUIDANCE,
             ]
         )
         if "MOUSE" in _normalize_valid_actions(valid_actions):
             lines.append("If you use MOUSE, include integer row and col arguments.")
+        if _opener_commit_hint():
+            # last line of the opener, where the resumption prompt sits relative
+            # to the reply it influences
+            lines.append(
+                "When your evidence supports a choice, call `action(actions)` with the "
+                "sequence it supports. Probe with a single action when you genuinely "
+                "need to distinguish two possibilities; otherwise batch the route you "
+                "have."
+            )
         return "\n".join(lines)
 
     def _tools(self, state_path: Path) -> list[dict[str, Any]]:
@@ -1279,35 +5197,105 @@ class ToolAgent:
             }
         ]
 
+    def _harness_template_kwargs(self) -> dict[str, Any]:
+        """Chat-template kwargs the HARNESS sets per request.
+
+        Shared by the request builder and the request log so the log records
+        what was actually sent rather than a reconstruction - which matters
+        for the effort ladder, where the value changes mid-run, and for
+        preserve_thinking, where the question is whether it reaches the server
+        at all."""
+        kwargs: dict[str, Any] = {}
+        preserve_thinking = _preserve_thinking_kwarg()
+        if preserve_thinking is not None:
+            kwargs["preserve_thinking"] = preserve_thinking
+        ladder = _reasoning_effort_ladder()
+        rung = getattr(self, "_reasoning_effort_rung", -1)
+        if ladder and 0 <= rung < len(ladder):
+            kwargs["reasoning_effort"] = ladder[rung]
+        return kwargs
+
     def _chat_completion(
         self,
         messages: list[dict[str, Any]],
         *,
         tools: list[dict[str, Any]] | None,
         request_timeout_seconds: float | None = None,
+        max_tokens: int | None = None,
+        enable_thinking: bool | None = None,
     ) -> _ChatCompletionResult:
         payload = build_chat_payload(
             provider=self._model.provider,
             model=self._model.model_id,
             messages=messages,
-            max_tokens=self._max_output_tokens,
+            max_tokens=self._max_output_tokens if max_tokens is None else max_tokens,
             temperature=_LOCAL_ANALYZER_TEMPERATURE,
             top_p=_LOCAL_ANALYZER_TOP_P,
             top_k=_LOCAL_ANALYZER_TOP_K,
-            thinking=bool(_LOCAL_ANALYZER_ENABLE_THINKING),
+            thinking=(
+                bool(_LOCAL_ANALYZER_ENABLE_THINKING)
+                if enable_thinking is None
+                else enable_thinking
+            ),
             tools=tools,
             tool_choice=_request_tool_choice(tools),
             seed=_LOCAL_ANALYZER_SEED,
         )
+        provider_prefs = _openrouter_provider_prefs(self._model.provider)
+        if provider_prefs:
+            payload["provider"] = provider_prefs
+        # merge rather than assign: the payload builder may already have set
+        # chat_template_kwargs (enable_thinking), and replacing the dict would
+        # drop it - as a request-level dict can also drop a server-side default
+        template_kwargs = dict(payload.get("chat_template_kwargs") or {})
+        template_kwargs.update(self._harness_template_kwargs())
+        if enable_thinking is not None:
+            # after the merge: _harness_template_kwargs does not set
+            # enable_thinking today, but an override that a later edit could
+            # silently undo is the bug that keeps recurring here
+            template_kwargs["enable_thinking"] = bool(enable_thinking)
+        if template_kwargs:
+            payload["chat_template_kwargs"] = template_kwargs
+        payload["messages"] = _strip_control_keys(
+            _apply_summary_visibility(
+                payload.get("messages") or [],
+                evicted=getattr(self, "_has_evicted", False),
+            )
+        )
         def post_chat(request_payload: dict[str, Any]) -> requests.Response:
             return requests.post(
                 f"{self._model.base_url.rstrip('/')}/chat/completions",
                 headers=self._headers(),
                 json=request_payload,
-                timeout=request_timeout_seconds if request_timeout_seconds is not None else self._timeout,
+                # (connect, read): one scalar gives the connect phase the whole
+                # read budget, so an unreachable host stalls a single attempt
+                # for the rest of the game
+                timeout=(
+                    _CONNECT_TIMEOUT_SECONDS,
+                    request_timeout_seconds if request_timeout_seconds is not None else self._timeout,
+                ),
+                stream=bool(request_payload.get("stream")),
             )
 
-        response = post_chat(payload)
+        # Per agent, not per process: every game meets the cold server on its
+        # own first request, so one shared flag would grant the grace to
+        # whichever game happened to go first and let the rest fail fast.
+        initial_grace = 0.0
+        if not self._http_initial_grace_used:
+            initial_grace = _env_float("ARC3_HTTP_RETRY_INITIAL_SECONDS", 0.0)
+        _diag(self, "request")
+        try:
+            response = _post_with_retries(
+                lambda: post_chat(payload),
+                retries=_env_int("ARC3_HTTP_RETRIES", 3),
+                base_seconds=_env_float("ARC3_HTTP_RETRY_BASE_SECONDS", 5.0),
+                max_seconds=_env_float("ARC3_HTTP_RETRY_MAX_SECONDS", 5.0),
+                initial_seconds=initial_grace,
+            )
+        finally:
+            # consumed whether the call succeeded or failed: the grace covers
+            # the server coming up, and by now it either did or will not
+            self._http_initial_grace_used = True
         try:
             response.raise_for_status()
         except requests.HTTPError as exc:
@@ -1322,22 +5310,63 @@ class ToolAgent:
             if detail:
                 message += f" | response: {detail}"
             raise requests.RequestException(message)
-        payload = response.json()
+        if payload.get("stream"):
+            # decode_unicode=True decodes each transport chunk INDEPENDENTLY, so a
+            # multi-byte UTF-8 sequence straddling a chunk boundary is split into
+            # two broken halves - an em-dash (E2 80 94) becomes "\u00e2" plus
+            # fragments. Most characters survive, which is why short probes look
+            # clean and only long streamed replies are corrupted. The assembler
+            # already decodes bytes itself, and a complete SSE line is always a
+            # whole UTF-8 sequence, so hand it raw bytes instead.
+            payload = assemble_streamed_chat_response(
+                response.iter_lines(decode_unicode=False)
+            )
+        else:
+            payload = response.json()
         choices = payload.get("choices", [])
         if not choices:
-            raise requests.RequestException("server returned no choices")
+            detail = payload.get("error") or {
+                k: v for k, v in payload.items() if k not in ("choices",)
+            }
+            raise requests.RequestException(
+                f"server returned no choices | body: {str(detail)[:600]}"
+            )
         choice = choices[0]
         return _ChatCompletionResult(
-            message=choice.get("message", {}),
+            message=_extract_leaked_tool_calls(choice.get("message", {})),
             finish_reason=str(choice.get("finish_reason", "") or ""),
             usage=payload.get("usage"),
+            served_by=str(payload.get("provider", "") or ""),
         )
 
     def _trim_tool_text(self, text: str) -> tuple[str, bool]:
-        if len(text) <= self._tool_output_chars:
+        """Keep both ends and drop the middle.
+
+        Truncating the tail loses the half that matters. An `error` field ends
+        with `SomeError: <the explanation>` after the traceback, so a deep
+        traceback pushed the explanation out entirely. And a snippet's `stdout`
+        typically prints the state, acts, then prints the state again - so the
+        tail holds the result and the head holds the setup.
+
+        Weighted toward the HEAD: a snippet's output is mostly sequential
+        reasoning that reads from the top, so two thirds there. The remaining
+        third is ample for what has to survive at the end - the longest refusal
+        message measures ~550 characters and a user-frame traceback ~250, well
+        inside a 1365-character tail at the default budget.
+        """
+        limit = self._tool_output_chars
+        if len(text) <= limit:
             return text, False
-        omitted = len(text) - self._tool_output_chars
-        return f"{text[:self._tool_output_chars]}\n... [truncated {omitted} chars]", True
+        omitted = len(text) - limit
+        if not _get_env_bool("ARC3_MIDDLE_TRUNCATION", False):
+            return f"{text[:limit]}\n... [truncated {omitted} chars]", True
+        tail = limit // 3
+        head = limit - tail
+        return (
+            f"{text[:head]}\n... [truncated {omitted} chars from the middle] ...\n"
+            f"{text[-tail:]}",
+            True,
+        )
 
     def _summarize_planned_actions(self, value: Any) -> Any:
         if isinstance(value, dict):
@@ -1366,6 +5395,9 @@ class ToolAgent:
         for field in truncate_fields:
             value = result.get(field)
             if isinstance(value, str):
+                value, suppressed_rows = _suppress_board_dumps(value)
+                if suppressed_rows:
+                    result["board_print_suppressed_rows"] = suppressed_rows
                 result[field], field_truncated = self._trim_tool_text(value)
                 truncated = truncated or field_truncated
         if truncated:
@@ -1413,13 +5445,230 @@ class ToolAgent:
             raise TypeError(f"Action {index} must be a string or a dict.")
         return normalized
 
+    def _fatal_prefix_match(
+        self, state_hash: str, remaining_sigs: list[str]
+    ) -> tuple[int, str] | None:
+        """Does a recorded fatal sequence continue from here?
+
+        Returns ``(offset, action)`` when executing ``remaining_sigs`` from
+        this board would replay a recorded attempt into its death - offset 0
+        meaning the very next action is the fatal one. Returns None otherwise.
+
+        Generalizes the single-action check: an attempt recorded as actions
+        ``A[0..n-1]`` with pre-action hashes ``H[0..n-1]`` is fatal from here
+        if some ``H[k]`` equals this board AND ``A[k:]`` is a prefix of what
+        the model is about to run. The environment is deterministic and a
+        death resets to level start, so replaying ``A[k:]`` from ``H[k]``
+        reproduces ``H[k+1] ...`` and ends the attempt.
+
+        Catching it at the FIRST position of such a batch is what makes the
+        block worth having: the per-position check alone would let the harmless
+        prefix run and refuse only the final move, spending those actions to
+        arrive somewhere the model has already been. A model that genuinely
+        wants the prefix can submit it without the fatal tail, which no longer
+        matches and is not blocked.
+        """
+        index = getattr(self, "_death_ledger_index", None)
+        attempts = getattr(self, "_death_ledger_attempts", None)
+        if not index or not attempts or not remaining_sigs:
+            return None
+        for attempt_idx, position in index.get(state_hash, ()):  # type: ignore[union-attr]
+            try:
+                actions = attempts[attempt_idx]["actions"]
+            except (IndexError, KeyError, TypeError):
+                continue
+            tail = [action_signature(a) for a in actions[position:]]
+            if not tail or len(tail) > len(remaining_sigs):
+                continue
+            if remaining_sigs[: len(tail)] == tail:
+                return len(tail) - 1, tail[-1]
+        return None
+
+    def _action_guard_hook(
+        self,
+        grid: Any,
+        action_name: str,
+        action_data: dict[str, Any],
+        level: Any = None,
+        remaining: list[tuple[str, dict[str, Any]]] | None = None,
+    ) -> tuple[str, str] | None:
+        """Called by the solver before each action of a batch. Returns
+        ``(stop_reason, detail)`` to refuse, or None to proceed."""
+        guard = self._noop_repeat_guard
+        if guard is None or not grid:
+            return None
+        if not _guards_active(level):
+            # level 1 is where the model probes hardest and where an action is
+            # worth least, so the repeat guards can be told to start later. The
+            # guard still WATCHES - note_level and the state hash below run
+            # either way - so that by the time it starts refusing it already
+            # knows what has been tried.
+            guard.note_level(level)
+            self._noop_guard_pre_hash = _interior_state_hash(
+                [list(row) for row in grid]
+            )
+            return None
+        rows = [list(row) for row in grid]
+        state_hash = _interior_state_hash(rows)
+        guard.note_level(level)
+        # the solver passes ENGINE names (ACTION2); every other consumer - the
+        # death ledger, the transcript, the model itself - speaks model names
+        # (DOWN), so translate here or the guard both mismatches the ledger and
+        # reports "ACTION2" to a model that has never seen that label
+        action_sig = action_signature(
+            {"action": to_model_action(action_name), **(action_data or {})}
+        )
+        self._noop_guard_pre_hash = state_hash
+        if self._noop_guard_deaths_enabled and not self._guard_death_override_batch:
+            # with lookahead off, only THIS action is considered, so a match
+            # can be found solely at offset 0 - the per-position behaviour
+            window = (
+                (remaining or [(action_name, action_data)])
+                if _fatal_lookahead_enabled()
+                else [(action_name, action_data)]
+            )
+            remaining_sigs = [
+                action_signature({"action": to_model_action(name), **(data or {})})
+                for name, data in window
+            ]
+            match = self._fatal_prefix_match(state_hash, remaining_sigs)
+            if guard.should_block_death(
+                state_hash, action_sig, lambda _h, _a: match is not None
+            ):
+                offset, fatal_action = match  # type: ignore[misc]
+                if offset == 0:
+                    detail = (
+                        f"{fatal_action} ended a previous attempt on this level from this "
+                        "exact board state, so it was NOT executed."
+                    )
+                else:
+                    detail = (
+                        "this sequence replays a path that ended a previous attempt: "
+                        f"{fatal_action} is fatal {offset} action(s) from here, so NOTHING "
+                        "was executed and no actions were spent. Submit a different "
+                        "continuation, or the same prefix without the fatal tail."
+                    )
+                return (
+                    "known_death",
+                    detail
+                    + " This refusal is not catchable and ended the snippet, so anything "
+                    "after the call did not run. Issuing the same action again as the "
+                    "very next action will override the block and execute the action. "
+                    "That is only worth doing if you have a specific reason to believe "
+                    "it will not end the attempt this time. Otherwise, plan a different "
+                    "continuation from this state.",
+                )
+            # the override fired: let the rest of THIS call run without further
+            # death checks, or the model would have to insist again at every
+            # position of a sequence it has already deliberately confirmed
+            if match is not None:
+                self._guard_death_override_batch = True
+        if (
+            self._noop_guard_noops_enabled
+            and not (reset_exposed() and action_sig == "RESET")
+            and guard.should_block(state_hash, action_sig)
+        ):
+            return (
+                "known_noop",
+                (f"{action_sig} already produced no change inside the board area in this "
+                   "exact board state "
+                   if _new_noop_wording() else
+                   f"{action_sig} already changed nothing in this exact board state ")
+                +
+                "earlier on this level, so it was NOT executed and no action was "
+                "spent. This refusal is not catchable and ended the snippet, so "
+                "anything after the call did not run - a retry loop cannot probe "
+                "past it. It applies to that one action from this one board state; "
+                "it is not a constraint on sequences or batch size. Issuing the same "
+                "action again as the very next action will override the block and "
+                "execute the action. That is only worth doing if you have a specific "
+                "reason to believe it will behave differently now. Otherwise, plan a "
+                "different continuation from this state.",
+            )
+        if (
+            _repeat_state_guard_enabled()
+            and (state_hash, action_sig) in self._repeat_state_seen
+        ):
+            # after known_noop deliberately: a repeated no-op is better reported
+            # as a no-op, which also keeps its override intact. Reaching here
+            # means the action DID change the board and still returned to a
+            # state this snippet has already acted from.
+            return (
+                "repeated_action_in_state",
+                f"{action_sig} was already run from this exact board state earlier "
+                "in this snippet, so what it does here is known and it was NOT "
+                # not "no action budget was spent": by construction this fires
+                # after the snippet has already spent budget on the actions that
+                # got it here, and the model reads that phrase as covering the
+                # whole snippet
+                "executed. A sequence that returns to "
+                # examples, not a diagnosis: the harness knows the board
+                # repeated, not why, and stating the cause would have the model
+                # looking for a wrapping cursor it may not have
+                "a board it has already acted from is not advancing (e.g. a cursor "
+                "that wraps around, or a loop stepping the wrong way past its "
+                "target). "
+                "Nothing after the call ran. Re-read `current_frame`, work out why "
+                "the sequence is not converging, and write a different snippet - "
+                "this block is per snippet, so the next one starts with no history.",
+            )
+        guard.note_executed(state_hash, action_sig)
+        self._guard_hook_trace.append((state_hash, action_sig))
+        return None
+
+    def _record_guard_observations(
+        self,
+        guard: NoopRepeatGuard,
+        raw_payload: dict[str, Any],
+        compact_payload: dict[str, Any],
+    ) -> None:
+        """Record every executed position of this call, not just the first.
+
+        The guard hook fires once per position immediately before that action
+        runs, so `_guard_hook_trace` holds the board hash each action actually
+        executed against - including the intermediate states of a batch, which
+        appear nowhere in the returned payload. The solver reports outcomes as
+        `gameplay_changed_per_action`, index-aligned with `executed_actions`
+        and therefore with the trace; a single-action call has no such list, so
+        the aggregate verdict is used instead.
+
+        Zipping to the shorter of the two is deliberate: a batch stopped early
+        (blocked, fatal, invalid) leaves one more trace entry than there are
+        outcomes, and an unmatched entry must not be recorded against a verdict
+        that belongs to a different action.
+        """
+        trace = list(self._guard_hook_trace)
+        self._guard_hook_trace = []
+        if not trace:
+            return
+        verdicts = raw_payload.get("gameplay_changed_per_action")
+        if not isinstance(verdicts, list) or not verdicts:
+            if len(trace) != 1 or not compact_payload.get("executed"):
+                return
+            verdicts = [compact_payload.get("gameplay_changed")]
+        for (state_hash, action_sig), changed in zip(trace, verdicts):
+            if reset_exposed() and action_sig == "RESET":
+                # Retain its position in the trace so following verdicts stay
+                # aligned, but do not learn that a bar-only reset is ineffective.
+                continue
+            # unconditional: known_noop reaches a repeated no-op first, so
+            # filtering on `changed` here would buy nothing
+            self._repeat_state_seen.add((state_hash, action_sig))
+            guard.observe(state_hash, action_sig, gameplay_changed=changed)
+
     def _compact_action_result(self, payload: dict[str, Any]) -> dict[str, Any]:
         compact = {
             "executed": bool(payload.get("executed")),
             "action_num": payload.get("action_num"),
             "level": payload.get("level"),
             "score": payload.get("score"),
-            "reward": payload.get("reward"),
+            # `reward` is not carried: it is (levels_completed - previous) /
+            # number_of_levels, so it reads 0.0 on every action that does not
+            # finish a level. A model seeing 0.0 on almost every action can
+            # reasonably take it for a verdict on the action, which it is not,
+            # and `level_completed` in this same dict already says the thing it
+            # was trying to say. The solver still computes it for the viewer
+            # and the raw payload.
             "state": payload.get("state"),
             "valid_actions": payload.get("valid_actions", []),
             "board_changed": bool(payload.get("board_changed")),
@@ -1429,6 +5678,8 @@ class ToolAgent:
             "run_complete": bool(payload.get("run_complete")),
             "action_display": payload.get("action_display") or payload.get("action_name"),
         }
+        if "gameplay_changed" in payload:
+            compact["gameplay_changed"] = bool(payload.get("gameplay_changed"))
         executed_actions = payload.get("executed_actions")
         if isinstance(executed_actions, list) and executed_actions:
             compact["executed_actions"] = [str(action).strip() for action in executed_actions if str(action).strip()]
@@ -1439,6 +5690,11 @@ class ToolAgent:
             compact["requested_count"] = payload.get("requested_count", batch_size)
             compact["executed_count"] = payload.get("executed_count", batch_size)
             compact["stopped_early"] = bool(payload.get("stopped_early"))
+        # blocked_action names the refused action for the next turn's opener;
+        # without it the model was told "the sequence was stopped before None"
+        for passthrough_key in ("no_op_action_index", "no_op_action", "skipped_actions", "gameplay_changed_per_action", "action_trace", "no_op", "fatal_action", "blocked_action", "action_echo", "animation"):
+            if passthrough_key in payload:
+                compact[passthrough_key] = payload.get(passthrough_key)
         if payload.get("stop_reason"):
             compact["stop_reason"] = payload.get("stop_reason")
         if payload.get("stop_detail"):
@@ -1446,6 +5702,14 @@ class ToolAgent:
         for timing_key in ("run_elapsed_seconds", "time_remaining_seconds"):
             if timing_key in payload:
                 compact[timing_key] = payload.get(timing_key)
+        # the session's own clock, which is also what decides when the game
+        # stops - so the endgame phase cannot drift away from the deadline it
+        # is meant to anticipate. It starts at session construction, so a game
+        # waiting on the gate is already ageing, which is the same convention
+        # max_runtime_s_per_game uses.
+        elapsed = payload.get("run_elapsed_seconds")
+        if isinstance(elapsed, (int, float)):
+            self._game_elapsed_seconds = float(elapsed)
         if payload.get("error"):
             compact["error"] = payload.get("error")
         return compact
@@ -1466,7 +5730,7 @@ class ToolAgent:
         def _serialized_runtime_state(
             *,
             next_valid_actions: list[str] | None = None,
-            last_action_result: dict[str, Any] | None = None,
+            last_action_call_result: dict[str, Any] | None = None,
         ) -> dict[str, Any]:
             refreshed_frame, refreshed_history = load_runtime_state(state_path)
             current_frame_payload = _ascii_frame_view_payload(refreshed_frame)
@@ -1475,15 +5739,20 @@ class ToolAgent:
             else:
                 sanitized_actions = list(valid_actions)
             persisted_action_result = (
-                last_action_result
-                if isinstance(last_action_result, dict)
-                else self._last_action_result
+                last_action_call_result
+                if isinstance(last_action_call_result, dict)
+                else self._last_action_call_result
             )
             return {
                 "current_frame": current_frame_payload,
                 "history": _ascii_history_view_payload(refreshed_history),
                 "valid_actions": sanitized_actions,
-                "last_action_result": (
+                "death_ledger": (
+                    self._death_ledger_payload()
+                    if _get_env_bool("ARC3_DEATH_LEDGER", False)
+                    else None
+                ),
+                "last_action_call_result": (
                     dict(persisted_action_result)
                     if isinstance(persisted_action_result, dict)
                     else {}
@@ -1491,9 +5760,24 @@ class ToolAgent:
             }
 
         terminal_action_result: dict[str, Any] | None = None
+        snippet_executed_actions = 0
 
-        def _handle_action(actions: list[dict[str, Any]]) -> dict[str, Any]:
-            nonlocal terminal_action_result
+        def _handle_animation(request: dict[str, Any]) -> dict[str, Any]:
+            if self._step_env_callback is None:
+                return {"error": "Animation frames are not available in this session."}
+            raw = self._step_env_callback({"query": "animation"})
+            record = raw.get("record") if isinstance(raw, dict) else None
+            record = record if isinstance(record, dict) else None
+            if str(request.get("kind") or "") == "frames":
+                return build_animation_frames(record)
+            if not _animation_timeline_enabled():
+                return {}
+            return build_animation_view(record)
+
+        def _handle_action(
+            actions: list[dict[str, Any]], *, stale_after: str | None = None
+        ) -> dict[str, Any]:
+            nonlocal terminal_action_result, snippet_executed_actions
             if self._step_env_callback is None:
                 raise RuntimeError("action(actions) is not available in this session.")
             normalized_actions = self._normalize_python_actions(actions)
@@ -1504,7 +5788,6 @@ class ToolAgent:
                     "action_num": terminal_action_result.get("action_num"),
                     "level": terminal_action_result.get("level"),
                     "score": terminal_action_result.get("score"),
-                    "reward": 0.0,
                     "state": terminal_action_result.get("state"),
                     "valid_actions": [],
                     "board_changed": False,
@@ -1518,37 +5801,107 @@ class ToolAgent:
                     "stop_reason": f"previous_{reason}",
                     "stop_detail": _terminal_action_stop_detail(reason),
                 }
-                self._last_action_result = dict(compact_payload)
+                self._last_action_call_result = dict(compact_payload)
                 return {
                     "action_result": compact_payload,
                     "state": _serialized_runtime_state(
                         next_valid_actions=[],
-                        last_action_result=compact_payload,
+                        last_action_call_result=compact_payload,
                     ),
                 }
-            raw_payload = self._step_env_callback({"actions": normalized_actions})
+            if reset_exposed():
+                reset_positions = [
+                    index for index, item in enumerate(normalized_actions)
+                    if to_engine_action(item.get("action")) == "RESET"
+                ]
+                if reset_positions and (snippet_executed_actions or reset_positions != [0]):
+                    detail = (
+                        "RESET must be the first executed game action in a Python snippet. "
+                        "This call executed no actions. Start a new snippet with RESET "
+                        "as its first action; inspection or computation may come before it."
+                    )
+                    refusal = {
+                        "executed": False,
+                        "stop_reason": "reset_not_first",
+                        "stop_detail": detail,
+                        "error": detail,
+                        "valid_actions": list(self._current_valid_actions),
+                    }
+                    self._last_action_call_result = dict(refusal)
+                    return {
+                        "action_result": refusal,
+                        "state": _serialized_runtime_state(last_action_call_result=refusal),
+                    }
+            guard = self._noop_repeat_guard
+            self._guard_hook_trace = []
+            self._guard_death_override_batch = False
+            # No pre-check here: the solver's per-position hook covers every
+            # action of every call, including the first. Checking in both
+            # places consumed the override slot twice - the pre-check armed it,
+            # the retry cleared it, and the hook then blocked the very action
+            # the model had been invited to repeat.
+            step_arguments: dict[str, Any] = {"actions": normalized_actions}
+            if stale_after:
+                # forwarded as an argument so the solver can weigh it AFTER its
+                # own per-position guards, which name a specific action and
+                # carry an override
+                step_arguments["stale_after"] = stale_after
+            raw_payload = self._step_env_callback(step_arguments)
             if not isinstance(raw_payload, dict):
                 raise RuntimeError("action(actions) did not return a JSON-like payload.")
+            if reset_exposed() and raw_payload.get("executed"):
+                # Use actual execution, not submitted batch size: a refused
+                # call does not consume the snippet's first-action allowance.
+                snippet_executed_actions += max(1, int(raw_payload.get("executed_count") or 1))
+            if _animation_enabled():
+                # Held on the agent, not routed through the sandbox: the chain
+                # is dropped by compaction so the raw grids never become text,
+                # which also meant it never reached the summary. It is only
+                # ever used to render an image here, so it has no business
+                # crossing the pipe twice.
+                self._last_animation_chain = raw_payload.get("animation_chain") or []
+                # Whether the LAST executed action moved anything, which is not
+                # the same as whether the sequence did: a batch of UP, DOWN,
+                # LEFT, SPACE reports gameplay_changed for the batch, so an
+                # inert SPACE at the end was hidden behind three moves that
+                # were not. The image exists for the action that animated, so
+                # it is that action's flag that decides.
+                per_action = raw_payload.get("gameplay_changed_per_action")
+                if isinstance(per_action, list) and per_action:
+                    self._last_action_gameplay_changed = bool(per_action[-1])
+                else:
+                    self._last_action_gameplay_changed = raw_payload.get(
+                        "gameplay_changed"
+                    )
             compact_payload = self._compact_action_result(raw_payload)
+            if guard is not None and normalized_actions:
+                self._record_guard_observations(guard, raw_payload, compact_payload)
             next_valid_actions = raw_payload.get("valid_actions")
             if isinstance(next_valid_actions, list):
                 self._current_valid_actions = _normalize_valid_actions(next_valid_actions)
             if compact_payload.get("executed") and _terminal_action_reason(compact_payload):
                 terminal_action_result = compact_payload
-            self._last_action_result = dict(compact_payload)
+            self._last_action_call_result = dict(compact_payload)
             return {
                 "action_result": compact_payload,
                 "state": _serialized_runtime_state(
                     next_valid_actions=next_valid_actions if isinstance(next_valid_actions, list) else None,
-                    last_action_result=compact_payload,
+                    last_action_call_result=compact_payload,
                 ),
             }
 
+        # per snippet: a repeat across snippets is the model choosing, not a
+        # loop failing to converge
+        self._repeat_state_seen.clear()
         sandbox_result = run_sandboxed_python(
             code=code,
             timeout_seconds=self._python_timeout,
             initial_state=_serialized_runtime_state(),
             action_handler=_handle_action,
+            animation_handler=_handle_animation if _animation_enabled() else None,
+            kept_functions=self._retained_sources(),
+            retain_imports=_get_env_bool("ARC3_PERSISTENT_FUNCTIONS_IMPORTS", False),
+            repair_hints=_get_env_bool("ARC3_PERSISTENT_FUNCTIONS_REPAIR_HINTS", False),
         )
 
         action_results = [
@@ -1557,6 +5910,7 @@ class ToolAgent:
             if isinstance(item, dict)
         ]
         payload: dict[str, Any] = {"tool": "python"}
+        self._record_retained_functions(sandbox_result, payload)
         rendered_stdout = str(sandbox_result.get("stdout", "") or "")
         rendered_error = str(sandbox_result.get("error", "") or "")
         if rendered_error:
@@ -1575,16 +5929,28 @@ class ToolAgent:
                 else:
                     payload["result"] = {
                         "action_calls": len(action_results),
-                        "last_action_result": action_results[-1],
+                        "last_action_call_result": action_results[-1],
                     }
 
         step_executed = any(bool(item.get("executed")) for item in action_results)
+        made_progress = any(
+            bool(item.get("gameplay_changed"))
+            or bool(item.get("level_completed"))
+            or bool(item.get("game_over"))
+            or bool(item.get("run_complete"))
+            for item in action_results
+        )
         if step_executed:
             self._last_step_summary = self._summarize_step_sequence(action_results)
             self._update_summarized_knowledge_from_step_summary()
+        elif self._last_step_summary and not self._last_step_summary.get("stale"):
+            # carried over across an exchange that executed nothing: mark it so
+            # the next prompt reminds instead of re-narrating it as fresh
+            self._last_step_summary = {**self._last_step_summary, "stale": True}
         return _ToolDispatchResult(
             self._render_tool_payload(payload, truncate_fields=("stdout", "error", "result")),
             step_executed=step_executed,
+            made_progress=made_progress,
         )
 
     def _dispatch_tool(self, state_path: Path, name: str, arguments: dict[str, Any]) -> _ToolDispatchResult:
@@ -1593,22 +5959,284 @@ class ToolAgent:
             return self._run_python_tool(state_path, arguments)
         return _ToolDispatchResult(json.dumps({"error": f"Unknown tool: {name}"}, indent=2))
 
+    def _calibrate_from_usage(
+        self,
+        messages: list[dict[str, Any]],
+        tools: list[dict[str, Any]] | None,
+        usage: dict[str, Any] | None,
+    ) -> str:
+        """Update the characters-per-token divisor from a served request.
+
+        The images are subtracted using the SAME formula that will re-add them,
+        so the two cancel and the divisor measures text alone. Clamped because
+        an under-count risks context overflow, which is the dangerous
+        direction; a bad measurement should cost accuracy, not a rejected
+        request."""
+        if not _calibrate_text_tokens() or not isinstance(usage, dict):
+            return ""
+        try:
+            prompt_tokens = int(usage.get("prompt_tokens") or 0)
+        except (TypeError, ValueError):
+            return ""
+        if prompt_tokens <= 0:
+            return ""
+        scrubbed, image_tokens = _split_messages_for_estimate(list(messages))
+        payload: dict[str, Any] = {"messages": scrubbed}
+        if tools:
+            payload["tools"] = tools
+            payload["tool_choice"] = _request_tool_choice(tools)
+        try:
+            rendered = json.dumps(payload, ensure_ascii=False, sort_keys=True, default=str)
+        except TypeError:
+            return ""
+        text_tokens = prompt_tokens - image_tokens
+        if text_tokens <= 0 or not rendered:
+            return ""
+        measured = len(rendered) / text_tokens
+        clamped = min(_text_token_chars_max(), max(_text_token_chars_min(), measured))
+        previous = self._text_chars_per_token
+        self._text_chars_per_token = clamped
+        if previous is None or abs(clamped - previous) >= 0.05:
+            note = "" if clamped == measured else f" (clamped from {measured:.2f})"
+            # returned rather than written here: append_transcript is a closure
+            # inside analyze(), not reachable from a method
+            return (
+                f"text_token_calibration: {clamped:.2f} chars/token{note} "
+                f"[prompt_tokens {prompt_tokens}, images {image_tokens}, "
+                f"text chars {len(rendered)}]"
+            )
+        return ""
+
+    def _append_context_message(
+        self, messages: list[dict[str, Any]], message: dict[str, Any]
+    ) -> None:
+        """Append and count new context once, before any history can be evicted.
+
+        Use the context estimator for all roles, including reasoning and image
+        tokens. Subtract the empty request envelope; static system/tool schema
+        tokens and re-sent history are not new context. The summary exchange
+        itself bypasses this helper and starts a fresh interval.
+        """
+        messages.append(message)
+        if _summary_interval_tokens() <= 0:
+            return
+        added = max(
+            0,
+            self._estimate_request_input_tokens([message])
+            - self._estimate_request_input_tokens([]),
+        )
+        self._summary_context_tokens_since_attempt = (
+            getattr(self, "_summary_context_tokens_since_attempt", 0) + added
+        )
+
+    def _maybe_append_rolling_summary(
+        self, append_transcript, tools: list[dict[str, Any]] | None,
+        *, request_timeout_seconds: float | None = None,
+    ) -> None:
+        """Attempt a summary after another interval of newly added context.
+
+        Called at turn start, which is the only clean boundary: nothing is in
+        flight, and a yield resumption arrives here too so no separate handling
+        is needed. Crossing the threshold is not urgent - the summary is not
+        needed until eviction reaches it - so overshooting by a turn costs
+        nothing and interrupting an investigation would.
+
+        The request carries the current history unchanged, so it is a
+        continuation rather than a prefix: both the KV prefix and the recurrent
+        state stay warm. Both messages are tagged, and the trimmer's
+        summary-aware drain navigates by that tag.
+        """
+        interval = _summary_interval_tokens()
+        if interval <= 0:
+            return
+        history = self._history_messages
+        if not history:
+            return
+
+        since = getattr(self, "_summary_context_tokens_since_attempt", 0)
+        if since < interval:
+            return
+        # before the instruction: context first, then the ask, so the ask is
+        # still the last thing read. Stored exactly as sent - a request message
+        # that differs from the one in history would diverge from the cached
+        # prefix at that point and cost a prefill on every later turn.
+        context_lines = self._summary_turn_context_lines() if _summary_turn_context() else []
+        request_text = (
+            "\n".join([*context_lines, "", SUMMARY_REQUEST_PROMPT])
+            if context_lines
+            else SUMMARY_REQUEST_PROMPT
+        )
+        request = _mark_control_message(
+            {"role": "user", "content": request_text},
+            _SUMMARY_CONTROL_KIND,
+        )
+        messages = [
+            {"role": "system", "content": self._system_prompt},
+            *history,
+            request,
+        ]
+        # Every attempt starts a new interval, including HTTP errors, empty
+        # replies and rejected tool calls. Reset before dispatch so all exit
+        # paths share that behavior; the summary exchange is excluded from this clock.
+        self._summary_context_tokens_since_attempt = 0
+        try:
+            result = self._chat_completion(
+                messages,
+                tools=tools,
+                max_tokens=_summary_max_gen_tokens(self._max_output_tokens),
+                enable_thinking=_summary_enable_thinking(),
+                request_timeout_seconds=request_timeout_seconds,
+            )
+        except BaseException as exc:  # a failed summary must not end the turn
+            log.warning("rolling summary request failed: %s", exc)
+            return
+        # before any rejection path: the tokens were generated whatever the
+        # reply turned out to be, and a summary that is thrown away still cost
+        # the time
+        self._accumulate_usage_tokens(
+            getattr(result, "usage", None), count_toward_turn=False
+        )
+        dropped_call = ""
+        message = result.message or {}
+        text = _normalize_message_content(message.get("content", "")).strip()
+        if message.get("tool_calls"):
+            # A reply that calls a tool may still have written a real summary
+            # first - the instruction not to act is advisory, and the model
+            # sometimes summarises and then reaches for the tool anyway. Keeping
+            # a usable one is better than asking again for something it has
+            # already produced, so the text is judged on its length: real
+            # summaries here run 4,000 to 6,000 characters, while a preamble
+            # introducing a tool call runs to tens.
+            #
+            # The tool call itself is dropped either way. It was never going to
+            # execute: this request is outside the turn loop, with no sandbox
+            # and no action budget attached.
+            names = ", ".join(
+                str((c.get("function") or {}).get("name", "?"))
+                for c in message.get("tool_calls") or []
+                if isinstance(c, dict)
+            )
+            if len(text) < _SUMMARY_MIN_USABLE_CHARS:
+                log.warning(
+                    "rolling summary tried to call %s with only %d chars of text; "
+                    "skipping",
+                    names or "a tool", len(text),
+                )
+                if text:
+                    append_transcript(
+                        "ROLLING SUMMARY REJECTED",
+                        f"[tried to call {names}, text too short to use]\n{text}",
+                    )
+                return
+            log.warning(
+                "rolling summary tried to call %s but wrote %d chars first; "
+                "keeping the text, dropping the call",
+                names or "a tool", len(text),
+            )
+            dropped_call = names or "a tool"
+        if not text:
+            # A reply that deliberated to the output limit without reaching the
+            # content wants a different fix from one that simply said nothing -
+            # a larger max_output or a shorter interval - so the two are logged
+            # apart rather than both as "empty".
+            reasoning_chars = len(_extract_reasoning_text(message))
+            if reasoning_chars:
+                log.warning(
+                    "rolling summary produced %d chars of reasoning and no summary "
+                    "(finish_reason=%s); skipping",
+                    reasoning_chars,
+                    str(getattr(result, "finish_reason", "") or "?"),
+                )
+            else:
+                log.warning("rolling summary came back empty; skipping")
+            return
+        truncated = str(getattr(result, "finish_reason", "")).strip() == "length"
+        if truncated:
+            # Kept rather than rejected: a summary cut at the output limit still
+            # covers the early material, and the alternative is no summary at
+            # all, which means the same span is evicted with nothing standing in
+            # for it. But the tail is where "what I was in the middle of" lives,
+            # so the model is told the text is incomplete rather than being left
+            # to infer it from a sentence that stops.
+            log.warning(
+                "rolling summary hit the output limit at %d chars of summary, after "
+                "%d chars of reasoning; keeping it, marked as incomplete",
+                len(text),
+                len(_extract_reasoning_text(message)),
+            )
+            text = f"{text}\n[This summary was cut off at the output limit and is incomplete.]"
+        reply = _mark_control_message(
+            {"role": "assistant", "content": text}, _SUMMARY_CONTROL_KIND
+        )
+        if _summary_replaces_history():
+            # only now, with a usable summary in hand - a failed or rejected one
+            # returned above and left history untouched
+            replaced_count = len(self._history_messages)
+            self._history_messages = [request, reply]
+            # this discards history without going through the trimmer, so the
+            # prefix is invalid and the flag has to be set by hand - otherwise
+            # the next request keeps a priority that assumes a warm cache
+            self._note_history_evicted()
+        else:
+            self._history_messages.append(request)
+            self._history_messages.append(reply)
+            replaced_count = 0
+        reasoning_chars = len(_extract_reasoning_text(message))
+        append_transcript(
+            "ANALYZER STATUS",
+            f"rolling_summary: {len(text)} chars of summary after "
+            f"{reasoning_chars} chars of reasoning, {since} new context tokens since the previous attempt"
+            + (" [TRUNCATED]" if truncated else "")
+            + (f" [tool call dropped: {dropped_call}]" if dropped_call else "")
+            + (f" [replaced {replaced_count} messages of history]"
+               if replaced_count else ""),
+        )
+        # the text itself, so a run can be read back and judged rather than only
+        # counted - this is the artefact the whole mechanism produces. Salvaged
+        # ones get their own section name so they can be grepped and read apart
+        # from the clean ones, which is the comparison that says whether a reply
+        # that also reached for a tool wrote a worse summary.
+        append_transcript(
+            "ROLLING SUMMARY SALVAGED" if dropped_call else "ROLLING SUMMARY",
+            text,
+        )
+
     def _estimate_request_input_tokens(
         self,
         messages: list[dict[str, Any]],
         *,
         tools: list[dict[str, Any]] | None = None,
     ) -> int:
+        messages = _apply_summary_visibility(
+            messages, evicted=getattr(self, "_has_evicted", False)
+        )
+        image_tokens = 0
+        if _image_token_estimate_enabled():
+            messages, image_tokens = _split_messages_for_estimate(messages)
         payload: dict[str, Any] = {"messages": messages}
         if tools:
             payload["tools"] = tools
             payload["tool_choice"] = _request_tool_choice(tools)
-        return _estimate_tokens(payload)
+        return _estimate_tokens(payload, self._text_chars_per_token) + image_tokens
 
     def _drop_oldest_history_block(self, history: list[dict[str, Any]], *, preserve_recent: int) -> bool:
         removable = len(history) - preserve_recent
         if removable <= 0:
             return False
+        # Never drop the last user message. Dropping one and draining to the
+        # next is fine while another exists; when none does, the drain runs to
+        # preserve_recent and _drop_until_first_user_message then empties the
+        # history outright - the request becomes the system prompt alone and the
+        # server answers "No user query found in messages".
+        #
+        # Reachable mid-turn, where the list ends with a tool result rather than
+        # the opener, so preserve_recent protects the tool message and the
+        # opener is the first thing popped. Seen in a live request log as a
+        # 1-message request at request_index_within_turn=2. Pre-existing and
+        # independent of the summary drain, which only makes it easier to reach
+        # by evicting further in one step.
+        if not any(str(m.get("role", "")).strip() == "user" for m in history[1:]):
+            return False
         first = history.pop(0)
         first_role = str(first.get("role", "")).strip()
         if first_role in {"assistant", "tool"}:
@@ -1617,6 +6245,13 @@ class ToolAgent:
             return True
         while history and history[0].get("role") == "tool" and len(history) > preserve_recent:
             history.pop(0)
+        # Only navigate by summaries when there is one left to land on, and only
+        # one that is a REQUEST (a user message) and sits within the removable
+        # range. Checking merely that a summary exists somewhere let the loop
+        # skip every remaining user message when the last one was already at the
+        # head, stripping history down to preserve_recent with no user message
+        # in it at all - which the chat template rejects outright:
+        # "No user query found in messages", HTTP 400, turn lost.
         while history and history[0].get("role") != "user" and len(history) > preserve_recent:
             history.pop(0)
         return True
@@ -1626,9 +6261,28 @@ class ToolAgent:
         messages: list[dict[str, Any]],
         *,
         max_turns: int,
+        drain_turns: int = 0,
+        force_drain: bool = False,
     ) -> list[dict[str, Any]]:
-        if max_turns <= 0 or not messages:
+        if not messages:
             return []
+        if max_turns <= 0:
+            # explicit opt-out: leave the token budget as the only limit
+            return list(messages)
+
+        drain_turns = min(max(0, drain_turns), max(0, max_turns - 1))
+        if drain_turns:
+            present = sum(
+                1
+                for message in messages
+                if str(message.get("role", "")).strip() == "assistant"
+            )
+            # trigger at the cap unless this commit has already invalidated the
+            # prefix for token reasons, in which case take the deep cut now
+            if present > max_turns or force_drain:
+                max_turns = max_turns - drain_turns
+            elif not force_drain:
+                return list(messages)
 
         kept_reversed: list[dict[str, Any]] = []
         assistant_turns = 0
@@ -1650,14 +6304,34 @@ class ToolAgent:
             trimmed.pop(0)
         return trimmed
 
+    def _note_history_evicted(self) -> None:
+        # The scheduler consumes the first flag at the next handover. The
+        # second stays set so summaries remain visible after history is lost.
+        self._context_was_trimmed = True
+        self._has_evicted = True
+
     def _persistent_history_messages(self, messages: list[dict[str, Any]], *, tools: list[dict[str, Any]] | None = None) -> list[dict[str, Any]]:
+        original_history = messages[1:]
+        if _any_control_pruning_enabled():
+            messages = _prune_control_messages(messages)
+        keep_images = _history_image_keep()
+        if keep_images is not None:
+            # after pruning, so a dropped resumption opener cannot consume a
+            # retention slot
+            messages = _apply_history_image_window(messages, keep_images)
+        before_trim = len(messages)
         trimmed = self._trim_messages_for_context(messages, tools=tools)
         if not trimmed:
+            if original_history:
+                self._note_history_evicted()
             return []
+        token_trim_dropped = len(trimmed) < before_trim
         trimmed_history = trimmed[1:]
         history = self._keep_recent_history_turns(
             trimmed_history,
-            max_turns=_PERSISTENT_HISTORY_ASSISTANT_TURNS,
+            max_turns=_persistent_history_assistant_turns(),
+            drain_turns=_history_turn_drain(),
+            force_drain=token_trim_dropped and _history_drain_coalesce(),
         )
         if (
             history
@@ -1667,7 +6341,115 @@ class ToolAgent:
             previous_message = trimmed_history[len(trimmed_history) - len(history) - 1]
             if str(previous_message.get("role", "")).strip() == "user":
                 history = [previous_message, *history]
-        return self._drop_until_first_user_message(history)
+        history = self._drop_until_first_user_message(history)
+        # Include turn-cap eviction, control pruning and image removal, which
+        # can change the cached prefix without the token trimmer dropping any
+        # messages. Image removal can leave the message count unchanged.
+        if history != original_history:
+            self._note_history_evicted()
+        return history
+
+    def _priority_total_levels(self) -> int:
+        """Read public game metadata through the existing session callback.
+
+        No new solver state or prompt fields. A custom callback, old object,
+        or unavailable metadata can fall back without interrupting a game.
+        """
+        try:
+            callback = getattr(self, "_step_env_callback", None)
+            session = getattr(callback, "__self__", None)
+            game = getattr(session, "game", None)
+            count = parse_total_levels(getattr(game, "number_of_levels", None))
+        except Exception:
+            count = None
+        if count is not None:
+            return count
+        if not getattr(self, "_priority_level_count_warned", False):
+            self._priority_level_count_warned = True
+            log.warning("priority: total level count unavailable or invalid; using 10")
+        return 10
+
+    def _maybe_handover(self) -> None:
+        """Re-price this game, and yield the slot if a gate is running.
+
+        Called right after the trimmer and before the request goes out, which is
+        where the prefix has just been invalidated. Consumes the trim flag: it
+        describes the trim that produced THIS request, and leaving it set would
+        keep the game demoted long after the prefix had been rebuilt.
+        """
+        if not self._context_was_trimmed:
+            return
+        self._context_was_trimmed = False
+        summary = self._last_step_summary or {}
+        level = int(summary.get("level") or 1)
+        actions = max(
+            0, _priority_action_count(summary) - self._actions_at_level_start
+        )
+        # getattr: an agent unpickled from before this field existed would
+        # otherwise raise here on every turn
+        tokens = max(
+            0,
+            self._session_generated_tokens
+            - getattr(self, "_tokens_at_level_start", 0),
+        )
+        gate = _priority_gate()
+        snapshot = None
+        options = _priority_level_options()
+        total_levels = (
+            self._priority_total_levels()
+            if options["normalize_score"] or options["tail_lookup"] else None
+        )
+        if gate is not None and _priority_refresh_enabled():
+            pace = self._progress_pace.cost_multiplier() if _get_env_bool("ARC3_PRIORITY_PACE", False) else 1.0
+            snapshot = PrioritySnapshot(level, actions, tokens, pace, total_levels)
+            estimate = gate._snapshot_priority(snapshot, time.monotonic())
+        else:
+            estimate = _game_priority(
+                level, actions, tokens, endgame=self._in_endgame(),
+                total_levels=total_levels,
+            )
+        # this request pays the prefill, so it drops to the bottom band; the one
+        # after it, with the prefix rebuilt, returns to the warm band
+        self._priority_current = estimate
+        self._priority_next = estimate + _PRIORITY_BAND
+        if gate is not None:
+            _diag(self, "gate")
+            gate.handover(estimate, snapshot)
+
+    def _in_endgame(self) -> bool:
+        """Whether this game has been running long enough to switch phase.
+
+        Read from the session's run_elapsed_seconds rather than a clock of its
+        own: that is the timer max_runtime_s_per_game uses to end the game, so
+        the two cannot disagree about how old a game is.
+        """
+        minutes = _endgame_start_minutes()
+        if minutes <= 0:
+            return False
+        elapsed = getattr(self, "_game_elapsed_seconds", None)
+        if elapsed is None:
+            return False          # no step has reported the clock yet
+        if elapsed < minutes * 60.0:
+            return False
+        if not getattr(self, "_endgame_announced", False):
+            self._endgame_announced = True
+            log.warning(
+                "endgame phase after %.0f minutes: ranking on what completing "
+                "the current level would bank, with no tail and no hazard",
+                elapsed / 60.0,
+            )
+        return True
+
+    def _note_request_succeeded(self) -> None:
+        """Promote to the warm band once a request has actually landed.
+
+        Called on a completed response rather than on dispatch: a retry after a
+        connection failure has rebuilt nothing, so the game stays in the bottom
+        band until one lands.
+        """
+        nxt = getattr(self, "_priority_next", None)
+        if nxt is not None:
+            self._priority_current = nxt
 
     def _trim_messages_for_context(
         self,
@@ -1676,6 +6458,7 @@ class ToolAgent:
         tools: list[dict[str, Any]] | None = None,
         preserve_recent: int = 1,
         extra_safety_tokens: int = 0,
+        force_drain: bool = False,
     ) -> list[dict[str, Any]]:
         if not messages:
             return []
@@ -1683,10 +6466,83 @@ class ToolAgent:
         history = list(messages[1:])
         preserve_recent = max(0, preserve_recent)
         budget_tokens = max(1, self._context_budget_tokens - max(0, extra_safety_tokens))
-        while history and self._estimate_request_input_tokens([system_message, *history], tools=tools) > budget_tokens:
+        # capped at half the budget: a mis-set knob should cost history depth,
+        # not erase history
+        drain_tokens = min(_context_drain_tokens(), budget_tokens // 2)
+        target_tokens = max(1, budget_tokens - drain_tokens) if drain_tokens else budget_tokens
+        floor_mode = bool(drain_tokens) and _context_drain_floor_mode()
+        # Normally eviction starts only once the estimate exceeds the budget.
+        # After the server has rejected a request the estimate is known to be
+        # wrong, so waiting for it to admit the request is too big would drop
+        # nothing at all; force_drain enters the loop already draining and lets
+        # the existing target - budget minus one drain, computed against the
+        # real budget - decide where to stop.
+        draining = bool(force_drain and drain_tokens)
+        def _head_is_summary() -> bool:
+            """A summary REQUEST at the head, not its reply.
+
+            Landing on the assistant reply leaves history starting with an
+            assistant message, which _drop_until_first_user_message then strips -
+            so the exit would drop the very thing it stopped for.
+            """
+            return bool(
+                history
+                and _message_is_summary(history[0])
+                and str(history[0].get("role", "")).strip() == "user"
+            )
+
+        # `draining` means two different things. On the normal path it is the
+        # hysteresis phase - we crossed the budget and are heading for the
+        # target - and stopping early there is exactly the point. Under
+        # force_drain it means the server rejected the request and the estimate
+        # cannot be trusted, so a summary-shaped head says nothing about whether
+        # the request now fits; stopping early there risks another rejection and
+        # another round trip.
+        def _stop_early(used_tokens: int) -> bool:
+            return (
+                _drain_stop_at_summaries()
+                and draining
+                and not force_drain
+                and used_tokens <= budget_tokens
+                and _head_is_summary()
+            )
+
+        while history:
+            used = self._estimate_request_input_tokens([system_message, *history], tools=tools)
+            over_budget = used > budget_tokens
+            if not over_budget and not draining:
+                break
+            if floor_mode and not over_budget:
+                # the mandatory phase is done; keep draining only while the NEXT
+                # drop would still leave us at or above the target. Looking
+                # ahead rather than dropping and checking is what puts a real
+                # floor under the retained history.
+                if _stop_early(used):
+                    break
+                probe = list(history)
+                if not self._drop_oldest_history_block(probe, preserve_recent=preserve_recent):
+                    break
+                after = self._estimate_request_input_tokens(
+                    [system_message, *probe], tools=tools
+                )
+                if after < target_tokens:
+                    break
+                history[:] = probe
+                continue
+            if not floor_mode and used <= (target_tokens if draining else budget_tokens):
+                break
+            if _stop_early(used):
+                break
+            draining = True
             if not self._drop_oldest_history_block(history, preserve_recent=preserve_recent):
                 break
         history = self._drop_until_first_user_message(history)
+        # A dropped message invalidates the server's prefix for this
+        # conversation, so the next request has to be prefilled from the
+        # divergence point regardless. That makes it the one moment when
+        # yielding the slot to another game costs nothing.
+        if len(history) != len(messages) - 1:
+            self._note_history_evicted()
         return [system_message, *history]
 
     def _force_reduce_messages(
@@ -1701,6 +6557,7 @@ class ToolAgent:
         history = list(messages[1:])
         if not self._drop_oldest_history_block(history, preserve_recent=max(0, preserve_recent)):
             return list(messages)
+        self._note_history_evicted()
         return [system_message, *history]
 
     def analyze(
@@ -1719,6 +6576,10 @@ class ToolAgent:
             return None
         self._ensure_session(state_path)
         self._step_env_callback = step_env
+        if self._noop_repeat_guard is not None:
+            solver_obj = getattr(step_env, "__self__", None)
+            if solver_obj is not None:
+                solver_obj.action_guard_hook = self._action_guard_hook
         self._current_valid_actions = _normalize_valid_actions(valid_actions)
 
         analyzer_log = transcript_path or (state_path.parent / f"{state_path.stem}_analyzer.txt")
@@ -1732,6 +6593,7 @@ class ToolAgent:
             previous_step_summary=self._last_step_summary,
         )
         display_action_num = _display_action_number(action_num)
+        self._wm_absorbed_in_turn = False
 
         with open(analyzer_log, "a", encoding="utf-8") as f:
             step_label = f"analysis_step={analysis_step} | " if analysis_step is not None else ""
@@ -1749,22 +6611,108 @@ class ToolAgent:
                 transcript_updated("".join(transcript_parts))
 
         append_transcript("SYSTEM PROMPT", self._system_prompt)
-        append_transcript("USER PROMPT", user_prompt)
 
         previous_history_messages = list(self._history_messages)
+        # Mirrors the history snapshot: a turn abandoned on a request failure is
+        # rolled back, and the per-game token allowance has to roll back with it
+        # or the limit stops measuring the game and starts measuring how flaky
+        # the endpoint was. The tokens were really generated - the server spent
+        # them - but the point of the limit is that the same game gets the same
+        # budget whatever the infrastructure did.
+        tokens_at_turn_start = self._session_generated_tokens
+        # The tools go with it, even though the summary must not call one. The
+        # template renders their definitions into a system block near the start
+        # of the prompt, so a request without them diverges from the first
+        # tokens and prefills the whole context: measured at 15% shared prefix
+        # against 99% with them, and as prefix reuse falling from 75% to 45%
+        # across a run. The instruction not to act, and the harness rejecting a
+        # reply that calls a tool, are what keep it from acting - not withholding
+        # the definitions.
+        self._maybe_append_rolling_summary(
+            append_transcript, self._tools(state_path),
+            request_timeout_seconds=request_timeout_seconds,
+        )
         preserve_history = True
-        messages: list[dict[str, Any]] = self._trim_messages_for_context(
-            [{"role": "system", "content": self._system_prompt}, *self._history_messages, self._build_user_message(user_prompt, current_frame)],
+        resuming_after_yield = bool(getattr(self, "_resume_after_yield", False))
+        resume_reason = str(getattr(self, "_resume_reason", "") or "yield_bare")
+        self._resume_after_yield = False
+        self._resume_reason = ""
+        if not resuming_after_yield and _persistent_functions_scope() == "turn":
+            # A resumption is the same turn, so helpers survive it either way.
+            if _persistent_functions() and getattr(self, "_kept_functions", None):
+                self._retained_clear_notice = (
+                    "Your retained functions were cleared at the start of this new analyzer turn; "
+                    "redefine any you still need."
+                )
+            self._kept_functions = {}
+        resume_mode = _yield_resume_prompt_mode() if resuming_after_yield else "full"
+        # state_only always carries the board - that IS the mode - so the
+        # image knob does not apply to it
+        resume_frame = (
+            current_frame
+            if not resuming_after_yield
+            or resume_mode == "state_only"
+            or _resume_prompt_images_enabled()
+            else None
+        )
+        if (
+            resuming_after_yield
+            and resume_mode in ("state_only", "short")
+            and _resume_diff_image_enabled()
+            and self._turn_diff_image is not None
+        ):
+            # _build_user_message consumes the pending slot, so re-arm it from
+            # the turn cache for this one message
+            self._pending_diff_image = self._turn_diff_image
+            self._pending_diff_image_label = self._turn_diff_image_label
+        if resume_mode == "state_only":
+            opening_message = self._build_user_message(
+                _yield_resume_prompt(resume_reason) + "\n\n" + _STATE_ONLY_CAPTION,
+                current_frame,
+            )
+        elif resume_mode == "short":
+            opening_message = self._build_user_message(
+                _yield_resume_prompt(resume_reason), resume_frame
+            )
+        else:
+            opening_message = self._build_user_message(
+                user_prompt, resume_frame if resuming_after_yield else current_frame
+            )
+        # log what was actually SENT, not the argument that was passed in: on a
+        # resumed invocation the short/state_only substitution happens here, so
+        # transcribing `user_prompt` would report a full opener the model never
+        # saw - and at a 60s yield budget that is most turns
+        append_transcript("USER PROMPT", _message_display_text(opening_message))
+        if resuming_after_yield:
+            # Tag EVERY resumption message, not just the short form. A yield
+            # leaves this turn's original opener in history, so the message
+            # appended here is a duplicate restatement of unchanged state
+            # (same step, level, board, diffs, ledger - a no-execution yield
+            # changes nothing). It is needed for the request that resumes the
+            # turn and is pure duplication afterwards, which is precisely what
+            # ARC3_PRUNE_CONTROL_CONTEXT exists to drop.
+            _mark_control_message(opening_message, "resume")
+        messages: list[dict[str, Any]] = [
+            {"role": "system", "content": self._system_prompt}, *self._history_messages,
+        ]
+        self._append_context_message(messages, opening_message)
+        messages = self._trim_messages_for_context(
+            messages,
             tools=self._tools(state_path),
             preserve_recent=1,
         )
         step_executed = False
+        turn_tool_call_count = 0
         captured_reasoning = ""
         latest_request_messages: list[dict[str, Any]] | None = None
         latest_request_tools: list[dict[str, Any]] | None = None
         latest_request_tool_choice: str | None = None
         latest_request_index = 0
         turn_started_at = time.monotonic()
+        # per TURN, not per session: the budget is how much the model may
+        # generate before the turn ends, and a resumption continues the same
+        # turn, so this resets where the clock does
+        self._turn_generated_tokens = 0
         yielded_control_reason: str | None = None
 
         def control_yield_reason() -> str | None:
@@ -1776,6 +6724,11 @@ class ToolAgent:
                     log.warning("analyzer stop check failed at action %d: %s", display_action_num, exc)
             if self._yield_seconds is not None and (time.monotonic() - turn_started_at) >= self._yield_seconds:
                 return "turn_time_budget"
+            if (
+                self._yield_tokens is not None
+                and self._turn_generated_tokens >= self._yield_tokens
+            ):
+                return "turn_token_budget"
             return None
 
         try:
@@ -1788,6 +6741,7 @@ class ToolAgent:
                 tools = self._tools(state_path)
                 tool_choice = _request_tool_choice(tools)
                 messages = self._trim_messages_for_context(messages, tools=tools)
+                self._maybe_handover()
                 latest_request_messages = json.loads(json.dumps(messages))
                 latest_request_tools = json.loads(json.dumps(tools))
                 latest_request_tool_choice = tool_choice
@@ -1818,9 +6772,17 @@ class ToolAgent:
                             analysis_step=analysis_step,
                             action=display_action_num,
                             request_index_within_turn=latest_request_index,
+                            chat_template_kwargs=self._harness_template_kwargs(),
                         )
                     result = self._chat_completion(messages, **request_kwargs)
+                    _calibration_note = self._calibrate_from_usage(
+                        messages, request_kwargs.get("tools"), result.usage
+                    )
+                    if _calibration_note:
+                        append_transcript("ANALYZER STATUS", _calibration_note)
                     self._accumulate_usage_tokens(result.usage)
+                    self._note_request_succeeded()
+                    _diag(self, "local")
                     if self._save_request_logs:
                         _append_request_snapshot(
                             _resolve_request_log_path(state_path),
@@ -1832,14 +6794,25 @@ class ToolAgent:
                             action=display_action_num,
                             request_index_within_turn=latest_request_index,
                             finish_reason=result.finish_reason,
+                            served_by=result.served_by,
+                            usage=result.usage,
+                            chat_template_kwargs=self._harness_template_kwargs(),
                         )
                 except requests.RequestException as exc:
                     if not _is_context_length_error(exc):
                         raise
+                    # A rejected request means the estimate was wrong, and 512
+                    # tokens only helps when it was wrong by less than that -
+                    # otherwise nothing is dropped and the fallback removes one
+                    # block per rejected round trip. force_drain evicts down to
+                    # the drain target instead, in one step, which is what the
+                    # drain is for: land clear of the ceiling rather than just
+                    # under it.
                     trimmed_messages = self._trim_messages_for_context(
                         messages,
                         tools=tools,
                         extra_safety_tokens=_CONTEXT_OVERFLOW_RETRY_TRIM_TOKENS,
+                        force_drain=True,
                     )
                     if trimmed_messages == messages:
                         trimmed_messages = self._force_reduce_messages(messages)
@@ -1851,6 +6824,21 @@ class ToolAgent:
                     )
                     messages = trimmed_messages
                     continue
+                truncated_tool_call = False
+                if str(result.finish_reason or "").strip() == "length":
+                    ladder = _reasoning_effort_ladder()
+                    if ladder and self._reasoning_effort_rung < len(ladder) - 1:
+                        self._reasoning_effort_rung += 1
+                        log.warning(
+                            "generation hit the output ceiling; reducing reasoning_effort to %r",
+                            ladder[self._reasoning_effort_rung],
+                        )
+                        append_transcript(
+                            "ANALYZER STATUS",
+                            "reasoning_effort_reduced: generation was truncated at the output "
+                            f"limit; subsequent requests use reasoning_effort="
+                            f"{ladder[self._reasoning_effort_rung]!r} until an action executes.",
+                        )
                 raw_reasoning = _extract_reasoning_text(result.message)
                 raw_content = _normalize_message_content(result.message.get("content", ""))
                 tool_calls = json.loads(json.dumps(result.message.get("tool_calls") or []))
@@ -1871,8 +6859,35 @@ class ToolAgent:
                             json.loads(raw_arguments)
                         except json.JSONDecodeError as exc:
                             malformed_argument_errors.append(f"{tool_name}: invalid JSON arguments ({exc})")
+                if malformed_argument_errors:
+                    # Drop ALL calls from this reply rather than the bad one.
+                    #
+                    # An unparseable arguments string means the reply was cut
+                    # off mid-call (finish_reason=length is the usual cause),
+                    # so anything after the truncation point is missing and an
+                    # earlier "valid" call in the same message expresses only
+                    # part of an intent - dispatching half a plan is worse than
+                    # dispatching none.
+                    #
+                    # Critically, the message must also never reach history:
+                    # servers re-parse tool calls in the incoming message list
+                    # when rendering the chat template, so one malformed call
+                    # committed to history makes EVERY later request fail at
+                    # render time, before inference - an unbreakable loop that
+                    # the HTTP retry cannot help with. Content and reasoning
+                    # are kept; only the calls are dropped, which routes this
+                    # into the existing no-tool-call path.
+                    log.warning(
+                        "dropping %d malformed tool call(s) (finish_reason=%s): %s",
+                        len(tool_calls),
+                        result.finish_reason or "?",
+                        "; ".join(malformed_argument_errors),
+                    )
+                    tool_calls = []
+                    truncated_tool_call = str(result.finish_reason or "").strip() == "length"
                 response_meta = _format_model_response_meta(
                     finish_reason=result.finish_reason,
+                    served_by=result.served_by,
                     reasoning=reasoning,
                     content=content,
                     tool_calls=tool_calls,
@@ -1889,7 +6904,8 @@ class ToolAgent:
                 if reasoning:
                     captured_reasoning = reasoning
                     append_transcript("THINKING", reasoning)
-                    assistant_message["reasoning"] = reasoning
+                    for _reasoning_key in _reasoning_history_keys():
+                        assistant_message[_reasoning_key] = reasoning
 
                 if not tool_calls:
                     if content:
@@ -1900,11 +6916,87 @@ class ToolAgent:
                         assistant_message["content"] = None
 
                     if content or reasoning:
-                        messages.append(assistant_message)
+                        if (
+                            not content
+                            and reasoning
+                            and _is_degenerate_text(reasoning)
+                        ):
+                            # A reasoning-only reply that is one character
+                            # repeated to the output cap is a serving fault, not
+                            # thinking: observed as exactly 6144 "!" from the
+                            # first token, with content_chars 0 and no tool
+                            # call, under speculative decoding. Keeping it cost
+                            # far more than the lost turn - 12k characters of
+                            # noise pushed the trimmer from 31 history messages
+                            # to 12, evicting real history, and the model then
+                            # concluded "my replies keep failing to emit the
+                            # tool call" and started writing shorter answers to
+                            # fix a formatting error it had not made.
+                            # transcript AND log: the transcript records it
+                            # where it happened, but a serving fault should be
+                            # visible while the run is going rather than only
+                            # on a later grep
+                            log.warning(
+                                "degenerate reasoning dropped: %d chars, no content "
+                                "and no tool call - the reply was a repetition loop. "
+                                "This is a serving fault, not the model reasoning.",
+                                len(reasoning),
+                            )
+                            append_transcript(
+                                "ANALYZER STATUS",
+                                "degenerate_reasoning_dropped: the reply was a "
+                                f"repetition loop ({len(reasoning)} chars) with no "
+                                "content or tool call, and was replaced in context.",
+                            )
+                            assistant_message = _mark_control_message(
+                                {"role": "assistant", "content": _DEGENERATE_CONTENT_STUB},
+                                "stub",
+                            )
+                        elif content and _is_degenerate_text(content):
+                            # a repetition loop feeds on its own output, so the
+                            # garbage must not reach the next request's context
+                            append_transcript(
+                                "ANALYZER STATUS",
+                                "degenerate_content_stubbed: assistant text was highly "
+                                "repetitive and was replaced in context.",
+                            )
+                            assistant_message = {
+                                "role": "assistant",
+                                "content": _DEGENERATE_CONTENT_STUB,
+                            }
+                        elif (
+                            not content
+                            and reasoning
+                            and _stub_dead_reasoning_enabled()
+                        ):
+                            # reasoning-only reply: nothing to continue from,
+                            # and re-sending it biases the retry toward the
+                            # same dead end. Transcript keeps the full text.
+                            assistant_message = _mark_control_message(
+                                {"role": "assistant", "content": _DEAD_REASONING_STUB},
+                                "stub",
+                            )
+                        self._append_context_message(messages, assistant_message)
                     yielded_control_reason = control_yield_reason()
                     if yielded_control_reason is not None:
                         break
                     followup_prefix = "You have not acted yet. Investigate first. "
+                    if truncated_tool_call:
+                        # the only signal the model gets that LENGTH, not
+                        # formatting, is why its call vanished
+                        followup_prefix = (
+                            "Your previous reply was CUT OFF before the tool call finished, so the "
+                            "call was incomplete and could not be executed. Keep your reasoning "
+                            "much shorter this time and emit the `python` tool call early enough "
+                            "that it completes within the output limit. Prefer one small snippet "
+                            "over a long analysis. "
+                        )
+                    elif malformed_argument_errors:
+                        followup_prefix = (
+                            "Your previous tool call had arguments that were not valid JSON, so it "
+                            "was not executed. Emit exactly one `python` tool call whose arguments "
+                            "are a well-formed JSON object. "
+                        )
                     if tool_call_markup_in_text:
                         followup_prefix = (
                             "You did not call a tool. We detected `<tool_call>` markup inside your reasoning or assistant text, "
@@ -1912,10 +7004,31 @@ class ToolAgent:
                             "Emit exactly one `python` tool call directly as your next response. "
                             "Do not place `<tool_call>` markup inside reasoning, explanation, or notes. "
                         )
+                    # the clause names where transient facts SHOULD go, so it
+                    # holds only while that section is carried; a mode that
+                    # drops it would be pointing the model nowhere
+                    _transient_clause = (
+                        " \u2014 current positions and this turn's events go in "
+                        "`Recent findings:`. "
+                        if "Recent findings" in _advertised_section_labels()
+                        # "only" wants something to contrast against, and the
+                        # clause that supplied it names a section this mode has
+                        # dropped; keep the contrast, drop the destination
+                        else ", not current positions or what just happened. "
+                    )
                     followup_prompt = (
                         f"{followup_prefix}"
                         "Then investigate and revise your working world model of what the level contains, what actions appear to do, what the current goal seems to be, and what plan looks best. "
-                        "If helpful, include short world-model update lines such as `World model:`, `Goal model:`, `Action model:`, `Recent findings:`, `Open questions:`, `Plan:`, or `Cross-level notes:`. "
+                        # the same narrowing the opener applies: advertising a section a
+                        # restricted mode drops invites the model to write
+                        # where it will never read it back
+                        f"If helpful, include memory update lines such as "
+                        f"{_section_list_phrase(_advertised_section_labels(), 'or')}. "
+                        "`World model:` holds durable environment rules"
+                        + (" only" if "Recent findings" in _advertised_section_labels() else "")
+                        + f"{_transient_clause}"
+                        "Each section you write replaces the stored one entirely, "
+                        "so restate what is still true, not only what is new. "
                         "Call the `python` tool with code that inspects `current_frame`, `previous_frame`, `last_transition`, `history`, or `valid_actions` -- use `current_frame.segmentation` as the primary view, and `.ascii` only for a small specific region -- "
                         "compare `previous_frame` to `current_frame` for the most recent change, "
                         "derives a compact board summary, programs a small search or scorer over candidate actions or short sequences, "
@@ -1923,7 +7036,12 @@ class ToolAgent:
                         f"{TOOL_CALL_FORMAT_GUIDANCE}"
                     )
                     append_transcript("USER PROMPT", followup_prompt)
-                    messages.append({"role": "user", "content": followup_prompt})
+                    self._append_context_message(
+                        messages,
+                        _mark_control_message(
+                            {"role": "user", "content": followup_prompt}, "nudge"
+                        )
+                    )
                     continue
 
                 if content:
@@ -1931,7 +7049,8 @@ class ToolAgent:
                     append_transcript("ASSISTANT", content)
                     assistant_message["content"] = content
                 assistant_message["tool_calls"] = tool_calls
-                messages.append(assistant_message)
+                turn_tool_call_count += len(tool_calls)
+                self._append_context_message(messages, assistant_message)
 
                 for tool_index, tool_call in enumerate(tool_calls):
                     function = tool_call.get("function", {}) if isinstance(tool_call, dict) else {}
@@ -1954,8 +7073,17 @@ class ToolAgent:
                     dispatch = self._dispatch_tool(state_path, tool_name, arguments)
                     if dispatch.step_executed:
                         step_executed = True
+                        if self._reasoning_effort_rung >= 0 and dispatch.made_progress:
+                            # the turn achieved something, so restore the server
+                            # default; an inert action leaves the reduction in
+                            # place rather than inviting another truncation
+                            log.info(
+                                "action changed the board; restoring default reasoning_effort"
+                            )
+                            self._reasoning_effort_rung = -1
                     append_transcript(f"TOOL RESULT: {tool_name}", _render_tool_result_display(dispatch.content))
-                    messages.append(
+                    self._append_context_message(
+                        messages,
                         {
                             "role": "tool",
                             "tool_call_id": tool_call.get("id", ""),
@@ -1977,8 +7105,44 @@ class ToolAgent:
                     break
 
         except requests.RequestException as exc:
+            if _is_read_timeout(exc) and _yield_on_timeout_enabled():
+                # Keep this turn's completed exchanges and hand control back to
+                # the solver, which re-invokes on the same game state.
+                append_transcript(
+                    "ANALYZER STATUS",
+                    f"request_timeout (yielding, history preserved): {exc}",
+                )
+                log.warning(
+                    "analyzer request timed out at action %d; yielding with %d tool call(s) preserved: %s",
+                    display_action_num,
+                    turn_tool_call_count,
+                    exc,
+                )
+                self._resume_after_yield = True
+                self._resume_reason = "timeout"
+                if latest_request_messages is not None:
+                    _write_prompt_log_snapshot(
+                        prompt_log,
+                        model_id=self._model.model_id,
+                        base_url=self._model.base_url,
+                        display_action_num=display_action_num,
+                        analysis_step=analysis_step,
+                        request_index=latest_request_index,
+                        messages=latest_request_messages,
+                        tools=latest_request_tools,
+                        tool_choice=latest_request_tool_choice,
+                        transcript="".join(transcript_parts),
+                    )
+                # preserve_history stays True, so the finally block commits
+                # this turn's exchanges exactly as a normal yield would.
+                return AnalyzerTurnResult(
+                    step_executed=False,
+                    reasoning=captured_reasoning,
+                    yielded_control=True,
+                )
             append_transcript("ANALYZER STATUS", f"request_error: {exc}")
             preserve_history = False
+            self._session_generated_tokens = tokens_at_turn_start
             if latest_request_messages is not None:
                 _write_prompt_log_snapshot(
                     prompt_log,
@@ -1997,6 +7161,7 @@ class ToolAgent:
         except Exception as exc:
             append_transcript("ANALYZER STATUS", f"error: {exc}")
             preserve_history = False
+            self._session_generated_tokens = tokens_at_turn_start
             if latest_request_messages is not None:
                 _write_prompt_log_snapshot(
                     prompt_log,
@@ -2013,10 +7178,15 @@ class ToolAgent:
             log.warning("analyzer failed at action %d: %s", display_action_num, exc)
             return None
         finally:
+            if step_executed and not self._wm_absorbed_in_turn:
+                self._turns_without_wm_update += 1
             if preserve_history:
                 self._history_messages = self._persistent_history_messages(messages, tools=self._tools(state_path))
             else:
                 self._history_messages = previous_history_messages
+            solver_obj = getattr(self._step_env_callback, "__self__", None)
+            if solver_obj is not None and hasattr(solver_obj, "action_guard_hook"):
+                solver_obj.action_guard_hook = None
             self._step_env_callback = None
             self._current_valid_actions = []
 
@@ -2036,6 +7206,7 @@ class ToolAgent:
             f"request_safety_margin_tokens: {self._request_safety_margin_tokens}\n"
             f"tool_output_tokens: {self._tool_output_tokens}\n"
             f"yield_seconds: {self._yield_seconds if self._yield_seconds is not None else 'disabled'}\n"
+            f"yield_tokens: {self._yield_tokens if self._yield_tokens is not None else 'disabled'}\n"
             f"available_tools: python\n"
             f"python_timeout_seconds: {self._python_timeout}\n"
             f"history_messages: {len(self._history_messages)}\n"
@@ -2056,6 +7227,31 @@ class ToolAgent:
                 tool_choice=latest_request_tool_choice,
                 transcript="".join(transcript_parts),
             )
+        # An invocation that executed nothing will be re-invoked by the solver
+        # on the same game state, so whatever it appends next is a duplicate
+        # restatement. This covers BOTH non-execution exits: the time budget
+        # (yielded_control_reason set) and the tool-step budget (loop condition
+        # went false, no yield reason). Execution and yielding are mutually
+        # exclusive, so `not step_executed` is the whole condition.
+        self._resume_after_yield = not step_executed
+        if yielded_control_reason is not None:
+            self._resume_reason = (
+                "yield_tools" if turn_tool_call_count > 0 else "yield_bare"
+            )
+        else:
+            self._resume_reason = (
+                "tool_steps" if turn_tool_call_count > 0 else "yield_bare"
+            )
+        if not step_executed and turn_tool_call_count == 0 and self._last_step_summary:
+            # no dispatch ran, so _dispatch_tool never marked the summary
+            # stale; without this the next opener re-narrates an older
+            # execution as if it had just happened (or claims nothing has run
+            # yet) directly beneath this turn's own reasoning.
+            self._last_step_summary = {
+                **self._last_step_summary,
+                "stale": True,
+                "stale_reason": "no_tool_call",
+            }
         return AnalyzerTurnResult(
             step_executed=step_executed,
             reasoning=captured_reasoning,
diff --git a/ARC3-Inference/inference/agent/vision_context.py b/ARC3-Inference/inference/agent/vision_context.py
index bb8232f..e3dadff 100644
--- a/ARC3-Inference/inference/agent/vision_context.py
+++ b/ARC3-Inference/inference/agent/vision_context.py
@@ -3,6 +3,7 @@ from __future__ import annotations
 
 import base64
 import io
+from collections import defaultdict
 import os
 from typing import Any
 
@@ -39,6 +40,24 @@ def current_grid_image_enabled() -> bool:
     return multimodal_context() == "current_grid"
 
 
+_RESAMPLE_FILTERS = {
+    "nearest": Image.Resampling.NEAREST,
+    "bilinear": Image.Resampling.BILINEAR,
+    "bicubic": Image.Resampling.BICUBIC,
+    "lanczos": Image.Resampling.LANCZOS,
+}
+
+
+def multimodal_resample() -> Image.Resampling:
+    """Upscale filter for grid images. Nearest neighbor (the default) keeps
+    cells as flat exact-palette blocks; interpolating filters produce softer
+    edges that sit closer to a vision encoder's natural-image training
+    distribution, which has measurably helped on some models. Model-dependent:
+    test against the model actually served."""
+    raw = os.environ.get("MULTIMODAL_RESAMPLE", "").strip().lower()
+    return _RESAMPLE_FILTERS.get(raw, Image.Resampling.NEAREST)
+
+
 def current_grid_image_upscale() -> int:
     raw = os.environ.get("MULTIMODAL_UPSCALE", "").strip()
     if not raw:
@@ -63,7 +82,7 @@ def frame_to_png_data_url(frame: Frame, *, upscale: int | None = None) -> str:
             value = row[col_idx] if col_idx < len(row) else 0
             pixels[col_idx, row_idx] = ARC_COLOR_MAP.get(int(value), ARC_COLOR_MAP[0])
     if scale > 1:
-        image = image.resize((cols * scale, rows * scale), Image.Resampling.NEAREST)
+        image = image.resize((cols * scale, rows * scale), multimodal_resample())
 
     buffer = io.BytesIO()
     image.save(buffer, format="PNG")
@@ -80,3 +99,234 @@ def current_grid_image_part(frame: Frame | None) -> dict[str, Any] | None:
             "url": frame_to_png_data_url(frame),
         },
     }
+
+
+# Off-palette neutral for unchanged cells in diff images. The ARC palette's
+# indices 0-5 span a white-to-black grayscale ramp, so no gray (or black, or
+# white) can be used without colliding with real cell values; a dark
+# desaturated navy is unambiguous.
+DIFF_UNCHANGED_RGB: tuple[int, int, int] = (16, 24, 48)
+
+
+_ANIMATION_IMAGE_MODES = ("off", "peak", "composite")
+
+
+def _animation_image_mode(name: str) -> str:
+    """One of off, peak, composite. Anything else raises.
+
+    These two are the only enum knobs among a set of otherwise boolean ARC3_*
+    variables, so `=1` is the natural thing to write - and silently meant off,
+    which cost a whole run to notice. Failing at startup is better than a
+    feature that is configured and quietly absent.
+    """
+    mode = os.environ.get(name, "off").strip().lower()
+    if mode not in _ANIMATION_IMAGE_MODES:
+        raise ValueError(
+            f"{name}={mode!r} is not valid. Use one of "
+            f"{', '.join(_ANIMATION_IMAGE_MODES)} - this knob is not a boolean."
+        )
+    return mode
+
+
+def animation_death_image_mode() -> str:
+    """off | peak | composite, for the fatal action of a game over.
+
+    Separate from ARC3_ANIMATION_IMAGE because the two fire on different
+    conditions and a game-over turn already carries two images, so a third is
+    a cost worth deciding on its own.
+    """
+    return _animation_image_mode("ARC3_ANIMATION_DEATH_IMAGE")
+
+
+def animation_image_mode() -> str:
+    """off | peak | composite.
+
+    Only ever attached when the action left the board area unchanged, which is
+    exactly when the board image the model receives is uninformative: the
+    action did something, and no frame it can reach shows what.
+    """
+    return _animation_image_mode("ARC3_ANIMATION_IMAGE")
+
+
+def animation_composite_part(
+    chain: list[tuple[tuple[int, ...], ...]],
+    transient_cells: list[tuple[int, int]],
+    *,
+    border: int,
+) -> dict[str, Any] | None:
+    """Every transient cell in the last colour it held before reverting.
+
+    Not a board state: cells from different moments composited onto one grid.
+    The alternative - a single real frame - misses a sweep that touches
+    different regions at different times, which has no peak moment.
+
+    Everything else is dimmed to the diff image's off-palette navy, so the
+    image carries only what no reachable frame contains. That includes cells
+    that changed just once (visible in current_frame anyway) and the border,
+    where the budget bar ticks on every action and would otherwise be the
+    brightest thing in the picture.
+    """
+    if not chain or not transient_cells:
+        return None
+    if not current_grid_image_enabled():
+        return None
+    final = chain[-1]
+    rows = len(final)
+    cols = max((len(row) for row in final), default=0)
+    if rows <= 0 or cols <= 0:
+        return None
+
+    wanted = set(transient_cells)
+    last_seen: dict[tuple[int, int], int] = {}
+    held: dict[tuple[int, int], int] = defaultdict(int)
+    for grid in chain[:-1]:
+        for row_idx, row in enumerate(grid):
+            final_row = final[row_idx] if row_idx < len(final) else ()
+            for col_idx, value in enumerate(row):
+                cell = (row_idx, col_idx)
+                if cell not in wanted:
+                    continue
+                if row_idx < border or row_idx >= rows - border:
+                    continue
+                if col_idx < border or col_idx >= cols - border:
+                    continue
+                final_value = final_row[col_idx] if col_idx < len(final_row) else None
+                if value == final_value:
+                    continue
+                value = int(value)
+                if last_seen.get(cell) != value:
+                    # a cell that flickers between colours restarts the count,
+                    # so the duration belongs to the colour actually painted
+                    held[cell] = 0
+                # later frames overwrite earlier ones, so this ends up holding
+                # the LAST non-final colour the cell showed
+                last_seen[cell] = value
+                held[cell] += 1
+    if not last_seen:
+        return None
+
+    # Brightness carries DURATION: a cell blended toward its true colour in
+    # proportion to how many frames it held it. This separates two mechanics
+    # that are otherwise identical in a flat composite - a 4x4 block falling
+    # down a column touches each cell for a frame or two and renders uniformly
+    # dim, while a column filling from the top holds its highest cell for the
+    # whole animation and its lowest for a moment, so it renders as a gradient
+    # pointing back at the source.
+    span = max(1, len(chain) - 1)
+    image = Image.new("RGB", (cols, rows), DIFF_UNCHANGED_RGB)
+    pixels = image.load()
+    for (row_idx, col_idx), value in last_seen.items():
+        colour = ARC_COLOR_MAP.get(value, ARC_COLOR_MAP[0])
+        # floored so a single-frame cell stays legible rather than vanishing
+        weight = max(0.35, min(1.0, held[(row_idx, col_idx)] / span))
+        pixels[col_idx, row_idx] = tuple(
+            int(round(base + weight * (target - base)))
+            for base, target in zip(DIFF_UNCHANGED_RGB, colour)
+        )
+    scale = current_grid_image_upscale()
+    if scale > 1:
+        image = image.resize((cols * scale, rows * scale), multimodal_resample())
+    buffer = io.BytesIO()
+    image.save(buffer, format="PNG")
+    encoded = base64.b64encode(buffer.getvalue()).decode("ascii")
+    return {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{encoded}"}}
+
+
+def animation_peak_part(
+    chain: list[tuple[tuple[int, ...], ...]],
+    transient_cells: list[tuple[int, int]],
+) -> tuple[dict[str, Any] | None, int]:
+    """The real frame showing the most transient cells at once, and its index.
+
+    Unlike the composite this is a state the game actually displayed, so
+    nothing in it is fabricated - at the cost of missing anything that was
+    hidden at a different moment.
+    """
+    if not chain or not transient_cells or len(chain) <= 2:
+        return None, -1
+    if not current_grid_image_enabled():
+        return None, -1
+    final = chain[-1]
+    wanted = set(transient_cells)
+    best_index, best_count = -1, -1
+    for index, grid in enumerate(chain[1:]):
+        count = 0
+        for row_idx, col_idx in wanted:
+            row = grid[row_idx] if row_idx < len(grid) else ()
+            final_row = final[row_idx] if row_idx < len(final) else ()
+            value = row[col_idx] if col_idx < len(row) else None
+            final_value = final_row[col_idx] if col_idx < len(final_row) else None
+            if value != final_value:
+                count += 1
+        if count > best_count:
+            best_index, best_count = index, count
+    if best_index < 0 or best_count <= 0:
+        return None, -1
+
+    grid = chain[1:][best_index]
+    rows = len(grid)
+    cols = max((len(row) for row in grid), default=0)
+    image = Image.new("RGB", (cols, rows), ARC_COLOR_MAP[0])
+    pixels = image.load()
+    for row_idx, row in enumerate(grid):
+        for col_idx in range(cols):
+            value = row[col_idx] if col_idx < len(row) else 0
+            pixels[col_idx, row_idx] = ARC_COLOR_MAP.get(int(value), ARC_COLOR_MAP[0])
+    scale = current_grid_image_upscale()
+    if scale > 1:
+        image = image.resize((cols * scale, rows * scale), multimodal_resample())
+    buffer = io.BytesIO()
+    image.save(buffer, format="PNG")
+    encoded = base64.b64encode(buffer.getvalue()).decode("ascii")
+    return (
+        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{encoded}"}},
+        best_index,
+    )
+
+
+def diff_image_part(
+    previous_frame: Frame | None, current_frame: Frame | None
+) -> dict[str, Any] | None:
+    """Render a change-highlight image: cells that differ between the two
+    frames appear in their (new) true palette color; unchanged cells are
+    dimmed to an off-palette dark navy. Returns None when disabled, frames
+    are missing, or shapes differ (a full redraw makes a diff meaningless)."""
+    if previous_frame is None or current_frame is None:
+        return None
+    if not current_grid_image_enabled():
+        return None
+    prev_grid, cur_grid = previous_frame.grid, current_frame.grid
+    rows = len(cur_grid)
+    cols = max((len(row) for row in cur_grid), default=0)
+    if rows <= 0 or cols <= 0:
+        return None
+    if len(prev_grid) != rows or max(
+        (len(row) for row in prev_grid), default=0
+    ) != cols:
+        return None
+
+    scale = current_grid_image_upscale()
+    image = Image.new("RGB", (cols, rows), DIFF_UNCHANGED_RGB)
+    pixels = image.load()
+    for row_idx in range(rows):
+        prev_row = prev_grid[row_idx]
+        cur_row = cur_grid[row_idx]
+        for col_idx in range(cols):
+            prev_value = prev_row[col_idx] if col_idx < len(prev_row) else 0
+            cur_value = cur_row[col_idx] if col_idx < len(cur_row) else 0
+            if prev_value != cur_value:
+                pixels[col_idx, row_idx] = ARC_COLOR_MAP.get(
+                    int(cur_value), ARC_COLOR_MAP[0]
+                )
+    if scale > 1:
+        image = image.resize((cols * scale, rows * scale), multimodal_resample())
+
+    buffer = io.BytesIO()
+    image.save(buffer, format="PNG")
+    encoded = base64.b64encode(buffer.getvalue()).decode("ascii")
+    return {
+        "type": "image_url",
+        "image_url": {
+            "url": f"data:image/png;base64,{encoded}",
+        },
+    }
diff --git a/ARC3-Inference/inference/framework/run.py b/ARC3-Inference/inference/framework/run.py
index 4d4accf..fc7d80c 100644
--- a/ARC3-Inference/inference/framework/run.py
+++ b/ARC3-Inference/inference/framework/run.py
@@ -419,6 +419,7 @@ def _make_solver(
         analyzer_timeout=getattr(args, "analyzer_timeout", 120),
         max_actions_per_game=args.max_actions,
         max_runtime_s_per_game=max_runtime_minutes_per_game * 60.0,
+        max_generated_tokens_per_game=args.max_generated_tokens_per_game,
         concurrency=effective_concurrency,
         save_request_logs=bool(args.analyzer_save_request_logs),
         start_local_server=start_local_server,
@@ -1222,6 +1223,15 @@ def main() -> None:
     parser.add_argument("--experiment-dir", dest="experiment_dir", default="")
     parser.add_argument("--max-actions", type=int, default=None)
     parser.add_argument("--max-runtime-minutes", type=float, default=None)
+    parser.add_argument(
+        "--max-generated-tokens-per-game",
+        type=int,
+        default=None,
+        help="Give up on a game once it has generated this many tokens. A "
+             "wall-clock limit means something different on every backend and "
+             "moves hour to hour on a hosted API; this is what makes an "
+             "offline run comparable with a Kaggle one.",
+    )
     parser.add_argument("--max-experiment-runtime-minutes", type=float, default=None)
     parser.add_argument("--max-experiment-runtime-hours", type=float, default=None)
     parser.add_argument("--n-passes", dest="n_passes", type=int, default=1)
diff --git a/ARC3-Inference/inference/framework/solver.py b/ARC3-Inference/inference/framework/solver.py
index 0b69d62..e40976c 100644
--- a/ARC3-Inference/inference/framework/solver.py
+++ b/ARC3-Inference/inference/framework/solver.py
@@ -7,6 +7,7 @@ import contextlib
 import copy
 import functools
 import html
+import logging
 import json
 import os
 import re
@@ -23,7 +24,13 @@ import arcengine
 import taaf.game
 from taaf.solver import Solver
 
+from inference.utils.animation import (
+    collapse_frames,
+    normalize_frames,
+    summarize_animation,
+)
 from inference.agent.action_names import (
+    reset_exposed,
     to_engine_action,
     to_model_action,
     to_model_actions,
@@ -34,7 +41,11 @@ from inference.agent.runtime_state import (
     RUNTIME_STATE_FILENAME,
     write_runtime_state,
 )
-from inference.agent.tool_agent import ToolAgent
+from inference.agent.tool_agent import (
+    _PRIORITY_UNTRIMMED_BASE,
+    ToolAgent,
+    _priority_gate,
+)
 from inference.framework.kaggle import (
     DEFAULT_QWEN_MODEL_DATASET_SOURCE,
     DEFAULT_SERVED_MODEL_NAME,
@@ -53,10 +64,242 @@ from inference.utils.viewer_artifacts import (
     reset_raw_events_sidecar,
 )
 
+log = logging.getLogger(__name__)
+
 AnalyzerFactory = Callable[[taaf.game.Game, int], Any]
 
-ANALYZER_RETRY_BACKOFF_SECONDS = 1.0
+def _env_number(name: str, default: float, cast=float) -> float:
+    """A numeric env override that cannot take the run down at import time."""
+    try:
+        return cast(os.environ.get(name, "") or default)
+    except (TypeError, ValueError):
+        return cast(default)
+
+
+ANALYZER_RETRY_BACKOFF_SECONDS = max(
+    0.0, _env_number("ARC3_ANALYZER_RETRY_BACKOFF_SECONDS", 10.0)
+)
+# A request that fails outright - a dead endpoint, a refused connection - is
+# reported as retryable and the solver goes back to the top of the loop. With
+# no limit that is an infinite spin: nothing monitors the local server, so if
+# it exits mid-run every game retries once a second until its runtime budget is
+# gone, and the run finishes reporting whatever each game had reached with no
+# crash marker anywhere. Giving up after a run of consecutive failures makes
+# that visible - the run ends in seconds rather than at the wall clock, and the
+# games carry a note saying why. 0 restores the unlimited behaviour.
+# Negative means keep retrying; 0 means give up on the first failure. No
+# max(0, ...) clamp here - it would flatten a negative to zero and turn
+# "never give up" into "give up at once".
+# Kaggle requires an action within 15 minutes of the run starting. SGLang
+# needs about 10.5 of those to load weights - 6 on the main model, 3 more on the
+# MTP draft head, which is a second full pass over the same 206 shards - and the
+# notebook blocks on /health before playing anything, so a slow load spends the
+# whole allowance before a single action is issued and the submission scores
+# zero. With this set, the first N games to start execute one RESET straight
+# against the engine before touching the analyzer, which satisfies the clock
+# while the server is still loading; the HTTP retry loop then covers the rest of
+# the wait. 0 disables.
+WARMUP_ACTION_GAMES = _env_number("ARC3_WARMUP_ACTION_GAMES", 0, int)
+
+
+MAX_CONSECUTIVE_ANALYZER_FAILURES = _env_number("ARC3_MAX_ANALYZER_FAILURES", 10, int)
 DEFAULT_CANCEL_DRAIN_TIMEOUT_SECONDS = 120.0
+
+# No-op guard: width of the border frame treated as HUD (progress bars,
+# timers). 0 disables the guard. Simple v1 heuristic; see step_env.
+# arcengine renders a frame after every internal step(), so one action can come
+# back as a short animation whose intermediate frames the harness has always
+# discarded. Off by default: new information for the model, not a fix.
+ANIMATION_ENABLED = os.environ.get("ARC3_ANIMATION", "0").strip().lower() in (
+    "1",
+    "true",
+    "yes",
+    "on",
+)
+NOOP_GUARD_BORDER_WIDTH = max(0, int(os.environ.get("ARC3_NOOP_GUARD_BORDER", "4")))
+# Print one line per action(...) call into the snippet's stdout. Costs a line of
+# tool output per call, which is charged against tool_output_tokens.
+# The patch-110 rewordings, behind one switch. They replaced "changed nothing"
+# with "no change inside the board area" across every guard message, because the
+# old phrasing was false in the case that matters - the HUD usually DID change.
+# The reasoning still looks right, but they shipped alongside the outcome line
+# in a run that lost 43% of the previous competition score, so the two need to
+# be separable before either can be judged. Off restores the wording of the run
+# that scored 4.05.
+NEW_NOOP_WORDING = os.environ.get(
+    "ARC3_NEW_CHANGED_PROMPTS", "0"
+).strip().lower() in ("1", "true", "yes", "on")
+ACTION_ECHO = os.environ.get("ARC3_ACTION_ECHO", "0").strip().lower() in (
+    "1",
+    "true",
+    "yes",
+    "on",
+)
+# When an action changes nothing, the sandbox arms a per-snippet latch and the
+# NEXT action(...) call is refused. Off restores the older behaviour of letting
+# the snippet keep acting on a state its own beliefs no longer describe.
+STALE_STATE_BLOCK = os.environ.get(
+    "ARC3_STALE_STATE_BLOCK", "0"
+).strip().lower() in ("1", "true", "yes", "on")
+
+
+# Stops a batch at the first action that changes nothing inside the board, so a
+# wrong movement model costs one action rather than the whole sequence. It used
+# to be implied by ARC3_NOOP_GUARD_BORDER being positive, which conflated the
+# crop width with whether the guard runs at all - the border is also what
+# gameplay_changed means, and that is wanted independently. Off by default:
+# the guard converts actions into turns, which is a good trade only when tokens
+# are cheap relative to the action budget.
+# Level 1 pays the smallest share of the score - one part in N - and it is
+# where the model knows least, so it probes hardest: repeating an action to see
+# whether the repeat differs, retrying something that killed it to learn what
+# killed it. The guards read that as waste. Set to 2 to let level 1 run
+# unguarded and start guarding at level 2, where an action is worth more and
+# the model has a working model of the game.
+#
+# The batch no-op stop is deliberately NOT covered: it costs one action of a
+# planned sequence rather than refusing a deliberate probe, and a wrong
+# movement model is exactly as expensive on level 1 as anywhere. Nor is the
+# terminal latch, which stops a snippet acting after a level-up or a death -
+# that is a board that no longer exists, not a judgement about the action.
+GUARDS_FROM_LEVEL = max(1, int(os.environ.get("ARC3_GUARDS_FROM_LEVEL", "1") or 1))
+
+
+def _guards_active(level: Any) -> bool:
+    """Whether the per-action guards apply at this level."""
+    try:
+        return int(level) >= GUARDS_FROM_LEVEL
+    except (TypeError, ValueError):
+        return True  # unknown level: guard rather than not
+
+
+BATCH_NOOP_BLOCK = os.environ.get(
+    "ARC3_BATCH_NOOP_BLOCK", "0"
+).strip().lower() in ("1", "true", "yes", "on")
+
+
+_BLOCKED_VERDICTS = {
+    "known_noop": "BLOCKED (known no-op in this exact board state; never executed)",
+    "known_death": "BLOCKED (ended a previous attempt from this exact board state; never executed)",
+}
+
+
+def _action_verdict(item: dict) -> str:
+    """The one-word account of what an executed action did.
+
+    Shared with the per-action trace so the echo introduces no second
+    vocabulary for the model to learn."""
+    if item.get("game_over"):
+        return "GAME OVER (attempt ended immediately after this action)"
+    if item.get("run_complete") or item.get("done"):
+        return "RUN COMPLETE"
+    if item.get("level_completed"):
+        return "LEVEL COMPLETED"
+    changed = item.get("gameplay_changed")
+    if changed is False:
+        return (
+            "NO-OP (executed, no change inside the board area)"
+            if NEW_NOOP_WORDING
+            else "NO-OP (executed, changed nothing)"
+        )
+    if changed is True:
+        return "effect"
+    return "executed"
+
+
+def _build_action_echo(
+    requested_displays: list[str],
+    executed_payloads: list[dict],
+    stop_reason: str | None,
+) -> str:
+    """One line per action(...) call, for the snippet's own stdout.
+
+    A single action produces no per-action trace at all - the trace is built
+    only for batches, game overs and refusals - so the snippet's only account
+    of it arrives a turn later in the opener. Inside a loop that is too late to
+    branch on. This fills the gap at the moment of the call, in the trace's own
+    words, without repeating the trace for batches that already get one.
+    """
+    if not requested_displays:
+        return ""
+    if len(requested_displays) == 1:
+        if executed_payloads:
+            return f"[action] {requested_displays[0]} -> {_action_verdict(executed_payloads[0])}"
+        reason = f" | stop_reason: {stop_reason}" if stop_reason else ""
+        return f"[action] {requested_displays[0]} -> not executed{reason}"
+    ran = ", ".join(
+        str(item.get("action_display") or item.get("action_name") or "?")
+        for item in executed_payloads
+    )
+    line = (
+        f"[action] batch of {len(requested_displays)} actions, "
+        f"{len(executed_payloads)} executed"
+    )
+    if ran:
+        line += f": {ran}"
+    if stop_reason:
+        line += f" | stop_reason: {stop_reason}"
+    return line
+
+
+def _build_action_trace(
+    requested_displays: list[str],
+    executed_payloads: list[dict],
+    blocked_kind: str | None = None,
+) -> list[str]:
+    """One self-describing line per submitted action: index, name, verdict.
+
+    A guard refusal is called out at its own position rather than lumped in
+    with the generic SKIPPED verdict: the prompt tells the model to treat this
+    trace as the authoritative account of what happened, so a refusal that
+    reads as "skipped" would send it looking for an upstream cause that does
+    not exist. Positions AFTER the refusal are genuinely skipped and keep the
+    plain verdict.
+    """
+    trace: list[str] = []
+    blocked_position = len(executed_payloads) + 1 if blocked_kind else None
+    for position, name in enumerate(requested_displays, 1):
+        if blocked_position is not None and position == blocked_position:
+            trace.append(
+                f"{position}: {name} -> "
+                + _BLOCKED_VERDICTS.get(blocked_kind or "", "BLOCKED (never executed)")
+            )
+            continue
+        if position <= len(executed_payloads):
+            verdict = _action_verdict(executed_payloads[position - 1])
+        else:
+            verdict = "SKIPPED (never executed)"
+        trace.append(f"{position}: {name} -> {verdict}")
+    return trace
+
+
+def _interior_gameplay_changed(
+    previous_grid: list[list[int]] | None,
+    new_grid: list[list[int]] | None,
+    *,
+    border: int = NOOP_GUARD_BORDER_WIDTH,
+) -> bool:
+    """True when any cell outside the border frame differs between grids.
+
+    Falls back to a plain inequality check (treat as changed) when the guard
+    is disabled, grids are missing, shapes differ, or the border swallows the
+    whole grid — the guard must fail open.
+    """
+    if border <= 0 or previous_grid is None or new_grid is None:
+        return previous_grid != new_grid
+    rows = len(previous_grid)
+    cols = len(previous_grid[0]) if rows else 0
+    if rows != len(new_grid) or any(len(r) != cols for r in (previous_grid[0:1] + new_grid[0:1])):
+        return True
+    if rows <= 2 * border or cols <= 2 * border:
+        return previous_grid != new_grid
+    for r in range(border, rows - border):
+        prev_row = previous_grid[r]
+        new_row = new_grid[r]
+        if prev_row[border:cols - border] != new_row[border:cols - border]:
+            return True
+    return False
+
 _LOCAL_SERVER_PROCESS_ENV_KEYS = (
     "LOCAL_ANALYZER_API_KEY",
     "OPENAI_API_KEY",
@@ -90,6 +333,24 @@ def artifact_stem(value: str) -> str:
     return re.sub(r"[^A-Za-z0-9_.-]+", "_", value)
 
 
+def _animation_chain(
+    previous_grid: tuple[tuple[int, ...], ...],
+    state: taaf.game.GameState | None,
+) -> list[tuple[tuple[int, ...], ...]]:
+    """The pre-action board, then each distinct state the action passed through.
+
+    `state.frame` is only `raw.frame[-1]`; the rest of that list is the
+    animation. Runs of identical grids collapse here rather than at render
+    time, because a single action can return dozens of frames and they stay in
+    memory until the next action replaces them.
+    """
+    if state is None:
+        return []
+    return collapse_frames(
+        previous_grid, normalize_frames(getattr(state.raw, "frame", None))
+    )
+
+
 def _grid_from_state(state: taaf.game.GameState | None) -> tuple[tuple[int, ...], ...]:
     if state is None:
         return ()
@@ -113,7 +374,7 @@ def _engine_action_names(game: taaf.game.Game) -> list[str]:
             name = arcengine.GameAction.from_id(int(action_id)).name
         except Exception:
             continue
-        if name == "RESET":
+        if name == "RESET" and not reset_exposed():
             continue
         if name not in names:
             names.append(name)
@@ -127,6 +388,46 @@ def _model_mouse_action_data(
     return {"row": int(data.get("y", 0)), "col": int(data.get("x", 0))}
 
 
+def _guard_action_data(action: Any) -> dict[str, Any]:
+    """Action data in the row/col form the guard signature expects.
+
+    ACTION6 carries x/y from the engine; `_format_action_display` already
+    translates it the same way, so the ledger and the guard agree on what
+    identifies a click."""
+    data = dict(getattr(action, "data", None) or {})
+    if getattr(getattr(action, "id", None), "name", "") == "ACTION6":
+        return _model_mouse_action_data(data)
+    return data
+
+
+def _stale_repeat(stale_after: str, action: Any) -> bool:
+    """Is this the same action that just changed nothing?
+
+    Compared on the DISPLAY form, which carries mouse coordinates - a click
+    somewhere else after a no-op click is a different action, and the repeat
+    guard's override is meaningful for it.
+    """
+    return str(stale_after).strip() == _format_action_display(
+        action.id.name, dict(action.data)
+    ).strip()
+
+
+def _stale_detail(action: Any, stale_after: str) -> str:
+    return (
+        f"{_format_action_display(action.id.name, dict(action.data))} was not "
+        f"executed. The action immediately before it, {stale_after}, was a "
+        + ("no-op - it produced no change inside the board area - so positions "
+           if NEW_NOOP_WORDING else
+           "no-op - it changed nothing outside the board border - so positions ")
+        +
+        "and paths computed since then may be wrong, which is why execution "
+        "has been halted here. Re-read current_frame and last_action_call_result, "
+        "work out why nothing moved, then act on what you find. "
+        "(StaleStateActionError can be caught if a snippet has good reason to "
+        "keep acting through no-ops.)"
+    )
+
+
 def _format_action_display(
     action_name: str, action_data: dict[str, Any] | None = None
 ) -> str:
@@ -182,6 +483,14 @@ class _HarnessGameSession:
     analysis_step: int = 0
     last_engine_action: str | None = None
     token_baseline: int = 0
+    # token_baseline is a cursor: it is dragged forward after every step so the
+    # engine can be told what that step cost, and so cannot answer "how much
+    # has this game spent". This one is captured once and never moved.
+    game_token_baseline: int = 0
+    # The last executed action's animation, or None. One record, not a
+    # history: the model reads it via animation() in the turn that follows,
+    # and older animations are almost never what it is asking about.
+    animation_record: dict[str, Any] | None = None
     _viewer_events_flushed: int = field(default=0, init=False, repr=False)
 
     def current_frame(self) -> Frame:
@@ -243,6 +552,30 @@ class _HarnessGameSession:
             return None
         return max(0.1, min(candidates))
 
+    def token_limit_reached(self) -> bool:
+        """Whether this game has generated its allowance.
+
+        Read from the analyzer rather than summed per step: should_stop is
+        passed into the analyzer and runs at every loop boundary, so reading a
+        counter that only advances when an action executes would leave a game
+        that deliberates without acting running past its allowance. The
+        analyzer's accumulator advances on every response.
+
+        Measured against a baseline captured at game start, which is 0 for the
+        per-game analyzer the solver builds but need not be if an
+        analyzer_factory hands back a shared one.
+        """
+        # getattr, not attribute access: the notebook unpickles a solver whose
+        # __dict__ was captured before this field existed, and __setstate__ does
+        # a plain dict update - it applies no dataclass defaults. A bare access
+        # would raise AttributeError here on the first turn of every game, since
+        # should_stop() runs this every time.
+        limit = getattr(self.solver, "max_generated_tokens_per_game", None)
+        if limit is None or limit <= 0:
+            return False
+        spent = _analyzer_reported_tokens(self.analyzer) - self.game_token_baseline
+        return spent >= limit
+
     def should_stop(self) -> bool:
         run = self.game.game_run
         if run is None or run.state != "playing":
@@ -258,8 +591,15 @@ class _HarnessGameSession:
             and self.action_count >= self.solver.max_actions_per_game
         ):
             return True
+        if self.token_limit_reached():
+            return True
         return False
 
+    def _queue_position(self) -> int:
+        """Position in the dispatch order, used only to break ties in the
+        untrimmed priority band so games do not all queue at one value."""
+        return max(0, int(self.game_index)) * 100 + max(0, int(self.pass_index))
+
     def play(self) -> None:
         run = self.game.game_run
         assert run is not None, "TAAF starts games before invoking the solver."
@@ -267,12 +607,51 @@ class _HarnessGameSession:
         self.transcript_path.parent.mkdir(parents=True, exist_ok=True)
         self.transcript_path.touch(exist_ok=True)
         self.token_baseline = _analyzer_reported_tokens(self.analyzer)
-        self.seed_initial_history()
-        self.write_runtime_state()
-        self._append_initial_viewer_event()
-        self.write_viewer_payload()
+        self.game_token_baseline = self.token_baseline
+        # A game that has not started yet sits in the top band, so every game
+        # gets going in queue order before any that has already trimmed comes
+        # back for a second turn.
+        gate = _priority_gate()
+        if gate is not None:
+            deadline = (
+                self.started_at + self.solver.max_runtime_s_per_game
+                if self.solver.max_runtime_s_per_game is not None else None
+            )
+            soft_remaining = self.solver.soft_time_remaining_seconds()
+            if soft_remaining is not None:
+                soft_deadline = time.monotonic() + soft_remaining
+                deadline = min(deadline, soft_deadline) if deadline is not None else soft_deadline
+            gate.configure_clock(self.started_at, deadline)
+            gate.acquire(_PRIORITY_UNTRIMMED_BASE - self._queue_position())
+        try:
+            self.seed_initial_history()
+            self.write_runtime_state()
+            self._append_initial_viewer_event()
+            self.write_viewer_payload()
+            self._play_inner()
+        finally:
+            if gate is not None:
+                gate.release()
+
+    def _play_inner(self) -> None:
+        run = self.game.game_run
+        if self.solver._claim_warmup_slot():
+            # Straight to the engine - _execute_auto_reset takes the same path
+            # and needs no analyzer, so this works with the endpoint still
+            # loading. One action off this game's budget buys the whole
+            # submission a deadline it would otherwise miss.
+            # warning, not info: _configure_logging only runs under run.py, so
+            # the notebook has no handler and falls back to logging's lastResort,
+            # which emits WARNING and above. An info line would vanish exactly
+            # where this message matters most.
+            log.warning(
+                "%s: warmup RESET issued before the analyzer is reachable",
+                run.game_id,
+            )
+            self._execute_auto_reset()
         try:
             retry_analysis_step: int | None = None
+            consecutive_failures = 0
             while not self.should_stop():
                 if (
                     _is_engine_game_over(self.game)
@@ -310,15 +689,42 @@ class _HarnessGameSession:
                 if result is None:
                     raise RuntimeError("Analyzer did not return a result.")
                 if result.retryable_failure:
+                    consecutive_failures += 1
+                    if (
+                        MAX_CONSECUTIVE_ANALYZER_FAILURES >= 0
+                        and consecutive_failures > MAX_CONSECUTIVE_ANALYZER_FAILURES
+                    ):
+                        log.error(
+                            "%s: analyzer failed %d times in a row; giving up on this "
+                            "game. The endpoint is probably down - nothing here can "
+                            "restart it.",
+                            run.game_id,
+                            consecutive_failures,
+                        )
+                        run.solver_note = (
+                            f"analyzer failed {consecutive_failures} times consecutively"
+                        )
+                        break
                     retry_analysis_step = analysis_step
                     if self.should_stop():
                         break
                     time.sleep(ANALYZER_RETRY_BACKOFF_SECONDS)
                     continue
 
+                # a turn that came back at all clears the run: the limit is for
+                # an endpoint that has stopped answering, not for a game that
+                # hits an occasional blip
+                consecutive_failures = 0
                 retry_analysis_step = None
                 if getattr(result, "yielded_control", False):
-                    retry_analysis_step = analysis_step
+                    # Reuse the step id only for a pure thinking continuation.
+                    # If this invocation already executed actions, the resumed
+                    # invocation opens with a fresh batch-summary prompt - a new
+                    # logical turn - and must get a fresh analysis_step, or the
+                    # viewer merges two turns into one card (and sums their
+                    # token usage).
+                    if not result.step_executed:
+                        retry_analysis_step = analysis_step
                     continue
                 if not result.step_executed:
                     continue
@@ -586,6 +992,14 @@ class _HarnessGameSession:
         }
 
     def step_env(self, arguments: dict[str, Any]) -> dict[str, Any]:
+        if str(arguments.get("query") or "").strip() == "animation":
+            # Routed through step_env so no analyzer-protocol signature has to
+            # change. Executes nothing and spends no action budget.
+            return {
+                "executed": False,
+                "query": "animation",
+                "record": self.animation_record if ANIMATION_ENABLED else None,
+            }
         requested_actions, error = self._normalize_actions(arguments)
         if error is not None or requested_actions is None:
             return self._error_payload(error or "Could not parse action request.")
@@ -601,10 +1015,84 @@ class _HarnessGameSession:
             for action in requested_actions
         ]
 
+        guard_hook = getattr(self, "action_guard_hook", None)
+        blocked_detail: str | None = None
+        blocked_kind: str | None = None
+
+        # Checked AFTER the per-position guards below, so a known death or a
+        # known no-op takes precedence: both name a specific action and offer an
+        # override, which is more actionable than a general staleness warning.
+        stale_after = (
+            arguments.get("stale_after")
+            if STALE_STATE_BLOCK and _guards_active(_level_number(self.game))
+            else None
+        )
+
         for batch_index, action in enumerate(requested_actions, start=1):
             if self.should_stop():
                 stop_reason = "stopped"
                 break
+            if (
+                stale_after
+                and batch_index == 1
+                and _stale_repeat(stale_after, action)
+            ):
+                # the model is re-issuing the action that just changed nothing;
+                # the staleness message is the correct one here, so take it
+                # before the repeat guard can claim the turn
+                blocked_kind = "stale_state"
+                blocked_detail = _stale_detail(action, stale_after)
+                stop_reason = blocked_kind
+                break
+            if guard_hook is not None:
+                # Per-position check: the hash is recomputed from the CURRENT
+                # board, so position N of a batch is judged against the state
+                # its predecessors actually produced - not the state the batch
+                # started in. This is what lets a fatal move be caught no
+                # matter which route reached it.
+                # The engine stores mouse coordinates as x/y; every consumer
+                # that identifies an action - the ledger, the action display,
+                # the guard signature - works in row/col. Passing action.data
+                # raw made every click share the signature "MOUSE" with no
+                # coordinates, so one no-op click poisoned clicking ANYWHERE
+                # from that board state. On a mouse-only game that is terminal:
+                # observed as 175 replies and one executed action in a run.
+                verdict = guard_hook(
+                    _grid_from_state(self.game.current_state),
+                    action.id.name,
+                    _guard_action_data(action),
+                    _level_number(self.game),
+                    [
+                        (later.id.name, _guard_action_data(later))
+                        for later in requested_actions[batch_index - 1:]
+                    ],
+                )
+                if verdict:
+                    blocked_kind, blocked_detail = verdict
+                    stop_reason = blocked_kind
+                    break
+            if stale_after and batch_index == 1:
+                # Ordering against the repeat guards above:
+                #
+                # Different action - the repeat guard wins. It names the action
+                # and its override is meaningful: the model may have reason to
+                # believe circumstances changed since that earlier visit.
+                #
+                # SAME action as the one that just no-opped - this message wins,
+                # and the check is placed before the guards for that case (see
+                # the early branch above). Telling a model that just watched X
+                # do nothing to "issue it again and it will execute" is advice
+                # to repeat a failure; worse, the guard's own override is
+                # already satisfied by this call, so without the swap the action
+                # would simply run with no message at all.
+                blocked_kind = "stale_state"
+                blocked_detail = _stale_detail(action, stale_after)
+                stop_reason = blocked_kind
+                break
+            # anything that reaches execution clears the flag: it can only
+            # happen when the check above declined to fire, and leaving a stale
+            # marker set behind an executed action would be a latent surprise
+            stale_after = None
             if action.id.value not in self.game.current_state.available_actions:
                 message = f"{_format_action_display(action.id.name, dict(action.data))} is not valid right now."
                 if executed_payloads:
@@ -636,8 +1124,35 @@ class _HarnessGameSession:
             if payload.get("level_completed"):
                 stop_reason = "level_completed"
                 break
+            if (
+                BATCH_NOOP_BLOCK
+                and NOOP_GUARD_BORDER_WIDTH > 0
+                and batch_size >= 2
+                and not payload.get("gameplay_changed", True)
+            ):
+                stop_reason = "no_op_action"
+                break
 
         if not executed_payloads:
+            if blocked_detail is not None:
+                # Nothing ran at all: report the refusal itself rather than a
+                # generic error, so the sandbox can raise the right exception.
+                return {
+                    **self._error_payload(blocked_detail),
+                    "executed": False,
+                    "stop_reason": blocked_kind,
+                    "stop_detail": blocked_detail,
+                    "requested_count": batch_size,
+                    "executed_count": 0,
+                    "stopped_early": True,
+                    "requested_actions": requested_displays,
+                    "skipped_actions": list(requested_displays),
+                    # the refused action is position 1 here, since nothing ran.
+                    # Omitting it left the next turn's opener saying "stopped
+                    # before None" - the later assignment for the partial-batch
+                    # case is never reached on this path.
+                    "blocked_action": requested_displays[0] if requested_displays else None,
+                }
             return self._error_payload("No action was executed.")
 
         final_payload = dict(executed_payloads[-1])
@@ -657,12 +1172,64 @@ class _HarnessGameSession:
         final_payload["stopped_early"] = len(executed_payloads) < batch_size
         if stop_reason is not None:
             final_payload["stop_reason"] = stop_reason
+        if final_payload["stopped_early"]:
+            final_payload["skipped_actions"] = requested_displays[len(executed_payloads):]
+        for item in executed_payloads:
+            if item.get("game_over"):
+                final_payload["fatal_action"] = str(item.get("action_display") or "")
+                break
+        if batch_size > 1 or final_payload.get("game_over") or blocked_kind:
+            final_payload["action_trace"] = _build_action_trace(
+                requested_displays, executed_payloads, blocked_kind
+            )
+            final_payload["gameplay_changed_per_action"] = [
+                bool(item.get("gameplay_changed", True)) for item in executed_payloads
+            ]
+        if ACTION_ECHO:
+            echo = _build_action_echo(requested_displays, executed_payloads, stop_reason)
+            if echo:
+                # consumed by the sandbox, which prints it and pops it: leaving
+                # it in the returned dict shows the model the same line twice,
+                # once on stdout and once in the payload it reads back
+                final_payload["action_echo"] = echo
+        if blocked_detail is not None:
+            final_payload["stop_detail"] = blocked_detail
+            final_payload["blocked_action"] = requested_displays[len(executed_payloads)]
+        if stop_reason == "no_op_action":
+            noop_index = len(executed_payloads)
+            final_payload["no_op_action_index"] = noop_index
+            final_payload["no_op_action"] = final_payload["executed_actions"][-1]
+            skipped = requested_displays[noop_index:]
+            final_payload["stop_detail"] = (
+                f"Action {noop_index} of {batch_size} "
+                + (
+                    f"({final_payload['executed_actions'][-1]!r}) EXECUTED but produced no "
+                    f"change inside the board area - only the outer {NOOP_GUARD_BORDER_WIDTH} "
+                    "cells, where a timer or remaining-steps bar usually sits, could have moved. "
+                    if NEW_NOOP_WORDING else
+                    f"({final_payload['executed_actions'][-1]!r}) EXECUTED but changed nothing "
+                    f"outside the {NOOP_GUARD_BORDER_WIDTH}-cell border (treated as HUD). "
+                )
+                +
+                f"Actions {noop_index + 1}-{batch_size} ({skipped!r}) were SKIPPED and did "
+                "not execute. Indices are 1-based positions in your submitted list. "
+                f"Net effect: only the first {max(0, noop_index - 1)} executed action(s) "
+                "changed the board; the final executed action moved nothing. Do NOT "
+                "update your position estimate by counting submitted moves - re-locate "
+                "the player from current_frame before planning. "
+                "The per-action verdicts are listed in action_trace. "
+                "This stop is about one action from one board state, not about batch "
+                "size - batching is still the right approach, and the stop is what keeps "
+                "a wrong movement model from costing the whole sequence."
+            )
         self.write_viewer_payload()
         return final_payload
 
     def _execute_auto_reset(self) -> None:
         action = arcengine.ActionInput(id=arcengine.GameAction.RESET, data={})
-        self._execute_action(action, batch_index=1, batch_size=1, generated_tokens=0)
+        self._execute_action(
+            action, batch_index=1, batch_size=1, generated_tokens=0, automatic=True
+        )
 
     def _execute_action(
         self,
@@ -672,6 +1239,7 @@ class _HarnessGameSession:
         batch_size: int,
         generated_tokens: int | None = None,
         flush_viewer_payload: bool = True,
+        automatic: bool = False,
     ) -> dict[str, Any]:
         previous_grid = _grid_from_state(self.game.current_state)
         previous_completed = int(self.game.current_state.levels_completed)
@@ -690,17 +1258,40 @@ class _HarnessGameSession:
             step=self.action_count,
             level=_level_number(self.game),
         )
-        self.history_entries.append(
-            HistoryEntry(action=action_display, frame=current_frame)
-        )
-        self.write_runtime_state()
 
         completed = int(new_state.levels_completed)
         reward = float(completed - previous_completed) / max(
             1.0, float(self.game.number_of_levels)
         )
         raw_state = new_state.raw.state
-        board_changed = previous_grid != _grid_from_state(new_state)
+        new_grid = _grid_from_state(new_state)
+        board_changed = previous_grid != new_grid
+        gameplay_changed = _interior_gameplay_changed(previous_grid, new_grid)
+        no_op = gameplay_changed is False
+        animation_summary = None
+        if ANIMATION_ENABLED and action.id.name != "RESET":
+            # The auto-reset after a death runs through here too. Letting it
+            # touch the record would clear the fatal action's animation before
+            # the model ever reads it - and a death is the case where those
+            # frames matter most, since the board it now sees is the restarted
+            # level and the death exists nowhere else.
+            chain = _animation_chain(previous_grid, new_state)
+            animation_summary = summarize_animation(chain)
+            # ONE record, replaced by every executed action - animated or not.
+            # A no-op that animated is the case this exists for, and the guards
+            # refuse the NEXT action without executing it, so the record still
+            # describes the animated one when the model reads the opener.
+            self.animation_record = (
+                {
+                    "action_display": action_display,
+                    "chain": chain,
+                    "summary": animation_summary,
+                    "step": self.action_count,
+                    "level": _level_number(self.game),
+                }
+                if animation_summary
+                else None
+            )
         level_completed = bool(
             new_state.just_won_level and raw_state != arcengine.GameState.WIN
         )
@@ -713,6 +1304,8 @@ class _HarnessGameSession:
             "state": raw_state.name,
             "valid_actions": to_model_actions(_engine_action_names(self.game)),
             "board_changed": board_changed,
+            "gameplay_changed": gameplay_changed,
+            "no_op": no_op,
             "done": raw_state == arcengine.GameState.WIN,
             "level_completed": level_completed,
             "game_over": raw_state == arcengine.GameState.GAME_OVER,
@@ -728,6 +1321,25 @@ class _HarnessGameSession:
             "batch_size": batch_size,
             **self.timing_payload(),
         }
+        if animation_summary:
+            payload["animation"] = animation_summary
+            # the raw chain, for rendering an image in the opener. Dropped by
+            # _compact_action_result, so it never reaches the model as text.
+            payload["animation_chain"] = chain
+        # Store the individual action before step_env folds results into a batch.
+        # Keep only compact metadata, without grids, animation chains, or clocks.
+        payload["automatic"] = automatic
+        result_keys = (
+            "executed", "action_num", "level", "score", "state", "valid_actions",
+            "board_changed", "gameplay_changed", "no_op", "done", "level_completed",
+            "game_over", "run_complete", "action_name", "action_data", "action_display",
+            "animation", "automatic",
+        )
+        individual_result = copy.deepcopy({k: payload[k] for k in result_keys if k in payload})
+        self.history_entries.append(
+            HistoryEntry(action=action_display, frame=current_frame, result=individual_result)
+        )
+        self.write_runtime_state()
         self._append_action_viewer_event(payload, current_frame)
         if flush_viewer_payload:
             self.write_viewer_payload()
@@ -743,6 +1355,12 @@ class HarnessSolver(Solver):
     analyzer_timeout: float | None = 120.0
     max_actions_per_game: int | None = None
     max_runtime_s_per_game: float | None = None
+    # Generated tokens a game may spend before it gives up, counted from the
+    # analyzer's own accumulator so it measures what the model produced rather
+    # than what was billed. A wall-clock limit means something different on
+    # every backend - and on a hosted API it moves hour to hour - so this is
+    # what makes an offline run comparable with a Kaggle one. None disables.
+    max_generated_tokens_per_game: int | None = None
     concurrency: int = 16
     save_request_logs: bool = False
     start_local_server: bool = False
@@ -816,6 +1434,7 @@ class HarnessSolver(Solver):
         state.pop("_local_servers", None)
         state.pop("_local_server_original_env", None)
         state.pop("_worker_pool", None)
+        state.pop("_warmup_lock", None)
         return state
 
     def __setstate__(self, state: dict[str, Any]) -> None:
@@ -829,6 +1448,7 @@ class HarnessSolver(Solver):
         self._local_servers = []
         self._local_server_original_env = {}
         self._worker_pool = None
+        self._warmup_lock = threading.Lock()
 
     def __deepcopy__(self, memo: dict[int, Any]) -> "HarnessSolver":
         cls = type(self)
@@ -845,6 +1465,8 @@ class HarnessSolver(Solver):
                 object.__setattr__(new, key, {})
             elif key == "_worker_pool":
                 object.__setattr__(new, key, None)
+            elif key == "_warmup_lock":
+                object.__setattr__(new, key, threading.Lock())
             else:
                 object.__setattr__(new, key, copy.deepcopy(value, memo))
         return new
@@ -879,6 +1501,12 @@ class HarnessSolver(Solver):
         )
 
     def _setup(self) -> None:
+        # Created here rather than in __init__ because the notebook unpickles a
+        # solver whose __dict__ predates these attributes, and __setstate__
+        # applies no dataclass defaults. _setup runs after unpickling, so the
+        # attributes always exist by the time a game asks for a slot.
+        self._warmup_remaining = WARMUP_ACTION_GAMES
+        self._warmup_lock = threading.Lock()
         if self.start_local_server:
             self._start_local_servers()
         self._worker_pool = ThreadPoolExecutor(
@@ -1178,6 +1806,23 @@ class HarnessSolver(Solver):
             return None
         return self._local_servers[int(game_index) % len(self._local_servers)]
 
+    def _claim_warmup_slot(self) -> bool:
+        """Whether this game should spend one action before the analyzer is up.
+
+        Claimed by whichever runners start first rather than by index: task
+        creation order and semaphore fairness both happen to give indices 0..N,
+        but neither is guaranteed by the API, and the cost of being wrong is a
+        game that never fires the action the deadline depends on.
+        """
+        lock = getattr(self, "_warmup_lock", None)
+        if lock is None:
+            return False
+        with lock:
+            if getattr(self, "_warmup_remaining", 0) <= 0:
+                return False
+            self._warmup_remaining -= 1
+            return True
+
     def _make_analyzer(
         self,
         game: taaf.game.Game,
@@ -1190,6 +1835,9 @@ class HarnessSolver(Solver):
             model=self.model,
             timeout=self.analyzer_timeout,
             save_request_logs=self.save_request_logs,
+            # queue position, so the untrimmed priority band has a deterministic
+            # order instead of every game tying at the same value
+            dispatch_index=index,
             api_key=(
                 local_server.api_key
                 if local_server is not None
diff --git a/ARC3-Inference/inference/utils/animation.py b/ARC3-Inference/inference/utils/animation.py
new file mode 100644
index 0000000..d2a1484
--- /dev/null
+++ b/ARC3-Inference/inference/utils/animation.py
@@ -0,0 +1,333 @@
+"""Animation awareness.
+
+``arcengine`` renders a frame after every internal ``step()``, so a single
+action can come back as a short animation. ``GameState.frame`` is only the last
+of those, and the harness consumed nothing else - so every intermediate frame
+was discarded before the agent could see it. On some games the whole effect of
+an action lives there: the board before and the board after are identical, and
+reading the final frame cannot possibly reveal what happened.
+
+Two shapes occur. Where cells change and change BACK, the information is
+genuinely hidden and worth spending prompt tokens on. Where every cell changes
+at most once, the frames are motion interpolation toward a state the final
+frame already shows, and there is nothing to find - those animations are
+reported nowhere, because a line about them would cost tokens to say that
+nothing is missing.
+"""
+from __future__ import annotations
+
+from collections import defaultdict
+from typing import Any, Sequence
+
+from inference.utils.grid_utils import ARC_COLOR_CHARS, format_grid_ascii
+
+Grid = tuple[tuple[int, ...], ...]
+
+# Retrieval budget.
+#
+# Why a diff timeline rather than the frames themselves: one 64x64 grid renders
+# as roughly 4,100 characters, and a single action can return dozens of frames.
+# Whole frames are never affordable against a ~1,024-token tool budget, so
+# retrieval collapses identical frames and reports only the cells that changed
+# between the rest. A single frame can still be read verbatim, cropped.
+# No step limit: the timeline crosses the pipe as data and the model prints
+# only what it chooses, so dropping steps served a prompt budget that no longer
+# applies. It also cost a real misreading - a step count reported as omitted
+# was read as "those steps had no changes", the exact inverse.
+ANIMATION_MAX_CELLS_PER_STEP = 24
+ANIMATION_MAX_TOTAL_CELLS = 80
+
+
+def normalize_frames(raw_frames: Any) -> list[Grid]:
+    """Engine frame list to hashable, comparable grids."""
+    frames: list[Grid] = []
+    for frame in raw_frames or ():
+        rows = frame.tolist() if hasattr(frame, "tolist") else frame
+        frames.append(tuple(tuple(int(cell) for cell in row) for row in rows or ()))
+    return frames
+
+
+def collapse_frames(before: Grid | None, frames: Sequence[Grid]) -> list[Grid]:
+    """The pre-action board, then each distinct state the action passed through.
+
+    Runs of identical grids collapse to one entry; non-adjacent repeats stay
+    separate, because a grid at positions 2 and 7 with something between them
+    is two distinct moments and the timeline should show both transitions.
+
+    ``before`` is prepended and is NOT optional. A cell showing X before the
+    action, Y during it and X again at the end arrives from the engine as
+    [Y, Y, X] - one transition, which would not register as transient. With the
+    board before it is [X, Y, Y, X], which gives two. Without this the very
+    case the feature exists for is invisible.
+    """
+    collapsed: list[Grid] = []
+    if before is not None:
+        collapsed.append(before)
+    for grid in frames:
+        if collapsed and collapsed[-1] == grid:
+            continue
+        collapsed.append(grid)
+    return collapsed
+
+
+def _transient_cells(chain: Sequence[Grid]) -> list[tuple[int, int]]:
+    """Cells whose value changes two or more times across the chain.
+
+    A cell that changes once and stays changed is visible in the final frame,
+    so the agent loses nothing by never seeing the frames between. A cell that
+    changes twice has shown something that is no longer there - a rejected
+    click, a consumed attempt, a sprite crossing a background cell. Those are
+    exactly the pixels no frame the agent can reach will ever contain.
+    """
+    changes: defaultdict[tuple[int, int], int] = defaultdict(int)
+    for before, after in zip(chain, chain[1:]):
+        rows = max(len(before), len(after))
+        for row in range(rows):
+            before_row = before[row] if row < len(before) else ()
+            after_row = after[row] if row < len(after) else ()
+            for col in range(max(len(before_row), len(after_row))):
+                old = before_row[col] if col < len(before_row) else None
+                new = after_row[col] if col < len(after_row) else None
+                if old != new:
+                    changes[(row, col)] += 1
+    return sorted(cell for cell, count in changes.items() if count >= 2)
+
+
+def summarize_animation(chain: Sequence[Grid]) -> dict[str, Any] | None:
+    """Compact metadata for one action, or ``None`` if it did not animate.
+
+    ``None`` for the single-frame case means ordinary actions carry no extra
+    tokens at all - the key is simply absent from the action result.
+
+    ``frames`` counts engine frames after collapsing and excludes the prepended
+    pre-action board, so it answers "how many distinct states did this action
+    pass through" rather than describing the internal representation.
+    """
+    if len(chain) <= 2:
+        return None
+    transient = _transient_cells(chain)
+    summary: dict[str, Any] = {
+        "frames": len(chain) - 1,
+        "transient_pixels": len(transient),
+    }
+    if transient:
+        rows = [cell[0] for cell in transient]
+        cols = [cell[1] for cell in transient]
+        # Inclusive [top, left, bottom, right]: enough to point at the region
+        # without spending tokens on a coordinate list.
+        summary["transient_bbox"] = [min(rows), min(cols), max(rows), max(cols)]
+    return summary
+
+
+def _bbox_phrase(bbox: Sequence[int] | None) -> str:
+    """Where the transient cells are, collapsing degenerate ranges.
+
+    A single cell reads as "at (20,30)", a single row or column as a span in
+    the other axis only; "rows 20-20, cols 30-30" is noise the model has to
+    parse before noticing it means one cell.
+    """
+    if not bbox:
+        return ""
+    top, left, bottom, right = bbox
+    if top == bottom and left == right:
+        return f" at ({top},{left})"
+    if top == bottom:
+        return f" in row {top}, cols {left}-{right}"
+    if left == right:
+        return f" in col {left}, rows {top}-{bottom}"
+    return f" around rows {top}-{bottom}, cols {left}-{right}"
+
+
+def describe_animation(
+    summary: dict[str, Any] | None,
+    *,
+    action_display: str | None,
+    gameplay_changed: bool | None,
+    game_over: bool = False,
+    timeline: bool = True,
+) -> str:
+    """The opener line, or "" when there is nothing worth saying.
+
+    Silent unless cells changed and changed back. Where every cell changed at
+    most once the final frame already shows the outcome, and a line saying so
+    would spend tokens to report that nothing is missing.
+    """
+    if not summary:
+        return ""
+    transient = int(summary.get("transient_pixels") or 0)
+    if transient <= 0:
+        return ""
+    frames = summary.get("frames")
+    cells = "1 cell changed and changed back" if transient == 1 else (
+        f"{transient} cells changed and changed back"
+    )
+    where = _bbox_phrase(summary.get("transient_bbox"))
+    name = f" (`{action_display}`)" if action_display else ""
+    # point at whichever accessor is actually available
+    where_to_look = (
+        "`last_animation_timeline`"
+        if timeline
+        else "`last_animation_frames`"
+    )
+
+    if game_over:
+        # Follows the BAR RULE block, which has just walked the model through
+        # history[-3] and history[-2]. This says what those two frames cannot
+        # contain, which is precisely what "transient" means.
+        return (
+            f"That fatal action animated over {frames} frames, and inside it "
+            f"{cells}{where}. Those cells are in neither "
+            f"`history[-3].frame` nor `history[-2].frame` - read {where_to_look} to see "
+            "what happened between them."
+        )
+    if gameplay_changed is False:
+        return (
+            f"The last executed action{name} animated over {frames} frames and left the "
+            f"board area unchanged, but it was not a no-op: {cells}{where}. "
+            "Whatever this action did is visible only there - "
+            f"read {where_to_look} before concluding it failed."
+        )
+    return (
+        f"The last executed action{name} animated over {frames} frames. "
+        f"{cells[0].upper()}{cells[1:]}{where}. {where_to_look} shows them."
+    )
+
+
+def _color_char(value: int | None) -> str:
+    if value is None:
+        return "?"
+    return ARC_COLOR_CHARS[max(0, min(15, int(value)))]
+
+
+def _cell_at(grid: Grid, row: int, col: int) -> int | None:
+    if row >= len(grid):
+        return None
+    line = grid[row]
+    return line[col] if col < len(line) else None
+
+
+def _diff_cells(before: Grid, after: Grid) -> list[tuple[int, int, int | None, int | None]]:
+    rows = max(len(before), len(after))
+    changes: list[tuple[int, int, int | None, int | None]] = []
+    for row in range(rows):
+        cols = max(
+            len(before[row]) if row < len(before) else 0,
+            len(after[row]) if row < len(after) else 0,
+        )
+        for col in range(cols):
+            old = _cell_at(before, row, col)
+            new = _cell_at(after, row, col)
+            if old != new:
+                changes.append((row, col, old, new))
+    return changes
+
+
+def _bbox_text(cells: Sequence[tuple[int, int, Any, Any]]) -> str:
+    rows = [cell[0] for cell in cells]
+    cols = [cell[1] for cell in cells]
+    return f"rows {min(rows)}-{max(rows)}, cols {min(cols)}-{max(cols)}"
+
+
+def _format_changes(
+    cells: Sequence[tuple[int, int, int | None, int | None]], budget: int
+) -> list[str]:
+    """Changed cells grouped by colour transition, one compact line each.
+
+    Strings rather than nested coordinate lists because the caller renders with
+    json.dumps(indent=2), which would put every [row, col] pair on four lines.
+    """
+    grouped: dict[str, list[str]] = {}
+    for row, col, old, new in cells[:budget]:
+        key = f"{_color_char(old)}>{_color_char(new)}"
+        grouped.setdefault(key, []).append(f"({row},{col})")
+    lines = [f"{key} @ {' '.join(points)}" for key, points in grouped.items()]
+    if len(cells) > budget:
+        lines.append(f"... {len(cells) - budget} further changed cells omitted")
+    return lines
+
+
+def build_animation_frames(record: dict[str, Any] | None) -> list[dict[str, Any]]:
+    """Every distinct state the action passed through, as FrameView payloads.
+
+    The pre-action board is dropped here: the model already has it as
+    `previous_frame`, and including it would make index 0 mean something
+    different from the step numbers in the timeline.
+
+    Sends the grid as well as the ascii so `segmentation` works on the far
+    side. That doubles the payload, but it crosses a pipe rather than the
+    prompt - what reaches the model is only what its own code prints.
+    """
+    if not record:
+        return []
+    chain = list(record.get("chain") or ())
+    if len(chain) <= 2:
+        return []
+    step = record.get("step") or 0
+    level = record.get("level") or 0
+    return [
+        {
+            "ascii": format_grid_ascii(grid),
+            "step": step,
+            "level": level,
+            "shape": (len(grid), max((len(row) for row in grid), default=0)),
+            "grid": [list(row) for row in grid],
+        }
+        for grid in chain[1:]
+    ]
+
+
+def build_animation_view(record: dict[str, Any] | None) -> dict[str, Any]:
+    """The diff timeline: which cells changed at each step.
+
+    The cheap first look. Reading a frame verbatim is no longer this
+    function's job - `last_transition.frames()` hands back FrameView objects
+    and the model crops their ascii the same way it crops current_frame, with
+    its own helpers rather than a region parameter it has to guess.
+    """
+    empty = {
+        "error": (
+            "No animation is available: the last executed action returned a single frame."
+        )
+    }
+    if not record:
+        return empty
+    chain: list[Grid] = list(record.get("chain") or ())
+    if len(chain) <= 2:
+        return empty
+
+    header: dict[str, Any] = {
+        "action": record.get("action_display"),
+        # excludes the prepended pre-action board, matching the metadata
+        "frames": len(chain) - 1,
+    }
+
+    steps: list[dict[str, Any]] = []
+    for position in range(1, len(chain)):
+        cells = _diff_cells(chain[position - 1], chain[position])
+        if cells:
+            steps.append({"_position": position, "_cells": cells})
+
+    rendered: list[dict[str, Any]] = []
+    remaining = ANIMATION_MAX_TOTAL_CELLS
+    for step in steps:
+        cells = step["_cells"]
+        entry: dict[str, Any] = {
+            "step": step["_position"],
+            "changed": len(cells),
+            "bbox": _bbox_text(cells),
+        }
+        budget = min(ANIMATION_MAX_CELLS_PER_STEP, max(0, remaining))
+        if len(cells) > ANIMATION_MAX_CELLS_PER_STEP or budget == 0:
+            # Too broad to enumerate usefully: a colour-transition census says
+            # more per token than an arbitrarily truncated cell list.
+            census: dict[str, int] = {}
+            for _, _, old, new in cells:
+                key = f"{_color_char(old)}>{_color_char(new)}"
+                census[key] = census.get(key, 0) + 1
+            entry["transitions"] = census
+        else:
+            entry["changes"] = _format_changes(cells, budget)
+            remaining -= min(len(cells), budget)
+        rendered.append(entry)
+
+    return {**header, "steps": rendered}
diff --git a/ARC3-Inference/inference/utils/frame_diff.py b/ARC3-Inference/inference/utils/frame_diff.py
new file mode 100644
index 0000000..a49b067
--- /dev/null
+++ b/ARC3-Inference/inference/utils/frame_diff.py
@@ -0,0 +1,307 @@
+"""Object-level frame diff core, shared by the sandbox runtime and the host.
+
+This module must stay dependency-free (stdlib only, no imports): its source is
+injected verbatim into the sandbox bootstrap template (see
+python_tool_sandbox.py), which runs under ``python -I -S`` and cannot import
+project modules. The host imports it normally for the auto frame-diff feature.
+"""
+
+
+def _node_topleft(node):
+    points = node.get("boundary") or []
+    if not points:
+        return (0, 0)
+    return (min(p[0] for p in points), min(p[1] for p in points))
+
+def _hungarian(cost):
+    # Minimum-cost assignment (Jonker-Volgonant style with potentials).
+    # cost: n x m matrix with n <= m. Returns list row -> column.
+    n = len(cost)
+    m = len(cost[0]) if n else 0
+    INF = float("inf")
+    u = [0.0] * (n + 1)
+    v = [0.0] * (m + 1)
+    p = [0] * (m + 1)      # p[j] = row assigned to column j (1-based; 0 = none)
+    way = [0] * (m + 1)
+    for i in range(1, n + 1):
+        p[0] = i
+        j0 = 0
+        minv = [INF] * (m + 1)
+        used = [False] * (m + 1)
+        while True:
+            used[j0] = True
+            i0 = p[j0]
+            delta = INF
+            j1 = 0
+            for j in range(1, m + 1):
+                if used[j]:
+                    continue
+                cur = cost[i0 - 1][j - 1] - u[i0] - v[j]
+                if cur < minv[j]:
+                    minv[j] = cur
+                    way[j] = j0
+                if minv[j] < delta:
+                    delta = minv[j]
+                    j1 = j
+            for j in range(m + 1):
+                if used[j]:
+                    u[p[j]] += delta
+                    v[j] -= delta
+                else:
+                    minv[j] -= delta
+            j0 = j1
+            if p[j0] == 0:
+                break
+        while True:
+            j1 = way[j0]
+            p[j0] = p[j1]
+            j0 = j1
+            if j0 == 0:
+                break
+    assignment = [-1] * n
+    for j in range(1, m + 1):
+        if p[j]:
+            assignment[p[j] - 1] = j - 1
+    return assignment
+
+def _node_bbox(node):
+    points = node.get("boundary") or []
+    if not points:
+        return (0, 0, 0, 0)
+    rows = [p[0] for p in points]; cols = [p[1] for p in points]
+    return (min(rows), min(cols), max(rows), max(cols))
+
+def _node_center2(node):
+    # bbox center in half-cell units (ints): (r0+r1, c0+c1)
+    points = node.get("boundary") or []
+    if not points:
+        return (0, 0)
+    rows = [p[0] for p in points]; cols = [p[1] for p in points]
+    return (min(rows) + max(rows), min(cols) + max(cols))
+
+def compute_frame_diff(grid_a, grid_b, nodes_a, nodes_b, max_group_match=None):
+    # Object-level diff between two same-shaped grids with precomputed
+    # segmentation nodes. Returns only what changed; unchanged objects are
+    # omitted. When max_group_match is set, identical-hash groups larger than
+    # it skip pairwise matching and report plain appeared/disappeared entries
+    # (guards against O(n^3) assignment on e.g. hundreds of identical pixels).
+    changed = set()
+    for r, (row_a, row_b) in enumerate(zip(grid_a, grid_b)):
+        if row_a == row_b:
+            continue
+        for c, (va, vb) in enumerate(zip(row_a, row_b)):
+            if va != vb:
+                changed.add((r, c))
+    if not changed:
+        return {"changed_cell_count": 0, "moved": [], "rotated": [],
+                "appeared": [], "disappeared": [],
+                "changed_color": [], "resized": []}
+
+    def touched_nodes(nodes):
+        touched = []
+        for node in nodes:
+            points = node.get("boundary") or []
+            if not points:
+                continue
+            r0 = min(p[0] for p in points); r1 = max(p[0] for p in points)
+            c0 = min(p[1] for p in points); c1 = max(p[1] for p in points)
+            # dilate by 1 so objects that shrank/grew still register as
+            # touched by the cells they vacated/claimed
+            if any(r0 - 1 <= r <= r1 + 1 and c0 - 1 <= c <= c1 + 1 for (r, c) in changed):
+                touched.append(node)
+        return touched
+
+    before_nodes = touched_nodes(nodes_a)
+    after_nodes = touched_nodes(nodes_b)
+
+    moved, appeared, disappeared = [], [], []
+    rotated = []
+
+    def _classify_match(node_a, node_b):
+        rot_a = node_a.get("rotation")
+        rot_b = node_b.get("rotation")
+        symmetry = node_a.get("rotational_symmetry") or 1
+        rotated_by = None
+        if rot_a is not None and rot_b is not None:
+            modulus = 180 if symmetry == 2 else 360
+            delta = (rot_b - rot_a) % modulus
+            if delta:
+                rotated_by = delta
+        pa, pb = _node_topleft(node_a), _node_topleft(node_b)
+        base_entry = {"color": node_a.get("color"),
+                      "pixels": node_a.get("pixels"),
+                      "hash": node_a.get("hash")}
+        if rotated_by is not None:
+            ca, cb = _node_center2(node_a), _node_center2(node_b)
+            in_place = (abs(ca[0] - cb[0]) <= 2 and abs(ca[1] - cb[1]) <= 2)
+            base_entry["rotated_by"] = rotated_by
+            if in_place:
+                base_entry["at"] = list(pb)
+                rotated.append(base_entry)
+            else:
+                base_entry["from"] = list(pa); base_entry["to"] = list(pb)
+                moved.append(base_entry)
+        elif pa != pb:
+            base_entry["from"] = list(pa); base_entry["to"] = list(pb)
+            moved.append(base_entry)
+        # same hash, same position, same rotation: unchanged -> omit
+
+    groups_before = {}
+    for node in before_nodes:
+        groups_before.setdefault(node.get("hash"), []).append(node)
+    groups_after = {}
+    for node in after_nodes:
+        groups_after.setdefault(node.get("hash"), []).append(node)
+
+    for key, group_a in groups_before.items():
+        group_b = groups_after.pop(key, [])
+        if (
+            max_group_match is not None
+            and max(len(group_a), len(group_b)) > max_group_match
+        ):
+            # too many identical objects to match affordably
+            for node_a in group_a:
+                disappeared.append({"color": node_a.get("color"),
+                                    "pixels": node_a.get("pixels"),
+                                    "hash": node_a.get("hash"),
+                                    "shape_hash": node_a.get("shape_hash"),
+                                    "at": list(_node_topleft(node_a)),
+                                    "bbox": list(_node_bbox(node_a))})
+            for node_b in group_b:
+                appeared.append({"color": node_b.get("color"),
+                                 "pixels": node_b.get("pixels"),
+                                 "hash": node_b.get("hash"),
+                                 "shape_hash": node_b.get("shape_hash"),
+                                 "at": list(_node_topleft(node_b)),
+                                 "bbox": list(_node_bbox(node_b))})
+            continue
+        if not group_b:
+            for node_a in group_a:
+                disappeared.append({"color": node_a.get("color"),
+                                    "pixels": node_a.get("pixels"),
+                                    "hash": node_a.get("hash"),
+                                    "shape_hash": node_a.get("shape_hash"),
+                                    "at": list(_node_topleft(node_a)),
+                                    "bbox": list(_node_bbox(node_a))})
+            continue
+        # optimal assignment over bbox-center distances; rows = smaller side
+        transposed = len(group_a) > len(group_b)
+        rows, cols = (group_b, group_a) if transposed else (group_a, group_b)
+        cost = []
+        for row_node in rows:
+            cr = _node_center2(row_node)
+            cost.append([
+                abs(cr[0] - _node_center2(col_node)[0])
+                + abs(cr[1] - _node_center2(col_node)[1])
+                for col_node in cols
+            ])
+        assignment = _hungarian(cost)
+        matched_cols = set()
+        for row_index, col_index in enumerate(assignment):
+            if col_index < 0:
+                continue
+            matched_cols.add(col_index)
+            if transposed:
+                _classify_match(cols[col_index], rows[row_index])
+            else:
+                _classify_match(rows[row_index], cols[col_index])
+        if transposed:
+            for col_index, node_a in enumerate(cols):
+                if col_index not in matched_cols:
+                    disappeared.append({"color": node_a.get("color"),
+                                        "pixels": node_a.get("pixels"),
+                                        "hash": node_a.get("hash"),
+                                    "shape_hash": node_a.get("shape_hash"),
+                                        "at": list(_node_topleft(node_a)),
+                                        "bbox": list(_node_bbox(node_a))})
+        else:
+            for col_index, node_b in enumerate(cols):
+                if col_index not in matched_cols:
+                    appeared.append({"color": node_b.get("color"),
+                                     "pixels": node_b.get("pixels"),
+                                     "hash": node_b.get("hash"),
+                             "shape_hash": node_b.get("shape_hash"),
+                                     "at": list(_node_topleft(node_b)),
+                                     "bbox": list(_node_bbox(node_b))})
+            continue
+        # transposed: every after-node (a row) got matched; nothing appeared
+    for group_b in groups_after.values():
+        for node_b in group_b:
+            appeared.append({"color": node_b.get("color"),
+                             "pixels": node_b.get("pixels"),
+                             "hash": node_b.get("hash"),
+                             "shape_hash": node_b.get("shape_hash"),
+                             "at": list(_node_topleft(node_b)),
+                             "bbox": list(_node_bbox(node_b))})
+    # Phase A - changed_color: EXACT pairs only. Same position, byte-
+    # identical shape (color-independent pose signature), different
+    # color. This asserts "these exact cells changed color" and nothing
+    # more; fuzzy transformation claims are deliberately not made.
+    changed_color = []
+    used_gone, used_new = set(), set()
+    for gi, gone in enumerate(disappeared):
+        for ni, new in enumerate(appeared):
+            if ni in used_new:
+                continue
+            if (gone["at"] == new["at"]
+                    and gone.get("shape_hash")
+                    and gone.get("shape_hash") == new.get("shape_hash")
+                    and gone["color"] != new["color"]):
+                changed_color.append({"at": gone["at"],
+                                      "pixels": gone["pixels"],
+                                      "before_color": gone["color"],
+                                      "after_color": new["color"],
+                                      "before_hash": gone.get("hash"),
+                                      "after_hash": new.get("hash")})
+                used_gone.add(gi); used_new.add(ni)
+                break
+    disappeared = [entry for index, entry in enumerate(disappeared) if index not in used_gone]
+    appeared = [entry for index, entry in enumerate(appeared) if index not in used_new]
+
+    # Phase B - resized: same-color pairs whose bboxes substantially
+    # overlap (overlap area / smaller bbox area >= 0.5); greedy
+    # one-to-one by descending overlap then size similarity.
+    resized = []
+
+    def _bbox_area(b):
+        return max(0, b[2] - b[0] + 1) * max(0, b[3] - b[1] + 1)
+
+    def _overlap_ratio(b1, b2):
+        inter_r = min(b1[2], b2[2]) - max(b1[0], b2[0]) + 1
+        inter_c = min(b1[3], b2[3]) - max(b1[1], b2[1]) + 1
+        if inter_r <= 0 or inter_c <= 0:
+            return 0.0
+        smaller = min(_bbox_area(b1), _bbox_area(b2))
+        return (inter_r * inter_c) / smaller if smaller else 0.0
+
+    candidates = []
+    for gi, gone in enumerate(disappeared):
+        for ni, new in enumerate(appeared):
+            if gone["color"] != new["color"]:
+                continue
+            bg = gone.get("bbox") or (0, 0, -1, -1)
+            bn = new.get("bbox") or (0, 0, -1, -1)
+            ratio = _overlap_ratio(bg, bn)
+            if ratio >= 0.5:
+                area_g, area_n = _bbox_area(bg), _bbox_area(bn)
+                similarity = (min(area_g, area_n) / max(area_g, area_n)) if max(area_g, area_n) else 0.0
+                candidates.append((ratio, similarity, gi, ni))
+    candidates.sort(key=lambda item: (-item[0], -item[1]))
+    used_gone, used_new = set(), set()
+    for ratio, similarity, gi, ni in candidates:
+        if gi in used_gone or ni in used_new:
+            continue
+        used_gone.add(gi); used_new.add(ni)
+        gone, new = disappeared[gi], appeared[ni]
+        resized.append({"at_before": gone["at"], "at_after": new["at"],
+                        "color": gone["color"],
+                        "before_pixels": gone["pixels"],
+                        "after_pixels": new["pixels"],
+                        "before_hash": gone.get("hash"),
+                        "after_hash": new.get("hash")})
+    disappeared = [entry for index, entry in enumerate(disappeared) if index not in used_gone]
+    appeared = [entry for index, entry in enumerate(appeared) if index not in used_new]
+    return {"changed_cell_count": len(changed), "moved": moved,
+            "rotated": rotated,
+            "appeared": appeared, "disappeared": disappeared,
+            "changed_color": changed_color, "resized": resized}
diff --git a/ARC3-Inference/inference/utils/openai_compat.py b/ARC3-Inference/inference/utils/openai_compat.py
index 157e113..2fadb51 100644
--- a/ARC3-Inference/inference/utils/openai_compat.py
+++ b/ARC3-Inference/inference/utils/openai_compat.py
@@ -1,6 +1,8 @@
 """Helpers for provider-specific OpenAI-compatible requests."""
 from __future__ import annotations
 
+import json
+
 from typing import Any
 
 
@@ -46,22 +48,46 @@ def build_chat_payload(
     tools: list[dict[str, Any]] | None = None,
     tool_choice: str | None = None,
     seed: int | None = None,
+    priority: int | None = None,
+    stream: bool = False,
 ) -> dict[str, Any]:
     payload: dict[str, Any] = {
         "model": model,
         "messages": messages,
-        "stream": False,
+        "stream": bool(stream),
         "temperature": temperature,
         "top_p": top_p,
     }
     if max_tokens is not None:
         payload["max_tokens"] = max_tokens
+    if priority is not None:
+        # scheduling hint, ignored by servers that do not implement it
+        payload["priority"] = int(priority)
     if tools:
         payload["tools"] = tools
         if tool_choice:
             payload["tool_choice"] = tool_choice
 
     normalized = normalize_provider(provider)
+    if stream:
+        payload["stream_options"] = {"include_usage": True}
+    if normalized == "openrouter":
+        import os
+        if not payload["stream"] and os.environ.get(
+            "OPENROUTER_STREAM", ""
+        ).strip().lower() not in ("0", "false", "no", "off"):
+            payload["stream"] = True
+        if payload["stream"]:
+            payload["stream_options"] = {"include_usage": True}
+        payload["reasoning"] = {"enabled": bool(thinking)}
+        _order = os.environ.get("OPENROUTER_PROVIDER_ORDER", "").strip()
+        if _order:
+            payload["provider"] = {
+                "order": [s.strip() for s in _order.split(",") if s.strip()],
+                "allow_fallbacks": os.environ.get(
+                    "OPENROUTER_ALLOW_FALLBACKS", ""
+                ).strip().lower() in ("1", "true", "yes"),
+            }
     if normalized == "vllm":
         if top_k > 0:
             payload["top_k"] = top_k
@@ -70,3 +96,101 @@ def build_chat_payload(
             payload["seed"] = seed
 
     return payload
+
+
+def assemble_streamed_chat_response(lines) -> dict[str, Any]:
+    """Consume an SSE chat-completion stream and rebuild the non-streaming
+    response shape: {"choices": [{"message", "finish_reason"}], "usage"}.
+
+    Tolerates keep-alive comments (": ..."), event: lines, usage-only chunks,
+    and mid-stream error events. If an error arrives and no assistant content
+    was produced, returns {"error": ..., "choices": []} so callers surface it.
+    """
+    content_parts: list[str] = []
+    reasoning_parts: list[str] = []
+    tool_calls_by_index: dict[int, dict[str, Any]] = {}
+    finish_reason = ""
+    usage: dict[str, Any] | None = None
+    error: Any = None
+    provider_name = [""]
+
+    for raw_line in lines:
+        if raw_line is None:
+            continue
+        line = raw_line.decode("utf-8", "replace") if isinstance(raw_line, bytes) else str(raw_line)
+        line = line.strip()
+        if not line or line.startswith(":") or line.startswith("event:"):
+            continue
+        if line.startswith("data:"):
+            line = line[5:].strip()
+        if line == "[DONE]":
+            break
+        try:
+            chunk = json.loads(line)
+        except json.JSONDecodeError:
+            continue
+        if not isinstance(chunk, dict):
+            continue
+        if chunk.get("error"):
+            error = chunk["error"]
+            continue
+        if isinstance(chunk.get("usage"), dict):
+            usage = chunk["usage"]
+        if chunk.get("provider") and not provider_name[0]:
+            provider_name[0] = str(chunk["provider"])
+        for choice in chunk.get("choices") or []:
+            if not isinstance(choice, dict):
+                continue
+            if choice.get("finish_reason"):
+                finish_reason = str(choice["finish_reason"])
+            delta = choice.get("delta") or {}
+            if not isinstance(delta, dict):
+                continue
+            if isinstance(delta.get("content"), str):
+                content_parts.append(delta["content"])
+            if isinstance(delta.get("reasoning"), str):
+                reasoning_parts.append(delta["reasoning"])
+            elif isinstance(delta.get("reasoning_content"), str):
+                reasoning_parts.append(delta["reasoning_content"])
+            elif isinstance(delta.get("reasoning_details"), list):
+                for detail in delta["reasoning_details"]:
+                    if isinstance(detail, dict) and isinstance(detail.get("text"), str):
+                        reasoning_parts.append(detail["text"])
+            for fragment in delta.get("tool_calls") or []:
+                if not isinstance(fragment, dict):
+                    continue
+                index = int(fragment.get("index", 0) or 0)
+                entry = tool_calls_by_index.setdefault(
+                    index,
+                    {"id": "", "type": "function", "function": {"name": "", "arguments": ""}},
+                )
+                if fragment.get("id"):
+                    entry["id"] = str(fragment["id"])
+                if fragment.get("type"):
+                    entry["type"] = str(fragment["type"])
+                function = fragment.get("function") or {}
+                if isinstance(function, dict):
+                    if function.get("name"):
+                        entry["function"]["name"] = str(function["name"])
+                    if isinstance(function.get("arguments"), str):
+                        entry["function"]["arguments"] += function["arguments"]
+
+    produced = bool(content_parts or reasoning_parts or tool_calls_by_index)
+    if error is not None and not produced:
+        return {"error": error, "choices": []}
+
+    message: dict[str, Any] = {"role": "assistant", "content": "".join(content_parts)}
+    if reasoning_parts:
+        message["reasoning"] = "".join(reasoning_parts)
+    if tool_calls_by_index:
+        message["tool_calls"] = [tool_calls_by_index[i] for i in sorted(tool_calls_by_index)]
+    result: dict[str, Any] = {
+        "choices": [{"message": message, "finish_reason": finish_reason}],
+    }
+    if usage is not None:
+        result["usage"] = usage
+    if error is not None:
+        result["error"] = error
+    if provider_name[0]:
+        result["provider"] = provider_name[0]
+    return result
diff --git a/ARC3-Inference/inference/utils/retained_functions.py b/ARC3-Inference/inference/utils/retained_functions.py
new file mode 100644
index 0000000..66aa5a2
--- /dev/null
+++ b/ARC3-Inference/inference/utils/retained_functions.py
@@ -0,0 +1,301 @@
+"""Conservative source-only function retention; no snippet state is serialized.
+
+Also embedded into the isolated Python runner. Keep this module stdlib-only.
+"""
+import ast
+import dis
+import sys
+import types
+
+
+def _import_statements(nodes):
+    """One explicit absolute import per binding; never replay wildcard imports."""
+    statements = {}
+    for node in nodes:
+        if isinstance(node, ast.Import):
+            for alias in node.names:
+                binding = alias.asname or alias.name.split('.')[0]
+                statements[binding] = ast.unparse(ast.Import(names=[alias]))
+        elif isinstance(node, ast.ImportFrom) and not node.level and node.module:
+            for alias in node.names:
+                if alias.name == '*':
+                    raise ValueError('wildcard imports are not retained; use explicit imports inside the function')
+                statements[alias.asname or alias.name] = ast.unparse(
+                    ast.ImportFrom(module=node.module, names=[alias], level=0))
+        else:
+            raise ValueError('only explicit absolute imports can accompany retained functions')
+    return statements
+
+
+def _source_parts(source, retain_imports=True):
+    tree = ast.parse(source, '<retained>', 'exec')
+    if (not tree.body or not isinstance(tree.body[-1], ast.FunctionDef)
+            or (not retain_imports and len(tree.body) != 1)):
+        raise ValueError('only top-level function definitions can be retained')
+    imports = _import_statements(tree.body[:-1])
+    return tree.body[-1], imports
+
+
+def function_signature(source):
+    node, _ = _source_parts(source)
+    return node.name + '(' + ast.unparse(node.args) + ')'
+
+
+def _function_record(source, retain_imports=False):
+    node, imports = _source_parts(source, retain_imports)
+    if getattr(node, "type_params", []):
+        raise ValueError("generic type parameters are not retained")
+    if node.decorator_list:
+        raise ValueError('decorated functions are not retained')
+    args = [*node.args.posonlyargs, *node.args.args, *node.args.kwonlyargs]
+    args += [a for a in (node.args.vararg, node.args.kwarg) if a is not None]
+    if node.returns is not None or any(a.annotation is not None for a in args):
+        raise ValueError('use an unannotated definition for retention')
+    # Restoration must never evaluate model calls or snapshot a frame via defaults.
+    for default in [*node.args.defaults, *node.args.kw_defaults]:
+        if default is not None:
+            try:
+                ast.literal_eval(default)
+            except Exception:
+                raise ValueError('defaults must be literals; pass changing data as arguments') from None
+    tree = ast.Module(body=[node], type_ignores=[])
+    code = next(c for c in compile(tree, '<retained>', 'exec').co_consts
+                if isinstance(c, types.CodeType))
+    reads = set()
+    pending = [code]
+    while pending:
+        current = pending.pop()
+        for instruction in dis.get_instructions(current):
+            if instruction.opname in ('LOAD_GLOBAL', 'LOAD_NAME'):
+                reads.add(instruction.argval)
+            elif instruction.opname in ('STORE_GLOBAL', 'DELETE_GLOBAL'):
+                raise ValueError('functions that assign global variables are not retained')
+        pending.extend(c for c in current.co_consts if isinstance(c, types.CodeType))
+    return node.name, source, reads - imports.keys()
+
+
+def _rejection(name, reason, hint=None, **metadata):
+    result = {'name': name, 'reason': reason, **metadata}
+    if hint:
+        result['hint'] = hint
+    return result
+
+
+def _validation_hint(reason):
+    if 'decorat' in reason:
+        return 'Use a plain top-level def without decorators if you want to reuse it.'
+    if 'annotat' in reason:
+        return 'Remove parameter and return annotations if you want to retain this function.'
+    if 'default' in reason:
+        return 'Use literal defaults and pass changing values explicitly as arguments.'
+    if 'global variables' in reason:
+        return 'Return updated values instead of assigning global variables.'
+    if 'import' in reason:
+        return 'Put explicit, sandbox-supported imports inside the function.'
+    return 'Use a plain top-level def if you want to retain this function.'
+
+
+def prepare_functions(sources, available, *, retain_imports=False):
+    """Validate sources without executing them; drop invalid dependency chains."""
+    records, rejected = {}, []
+    for source in sources:
+        name = '<unknown>'
+        try:
+            parsed = ast.parse(source)
+            if (parsed.body and isinstance(parsed.body[-1], ast.FunctionDef)
+                    and (retain_imports or len(parsed.body) == 1)):
+                name = parsed.body[-1].name
+                records.pop(name, None)
+            name, source, reads = _function_record(source, retain_imports)
+            _, imports = _source_parts(source)
+            if imports.keys() & set(available):
+                raise ValueError('import alias conflicts with a provided global or builtin')
+            records[name] = (source, reads)
+        except Exception as exc:
+            reason = str(exc)[:180]
+            rejected.append(_rejection(name, reason, _validation_hint(reason)))
+    # Import bundles must not overwrite another retained function at restore.
+    function_names = set(records)
+    for name, (source, _) in list(records.items()):
+        _, imports = _source_parts(source)
+        if imports.keys() & function_names:
+            del records[name]
+            rejected.append(_rejection(name, 'import alias conflicts with a retained function',
+                                       'Choose a different import alias or function name.'))
+    # A wrapper must not survive its unavailable dependency. Re-run to a fixed point.
+    while True:
+        dropped = []
+        for name, (source, reads) in records.items():
+            missing = reads - set(available) - records.keys()
+            if missing:
+                dropped.append(name)
+                names = ', '.join(sorted(missing))
+                rejected.append(_rejection(
+                    name, 'unavailable dependencies: ' + names,
+                    'Pass ' + names + ' as arguments, or put their imports inside the function.',
+                    missing_dependencies=sorted(missing)))
+        if not dropped:
+            break
+        for name in dropped:
+            del records[name]
+    return {name: value[0] for name, value in records.items()}, rejected
+
+
+def restore_functions(sources, namespace, available, *, retain_imports=False,
+                      repair_hints=False):
+    """Restore definitions and explicit imports through the sandbox importer.
+
+    Never replay assignments, expressions, decorators, or previous actions.
+    """
+    kept, rejected = prepare_functions(sources, available, retain_imports=retain_imports)
+    failed = set()
+    for name, source in kept.items():
+        before = dict(namespace)
+        try:
+            if name in available:
+                raise ValueError('name conflicts with a provided global or builtin')
+            exec(compile(source, '<retained>', 'exec'), namespace, namespace)
+        except Exception:
+            # A multi-import bundle can fail partway through. Restore bindings
+            # so one failed helper does not replace another helper's globals.
+            namespace.clear()
+            namespace.update(before)
+            failed.add(name)
+            reason = ('could not restore this definition or its imports' if repair_hints else
+                      'could not restore this definition; redefine it if needed')
+            rejected.append(_rejection(name, reason,
+                                       'Check its imports are supported by the sandbox, then redefine it if needed.'))
+    survivors, dependencies = prepare_functions(
+        [source for name, source in kept.items() if name not in failed], available,
+        retain_imports=retain_imports)
+    rejected.extend(dependencies)
+    for name in kept.keys() - survivors.keys():
+        if name not in available:
+            namespace.pop(name, None)
+    return survivors, rejected
+
+
+def _matching_imports(nodes, namespace):
+    """Recognize live import bindings without importing or evaluating anything.
+
+    Only module bindings and callable/immutable exports are eligible. Ordinary
+    snippet assignments and mutable imported data are never serialized. Looking
+    in module dictionaries avoids invoking __getattr__ during bookkeeping.
+    """
+    matches, declared = {}, {}
+    for node in nodes:
+        if not isinstance(node, (ast.Import, ast.ImportFrom)):
+            continue
+        try:
+            statements = _import_statements([node])
+        except ValueError:
+            continue
+        declared.update(statements)
+        for binding, statement in statements.items():
+            single = ast.parse(statement).body[0]
+            alias = single.names[0]
+            if isinstance(single, ast.Import):
+                # Even unaliased dotted imports must already have executed.
+                if alias.name not in sys.modules:
+                    continue
+                module_name = alias.name if alias.asname else alias.name.split('.')[0]
+                expected = sys.modules.get(module_name)
+            else:
+                module = sys.modules.get(single.module)
+                expected = vars(module).get(alias.name) if type(module) is types.ModuleType else None
+            eligible = type(expected) in (
+                types.ModuleType, types.FunctionType, types.BuiltinFunctionType, types.MethodType,
+                type, str, bytes, int, float, complex, bool,
+            )
+            if eligible and binding in namespace and namespace[binding] is expected:
+                matches[binding] = statement
+    return matches, declared
+
+
+def collect_functions(code_text, previous, namespace, available, *, retain_imports=False,
+                      repair_hints=False):
+    """Collect from the actual namespace, including after partial execution.
+
+    A later def may never have run. Match the live function's source location
+    rather than choosing the last definition in the submitted snippet.
+    """
+    candidates = {}
+    rejected = []
+    try:
+        tree = ast.parse(code_text, '<python_tool>')
+    except SyntaxError:
+        # Parsing failed before execution; restored functions are still valid.
+        tree = ast.Module(body=[], type_ignores=[])
+    fresh = {}
+    for node in tree.body:
+        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
+            fresh.setdefault(node.name, []).append(node)
+    import_nodes = []
+    for source in previous.values():
+        # Old source-only records remain valid; no migration is necessary.
+        import_nodes.extend(ast.parse(source).body[:-1])
+    import_nodes.extend(tree.body)
+    imports, declared_imports = (_matching_imports(import_nodes, namespace)
+                                if retain_imports or repair_hints else ({}, {}))
+    for name in dict.fromkeys([*previous, *fresh]):
+        value = namespace.get(name)
+        if not repair_hints and (name in available or type(value) is not types.FunctionType
+                                 or value.__name__ != name):
+            rejected.append(_rejection(name, 'definition did not execute, was removed/replaced, or conflicts with a provided name'))
+            continue
+        if name in available:
+            rejected.append(_rejection(name, 'name conflicts with a provided global or builtin',
+                                       'Choose a different function name.'))
+            continue
+        if name not in namespace:
+            reason = 'function was removed' if name in previous else 'definition was not reached or was removed'
+            rejected.append(_rejection(name, reason, 'Define it before calling it if it is still needed.'))
+            continue
+        if type(value) is not types.FunctionType or value.__name__ != name:
+            rejected.append(_rejection(name, 'function name was replaced by another value',
+                                       'Use a separate variable name; redefine the function only if needed.'))
+            continue
+        if name in previous and value.__code__.co_filename == '<retained>':
+            # Preserve the restored definition when execution stopped before a
+            # replacement (or while evaluating that replacement's defaults).
+            candidates[name] = previous[name]
+            continue
+        node = next((node for node in fresh.get(name, [])
+                     if isinstance(node, ast.FunctionDef) and not node.decorator_list
+                     and value.__code__.co_filename == '<python_tool>'
+                     and value.__code__.co_firstlineno == node.lineno), None)
+        if node is None:
+            rejected.append(_rejection(name, 'definition was decorated, replaced outside a top-level def, or is unsupported',
+                                       'Use a plain top-level def without decorators if you want to reuse it.'))
+            continue
+        candidates[name] = ast.get_source_segment(code_text, node)
+    # Rebuild every bundle from the LIVE namespace. An old helper must not
+    # silently restore an obsolete import when that alias was rebound/deleted.
+    for name, source in (list(candidates.items()) if retain_imports else []):
+        try:
+            node, _ = _source_parts(source)
+            definition = ast.get_source_segment(source, node)
+            _, _, reads = _function_record(definition)
+            statements = [imports[n] for n in sorted(reads & imports.keys()) if n not in available]
+            candidates[name] = '\n'.join([*statements, definition])
+        except Exception:
+            # Validation below produces the specific rejection and prunes callers.
+            pass
+    kept, invalid = prepare_functions(candidates.values(), available, retain_imports=retain_imports)
+    for item in invalid:
+        missing = item.get('missing_dependencies', [])
+        if not missing:
+            continue
+        hints = []
+        missing_imports = [n for n in missing if n in declared_imports]
+        missing_helpers = [n for n in missing if n in candidates and n not in declared_imports]
+        missing_data = sorted(set(missing) - set(missing_imports) - set(missing_helpers))
+        if missing_imports:
+            statements = '; '.join(declared_imports[n] for n in missing_imports)
+            hints.append('Put `' + statements + '` inside this function if you want to reuse it.')
+        if missing_helpers:
+            hints.append('Make the helper dependencies retainable too: ' + ', '.join(missing_helpers) + '.')
+        if missing_data:
+            hints.append('Pass ' + ', '.join(missing_data) + ' as arguments; snippet variables are not retained.')
+        item['hint'] = ' '.join(hints)
+    return kept, rejected + invalid
diff --git a/ARC3-Inference/inference/utils/segmentation.py b/ARC3-Inference/inference/utils/segmentation.py
index 20f10ed..33f8848 100644
--- a/ARC3-Inference/inference/utils/segmentation.py
+++ b/ARC3-Inference/inference/utils/segmentation.py
@@ -62,17 +62,61 @@ def _corner_points(contour):
     return corners
 
 
-def _object_hash(cells, color):
-    """Translation-invariant signature of an object: its color plus its cell shape,
-    normalized so the top-left of its bounding box is the origin. Same shape + color
-    => same hash regardless of position, so objects can be matched across frames."""
+def _normalize_cells(cells):
     min_r = min(r for r, _ in cells)
     min_c = min(c for _, c in cells)
-    norm = sorted((r - min_r, c - min_c) for r, c in cells)
-    payload = repr((color, norm)).encode()
+    return sorted((r - min_r, c - min_c) for r, c in cells)
+
+
+def _rotate_cells_90(norm_cells):
+    """Rotate a normalized cell set 90 degrees clockwise; returns normalized."""
+    max_r = max(r for r, _ in norm_cells)
+    return sorted((c, max_r - r) for r, c in norm_cells)
+
+
+def _cells_hash(norm_cells, color):
+    payload = repr((color, norm_cells)).encode()
     return hashlib.sha1(payload).hexdigest()[:16]
 
 
+def _object_signature(cells, color):
+    """Rotation-canonical identity of an object.
+
+    Hashes the shape in all four 90-degree rotations (extend the transform list
+    with reflections for full D4 canonicalization later) and takes the lowest
+    hash as the canonical id. Returns (canonical_hash, rotation, symmetry,
+    pose_hash) where:
+      - canonical_hash: identical for all rotated copies of the same shape+color.
+      - rotation: degrees CLOCKWISE to rotate the canonical form to obtain the
+        observed object. None for 4-fold symmetric shapes (all rotations equal);
+        reduced mod 180 (i.e. 0 or 90) for 2-fold symmetric shapes.
+      - symmetry: rotational symmetry order (1, 2, or 4).
+      - pose_hash: the classic rotation-SENSITIVE hash of the observed pose.
+    """
+    norm = _normalize_cells(cells)
+    variants = [norm]
+    for _ in range(3):
+        variants.append(_rotate_cells_90(variants[-1]))
+    hashes = [_cells_hash(v, color) for v in variants]
+    # The reference orientation is picked from the COLOUR-FREE hash. Choosing it
+    # from the colour-bearing one made the argmin depend on the colour, so the
+    # same shape in the same orientation reported a different rotation for a red
+    # object than for a blue one - and a model comparing rotations across
+    # objects was reading noise.
+    shape_hashes = [_cells_hash(v, "__shape__") for v in variants]
+    k = shape_hashes.index(min(shape_hashes))
+    canonical = hashes[k]
+    distinct = len(set(shape_hashes))
+    symmetry = {1: 4, 2: 2, 4: 1}.get(distinct, 1)
+    rotation = ((4 - k) % 4) * 90  # so observed = canonical rotated this much CW
+    if symmetry == 4:
+        rotation = None
+    elif symmetry == 2:
+        rotation = rotation % 180
+    shape_hash = _cells_hash(norm, "__shape__")
+    return canonical, rotation, symmetry, hashes[0], shape_hash
+
+
 def segment_layer(layer, color_chars):
     """Segment one frame layer into connected-component nodes.
 
@@ -189,11 +233,16 @@ def segment_layer(layer, color_chars):
         comp = components[cid]
         color = color_chars[max(0, min(15, comp["value"]))]
         boundary = _corner_points(_trace_outer_contour(comp["cells"], comp["start"]))
+        signature = _object_signature(comp["cells"], color)
         nodes.append(
             {
                 "id": cid,
                 "color": color,
-                "hash": _object_hash(comp["cells"], color),
+                "hash": signature[0],
+                "rotation": signature[1],
+                "rotational_symmetry": signature[2],
+                "pose_hash": signature[3],
+                "shape_hash": signature[4],
                 "pixels": len(comp["cells"]),
                 "boundary": [[r, c] for r, c in boundary],
                 "children": children[cid],

## 1. Environment and submission mode

Detect whether this is a real competition rerun (which minimises diagnostics), set the
framework's environment flags, and put the CUDA libraries on the linker path.

In [ ]:
import json
import os
import pickle
import subprocess
import sys
import sysconfig
import time
from datetime import datetime, timedelta
from pathlib import Path
from urllib.request import urlopen

# Kaggle might cancel the run in no action is done in some timeframe.
# To prevent this, harness will start up even if backend is not yet ready,
# and the first started game will perform a single reset action.
NOTEBOOK_START_TIME = time.time()
SERVER_STARTUP_TIMEOUT = 12 * 60
os.environ['ARC3_WARMUP_ACTION_GAMES'] = '1'
os.environ['ARC3_HTTP_RETRY_INITIAL_SECONDS'] = '900'

# model and dataset paths
WHEELHOUSE_DIR    = '/kaggle/input/datasets/dfranzen/pennyroyal-v253'
MODEL_DIR         = '/kaggle/input/models/dfranzen/intel-qwen3.8-flash-next-w4a16-autoround/transformers/default/1'
DRAFT_MODEL_DIR   = '/kaggle/input/models/dfranzen/albucino-qwen3-8-flash-next-drafter/transformers/default/1'
ORIG_BUNDLE_DIR   = '/kaggle/input/datasets/dfranzen/taaf-kaggle-source-bundle-copy'
SERVED_MODEL_NAME = 'flashnext'

# Apply harness patch
BUNDLE_DIR = Path('/kaggle/taaf-kaggle-source-share')
!rm -Rf $BUNDLE_DIR
!cp -a $ORIG_BUNDLE_DIR $BUNDLE_DIR
subprocess.run(["git", "apply", "--include=ARC3-Inference/*", "-v", "/kaggle/harness-changes.patch"], cwd=f"{BUNDLE_DIR}/src", check=True)
print('harness patch applied successfully')

# --- INJECT FRONTIER 70% UPGRADES INTO BUNDLE ---
import base64
import gzip
from pathlib import Path

_UPGRADE_PAYLOAD = {'inference/utils/sim_helpers.py': 'H4sIAF414GoC/81Y3XPjthF/11+x9T2YrGWd1FzzoDllJm3ukk7zcRO7zYNG5YAkJMGmQAYEbSuu/vfuLvgFibIv7Us9c0dqsd+7WPzAi4uLj6K0oPR1KXQa509Qql2VCatyPYZC2O1a6VTpzRhwHYy0Jk9VQsuwlVkhTTkZjW63qoRdnlaZBHxT2kpNLCLL9lDKbH2d5NoKpWUKQVEZCaVFdcKkkKnYCLMPocxB2RIflUnkKBEaYmQrMpWg0IM0Mfq0I9U52K2ET3u7zfW1zfMMGtfjPLelNaKYjC4uLkajtcl3kORZJtlhdGxX5MZCKn+t5GgUffPh+9uvb2ABzyPAv9kcguvZGKbhGOAN/OMTU/+E1IaI1G9++uVHpn+B9OkYrmeO+/sPH2+Z/s7RZzX/z3/79rvb0WE0SuUaKJdRuUUfZGkjym6AiTB2DJtcZCUlvCxFnMkokRn93hhFAqKQi+DLd2P48l0YztkMBvit1NKIDP7y8QYapW3JpIEcs9ZqZFXAaiesgP/72mxKp4/+2Jc52KpA/sCMIQnbNXZwjsW0bzOFdvK14yvh2XGOYTKZHCA32EF6gwoG1fgBOn3KYhjk4hmdnQ9tNjDHW6k2W8zco0rttjPALz9LWxndi6xxmTpzg1kRdUMsZ5PJu1WTUH4q5MIIqEETGdR1YbfCuZ8Nahx+cR7KrJSnHBigU3LkIprhdKNLdW7bdcPew3LV1amtolPoZ9FpxpauaJ2bO1g2jbVchSvH8IBhWdxL6DWvOa+3mEEkdantjD5uFVpkvZ1zSWWMGwwoxWuTIsfkrG3QRYjBEd9AbL34SIefE17EuqNmEm9p65wrNoYgxeU0CUlxvXsnGNOuDELfhEY+TXoMXEHKOvGZ+DxPFhkCxxp6S+j/FN4vUA28hy1PPvc7wd+PvqmaPyB1SneVQp9rkis/ayGKzplal+NUWa9WE5GmpDgcZHLZF0UhdRoQW12XK1hitlahk+q6iUcQzgS13kf9QR7UIz830brSCXa7ERrt41KEgx3p+27o/JPkFW5RuxUWBOCgTlUqrOxJQauwOzBKyoztMb0whjx35mgiyzil1NNWjuvdG8L1V6Dlk42Y3Iqfej9v97/zZPl8ySKXOH8izNqlU4g/Bf/slDLHs72aHQ6rV0ZMPe7wDMrGbCfEIaXK6EFkKh3DWqgMD74olXgOZqE3cqi/Vfo0xokgC+oNqasdzkQrg7Ol4ERF1MEkNNngVKiD6ppFHK3XYXYM8qnAo5FnQsfViz7sJXXvd2phJAXpZL2CBZxTtH2ypzqZPyxa26cboO7Yj7hrsNjPg71/Sf5GdM49YY0oeefYujoPc3i1H2Zp/Uau9v0MbxMWsjavp5yHXgUSWVj4wA/aOqIk2uDAPJuQz0jGy4l4JQmX0pjckLA1AXoX+iyH/py5NRX5WAOeMs8eZFRUeLLwcUU9XbqjKSoysZc4fBG5STpihcH2w5dH8TuhD9kwvIkaQ8CG4C3c5Pd5jFByJ5Ot0Cp5BfvUPp2BQOzoHBBV/iY1ARf89ypmqcP6vWKcBIZH/x+oqA4DN3sbR1DTnBVOjrfMFLfI0dTYhd/D17AHTota46JN4UsASWkckiJzc4uO9YEeG8ZJnuQZvOTxHDqrb+AGrxTUb7lOxIOye1Qr0ixP7iHZyuSeuWgj4EHQrMg0iCPMRG+QU+/G5GZC059X58fDM3AMYYMfTrLS/L2Bv5Jp9Mlo6uXgx7c3XAEHQYIPb39xP08kNd4hthGtUQZjA9cwczbRnqshO0pFmZ5I483Nl746L70l1ScaHulW1CngfCDfkYJk2LwUp8JXw8KPg9YZxHUZoItMG1Gdus5BXG0NhsMYrjcQR0OTfMTUnXiK5IO7K8z+PJ26wBrKdBCNszOO5X2noPPCLV0tekEGRVTkOFVd530OgGfOwe13DsN7OL6gWwIhcDb8v0B5VkBFLRowXzg0Hx7xEZBnk9q3WQdE+yZwML44xvUFA/uQQbuz13TNaW3pU4bSvaKexN4YbDUN7Gi3Uz/hgYW3ZZpPp7sxbgLXbeR6MHTHTcsxhx+fhH+agvg4BXEvBbGXAp90JpYXM8OQsjkh3PTDLfjsEnQI4d/4zgYOQ177s7PTFf43XrQHhK53RHdg9TT7+fVv9i+rcxvs9d7ohF+7D3p3wVbs1MTRdbDl/JxbocNpsSrw0ERjEV5i1UbvpLaB+yDXAbQOhH1X6Y0wir7UNYKww6NyV+3gB6HxfmhxLVXuUwoOKptQs/cg2CehDOJdkWzr736IW9BV+r6XIxzeIcJzVsHmTrX6DS+aucWFRu9kGNsQSIdnpzVi/DSvdblfBw/e1EFiIQkTNTGHR9CH1zzUg5XMpG756V5Dv/1Uua2OMz+LBZ7LGMrGSJnusQmEoW92mkBbnHcfPMQD3hF733s8m+57kDI0NxaEtPuzlbdo7c58aAK2qk8bLTZS3HvUmLxbUOaDVm4M93K/yMQuTgVYvDHECOiX0xVuaIuPEBvNkWaONFsd7aba92W5QtVkwVtt7UyM3OUPMiCOcHRy7rAOH99qWHjVGLlpgKc0tp5RT7i8bCWW7KTjXKpVHYDL8/Ju1Y+kY5odMWFsnPU7yjre1Dcy0OHKK4jqL/HKqo8fuz0ksk1usPt3EPy92urrHyp9b6inSMtPgf7XF2G7hVi4onjQ7T/i8GGk41DrILUYpD6K/SB9NOA8QjnH0fVNQZILUC3hbgp9ZIZtw96ss1zY4FLp9WV4bIoDKRllLxkVDTE46ENAym9aElzesRMeyuJ2J18KWvXIqcysoGtKzyeP4W4Gx+jyqMIDmTjaY86v1ZnjqTL0fbNryyW6inh0tbzjB/5XIYmeD6hkUIf7vkrgD1N81lJTA2RwX1TPcmEjOKa76Tl7jaL3LoXnLTYZrgXO8nGi715K9AtZfjHD1RLrvloRAGZnhr+wUjDX5ziGT/x+QgdFeQPczU6+e7kmxb56beie63ROFlfpqJ9r1fScrT7HFyJ/hiPeYMUxVGW2O2xe3xAu6BV8dWzJqVo2M5WZqOsphnawMqUPU5zU6D/KI9l6rhwAAA==', 'inference/agent/prompts.py': 'H4sIAF414GoC/81963IbR5bmfz1FNTt2RW4AkGQrPLNU9OzCJGQxmhctL/JoPQ6gCBSAMgtVcF0I0QpFzENMxMY+z77JPMmc7zsns7IAUtPd07PR/iGTRFXmyZPnfsPe3t77slit66hOVussrpMqmhdlVC+TKM7j7OG3pIziRZLXg729vWfP5vJwlObzpEzyaTJo6jSrBosynY35Y5Su1kVZR8PLo/HRxenF5fh09MPo/PjZs9PRh9Hp+PpyeH71dnQ5vvp4dT06G/9wc3I8PD8aRX+I9p9F8t9ePzpN7hNZqamaOMseotsmzWZRkUerZLqM83RaRVkSl3kyEzgi+SlLBcSML/WipFon05Qv4giroqqjPV25TKZyDFkpGURHcVk+4KCbuJxFVbMG1LLiXV5ssmS2kMNXURxVdVzWab6Ilg/rQtarUtlis0yzpF20P10m0zs8FOcP9RI/TIu8LuNZOsWStw9Rnmyi5D6dEWXRufym4EbFvE5yOUZdFrNm6heNZ7O0TgtBf3voXlQVq6ROV3JB9bIsmsUyavJ5vEqzNC6j2wIHSbJkJWesBtG1ACuHsIUqv3KZuE15UbEgBNddFdk9IAfOCBtXiBYFQIgfojJZxYJtfFzFq0QupZa//dqkpQcah2yvqC7k83i6jNK6Fxk9cbW0rpJszkXx7CIZ/FO+9+zAEcjV9fDyenxzJSTy/vLi7P11SxkfiyZaxveJoFcoNQFuseq6FNQWTeXgnkybUoizHs9LgXQSyY3yQgT+Sg/gLtUAV9QVc36YJ59qXekN7jOKp9hNfuK96sp90LsgUNgiqpbFBsumFW+Zaw3cyt+TdGN+IryVG2fJs1n8UACFseyJP8lngDHNG3cL3WM9F2qc4ibd0pUgn+T0Tznwx79d4WAROZS07y9D9hLGqOPbLK2WCZlpd4s30awQZOFeZ2k1Le4FMFtYHl5Fm7ReEuYkrop8EJ3kYLXasGZH5wkDsnQE2bPLnrWEWOJ3+1BeEiZISj0jAS5b8Jp8lpQCfj4DcmaFcADgTD4JTtNcGcqWdfvtcFaLjDxJZiCdgkSf2H0AA2Ax2TUtSmGa3+RuVyJHem5pEXpFuYrrFO8INisRBAkRmQr7JUKZKY4BcKcQUOHNXArOqqSqPBsc6jE6t1Dc/iL4xPKViNIsitfr7KFH+lVWc6SRt+cNxINs7td0wkYk22IhsAoNztI5hXbd7vM/oqNidZvmiawP9mhvpxWDuHVuOE+JfnJ2l5aV5snDz2bJXOEeE6NjEYN5JfuOq4dKFMx40aSzWADbX1PpHMpRy4Oo/w/4/6Huvbd3sa77Imz0lWhTlNj4DQ82lRsRulxQT4mS+Xa8pVecQhk842rvSxGDJW9ZBF+ZLlIIVWE1kFPdqjqBO24ywVO5Usk3T4XiRAcloEOAyuVioarSyR9w40LQWLlL6SsLiJypy0YpWaG2kyiWZdMsXnA52TXNTUALdQsSinXndcI3j9OsKRM+UTZ5sP3AYYz/l+WbMjdpif8U8EGZyH0Jyv3fu3pWOcVrWadQA8qCsFeBpWB2WHUqpKD0qhwurFBvEllRVRyke6+z89cNgfbZA//Tkye4EvKqk0UqVwB94ogtyopClAyEd1lUlbcNts/RygQcwrRRdGkafUuw3SZyF+CUdRJTe8Rkg93z7e39WWeQ7RaizXMvlWVLcCy1lxEcFJFIZNk+zaeQv6DlKL4tG7HbKrFqEoNeyU0JAaYHIQdq4kxem4kxlWxpuf8w/JMfT84ngsrYtOtmWYh1tADwIhsoYkUhnqWzPmiXe4YMJM+scLAsvcONiXAR8SE0aPZX99BqeQkcD05bJo8Q2J8KVOfFP+GkJyLacqeGRa5AXsizYkVV0QRrj6EzJ71oIicdOyMFv6ss9GbLhMJ6MhNLVH4k02NVoal0tRLlKzQN4gW77xKHXBxu99Fj/8UQ/pXBUWRCH1xfXJyOj4anp+O3F5dnw+tHDP4foUUh17HjZC0mdJELHMkqhYoXSMyUr4si6+O5SPUwTS+RV1mViCImmag+UrHX6qdVMYNdaKrtBsyTc824tqW86SP2srBLeTfDynNoUBEbsh5orwRhlqpo40zYMhfEBFBVD2JjfvL7HOuKvzZFTW4lTQWPY5uG0l5UtdoysmAhXkktSH2jxgUMt5n4MzO8SSR5VBiG3K/6oRrXqot/GJ6NxhcfRpcfTkY/jofHx+KG3Zy1eId58gM4AjRxnyabQ2+u9CPY2vAVnF8QRyshprSv/Ev7d9389psx1SB65FWq6pkJzGQmGKdJSS0u5h2uQcgVGk00rvC33Db8BeowYX6hRrFUqLz7wE1fKCGaPohpdQg6bG1CXVEtc9j2gm5vAoeytBc16xk+pXEpVgV9Nvm/6D2SCAjoPckP1ipUSEvjtzSilLHgLi3h5DobXPxj6PW6dXIEfoNIjm2gFsVM7uxTMm0EBnhFgF3unTY0vKV+ch9nDSDc5q72QHqEmZ5z0MV6Gf1S3EK+BaatUJbIydJkn7iiNE1xcLlIoTWze0GpvzR2Ao9LkbpJHm7y/81icBuKJZiuYIqDmQWpc1HHad7nYdxy8ue1aPn0NnMKQhe/TXDMMhE/RD4KVz0xQZHk92lZ5HAYeqorK1LuGkZjDjTL2t+9jj7hn2mRCQyg/EoWxZXK57SOxQy1T6uH1W0RnGAumx3xkyxZyDuH0eftwMgX5/5+OLm6GZ6OybWPM+uHFBERPbyzo0OeDZhIFb+pUJgHBFQNf1xPVkzvKMvEY1wk+FM8+0XkUz4Vh8OcZUXLqgCWlVjleKukLh9CXFKCgMZqWGHAno/bJC2ewA9FnicMiKgkqXlTy3gtr1WNwCzPfPP//s83Pfz7bS/6lv8CdYXQiMjPuIbMJe3DPBaLuBpEVxYTAeE+RKt0sRQ2zcSNk91ffXolrHCX5NXAg4vnyQsKa1Ys0ik3UX9cZVplXk9BsZxORczC5oAhdh8LxvACQzxFloG2okqcxpZtLQZgCkAcvwbU6pZIPqXiOzpDTUg+9mILhtot7Hy7qZ48sCD2hSWrohTlLC9Ma/wkINCoMZ9DwJmni0bMYbgFwf18H0/vTGeTRvXcysHCLaqbFmX88OJWFNRdP62WggwhCtvbmJknyTbxg+D8Q1Kmc4G0XdlFxwRzImJkgxh2RHybZmn9oLSjJxLcyPMigOTJMPZBJJl7u9pi1RWsX16ZrERqiET6pr+BSjNAL4KsTkXbiA2ZQ2jCeBSpBO81BQeAPnhnGsSSbfrijq0FVnHSo5PasFEtRS7eVbxcyqVKeQnPDmBaQRnDra5xeiVZLNFzxoMwknjZQp2xi/sJ4cAIIV+mtXKByWIasXrtTZ5ZaKAkxHKXInnrwIunqSILODFqBMrogWphL3tb5A1hZq+EJp3/CCVn6mElYMZ3oMoqWaxU1hFhch7K2qmQ/R1p2zZYp8kUDHdCnW06HayYrqHHGSpRuSLsnMKzzhohpVAzimEJMVvXxaonAnGugcGSPHsrTj4dnZmX6zTSTEMQAbqZinisR2wg5qIsICd6qHyEUngmE6rCVSnm390cv1BKINZ7xpwtt9HCwrF9eNVwg5u0H4WrL86fX8u/0fW7k6vfBZQKu3JaMFbB4ITamIVJ2PZUpRh0olnTW0gfUG0bGDaJoZ4fAxY+IhRXGk5VW4wx7rlIrvhWFD3sibLYqFTKmlWH//8G3Mtnf0Oe4rO/rciBA+ctHLCzi5urkXg+a6Fb8dCKzYR7TeRSJyR2qEGhpsZC/PqIoMULQLGDUlXX9lIVikr3qbM4hkfXJxfn45PztxdPWBxDNXhxZQz97TfOefK2nLdyRTWm1IYHoU0yuXkPd/P44sdz/P909PZ6ogphcnnyw7vrCdXRTCzUqSU8TLFW8IDiWvV6PJ+DQW3VmSAyn1WOoOmABFvSn/TQyKZX74dHo0kkzhu8PRALXum3TDRVnDkrxBur+aLXhtuhePmXsqhj/UksEmXJF7PEfqJIUyNffhPHBrwGcZjoEUzp+bh6qPs2JuURxf3qgZRQVFjhOEpmkCa83ug9CMgRzDYh2bLzNMkEh3QsXoKZvvt2ILZqK79wL78lZdG/FX6atR6ISPE+hPehreyj48p5VQTfmVm1kHjdh5T2+NQR4c358cVTJPjI0fH4RFhJSB7LxW303N2iMz+ZyYorC084MI/Inw7TYkKKMUW1VoOTNfuhqY+YehO2a43MaG2ycuE8ZneAy9HV6PrPOAGfn3gwHQgdz0tln1wKKGc7XWXqS3CaNTOXEmhNG8UDVZ1aJyoK75JkralJlzszbyuMi6TmiSoGeOVCoE1uf1F7wKQI3WAmnMLcgojkshBW8MvyuC7pJ34/gu7m3bsbC5W+UyTCnDSL6SXiLQGMRGXL0oenzBexyy1WcF9N/2jwPvAS4drH5txHVZ6K4qjfIPayTjpZtUKD+00dq+AT1bBG2hiGU+1uHC7a+9Phx/HRu+H5D6Pj3Xv/fXSRJ954zwUOqN+phfATwRWcIhjQtBA2KbT/HrQLIz2gzpUI+tRkwO/53p6Fivix2kt7MKeWsAxmgRv8sEUb+qytNE2yjMS2khu5T0TM7qWL3CgQIQ+agXxqTy0lJc2yyRKnycResNUWSUGPUC9xb96UFG8CuGMneoEKXgvLwKsHiq6x5QapsMS5SzT4NTz/SECMvKqQ5vUYsILlZj/rql9cDtflApm/olX9dSfAMyMM1zcaKYXpEMDVQVG4vgLCGIfBTv3IA8QKfosVd5JBNPRhWl4n33EOiMuQGswAkKHN7u2ISYqo7hYCna+2ewZHvVfXlzdH1zeXQreXN+fXJ2cj5Nyvnwo6XArvQZTci9MG/q98xJLxIxevbWOQHe2/lYqnO5ZRBSQ0+9UKRfDRZ+JYglKHsZndYNf2uuKmF5WzuSeDuJqmKcyNAa6YP1DY6Z8QcnA2yMDserL75OktbEk9QCXEk9HAg8PZnyVZukqdN2TVH0pi4YFUSXcjR1ptgbhGf1ogKf10KGkXpA7oYjRSIfr4A/R/4SI+1G9qtgvJfH6ey2bV88Pop8Fg8HMveu6DP+NMHEP3wZcOQkbwhXO6kHk0CTf/ydb7mfiBZH/db6M9cMb6eibzs3B0sR5SIdf9VNDxCUaTYgXGBUp2aGHwpwO1ZYtSHm6xI0RNL7vEx8u4WsqnuJiF2C4N4xuB//S8srcYw8L1m0NtTO012vD82Kw7+aXfj5JfRSxEWJ5kPxMbm1GaempVPt1FmzW1py3wRk1Xiw346guHBA0k1A84gHtFDjFLFmUiu03hSW/SKohlt4eJ8yKnvS+2vbuEw+jli//+8sWrv3/54pu/e4nIEcoeWOaAD8he3/TnRebDeGL5avhNIDhH7keTzQ0FoX9EPqhCGONs7MKAAu6rXvQNTd7XeAZMOLbboM6wd/pXo/Ork+uTDyMVI4grphpPQTEGAxJAqqZ7wlNhVQLpliXO+6Aa+AAQDtjULsDfP4FJP4l5g1cSZ0PgzxaAAvAtjkXzCguKgyCSThP84t+WsnRq5srkJ7F1ERnNhMhtQW8qH5gwEarIZsKgIGu4J3PHfoZUYbCsqDQyxPiz4Pagw+Bdrtriyp+9/KwQshKY0l70y88TMmUfCZlYzB541VXidxbEUGFTcfi9npIokJbUY4Et2hXEyVpR+RW5pKJ2exk1afNmdWum8xMgUDrrQSf7gvSKWK8ORMM0624YH2UaZbzBoglIVauy1PPJlVYF61YpBMQPGKT5qhiNfe1RKhbYg6omkyYqV0Uu+Di6RlR7no86kfMgog7yYEChrwrPZIBKlkehMn3jgkhqQrmCJO+5qoXps5cVghFtZEXhpaIx49UjXagPqUZD9HQpF1wwFs4oqtKXORIKcZXH62pZ1F2F1F3GzGv3vscSA20RqiF39IktgAQC/jU1Lhqce1Nfq4p3+lqT2pM3fMNlHHj25hZRtXVNJwUxomjCVYWPuJjwz8727gA/pT8PWhOFHkQQHgufMsDeaGhYn/pPsD5OfHGmw1AlZqsQ9bQ6VNuyg3vRB334qA+9Ftj+q+0zsXBTrZEXayhZs0B9smjLrKLVkGqtXadQzRIc14XzoFxZZe1dhMDwaR1zuRK3hrPcigC9/W88xJuO19yhue0VOhcW7M7KXyv5FfbJOoLMwYTtVfeJPsqL9oi6HMj6USAQWBgbLTgI/oQN6Qq+mKUVLPNw99ax0lf18ae23Dp6eJcKNxR5B8Y3cO2pY5Oda3ZB4Cd3ZnBUq9T+PYHR+BWi4C2kvUOHlDVv8mSllZgq8ZQj27celwaKqLETCvIZgA9+tzPtx1kaUwd3HznYEiS2rf/dB+crsxHF6rtPZ8G5zNXzFZMmX6uuLNu9uvZkRGKIVfDqpKUG8Q4RJGhgKtimCJ/VLHsNMQRQfDUsAjQbpiGZYSSu20f1eKJrEGlWJL7Qe4dLStW1WkOWbgPbxTh00c4TIX69LvLrbREbFtgVUU9SOjzKsbubThW3lTeZbjZZwzqOflpVjeBuomvsiy8j5sOUqrONINzS4qzFRGXbgNZlzF5UdwgO+YBUr63L1LpnxN076zq3ihadAmXu1oqwegHicwox0G7KMghBAUCNJPi+C67ccyQ3+fylIyni9j2HDU0udyooglKSzrqO7Zu6QFHzVMNovZ0LcwXuxDjifS50aHWycd3lDY3GsXYoEQpgjVRL6rtsbpAvoX88MH+4LhtzSGL5SKxxsZ1UbNQtV5CT23feyk36lzqUYEItGnYP+xjzPEl9hxChGpAMZUTiqrJiRJDJszPh1WWHou9FGs1s0WrXNDb5yadCTdG5yCNs40jPljogP7maIlZMlZoke0IBWcBfiS9cnSmD3dXN21Bz6qfnSN88N1H102dnWB1Gz5mReN6LnovFLr+/lp/ENpaf/u5L1+y68HHfIAKuDATwrVALUSwfZ/Fk7mmFwVwtwdJaErPW8nS1U/8wVBNu52S26w6993YsjNamwo+hMuxt36wplycFWMybmcuvy8RnQIbnJ2dDZuJcBG6MqNzpyXmnK8pWdWccIyqH+JOpZIjxyP1NpWJRGUIr0q5LH5CfN8tU2FADia4zwxUPWcFD7Ra2kI1DIo6ty2paydadMJyKWtI8MXt+zWS7uIzKHW4b65IyUaCPkv3NXnYLusglvPbb4pNh1wVfU8ZU7aHKIg48UI+2bJHN/gE52P8ZwY3sCTkeRCKxJwek3o5Nsx+7VTXVsGYIIIMfqMv3fKuHXOAG/9RFoYUpFk8vyCgHT99peJUBEyhVqzw2km7J+LH4KUsr2MSgZtN2nlDTH/jcqTRftvi0pqQI032TWe9JAhZXysEGKeA74KzA2IjCMwkAd1EY8kX7Z94nyj7tD4ZGdwueUljcmXvS+X549Mdo1vhGGA8OO3c0Vaw1ASIixFZSudCmjIhuVCqVDPtog1qfGgBBzW1J4P6+IxDwgc97tuLhESvGs6vjmNBdqdo7QcR7677INC4X4O9GSBsxCasAcqwSOp0aQXnEwey1XuCjzh5CsdFLQNi2fVkqzcIW8y7W3QUZ/GTktBODRo1sXWdMf/mU4ND8DerJqNZGRfPaunh5O7wennpnjYraQPCK3BOhbORKurOs2KA7VAyxo1Ist/Y4aekddMZ5kJV8g2BPboUZGjn57vWn717boTfQ+CzLCautXetgM1skss0IzrcXEu4GZ6LOGfzQ2+tZ+i92jtBKpAz6yIRSPTWFta1OmpzdnIpGuDgeno6PLs6vR/94/USy5gzaUDjbyijE5jvsViiZSWlmMAODWsSQuH5GQ7CzTRByR1xtO+6mD2vY87HAFe9zg8CVFdPGVv40b4nQ7bEjyD4apyJiwI3ULC60XlDQT4G7VQFOvGkMAn9CgayVelda6yMON/ndqixl8XkDWw6BZFyIlZM4qJp8mpRwN+oHdw/vP16/C9X0u9HwuHsBZtWTRB4r0f1Pij/+ZXHHr8P150QguxBqfKp7LgsjJTkK696IO4ICg8aXHDByiaxXs5Jjo+B7K4zZCd09kXnUcgSYuGJfsRrLmtGFIxpkLpn97bPpTj2oaSOCexWhcz0V/tVqYJeW2CZ5YsK6pln0C9IT0PpZelvirtw2QvuH0a0oGFTwyhVlifMlp8VabmVeeudyLlRGgu5FyyRe/yoOqshG+8svFaSeSI6lGFVik8Qs/BWVOUMBZZmwxraGr8/ucCYfHwWaGIIDaSh745pFrCwcBttaxJFskSHbNoPbt7PckOW/PuS3aze3VjLlwpbTEzgebHEEPQnZqFcN/vOJLsT0rXLuIUoeku1Cue1gm7PRfMhhO0SImMM2tD5C9heHOgZfi7V2ZButE4WOXVX1IxBZuaseIdShWgWErh6TQr8X7sugWQtlEjSw9LWBBba/liSKgKs2MSMZ8JZqa1N2HbCyiM969jttXkEE9eTs/cXl9fD8WpR4XIrkdBJNXgmKt2UtbJQjlKcxhZZ349lMPmhWO6LzenhyGprEfq/DaNROcPAK1RWJxreo0l/FNm5BB1OIr3yfLrQZC5UeLLjtsV66nSJRxXMqj0jd/Dhva/vtdHG2QOP3cuULAb9/ezWIzlixneRgDrRje5ja4ty0Cnp86Lzy+jotQ6hubnvDUeRZL62lWsRAhhfnKUSrbNqLjvHPbRKvDLie+gayYh8v+r+6ygPdpu8bktznSJWrjFsmYjZTVFA3g//JoF3nfGqV1T79oAq1Lf5oa7WsolGXq5q5aIN0t9+JkbN1Wdzqhc0csm2WBAeJlK6fSQg7nm516qiwceyEFVyc1UzFJ+SKxWhUsNiDSSLk2jnxOfUQ7T8Wbk6XBZO1bpgB34QqoYfWsqiAJGeUu585lRV0sZiWhUMoctnHV32+uGd1cDoXI80frBBcNPW6o5X/mMB3RsegUP26qZ1RF1XQkKsUZmWmUWDhmAoUoLlzgFVEzuH5tUmnd61w6YZ/tbZGvP3n9CuTgEUELBrcqjUi6IWedomt0DtAHlC4nONqjV5rBMeFXELlAYWC1g3XaC8UvxaGONxS+cR7T+nCGvOrFAQSxrK8U25wavMBqwFZ4mU0xiLGKaZGsO3PMUcb7W3ZFBTqujTzTrRDo7w99UOd7mNXQ4TKqIxYDkxR17G6CUIe2wrM9RuIHXBH/tfqB5ezR5ceDLseT5DF7Jpo1EgVvxnAfgLfw8AT9enRwe1ZHwZzcNqdLgJVyuqKsEchzjAMZws6NRWNbNvbcT64hVi0+KWT5FHXm+XQaeV/IdnJNiT4lZxIr9Bdi4uw8wqVD9DLUIlopinF1gVqVNtfLc7dEJ/Wozlf8V57lERYEwW+48bh2K3GNip9RuvateiOTb+LEm05rKHdqRtkVR5O0zZ6aZvPu5vjftC5wuqRbh+Pmi/31Nx3XeK40RSpMJElF+ZWkIqL84JGO5ArVLugStkmWuhDbSvLxGW4nooh6w5WzWcCNKQYDUtRbltAAqLHBLecrlsD0lk0nk6Tda29NVbZ5asCgiC12YYDVTz0eyBkGAzUgnuoNt/C6TnY6Xbt2zJHTo6sKZ20fsyXZFPgdPf8DCCv0VHEyUppTgi7lcJh2sjwBfHlkhlcVRtz/USXrIhhD7WK0+wqaw0o8sfbUv7mRgx4wZDFi7AGIQBGzDjmvlyd/Fsx7UbHKGWDpcJkDM0UeC5QGAfRv/7zv7SDphjL6TN+U7mSd58g9u78dtk6rY47ITJmttWdbDuL0D8kEG0YQ93GjMAbz+5hx854125sVAejLxR1PNt2pxKXPcbpUFiIAhCNRXsHxtWca9H7fjJYDHrMY/3GVk0RoAxCa309I+RWrs9oEkISaHmSuwKiNiRqTWpB6JgEYbGTdc1pd5xVDzPGEZbWa0zbiy+3C+tDNdYgOq9y3sXbSzQgH5+8fTt+d3J+/Wj8WiO9Y6iIfaXrPyBx3VPznz8j4ep9mU3hLakLIQRytWtj8gYBRNa247YbX5rA8dWUqdxaJ80elLhq1gIVYOVDWI7rUgpj3MGYiEH0P3fNMgfckYabVe1PKOMnrgBSf1Tdpj97Vae/+g1YsmqesEAsGnQmHuNN7hWPKSJ24ypSOqY4P7ZxC16VlIHFLArdV6K2Nba9Tu1tq3p9j71LCFnqRx9tOx5RLqhWhBVuiahYaCUdyhIWQRSJfXPIiFCwweGkW0q6qsCQppac7oe5UDTbAHtEYDn0Z5En/XsdHaN3EaRfg/NpcQuTWG03ivNxhfDOHK7a8WG+JBdjPeTO29lYvjr3UG1BryQqEWRZXHbOpalxmx8oNijos8hV4buO7tVtumg44wupCT40reFbOnNI53BZNYmvQbYwbeWpb3yLgq882ikTftMSaICfYCGdP9KtvOzQqhvnV9nhWV88nTbr1Bl9ClzY7cM3RQtaLZ5xmjBIPWkrcJG+08CKY4qJhlH8r/YxS3yNZfQJ/qUDs2Ol4JABLQX0oJEttZ/Z+DQvipqmlSuRrQKrFrUMqUZOXGi8bW9Xr9fEpohgloDUY4XahJT8TpAnXEMx1LWPnJDQ50Ox4WsZ3daa7mTq6afypdjaqOd+Jf/XMiE00JiAcMXpjc0qYVzdl12nuZVba4FqX3Y2RHXD8hHtrTZca6XrrrSyydNfm2SLNVJnuXve8KY1xnto2THB64RuP8VQsDAgAh3SURbbhYP/+s//t+q6rCrC3nSW8JG4by0S14t2ygq3dZILACE65KxNfYnEr9v0HqtFZBvN6P34w+jy5O3J0dOZ3mPNVbrpOWYYQOF6F6WNtbiAyu1DECw0ewIFaIsgmamWEYnczUo0lVeJ0TBnEypR3dt23Owaxb9Zo22YA44K78OFfXANfHmKTvX2q8TBhTookCetdYtDqvGts12KtWuAWqWVOre2MD3lFcOUnbZF7YjWdlmfeemOjTELGkMNRF0SN4WOg3DJOK1v3LKnNQWkEQEE3JhuY9c8i1AAj7keFmPSUKplNBYkJQEzQeF60Ldbd+KSaF4ndUSXCYYLpn7VYCgOJgP1fAAMl+M5LkS8PUCbueq6HgE2/FzGYOSrTnTQ+GebR3YwF5FyVR4gmF19UMIQlEVxl3RG9VjUIxybpMih5Rokia3szVjj5Hx8OXp/Or46Obs5fZo3Ts77eCxqH+tFl6Pry4vjEzamR/81uvp49v3F6clR9P50eH5+cv5DtP+/R5cXkXauR0cXV9dBt3kU/bfopJNAtZo63AhmsLhsICpa1IlXmQ4xtk7EyIFGuHw3HA2cAxi6azAKubtdwe86G2PUq06tlNUQpVkmGeyKoP5Pu9zZnhktsuI2ZsxqnYQniKI+qXTs4r1jxHv3mdlSCqp0QABWHBtLc/wyRfwf9r973fvu9cHBYSQXUsb9OTLz37+98vHmpAwc+wLjlQSMn14NBq9/dkiD/TvYgkmF1VheLQsTNPtiDDUZklJjpLF6QTBobLJXwOCLqSuntbq8yK9T4TB6G9s+ssDHubnwswYdXA9Pfxx+vLJEP/nwWw3KMKi4s4rKXj5gJ/ShwC7qX4kp84njW/jgfTvYSLRArJVFSC0Xcaf/R9nh6ayuRuHDT8fdlDr++QbW4IzGfjK9O5SdKzSykWTDkl8MC1irCHBRQ85ImykOOaCl4bs9i2pSUrzyQRykcnU6mFqlaf3QheRboX5EQYv5HPmkQ41v7BIlY2cWgbUkAwgtxbyIFefYFXxLDRN7k9fpoqkoPjK49iH/D7rXfHR5cXXV57hQERYfhpcnyEW5waHRvn7yCtNj9cfzg21W4nSqdljH/p0ciHUj4CLMp1kvHyr+tqE6y21c9oH6Wt4QttJdxijdDLTW4ILE+d3WzizGUveN0w9s0PHxRXR+cc1hb2J9gE6L0u6pBdM1wbsxzLBPT8KwDQbsOr0vD+SRZ8UXAZ93xsCBlWzIlZNDXWx/n7B8NXSGmJ3WtvwymBFsyDBXO5/5KSkqXx0oKlY9OJWbIb4sVOKCrEKLavjh4t83qOShKHxIlMTl6AhTDj9G+z9cXtycH4+Oo6vR6dv+1c17zD68kqe2lMT32xKijbmhOh5s1VNfVNOEgq3YldBtNNZSNNnMDSdigMeNBoOPShNuf4bJUNMDTfJkZpYn1TaRcyDFV0pG4S10beUwgmdG2DbdixCYwHR/YtmBaL3959st4s8PJocsGjBcbMQt0ElxjDZwUJIooqmaDHnRL9aD6ArGXteUszBsC2V38qJfZIXMBoP/SjJajmSTNSF91506an+ywCRucxXyj3DKwklj7hRc3WF0E4AQCNSWXH1Htlm+ZkC7IGl3ZA9tjTndCFgXJCRwGnmC6TF2E1kOwWZXuPGQvW4Q9hYVHd3io/DrBlQ60GyDBGWWXYhTM/FKe5xwE+YYdTZMZ1QpwFPb33pgnP/hh9M4Rvzx4vL0eHx2cTw6fZQHz1AUFbMI26UdHpmmeais4kcD0uvkRDntUujp586e9XN2khUE16zo+fk/qnBt7JN+AUPc9ld4Y1ZPxc+YxWNib47jhTMh/GB0UU5x+eSsyY7O2klAmDGjUeRuwYcb/+niE75g+fbBDx51KKKy5uQ+hQlz6NFmTLqJVwy1YJUV7pROkpECCPI+NZ/f1g8HmbalNB3DPLzYt5ej0ePFyX81NPB4vrD6P3BK4s1NlmkP15Z+Bmc8Hg2v341/uBleHj+uQLqDN6xsJRgjZFkM20WAHf2juBvtjLk60fYv7c8yY9NRl3MXWHBfxin71/6YF5uc+QKdpzUqS8a9REAzIlk2rslHZIbbn1gSOzwory/WY62W0gYyzOjPx8yyTOzbOhrmF6BRKsuHttN65swfULV1pr36eVAleLIqOPgIPT3MUTIo1mBUBzsGmGcpC06cS6xstGYnqE43sxqIfMv3xIiMWxSwURmo3Ze6NLoreSS82qfUqX49v7h4/yddqDObzCbCOD9V0A7RbkD0X+dGz4uLdfdCtalC1oKJNwWute8LeXkEHshNmlIUTea+NqVDBb7eW4nBGnQtY1PpWKRB9BQx5OL/TPysSn5XShzWauL5IB5wK9eSmH3odL5OzuWrPs7LLvqNuzjNNgVO9iPX6L8Sxt+mVofUuoHVX/i62ssRvgnAzwq/GneaX35P4sbsBJYTZDRTeVeoVYnL3PmWDAYJ0gWaPTFeXDpWs3dQzXsDv2CsTV+8fgLWR+hELmHKP2AG/AOHZmnLRVumCAtWvVvyrC2oE8WUZ3UKipsw7u7Xl/f3HSridKUJC1aVx2nWs8XknBCuCow6bY0mN1j+9vq/aOHkWm5Pv+kHd4wIXyZLWgWiW4pVN6xuSNy+NqjoEFVDCOCB8ghGYgXanJmr3TuIZiE4UrbrJfHKDRWLw9Hjle9g0Wo5C3VCprJcM3voTykybCkRuVrxJjLnm97Lly/dxgKCG5nUol3jL/PYSgfgjveZ/bDVtGShyf3YHR6NN7Chsa6NCPxItFcVfH0GhmRqTb7LNKtXgC3d9b2x7v5O7UC19mytnp5rB/jRzW4PvnWpdcox/0u2asvZFYEsV3LNe3rqw/YkYrBuEo73iOuu9nMY0mMSzEUDqzO2GeMOK3r1iAi5Eg8cjXabWiaxDixtF1cu586OV0fv343ORpdDsSBuzhmG2+JWKzzeKs+20bOc1BDfU8xqPnKqcZUKGsSVZn6lMFsbMH15NoTH6PLq5Op6dH79FYgsnNeWY2i/iAPCXwJKOrJ0wVnirferU/i1arwMMO/8YdeMwkLTz1k650jqLwOttyFu/cPtqj4c6L5Yqe2nsjmhWrMbfEkMwssOEL9S+00BbpiOqb1uIcxbVhD4lyj+PVg9Cjk3VGfKsagwENwo7TaISZGOk3rLhK2jptbaoZribMWczNrOK0UpDhKiEN92IZghL0dVLwemuTeXNUDaFlXulAG0HSG7uB1E791ybp2AE30HJk3cxJXwDKLPmovxX1L0hQVZ/JKpVmuuLSkffnePu0EIVJbp093hALw4aCPPUNwfZ67koZ29eA36sxZ9yyWbMdGSC9sOW5rT7w4gaDo14pekYzU9O7o4E6/wWrXq1ejq6vGQCrpWuD2+nopVR+hh2P4qCNoDrrSxyLUJ6tAzeWfg/ecnv/Pji1/1cxtlGvsjftkp2vJfeNGVJnGlvZdasqWGh0gv69y09tX8HkJ+oQO2rVAzqC+L9ZsQupWfIb9YVV/pNLCTEG0BuDcd3Rd+RVfgH8FTXcbhG3WhXl656pSFbtVkmPuaWAWeddU564keOui5Ozu8sLrFdzfHPhPkeYaBDH6VCbPWAfapg8YAkWGhAPXdfunOiAPrH3TT96lYWLcU2tDGWUEbASWGK6jDkytXyb39kEeZ/3I5ljTjfPa+4tONmOAFPhk62532s9sxpJSwZHAGJ9FCNTFAvn1ptk8nnej4dI2QT2XfcsImC3Qe0Dj7TKRqXfRYDZov7lsI3PB0fZ++VYPd5r22sdCK/mgm14mqD5oxO4XhmnxVuVsFJdjrpsTglD5la6eic9asWAqh8/1JdQjcVDYrMNdZBl6CXN2cnQ0vP44vR//rZnR1vfN9jFe+Npi+ppq5goX7xGI/3gkJv3tQ0UQ/TJOr3UedEmzYNNrUwecb/yU/KH6fzRCHnNv4WPfNm/g2IHgCnLHBon01i2gxuMWJviDLLD50Beu3ZpRMV1y7L47D9FudTR6G0qrt7/lh0ojzWJQ2taPaeq/6ligh/xBrD2Rbf6e/j84LTaVaVN36aGyeIkbbR/cpv+Nk77CzDMxu331Te2MYqSv2QXhfpSq8K6KJs4ruGBMxNAwBoiBhQOfImftoJoAzIq5iQZkFQm+/8gXV+YvchqyGYG04R1swWHt7P15aa0bcqjF6PdrVwcut/cy9OhKRWLH3k0a84tWWYgAxehBT275Citi1VsI6rBeluqJxI2JwoK2DAaHaF0ZhT/NM7+XE/vtLXPiiLHQmTK6OYTwNhvjZia383vDSNoUsCjchSIMrjlAD70DE2D1bKXALi05F/mmSL+qls5xjaiuR8sTWG3wNLr+ciV8My1V1EX7/078BU8aZ4tl3AAA=', 'inference/agent/python_tool_sandbox.py': 'H4sIAF414GoC/9U9f3fbuJH/+1Ogyru31FbmOu3de3faundOrGx869g5W2mv5/oxlARJbChSS5CWta6++80MABIAQVnJpr3evrexCAIDYDAYzE+w1+tdJotlueH4L0tEnsYln7GiyjJesHlesDiL0+3P8PB+Wy7zjJV5nrJpnKYi7PV6R/MiX7EomldlVfAoYslqnRcltMryMi6TPBNHR6osycSaT0v9+BeRZ/p3mi8WSbbQj7nQv36qeMX1g0gWMJj6qZqsi3zKRV1ZbOufJV+t50laNy2XBY9nRhclfyw3Rbyun5MVl5Mpt2uopydylm0H7DVMN54ANFkjyea84NmUh1WZpEJXnRfxikezZD5nsWBR8xit8lmV8r2NBV+seCZRRs3Ngr0tC17GScZnsAbZlDBO7dvF+/tPVtGSp2teyObGs7dduCiSWWSBOLt5Hb2+vry+iV6/Pbu5PTo6im7Prs5fXf939Or6enw7vjl7z05rzIczPoMJBkcM/iuQlvCHph9Rmo+TKklLoB+zbJpnCCpNJmZpkptPRGPGc25BQHoxHssinvJJPP10RKVlsR3SD6NKwUVeFVNO5fxxytclu6BXo6LIiyFjL9i6iBereMiyHIb4wIsahm4MOLjKMy57MRAG5YAEKo2i29EP70ZX47PxxfVVdHv94eb1KIr0yzc3Z+9G0fnFmzetVzej8dnF1eg8evPh6jU2vm1Vub14F70dXb4f3bjv3l7fjqPb8fn1hzEMBtATinKWV6V8e3v2ZhS9uz7/cDnCsT7VE+tNEgEbuzdoSqZ5mnJJdnbxems8A7KAaxwrziI47HYgMEDbah0XwIyyAWLx4rvrkL3jsYC6MxZPi1wI9tvfGFBg5TIxLZJ1CcRYsk2saRLq/xttbSDqbMY+3I7OAWLGWT4HlsBX7Bj/MNifPDXgAa+YLqEN8j+AB2PryTFxVm5yaAIMQvQIJDTP2KZISqiOoJqxG/AmW7aEyiE7S9N8Q9wFWRLsJ+iBA3VvBSvySSXKDNgZThrIO17HkyRNyu2QxQYsAZPdanpMYYcjitI8WwCLnuailMPYLPOUM5El6zUvw2YBkB3BjjHXBPhUe6GIaQCntwqXPF7/ZBbArItWLdxz5vMqLpfmcw48JS7zwiyD9ZvlK6uEm08C+aAok6mwSwvApSrZNTT66sPF5fjiyibSF2xE+xX5K7B4LkJ2naVbWOOqYBtYCti4XABDGjCRw9JuaZVZFsPGhteA1dhc0o9q99dAP7J5DAcOECgtreYO0zSuBJekkuef6JVaFgPaJimXQBMhGwPZQW0YDaxsQc2FUNTLH9e5wOWmkX3PbrcCjrnRY4LH7cyA9iPfTvK4mF0AgyyKCkYB+Nsy2Mhswqc0HhoGtJrkj6wSknZXit4FyzfmVHH/YFscg1hWwBE2GdEzUN/CoK0z2AXLFYdVImZortQZTKJALLXflLCIk6rkrTej6zetsotsxh9bpTDfVtklILtat4qv4ExuF+YlMPGU43nLZ63X18DE57BvWy9u+LQqhG9Wt2W+viiRzhN7M3zIJnmVzS5zkKBarf6HF/l58pB4QcYTi/ZBIrEes631KKZJYhZMEmsYE9i01vO25EDo8dYttLm3EoOssmVhM3hE5KNZNEvso2GWFPbjA7Bfs4Rn1QpRZ/VT7zOLS4FUwC1osE6x1RuQ9MopKfKfeSa4VbgAaQko0eJ2sfAULW2GaM00ySyYsI4ZbDw40exSEFxpW7uc1HxOuTXRFJifzVPX9qM1jpW92BlISRYDttcjLyzsr/ON9Vg4swJWveA2p17bvJzDfhHcAlog0Vus20a/SBMbS4IOb4fZW4+VdVyU1dqmyzGw+NYeQr5vPv8hTqt2rZsqQ5bXKv85WddnjZSIZnyOgno2C9bxFihv1m8kRkOWCkk+CPBgDGfVai109QEDQkS9ifbr6Zs4FbzPfs16f856fS+keVqJZdA3+y/49CEwOiaZoJbekixEzQcLgwZkMicZA0ubhiSjxiDKMc16g54+I5JsXeFplOPC9g2RFtS+jMTsECckAgSph0dUzt6gHvSHhG+anmjcUZIlZRQFgqfzAft2wAgJcPqWfD1gKVBRCg/LeM0HDJWNvj1SbBZSE5gs/W2/RlCICvjTfkk9wFv662mLPaO6gpQV0FO/XSvCkUEt/ON5a2l1ptiP//0HqK8gC5VbCy9mE8KNM+9k7oOdCAJuV+0ch3qM0njLi6CZyMBUR+zZqpVugztyVhX2qVpUZ+AgRoFoC4qBoBFoFPt6mYPAAAtaU05Aa3T61KzbTtKJKsKfO0Usp0/Y0e7xCXva9XvN+KIIWVUUQfdqmBadvgUemxfbUQaK30HkSlLzQKr9A9TuqrQ8xVXwUupU4V7+aFcgMPCe/rZfS/DwHo/TQD71kRiaU0aVDqhKn3FgJuxpd/j6WLh3sRHIcSuEy4dfFTs1fVVMvz8H52NU3BKE9TkYn5DaFCnEx3M4OyNrFb5oAUyoUM189EBrOkWQzVP3wu6t9Hdb3qDFIXpnhBB7KYJeq96810UCvrom+lQLs6ijlYEj3U1TgoRlVu8fQGaEFGmJQytWpA5ez3mtjkQD3/UhTQj3YpMYulPWcC2rhTziYWwabggyZ6Alddbr9fsDmy6Qv4HwZdfHUqh+4taWHLJVnYp99SWvtEFjEdS9O4H69059PB7s6liCtc2almCylCzkWcSDEotHwt19XYR6KEfOA3IHUy0YlN3dt85CZ82olXfFiOnkINlllb3/qP8wBo0c5LhWkxYfbNWgxZU7A1eXRqDWdioVFs/i1jMlUvdRqAGH3naCUAePUV+W9Dz1+0f+J0W6hApzCcuaKQg5OrWmgfpr4Nio2l7MBPX2QbOmtYrnAdTgk0S3IlC6mV5bC61IFfAnRFPQOuj7qENWP5AYnENAje6Oxs+O2ct7xc6RM1PZ79mJZMUWK3AQ0kldDtPdR1vq8PNWsZitdTr6IRpcVpLNnsoWeakj6HPpykCFSV2oNkQinvPgARUxmxlTkZZrcZmNXU7vBiyAVR/AOsCZSHo/SAZ5nvb9rJramB204XXz+Sekw08cyNQcNW6XPtE3vBrQ7kHiJmgh7aWgv9vfZYCa/UAqGSDA8rJj9Hfebq0e712s45glYi11NQapKvmZR1xbVAL4ZfTKH9ERgqbz08YlEqrSqJxg9TCK6ldR1Kx3JTRhEROQewVHqnZNZkAHfFBpiH66jPbbKev9bk3OgAjNyr/v3VtaLcHsjXXHLFjlAn0yUyBOckkyECphBXs286n7NkeHJ4weyt3xy6FzslB3etvOv2HsDQzSHd1A6tpPchr4O8t3SJC6CGe1+6Z/5Ifae5KYJFE4iuAX1o+i3ZDhm11bz0a7QPiXPMlIzRbOus55JN0CQUay8CLNJ3EqSCWBoaLJUT8gL0fKOw36StU+PTFIoMjzUjFfWhjNZNdpAsdLCBN/2b87uTdJm5rQYZxZjiKfecHwmAEW3pFzlH3zJLGFW57YNrpKkEwy01gdtpGifYOAPzl9kH8tBOi5N9NWc7bxx0EfTlZJKQJl8IYiOCNmwuZLtRfPq3LLMdVF03UlgQI2V/Fj8JLYVasDtPm8tGiWGg00E3TF9vpI1IOBs/Dm8uLdxTh6/f4DLA9poYOme0cU2NP+ze3F/4waCC+jk5MT/P9wCFfXby4uDRC//Y0pILYEOImeTvuF95S2XLOuezWElSwIaqCwqJmtZIb2WaXcNMH1LdHjgDU2QY8AuYadahJNhtblFLmp8qEpNdmhGYPvq/dovnA7qAVhVafZXzztAOI5sb4AinkI9TvAYZV6cgZI4TUg1tZXj7opgdSw0K3Fp6VgMZP+PNDoMy0A5pO/wLsBMqCYhoCOW/VO1hbfyToi7HXIIFpJwIn4xtoseNAaW8F/qpIC/W8lsIwY+weGL18jK6rB1YQw6xR+9WHdyL40JKSEuChPX7a3hrFWWNVHM42AGMkjlLrxisMtkZhaDP1Cn4uZubISsCeazQ43LF+ty63JkNvI0Gfdkxbah2bfu/6BGpoHFR0ano0LPMAIH2117O+DolUiBHqRgaI/Sogf2Tzh6cyHNqkdnbIOZPlGarwPK0B1EfRJinp3/eF2JAMUgt5jD8mOyA8P8m392D9wZt5a0mDjTJlcySlfxNMtozGwx++2csbie3yLVmAa1jRPw54XsHdJetCuHrd/2IS+O6p4r/eBVIehxA8UBnEYUKxoA8WSQyifABxI5y7vbKF3VQl04desUvJF3AoeuagZi3FirWIQHs3DCZWBGN0gyo9Uv5HCk+kQqF/ZMUu0ySSYGjV5EU2XcSFqBb0xE9iijy0XWWCcmrBvf3vSN05vJRFG083MNwrjddeWR1dKU8smgFyE0+UsKQKjRtNyn7TYHApqd0ql2bGKyMAqJKk8vKXFvLg2hlZIH2SkRFgrkoVO0SjScm8UAaN4apMici6mxTRdeUDlbXrEcypTapIVQ2PV3NlCoDY0DUl2a97tuqZxZw/7Hp+1zE77y9RiGmSDxroiX1P0kPCNQqRl925iLQNp/lB6Dlk6fD6ZF+yaxIZj6QWkoM0JLzecZxTiJbXEkJ3zeYyLN8TooIckr4SyDR3/nk2rokAvmtT1HPA3tAlBz8QYo80S5AfYENmCz4ANZuqnEm8ExfvkQEygi4buySfn0y0gq/enLq7lLrBH3WsZyAhB3cDl6w7Y1vzboO2BE6va35m2tfQ4ed6HGLkASGqWgwVZ7q5D/BAnFJTy7/3ezj8E5cr9lXLByMc93avViaY8TaNpDlOHsRwDi+qBYMBxXLJrgiOIdnjxPZtXaQpAZkVsBlC0xF9lnpPG9qE1RLRtkmlOvzTGu/NZhTDapiq5EWocdNCH8u/6F7jrpR6b4e2VK5/lM4wMsm3/NsjPaNR3fFgFnwNjWUYY9McD+jdquw3oFDUpENmHx45utfcSrk25yuSL0LwODA88Vc+FhMYopRdGaJ2KaiefB0ZHZWdbWScKAGovN/kPO4C5DsWOaq6HsVs4e6EHxKfLHDc2RQvBnp1s1Ru0L1GIbMFX+QO8kbHkwPNMs6zLPH/kfK2CKDHkcSaNe2rWCUaNJ1CiQ4FFNZGhw3WKAdncuxEXrvO1Vgdo6NpY0WW/R2J41hXSXvumiTSl6vZ3xy/vcSnMHhofgr3R3QPUpl08M62S/Y0xuUN8YVtN5PeNW2R/A2N21Mh0AjwzSgt31Ngp2w/AOfTuvRvFgWi7/9Eu5aygMkvSWeZ3+PQPmJYius8ZmBlq8Dcelw/VOnDii7puddacVNRX8/hMM1CA4FTEoEAYwhojyql5q3g/mAdeJPMtpsYUOfI4PWVP+X5AIk8fgIFXcDSD6BFN0Zwjqdz75hBgk2QdFyUomFEsMNEJT08Dou/1M5MF5W+m7ZIE6U7bQ2zvkec0stvSmX2/v7cZB+xHKZ8teCGH3YZq1SHN8KkH2gnakQRIO3f3u8N3kHlG3tdRParkyIhef5WXSzg65pVAPWoGK/3A5Sn0Kha8DiuWSRfGIyYBGGBAMuFA99C2Ffk//AgCPAa46jMvDMM+OwapP5kuZW6JkCkiZooJaH9JxkE5OIbTDbPlmNiQz4NC8UN2pgfNVhx2HB2HaivOkhmO1oRWZWiihPY0cCPJ4BtM5xLVCg5RWhHctHGKIaGglhQ52aZmDEN2twY88mspcZ2ynjApALpgcwxRRRl8lQiOGRSc+i4BIJ77L//l5MQAAwoxaMUwUUBbvQQxigk5JeDAkKlnhqI2doADp9QFmcODKRChAe+2jFN+i9OQhhEyk4CYilhLcHHTZEKmVUDrNC6ntAeHcgqoUgprSaFdwtUw3p39CQ0rAjtgAGGpMj5gyjBA5Al6eMr427h4KJDtxyzfZFf59doYWGARmCO6oieBVg5tPjMGmvZ4dHaO5m3+SCiTVkO14nrFAHEPIEzJ3BDMHzryjeMc99nfZiByzbIZ5rmgtIdandzB5p67ORAWrJkiU7aMG7oEijKAyaRMrMsfoaWkj5C9qhaLLUvzfA1kNuO4iZhAkxsMTGY/pckcqCkuDFjAhmgQgCXEHuy9GGUgkRdyZpiXCFuNNPebix/ejtEslaO9bUkph9tpys30tRgtvGX8CdbkI2x36BTkVfY79pchuxy9GX8EMk4/wcQ28baRfMu4gHEgC0biZ8cmPJlRxmDjrZIsRoc2ULE0IoDIhWPDYeaYJYf1FqQCT8sKdv4W10WYo1vVOXMl+80Jg4n89gQ3Vsk1qxKhzQUBZ58kZNS64wlQ29DMBFoqrVcOA6hxVk1l8hBwBVwLgfvwI/rnLGYoWaYvFwuYJwjtINPTsiUYBSH55iav0hnLcSdu0CxqcEe9JR3EfUKtYZFjUhJIIgxzLwsELNK8HJo8UQsuKiUYX1nJf8SoBrRWRFpQHROilW9qRrycSJllfFMDvbhtARJrZIA6yTBeEyROFjC25oXeFW12csPXxFDlJr7IiOcdtpc9+3CDCljM3t+M/nBx/eFW77/E2YLaKqUSzAaaGs3kQ1WVVLqySHDtc4TH4kWcZGgvM1gwcIgUSpl9yJpnszmXIasyi/yaEwPpUI4VTy8g4SIj/E8wN9Uhz/rYk+xdHMLN5VTjVh4oUTlypoIfI7E4BocV7OqUL4AYVnLCmxh00BJtVPmUm7Y8uare4ys4bDFbwpDvxCFJyPfiGTDugdHAcd90A/JOjgB533QD6qZ9gtb92mQHxaqm+/q8kfQtmgRKMc3XaGDM2fgt7F5N3CpH0iEszH2ppD3Tu4VJ8MqQagXAQF2EjiQQKtAJyGeG7wDxIa1bQ3Jqo1x+71i13c1b7zxr05GERnQsD2GUqaVFm85oU3Y6J8lXHCr6NgKvmfTawdh9sjDhDM02tiBsJr2iSLxPEMaE2NfSUUZsOlfYxrPCUvZeaEvoTPIhAahRciRt4SZDmj8mKMviAUCH7zSHDT0t3Z06Vudva7Mezn1bdN0Fk6i666Upk6Mo0KbqWrJthAapBLn07WHjRjawqClcuiM0jVMoNsnoFqUbwPbQfC3PYApi3WucgjgkZNgDnLApN0S0SE9C7g9nU0Tj0c27i6uzy+gcrzm4bLvFUCCK8OQHfdITe4Jj0dYhG4Ew2LotpWGrfYTcf8LJgIlzBFyiWND2XPdoOrX2JYlvm1d4BGaZQZhAegsKIiHUJPUwUOFB4veAVlQ/k+MiXMrINzh7MqEFZC6lEUSbGwvjuO1oZpFM1cUsy89HlQMBtVXs3EEZav58NvBi65eiqcGJB/rXQRPs4HqKX4Ajs7lcOdSUYSbzJEvEst4diZecuqbbOWQnE8iJZGrFFjkbzeMOk8EIXYzQ3Yp3DsC7k/t72/DYxCNowxKa1bsj6KzGMsv2ybMEW/KU1cE9XZhEA1N7BO3qSopEm1sBiJyCAoH2Cj7Fy2qmcGDH7BMKRoyMWTIAQxZk+XG+9sAjkRW0pgLFhRUXIl5wKY32tnj3hGR0KEtqA0RPyqNJ6YE2jQuSAGI0rkwLoK1a0fHyJJA0SEqA6ZuCBywPUYFR1Jhwbazs3ETQdbr1xIvU4bHwWgVw4NLIeCTta5GO3Q5aM/OuMbQ/MGDJlnXyh7pro9fvHzCAXzUD0AELh4ygd5GRGbTZ2EqdI0aiE6OXuSjDZ7x0xpDscUgb6K67cR0sbpW6TrjX8VSyExUmgZYwvA5nInUph0OFTE0M19xU7yQwkpBQ3JKxFigLpklMd5CkgpQydM4OUCggp0DMFhUyccKg43VD4QHBoKrpRnG01wCLw6Qem0NbjlvYwCkVaFy2Ixwsz59qkK+hIMaLaiR90uEOvAg0u6QQ5eEk2gV9hrdvpT76PGg8xrbsHItfm/QGHhwyUgrUkvZPhdBnsnwOnQtxx4jYZedc/GbLXzYXyZXncUme4o7T9GtNslBaqXZPJNkzq7fHwvPLpu2ZEsWzFSiu2HKUYW81rKxSJnOMQ72vSgxZnq89mHmhVfN6gegczHJz0GjjBd1mGaelZdMzdBL7LCe9Ey04KGKiU0GUaMrJ5zoIkwIpW86NfZTqGvq/BqGS+PDlhIqsNpqn8YKyRWzZFY7NGRzr+NcV/QemyuSJB/auK/XTETz8gn3Evj4aXiskL5zQJs++V+IxumRiYYnIXmiOUNmZR+mX2UnCIZwg6REKpJBDhQcGH1MgFNDwJ/s0ooiX0y+JK8FB4Usf/ZO3wTZjIm3IYFFlmJEuD/R2cC4jC0Ej9UqKaUquQrxcDGNNN7E0W8/zolHKUUPCXgCvxlWFtUkD43kCGq49CSQa2E/bSBuNT337363VDhJswUnwUoiK+5ATo8E+VWY8g77aJl80YChTrVQo22FSwBIseXh435VLICekggeSqRRhoH0Zk+Wm94OaQm0/x4ByvHkNnWPSWdU7ZNp0+480RLcxqRmBb49GNbc83de0due3aVznS/g3Vg0f45a8Neywsrr+gKybfZpTXegPs6ept8euimaJQGx1H23dTbN2bKp5JP77cwfb4YTgR14dw//NU6QSHOvlmuCVSmiAm7Gr65omNIEH+QQQOk3xTsk0pUvhcMh0TvTDQ8b9gp0VK8v/QWeYNEaok8Y6aRU1dh/M2jkH5JqDVplJl6yOcDaP4w3uPWBVn/DO3TY85EHHeAroiMHGGiQDsZSnLc5KIT0qCYb3+TRiGW/KfkzSVN2sSa7dcon3AEnXKw2MwvNRbpir2z89sJTHFINr2NnVuRwVuWDx2FbcmDKfDUxo27NvbPqGUKBg0NzRJM9uc5ZyG1nSIsRgwXnxoF1vHnA4CHkhKaAcN53hDzh7PdZOLSnFS2YYdiU4WW4I/743auiTF4UaouO+93qojigcCutt9L85B4wGQBluPK+y8jRGHd0E5RTMxeOom7GnHraFH/hn91XsE/XYvoaJQgP7fCtFPYwvM1TUHR9uq9C3q9hWihrvrYEjymmsMiS/1xlHb1zb8uQNXr9rtey4Oce+QcfI52uuzhl0SOB4g05dv746x19b3aBTV2+uzumATjfoGODV9TnBiee6HesanaaNukMn6D97V0udTKqC9gLrWhy77v0eL5QR6OlchEVXmhgJVMaeyDCOqiV7vGA/qOwky8WuL3SliDEnZFSlNJB70Y3WlNraMi7wDmQm+Q3aOW9Gt6MxU5e/gATsiUJtW7Gkt0Zqj6UZO0f+QDk17VABRpEXJIKJBDSHrQOKrPzYqXFWemApoy3wxvp+ZwwoiretkWEgvJRcQc9c4ekSshHm0krXXld3LTi4iygOC1NMU848eUjSmxldxj9vz/RyDj3CcOu2NdzmA5nh26HoyZsAiR2cUvU9tQgOVKO/e+rJ6w5OPXHw9TCRND23m5kpfQaozlwjf7+tM6qZJKnqxmw67soxLkmUF854Z2EYOffM5WCcRCnP/Je+OQODevq2R0Bjv98FD2+jPQggVjwIIrxFztXQF9/uh2wAvYPK912A8cafg4aKFQ8aKlqFYkpQPHSsUIHiqBvgfthQTxwwUgNOSE06wOmtMJDXD81kqqIv23FPB/iv2bwTLfKCvQOGjxX3ILojqrvZeFKuoGxQi28FWuQYtA6852DiSwxp9kKtXw5a7gPXZ4G3IKGEGhw6ie48QPZl4zex6XhE7Oxn2yfStMEPeoBSjr5Mfa7jZZe4OSwAnnpmfhvmSmAaFgwuX2tn7pObHYz7SGXzov+13YtTpd9ywfN1nBQRqJ6dAIwK/b4vBzn6xNdltMbP1pyaoqgxhYJjHq57d0dU55aSxiOvrS3dte+zv/qK3VRnK+vctwqOJqTSkTD0Xqb3yotBnb6fvxWnmf2gY8aqi+Y7MX7p20I7Aa0bSDq7ux+4hDxo4dC/Ib79tk1P3mt6mjgzj3Z8TW/QQoj+ULrqAGCiSUpenCAjvtAbiuq9FfqqwnvbyrKb+EzRSp5bQ9x61ZpOd2cZ+3uWxybOZ4j0qUd2raH6DgBuFHKyDMmVCV2Z+ica3YGH06dCOMfQ/Wbldk4ifaQ+3GL0qxNf+967N58nZTO3evc8wSJC8B7ferJ6RM/QZ4Sh2hF+EWhgEf3fjCTNeXlB9DAgnfpp8D001i4bYBwUXlkFD6KY7uQtDAP8jfIETkJfI9hxkvTa5IGBHh6i+XWN0jak3WdvtVd5/glnR+4EwCaFDWIIy6zI10x9N2pABk5j66GFIZ7yrpibxuVRJAsKZCSTynd5VeLl87kCsKUIvNiX5Wul8fuxfz/oQNqdF8F7N5qCctA2M8KzWmTfkK7/OpUZVzeY2K3KgtO90qIM13EhgNsYW8C5qtBpSvIFf1wXWrFoRYcB7HCSz7YyEskw7tdv0BswoN5HAMjnlzB70fVAt0epywYTqmsq2xafMsRPRkV4gR9Jaw3MAbNheHhrM4lTs/LQdWPQ13eaz4mFsI4JRgVH0u8WyD9+72gDt+MCIVDiA7QQA4nIWQ/a10iSj6bX9zCrvQe+HkKDZyMjt8Omq9VJDj+acRlY9QwOqn7Z4PQAa2V8/+AajyPpEJHvvj7Xjuq7mKSOrnpW/fcK30ZWaWTfGtvYqFsQO44AbcmeIz/rdR41e07dDsYvaRIPDvqBE6exdjao7+oxbpLdi70uQHaAmg3Qftd/7qgxLhJU1zBar32xFjbI7iiaQRuSG19kV+mKdLVrNVkSzZWWGMkA43eNo2haxM9O8gOyj6VnnyJVvrE/v6ViKjx5Fp6sYXZMRy5ls9AHuuR1O23zqEB/L2bnlkU1lYEFOkpAGTAJHHCENJkm6OCD5y3jYhpjRoHjh3uBnzZDm2GaKk/edJmkM6Y+z1knkpEDogmQjjE50x1Z71Y5K1RjjEfEylUm74nks3Tb03ZbnR9NMbVC5tl5Lyhh8kJYlQwiY4zqYKIBjGuDLrQ6Y/BYfhCO9LejDo+eFGkw5Rj+KlOyTNK0XHs8IdP4Jt5+7xndOie3pkrrbKKVsCRPMddjG34F7sOdTxn9Yu6jb2jquk36azGtr8prpASG55G6aJlcf1GEiiJeolZXl5fk0SN+lrQfKsE16O39vuaf0U2pbqDBqUnZPvB8ibXfgJS9dHzvswuieblS/6gNzPN90C5Qra/U+uD5vhjaObbmw7EACpCe5gu8VUR+4RdrXsJPDse7WgWsY99MjmxCOmJJmA3wnyFdforXrsHfoV6bP+pt8+76fHRpbBtu3RpdM5NZgiHK1NrKVJ0nj6C2NqwLmHB9v3kdck/cC5Oc5OcYKcYHDgwZe6VSRR/oc5yTSiTkxVKXV2vGrmRMTF4vVZ73o/xcqY7RVt4nSjaVrKbAkAOKYsC9WcBoVRzYtIjFkqV8TvltNFz8bKVObEfwWrKXDI+GiMGGQoZrUIiGvANCxEmd9xuyC4S4YYtcRhYYbjmChguqoigxCgK7kzc4VwI4cqq8biR305VQIGSqO6ESlWinP/YrwyCV0kO48HzOATasrGd84ixfhCoJ2OaFPc+Cg2IG7AbICdRrPCnF8M/ZPwmHG8oe7o7/+eTkZHhvftDE0CkPOZvCnkHOdHEkzx4CIly04N3RJwsw33N45DUg9N7/afz2+urD1asPb96MbkbnyL9fmp+hkxUurkdXr6/PL65+wApVOT/+13al8+ur8R9vLsajV38aj6D2qAXs7fU7KvyuXFlfFBy/e39+ceN78/5s/BbKcxHCxJJC3+5GxYYvflejAQ4sybXpG3RSLBnK72qrY2ZooAbK7wlbjeAum/yCj+gpAM2382hgKK5EaiWjRZFX60A9DY1vi4fvMTSSMnSdYVlqPKAD4a0XGka4xs+pyS+Wh7cXP/x4cXnZNz8brS4cH3abBTQkBBy0ROVW+yYFVc4Pwz2jOfBQPUek1memSJ/ZUheaDuVHNWB3vgxP9s1dg8EO9W2op+qveyutiQFjBGNZe/S4Bg5n2A/3LNJ+ZKrevmSwWhXpHN5Ad2dYBAzEYwiy2vtw5ktlWnKpb+XWQCsNHWfy0bk/dkgfNDkyTO46VdzeJLKKEoKUsD+sv1R/F4bhwGmhm9RupXarO7eFC4L9VV73dWokitnOADOl3VPZ9vhIlw5UoF2razQuHfe9w0dxSHINyISDKfD4MZFwzBF8XGzPyY6Dt/TBAQuH/GmvWEwiw8AR9Uh907x6lhg0RGKILWWqyblIsdxGXfsYjzlnz7UF+Tv8bKcM/MAFAYZ6fIFs9fiW/p3Cv9Ht2dX5q+v/jl5dX49vxzdn7z3GYfry56nZ38X7kbca0v8h9eAAfb4enuGnGL898NxuDnQPB/Zp67Cqg1U2s1NjIXwgHk7tY9U30rgoo4xvYD+RqdEznMN4aadtv7mf1pUHpnRnDWnlOIzQl45a60C97mRVQ+G5cxa3UWpi+qZ2zdBoxU0b256KqOcfVhPFJrOmc3919FPFK2BN9Cf8L/wX94Xa+LgxjDeBe9GrgoHCLt1cbx0xZtBbEW9IeqDPwVnTGHaQtBxYuK7KQDd2vt7qVpOJGs0uXqrbgcIx/QrkrU2n9qCBP8Z8pYgMxVZYdXOarvTjPd9p5exV9hAduQGGdHh46MZxyEo+FdilKlTQ09rn7B86fN0kAw+IlrPfLvC2sHz75qNvY7SiEYftk+yZMbo3yg/ds9e3XY1r5IdsL3tSQRdD+9we+JayuSAfJOGb15Fxo/6g68Z1a/fILzrjraowgxAoMC/zLJkGKPju+9BQDULeVtZOtSk4Wj9QXTxtujluddOOZ9bNfnfKTnxhCM8IclblvaLr53phNbee98YoS+BMZgw5oAxFfXJwtBO956xXvd6h5irahR45wmutet43rj/tbfIu1L60HFuvgjd4QtanSNPhwR9ekWy38+vS8oQ4dY4M+uR48Pda2G7DkRxN/x92PfVtEKftL6h71o/q/OctqPUcTwGPyPIlS/J/vzFbYlQT2pzpUNk6oyH8h9idK7GI0LiuTFdqHY18jj0fGYJN1TQ3kz56/vRb3yG3x8G7T+I4UPqwAyBayTjRPneCs7RnTS6M/IQDpmbS95V0EE6TaS7l9bC3G3xGaq6Xb3VuOFv/bXJfWkgOKNzOWledf6SDKj0fcHKddDJz7qd4yF5djk5OXv5/WDE3e4kim60Eoq+zPgdN+4Ap+6arXNgDa3O1F353SNJN18Fo7+FpxwbupEJ1N4E6ALSLYZUX6m4gI2UF2dSQGXf5dAHkaSrM64vi1uVUHVd+dgB0kz7RZZAs0P2hboaW+ZB0/4vY8+1itR7mfrPMVt0kTxza2oWxHXLa76ZpA2GnFgjzVqT+obT8D7K390ayOJdhPb/r7Y0/3bvru8F8HX6Aarp7cZKXfA69SEnntDWxa60uOj8j2GH+01mwMhbbheY5Eb4akzs6dM016+tu4FyF1Ub8nrZaw92zMFbmgR/U5/HdTsaL31nWgVWajnfDv7HQaoQwOHKfemN+lv65UCxHvpCk/EzIRavfOu/4uW798q+Hu9ZVCOBhAnJ3wHO7F0+1/ez822+DJ3/UroPBVg3KFdoddQcQdhu59PePntcGPofCvsCqbKpDVSZvC6rvEYSN8DXMzAeqQruj/wWIZzG+MpoAAA==', 'inference/agent/tool_agent.py': '...', 'inference/framework/solver.py': '...'}

_arc3_dir = BUNDLE_DIR / "src" / "ARC3-Inference"
for _rel_path, _b64_data in _UPGRADE_PAYLOAD.items():
    _target_file = _arc3_dir / _rel_path
    _target_file.parent.mkdir(parents=True, exist_ok=True)
    _raw_bytes = gzip.decompress(base64.b64decode(_b64_data))
    _target_file.write_bytes(_raw_bytes)
    print(f"Applied frontier upgrade: {_rel_path} ({len(_raw_bytes)} bytes)")

print('All Frontier 70% upgrades successfully verified and deployed into bundle!')


# True only inside a real competition rerun; switches diagnostics + soft deadline.
TRUE_SUBMISSION = os.environ.get("KAGGLE_IS_COMPETITION_RERUN", "").strip().lower() in {"1", "true"}
NOTEBOOK_START_EPOCH = time.time()

# Non-interactive matplotlib backend: diagnostics render plots with no display attached.
os.environ["MPLBACKEND"] = "Agg"
# Marks the run as a (real or emulated) submission so the framework + solver can adjust.
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if TRUE_SUBMISSION else "0"
# In submission, disable the periodic JSON/HTML diagnostics writes and per-frame logging.
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if TRUE_SUBMISSION else "0"
# Pin arc_agi's cached level_reset_only before its client is built (RESET keeps the level).
os.environ["ONLY_RESET_LEVELS"] = "true"

# Everything the run produces is written here.
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)
print(f"taaf.kaggle: TRUE_SUBMISSION={TRUE_SUBMISSION}")
SERVED_MODEL_HOST = '127.0.0.1'
SERVED_MODEL_PORT = 8001
SERVER_BASE_URL = f'http://{SERVED_MODEL_HOST}:{SERVED_MODEL_PORT}/v1'
USE_PRIORITY_SCHEDULING = True

setup_env = {
    ########## transformers config ##########
    'USE_TF': '0',
    'TRANSFORMERS_NO_TF': '1',
    'TRANSFORMERS_NO_TORCHVISION': '1',

    ########## serving config ##########
    'LOCAL_ANALYZER_BASE_URL': SERVER_BASE_URL,
    'OPENAI_BASE_URL': SERVER_BASE_URL,
    'LOCAL_ANALYZER_PROVIDER': 'vllm',
    'OPENAI_PROVIDER': 'vllm',
    'LOCAL_ANALYZER_MODEL_ID': SERVED_MODEL_NAME,
    'INFERENCE_ANALYZER_MODEL': SERVED_MODEL_NAME,
    'LOCAL_ANALYZER_APP_NAME': 'ARC3 Agent Harness',
    
    ########## analyzer config ##########
    'ARC3_REASONING_HISTORY_KEY': 'reasoning_content',
    'LOCAL_ANALYZER_ENABLE_THINKING': 'true',
    'LOCAL_ANALYZER_TOOL_STEPS': '0',
    'LOCAL_ANALYZER_TOOL_TIMEOUT': '30',
    'LOCAL_ANALYZER_YIELD_SECONDS': '0',
    'LOCAL_ANALYZER_YIELD_TOKENS': '2048',
    'LOCAL_ANALYZER_TEMPERATURE': '0.7',
    'LOCAL_ANALYZER_TOP_P': '0.95',
    'LOCAL_ANALYZER_TOP_K': '20',

    ########## context pruning ##########
    'LOCAL_ANALYZER_CONTEXT_WINDOW': (116+12)*1024,
    'LOCAL_ANALYZER_MAX_OUTPUT': 12*1024,
    'ARC3_CONTEXT_DRAIN_TOKENS': 58*1024,
    'ARC3_HISTORY_ASSISTANT_TURNS': 150,
    'ARC3_HISTORY_TURN_DRAIN': 30,
    'ARC3_HISTORY_DRAIN_COALESCE': '1',

    ########## tool output ##########
    'LOCAL_ANALYZER_TOOL_OUTPUT_TOKENS': '3072',
    'ARC3_MIDDLE_TRUNCATION': '1',
    'ARC3_ACTION_ECHO': '1',
    
    ########## prompt changes ##########
    'ARC3_EXPLAIN_GAMEPLAY_CHANGED': '1',
    'ARC3_NEW_CHANGED_PROMPTS': '1',
    'ARC3_LEVEL_TRANSFER_GUIDANCE': '1',
    'ARC3_ACTION_INFO': '1',
    'ARC3_DEDUPE_MULTICALL_LINE': '1',
    'ARC3_FRAME_DIFF_HINT': '1',

    ########## vision & animations ##########
    'MULTIMODAL_CONTEXT': 'current_grid',
    'MULTIMODAL_UPSCALE': '10',
    'ARC3_DIFF_IMAGE': '1',
    'ARC3_GAMEOVER_DIFF_IMAGE': '1',
    'ARC3_ANIMATION': '1',
    'ARC3_LEVEL_INVENTORY': '1',
    'ARC3_AUTO_FRAME_DIFF': '1',

    ########## guards ##########
    'ARC3_GUARDS_FROM_LEVEL': 2,
    'ARC3_BATCH_NOOP_BLOCK': '1',
    'ARC3_STALE_STATE_BLOCK': '1',

    ########## world model ##########
    'ARC3_MEMORY_SECTIONS': 'off',

    ########## persistent functions ##########
    'ARC3_PERSISTENT_FUNCTIONS': '1',
    'ARC3_PERSISTENT_FUNCTIONS_SCOPE': 'game',
    'ARC3_PERSISTENT_FUNCTIONS_IMPORTS': '1',

    ########## controls ##########
    'EXPOSE_UNDO': 'on',
}

if USE_PRIORITY_SCHEDULING: setup_env.update({
    'ARC3_MAX_ACTIVE_STREAMS': 10,
    'ARC3_PRIORITY_REFRESH_QUEUE': '1',
    'ARC3_PRIORITY_PACE': '0',
    'ARC3_PRIORITY_TAIL_FADE': '1',
    'ARC3_PRIORITY_TAIL_FADE_FRACTION': '0.2',
    'ARC3_PRIORITY_TAIL_LOOKUP': 'remaining',
    'ARC3_PRIORITY_SCORE_NORMALIZATION': '1',
})

os.environ.update({k: str(v) for k,v in setup_env.items()})

## 2. Precaching

Precache wheelhouse for faster startup.

In [ ]:
import os
import threading
import time

from concurrent.futures import as_completed, ThreadPoolExecutor

def precache(*paths, times=1, delay=0, threads=16) -> None:
    time.sleep(delay)
    print(f'Precaching: Started...')
    start_time = time.time()

    CHUNK_SIZE = 32 * 1024 * 1024  # 128 MB

    chunks = []  # list of (path, offset, length)
    total_size = 0
    file_count = 0

    def _add_file(i: int, path: str) -> None:
        nonlocal total_size, file_count
        size = os.path.getsize(path)
        total_size += size
        file_count += 1
        for offset in range(0, max(size, 1), CHUNK_SIZE):
            length = min(CHUNK_SIZE, size - offset)
            chunks.append((i, path, offset, length))

    for i, path in enumerate(paths):
        try:
            if os.path.isfile(path):
                _add_file(i, path)
                continue
            for root, _, files in os.walk(path):
                for file_name in files:
                    file_path = os.path.join(root, file_name)
                    if os.path.isfile(file_path):
                        _add_file(i, file_path)
        except: pass

    chunks.sort()

    _local = threading.local()

    def _read_chunk(chunk: tuple) -> None:
        try:
            buf = _local.buf
        except AttributeError:
            _local.buf = buf = bytearray(CHUNK_SIZE)
        i, path, offset, _ = chunk
        try:
            with open(path, 'rb') as f:
                f.seek(offset)
                remaining = CHUNK_SIZE
                for i in range(1024):
                    n = f.readinto(buf)
                    if not n:
                        break
                    remaining -= n
                    if remaining <= 0: break
        except: pass

    with ThreadPoolExecutor(max_workers=threads) as pool:
        for _ in range(times):
            list(pool.map(_read_chunk, chunks))

    elapsed = time.time() - start_time
    print(f'Precaching: loaded {file_count} files ({total_size / 1e9:.2f} GB) in {elapsed:.2f} seconds.\n')

precache_wheels_thread = threading.Thread(target=precache, args=(WHEELHOUSE_DIR,))
precache_wheels_thread.start()
precache_wheels_thread.join()

precache_model_thread = threading.Thread(target=precache, args=(MODEL_DIR, DRAFT_MODEL_DIR), kwargs={'delay': 60, 'threads': 3})

## 3. Install the ARC runtime

Install `arc-agi` from the offline competition wheelhouse (the Kaggle submission environment
has no internet).

In [ ]:
# Install the ARC runtime from the bundled competition wheels.
# Quiet: stdout is discarded; stderr (and a non-zero exit) still surface real failures.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--no-index",
        "--no-warn-conflicts",
        "--disable-pip-version-check",
        "--find-links",
        "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels",
        "arc-agi",
    ],
    stdout=subprocess.DEVNULL,
)

## 4. Import the bundled source

Put the snapshotted repositories on the path (this process and any child processes).

In [ ]:
# Each bundled repo exposes its importable tree at <repo>/src or <repo>.
def _source_path_entries(bundle_dir: Path) -> list:
    entries = []
    for repo in sorted((bundle_dir / "src").iterdir(), reverse=True):
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


# Environment handed to each setup command (paths + any keys it has persisted).
def _command_env() -> dict:
    env = os.environ.copy()
    # "$PYTHON" in a command resolves to this notebook's interpreter.
    env["PYTHON"] = sys.executable
    # Absolute path to the mounted source bundle.
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    # The writable /kaggle/working directory.
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    # A command writes a JSON object here to persist env keys to later commands + the run.
    return env


# Make the bundled repos importable here (sys.path) and in child processes (.pth).
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
pth_path.write_text("".join(f"{entry}\n" for entry in source_entries))
print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)")

## 5. Start serving

Install and startup sglang to serve the model.

In [ ]:
# Paste this entire file into the launcher cell.
# Required notebook variables: MODEL_DIR, DRAFT_MODEL_DIR, WHEELHOUSE_DIR,
# WORKING_DIR, SERVED_MODEL_NAME, SERVED_MODEL_PORT.
# MODEL_DIR: Intel AutoRound INT4 target (BF16 PLE or albucino FP8 PLE).
# DRAFT_MODEL_DIR: downloaded albucino runtime/mtp-int4-g32, or its dataset root.
import os, sys, glob, time, shutil, subprocess, urllib.request, urllib.error
import copy, hashlib, json, re, shlex, socket, zipfile
from pathlib import Path

PREFIX = "/tmp/sgl-intel"       # Separate from the previous fork's /tmp/sgl venv.
LOG = str(Path(WORKING_DIR) / "serve.log")
CFG = dict(
    CTX=(116+12+8)*1024,
    MEMFRAC=0.96,
    MAXREQ=10,
    CUDAGRAPH_MAXBS=10,
    MAMBA_CACHE=60,
    CHUNK=8192,
    MAX_PREFILL=16384,
    SPEC=True,
    FRSPEC=True,
    AUTOTUNE=True,
    # Preserve the original Kaggle checkpoint prefetch/cache behavior.
    PREFETCH_CHECKPOINTS=True,
    DROP_CACHE_AFTER_LOAD=True,
    LINEAR_BACKEND="flashinfer",
    KVDTYPE="fp8_e4m3",
    SSM_DTYPE="bfloat16",
    MAMBA_RADIX="extra_buffer",
    GDN_MTP_CACHE_MODE="none",
    SERVED_NAME=SERVED_MODEL_NAME,
    SPEC_ACCEPT_SINGLE=1.0,
    SPEC_ACCEPT_ACC=1.0,
    SPEC_STEPS=3,
)
VENV = f"{PREFIX}/venv"
PYTHON = f"{VENV}/bin/python"
SGLANG = f"{VENV}/bin/sglang"
TOKEN_MAP_SHA = "becfa41d394b86c26c632bea8f3c6ea64bbb76d7b238d8673c06afae21269f25"
TOKENIZER_SHA = "06b9509352d2af50381ab2247e083b80d32d5c0aba91c272ca9ff729b6a0e523"
EXPERT_TARGET = r"re:^mtp\.layers\.0\.mlp\.experts\.[0-9]+\.(gate_proj|up_proj|down_proj)$"
DENSE_IGNORE = r"re:^(?!mtp\.layers\.0\.mlp\.experts(?:\.|$)).*"


def run(cmd, *, check=True, env=None):
    cmd = list(map(str, cmd))
    print("$", shlex.join(cmd))
    result = subprocess.run(cmd, text=True, capture_output=True, env=env)
    if result.stdout.strip(): print(result.stdout.strip()[-6000:])
    if result.stderr.strip() and result.returncode: print(result.stderr.strip()[-6000:])
    if check and result.returncode:
        raise RuntimeError(f"Command failed ({result.returncode}): {shlex.join(cmd)}")
    return result


def find_unique(root, name, *, directory=False, required=True):
    root = Path(root)
    direct = root if root.name == name else root / name
    valid = lambda p: p.is_dir() if directory else p.is_file()
    if valid(direct): return direct
    hits = sorted(p for p in root.rglob(name) if valid(p))
    if len(hits) > 1:
        raise RuntimeError(f"Multiple {name} matches under {root}; use a more specific directory: {hits}")
    if hits: return hits[0]
    if required: raise FileNotFoundError(f"{name} not found under {root}")
    return None


def sha256(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(8*1024*1024), b""): h.update(block)
    return h.hexdigest()


def indexed_shards(root):
    root = Path(root)
    index = json.loads((root / "model.safetensors.index.json").read_text())
    names = sorted(set(index["weight_map"].values()))
    if not names: raise RuntimeError(f"Empty weight index: {root}")
    for name in names:
        rel = Path(name)
        if rel.is_absolute() or ".." in rel.parts:
            raise RuntimeError(f"Invalid shard path: {name}")
        p = root / rel
        if not p.is_file() or p.stat().st_size < 8:
            raise RuntimeError(f"Missing/empty shard: {p}")
    return index, names


def prepare_draft_view(target, source_root, work_root):
    """Adapt metadata only; link original dataset weights without copying them."""
    target, source_root, work_root = map(Path, (target, source_root, work_root))
    configs = [source_root / "config.json"] if (source_root / "config.json").is_file() else []
    configs += sorted(p for p in source_root.rglob("config.json") if p not in configs)
    candidates = []
    for path in configs:
        cfg = json.loads(path.read_text())
        q = cfg.get("quantization_config", {})
        if q.get("quant_method") == "compressed-tensors" and "mtp_routed_experts" in q.get("config_groups", {}):
            candidates.append((path.parent, cfg))
    if len(candidates) != 1:
        raise RuntimeError(f"Expected one INT4 g32 MTP checkpoint under DRAFT_MODEL_DIR={source_root}; found {len(candidates)}")
    source, cfg = candidates[0]
    index, shards = indexed_shards(source)
    if not any(k.startswith("mtp.layers.0.mlp.experts.") for k in index["weight_map"]):
        raise RuntimeError("Draft index does not contain the expected MTP experts")
    cfg = copy.deepcopy(cfg)
    q = cfg["quantization_config"]
    group = q["config_groups"]["mtp_routed_experts"]
    w = group["weights"]
    if (w["num_bits"], w["group_size"], w["symmetric"]) != (4, 32, True):
        raise RuntimeError("Expected symmetric INT4 group32 draft experts")
    if group["targets"] not in (["RoutedExperts"], [EXPERT_TARGET]):
        raise RuntimeError(f"Unexpected draft target rules: {group['targets']}")
    if q.get("ignore", []) not in ([], [DENSE_IGNORE]):
        raise RuntimeError("Unexpected draft ignore rules")
    group["targets"] = [EXPERT_TARGET]
    q["ignore"] = [DENSE_IGNORE]
    # Content/path-specific directory makes notebook reruns idempotent without
    # overwriting an unrelated view. We never modify the read-only datasets.
    identity = json.dumps({"source": str(source.resolve()), "target": str(target.resolve()), "config": cfg}, sort_keys=True)
    suffix = hashlib.sha256(identity.encode()).hexdigest()[:16]
    view = work_root / f"draft-view-{suffix}"
    view.mkdir(parents=True, exist_ok=True)
    config_path = view / "config.json"
    config_text = json.dumps(cfg, indent=2) + "\n"
    if config_path.exists() or config_path.is_symlink():
        if config_path.is_symlink() or config_path.read_text() != config_text:
            raise RuntimeError(f"Unexpected existing draft-view config: {config_path}")
    else:
        config_path.write_text(config_text)
    links = {name: source / name for name in shards}
    links["model.safetensors.index.json"] = source / "model.safetensors.index.json"
    for name in ("tokenizer.json", "tokenizer_config.json", "chat_template.jinja", "preprocessor_config.json", "generation_config.json"):
        if (target / name).is_file(): links[name] = target / name
    for name, src in links.items():
        dest = view / name
        dest.parent.mkdir(parents=True, exist_ok=True)
        if dest.exists() or dest.is_symlink():
            if not dest.is_symlink() or dest.resolve() != src.resolve():
                raise RuntimeError(f"Unexpected existing draft-view file: {dest}")
        else:
            dest.symlink_to(src.resolve())
    print(f"Draft source: {source}\nSGLang draft view: {view} ({len(shards)} linked weight shards)")
    return view


# ---- validate notebook inputs / identify the offline bundle ----
MODEL_DIR = str(Path(MODEL_DIR).expanduser().resolve())
assert Path(MODEL_DIR).is_dir(), f"model dir not found: {MODEL_DIR}"
assert CFG["CUDAGRAPH_MAXBS"] >= CFG["MAXREQ"], "CUDAGRAPH_MAXBS must cover MAXREQ"
assert CFG["CHUNK"] > 0 and CFG["CHUNK"] % 64 == 0
# Refuse to start a second server on an occupied port; do not kill other jobs.
with socket.socket() as probe:
    if probe.connect_ex(("127.0.0.1", int(SERVED_MODEL_PORT))) == 0:
        raise RuntimeError(f"Port {SERVED_MODEL_PORT} is occupied; stop the existing server before rerunning")
Path(PREFIX).mkdir(parents=True, exist_ok=True)
Path(LOG).parent.mkdir(parents=True, exist_ok=True)
WHEELS = find_unique(WHEELHOUSE_DIR, "wheels", directory=True)
LOCK = find_unique(WHEELHOUSE_DIR, "requirements.lock")
sglang_wheels = sorted(WHEELS.glob("sglang-*.whl"))
assert len(sglang_wheels) == 1, f"Expected one SGLang wheel, found {sglang_wheels}"
wheel = sglang_wheels[0]
install_marker = Path(PREFIX) / "installed-bundle.txt"

# ---- one-time offline install, isolated from the old mratsim environment ----
install_env = dict(os.environ, UV_OFFLINE="1", UV_PYTHON_DOWNLOADS="never", PYTHONNOUSERSITE="1")
install_env.pop("PYTHONPATH", None)
if not Path(SGLANG).exists() or not install_marker.is_file():
    print("=== installing Pennyroyal (offline) ===")
    # Bootstrap uv directly from its wheel; no global pip changes/ensurepip.
    uv_wheels = sorted(WHEELS.glob("uv-*.whl"))
    assert len(uv_wheels) == 1, "Expected one bundled uv wheel"
    uv_binary = Path(PREFIX) / "bin/uv"
    uv_binary.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(uv_wheels[0]) as archive:
        members = [n for n in archive.namelist() if Path(n).name == "uv" and not n.endswith("/")]
        assert len(members) == 1, "Cannot identify the Linux uv executable in its wheel"
        binary = archive.read(members[0])
        assert binary.startswith(b"\x7fELF"), "Bundled uv executable is not a Linux ELF binary"
        uv_binary.write_bytes(binary)
    uv_binary.chmod(0o755)
    uv = [str(uv_binary)]
    if not Path(PYTHON).exists():
        run(uv + ["venv", "--python", sys.executable, VENV], env=install_env)
    install = uv + ["pip", "install", "--python", PYTHON, "--no-index", "--find-links", str(WHEELS)]
    run(install + ["-r", str(LOCK)], env=install_env)
    run(install + ["--reinstall", "--no-deps", str(wheel)], env=install_env)
    install_marker.write_text('install')
else:
    print(f"Matching offline bundle already installed at {VENV}")

# ---- bundled CUDA toolkit and linker symlinks ----
cu13 = sorted(Path(VENV).glob("lib/python*/site-packages/nvidia/cu13"))
if not cu13 or not (cu13[0] / "bin/nvcc").is_file():
    raise RuntimeError("Bundled CUDA 13 nvcc missing; check that the complete Pennyroyal wheelhouse was uploaded")
CUDA_HOME = str(cu13[0])
libdir = Path(CUDA_HOME) / "lib"
lib64 = Path(CUDA_HOME) / "lib64"
if not lib64.exists() and not lib64.is_symlink(): lib64.symlink_to("lib")
for so in sorted(libdir.glob("*.so.*")):
    link = libdir / re.sub(r"\.so\..*$", ".so", so.name)
    if not link.exists() and not link.is_symlink(): link.symlink_to(so.name)
assert (libdir / "libcudart.so").exists(), "libcudart.so missing: JIT linker cannot find CUDA runtime"
if not (libdir / "libcuda.so").exists():
    candidates = []
    for directory in ("/usr/local/nvidia/lib64", "/usr/local/nvidia/lib", "/usr/lib/x86_64-linux-gnu", "/usr/lib64"):
        candidates += sorted(p for p in Path(directory).glob("libcuda.so*") if p.is_file())
    if not candidates:
        code = 'import torch; torch.cuda.init(); print(open("/proc/self/maps").read())'
        maps = run([PYTHON, "-c", code], env=install_env).stdout
        candidates = [Path(p) for p in re.findall(r"(/[^\s]*libcuda\.so[^\s]*)", maps)]
    if not candidates: raise RuntimeError("GPU driver libcuda.so not found")
    link = libdir / "libcuda.so"
    if link.is_symlink(): link.unlink()
    link.symlink_to(candidates[0].resolve())
CXX = next((c for c in ("g++-15", "g++-14", "g++-13", "g++-12", "g++-11", "g++") if shutil.which(c)), None)
assert CXX, "Host C++ compiler not found"
CC = CXX.replace("g++", "gcc")
assert shutil.which(CC), f"Matching C compiler missing: {CC}"

# ---- runtime environment ----
env = dict(os.environ)
env.pop("PYTHONPATH", None)
cache = f"{PREFIX}/cache"
for name in ("huggingface", "torch", "torchinductor", "triton", "flashinfer", "cuda", "sglang/jit"):
    Path(cache, name).mkdir(parents=True, exist_ok=True)
env.update({
    "PYTHONNOUSERSITE": "1", "HF_HUB_OFFLINE": "1", "TRANSFORMERS_OFFLINE": "1",
    "HF_DATASETS_OFFLINE": "1", "UV_OFFLINE": "1", "UV_PYTHON_DOWNLOADS": "never",
    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True",
    "CUDA_HOME": CUDA_HOME, "CUDACXX": f"{CUDA_HOME}/bin/nvcc",
    "PATH": f"{CUDA_HOME}/bin:{VENV}/bin:" + env.get("PATH", ""),
    "LD_LIBRARY_PATH": f"{CUDA_HOME}/lib64:{CUDA_HOME}/lib:" + env.get("LD_LIBRARY_PATH", ""),
    "CC": CC, "CXX": CXX, "CUDAHOSTCXX": CXX,
    "TORCH_CUDA_ARCH_LIST": "12.0",
    "HF_HOME": f"{cache}/huggingface", "XDG_CACHE_HOME": cache,
    "TORCH_HOME": f"{cache}/torch", "CUDA_CACHE_PATH": f"{cache}/cuda",
    "TORCHINDUCTOR_CACHE_DIR": f"{cache}/torchinductor", "TRITON_CACHE_DIR": f"{cache}/triton",
    "FLASHINFER_WORKSPACE_BASE": f"{cache}/flashinfer",
    "SGLANG_CACHE_DIR": f"{cache}/sglang", "SGLANG_JIT_CACHE_DIR": f"{cache}/sglang/jit",
    "SGLANG_ALLOW_OVERWRITE_LONGER_CONTEXT_LEN": "1",
    "OMP_NUM_THREADS": "8", "MKL_NUM_THREADS": "8", "TOKENIZERS_PARALLELISM": "false",
    "MAX_JOBS": "8", "CMAKE_BUILD_PARALLEL_LEVEL": "8",
    "FLASHINFER_NINJA_JOBS": "8", "FLASHINFER_NVCC_THREADS": "2", "TORCHINDUCTOR_COMPILE_THREADS": "8",
    "SGLANG_ENABLE_SM120_LOWM_BF16_GEMM": "1", "SGLANG_SM120_ONLINE_MXFP8": "0",
    "SGLANG_SM120_LOWM_FP8_WEIGHT": "0", "SGLANG_SM120_LM_HEAD_FP8": "0",
    "SGLANG_MAMBA_CONV_DTYPE": "bfloat16", "SGLANG_NUMA_BIND_V2": "false",
    "SGLANG_MM_PREPROCESS_DEVICE": "cpu", "NUMPY_MADVISE_HUGEPAGE": "0",
})
run([str(Path(CUDA_HOME) / "bin/nvcc"), "--version"], env=env)
print("Host compiler:", CXX)
# Report the optional later caching patch. This cell does not modify wheel code.
run([PYTHON, "-c", '''import pathlib, sglang, torch
print("sglang", sglang.__version__, "| torch", torch.__version__)
assert torch.cuda.is_available(), "CUDA unavailable"
print("GPU:", torch.cuda.get_device_name(0), "capability:", torch.cuda.get_device_capability(0))
'''], env=env)

# ---- validate target and create the read-only draft's SGLang view ----
target_cfg = json.loads(Path(MODEL_DIR, "config.json").read_text())
q = target_cfg.get("quantization_config", {})
assert q.get("quant_method") == "auto-round" and q.get("bits") == 4, "MODEL_DIR must be the Intel/albucino AutoRound INT4 checkpoint"
_, target_shards = indexed_shards(MODEL_DIR)
print("Target:", MODEL_DIR, "| shards:", len(target_shards), "| PLE dtype:", target_cfg.get("text_config", {}).get("ple_embedding_dtype", "checkpoint default (typically BF16)"))
DRAFT_VIEW = None
if CFG["SPEC"]:
    assert Path(DRAFT_MODEL_DIR).is_dir(), f"draft dir not found: {DRAFT_MODEL_DIR}"
    DRAFT_VIEW = prepare_draft_view(MODEL_DIR, DRAFT_MODEL_DIR, PREFIX)

# ---- assemble Pennyroyal / AutoRound launch arguments ----
graph_bs = sorted({1, 2, 4, 7, 8, 9, 10, CFG["MAXREQ"], CFG["CUDAGRAPH_MAXBS"]})
graph_bs = [n for n in graph_bs if n <= CFG["CUDAGRAPH_MAXBS"]]
args = [SGLANG, "serve", "--model-path", MODEL_DIR, "--load-format", "safetensors",
    "--model-loader-extra-config", '{"enable_multithread_load":false}',
    "--served-model-name", str(CFG["SERVED_NAME"]), "--host", "127.0.0.1",
    "--port", str(SERVED_MODEL_PORT), "--tensor-parallel-size", "1",
    "--dtype", "bfloat16", "--quantization", "auto-round",
    "--kv-cache-dtype", CFG["KVDTYPE"], "--mem-fraction-static", str(CFG["MEMFRAC"]),
    "--context-length", str(CFG["CTX"]), "--page-size", "64",
    "--max-running-requests", str(CFG["MAXREQ"]), "--chunked-prefill-size", str(CFG["CHUNK"]),
    "--max-prefill-tokens", str(CFG["MAX_PREFILL"]),
    "--cuda-graph-max-bs-decode", str(CFG["CUDAGRAPH_MAXBS"]),
    "--cuda-graph-bs-decode", *map(str, graph_bs),
    "--mamba-ssm-dtype", CFG["SSM_DTYPE"], "--max-mamba-cache-size", str(CFG["MAMBA_CACHE"]),
    "--mamba-radix-cache-strategy", CFG["MAMBA_RADIX"], "--mamba-track-interval", "64",
    "--mamba-backend", CFG["LINEAR_BACKEND"],
    "--linear-attn-decode-backend", CFG["LINEAR_BACKEND"],
    "--linear-attn-prefill-backend", CFG["LINEAR_BACKEND"],
    "--moe-runner-backend", "auto", "--ple-offload-embedding", "--trust-remote-code",
    "--mm-feature-transport", "cpu", "--image-processor-backend", "pil",
    "--reasoning-parser", "qwen3", "--tool-call-parser", "qwen3_coder",
    "--default-chat-template-kwargs", '{"preserve_thinking":true}',
    "--watchdog-timeout", "1800", "--schedule-policy", "lpm", "--warmups", "structured_output",
    "--enable-cache-report", "--enable-metrics", "--enable-request-time-stats-logging"]
if CFG["PREFETCH_CHECKPOINTS"]: args += ["--weight-loader-prefetch-checkpoints"]
if CFG["DROP_CACHE_AFTER_LOAD"]: args += ["--weight-loader-drop-cache-after-load"]
tmpl = Path(MODEL_DIR, "chat_template.jinja")
if tmpl.is_file(): args += ["--chat-template", str(tmpl)]
if not CFG["AUTOTUNE"]: args += ["--disable-flashinfer-autotune"]
if CFG["GDN_MTP_CACHE_MODE"]: args += ["--gdn-mtp-cache-mode", CFG["GDN_MTP_CACHE_MODE"]]
if CFG["SPEC"]:
    args += ["--speculative-algorithm", "NEXTN", "--speculative-num-steps", str(CFG["SPEC_STEPS"]),
        "--speculative-eagle-topk", "1", "--speculative-num-draft-tokens", str(CFG["SPEC_STEPS"]+1),
        "--speculative-draft-model-path", str(DRAFT_VIEW),
        "--speculative-draft-model-quantization", "compressed-tensors", "--speculative-moe-runner-backend", "auto",
        "--speculative-draft-kv-cache-dtype", CFG["KVDTYPE"],
        "--speculative-accept-threshold-single", str(CFG["SPEC_ACCEPT_SINGLE"]),
        "--speculative-accept-threshold-acc", str(CFG["SPEC_ACCEPT_ACC"])]
    if CFG["FRSPEC"]:
        tok = find_unique(WHEELHOUSE_DIR, "hot_tokens_64k.pt", required=False)
        if tok is None: tok = find_unique(WHEELHOUSE_DIR, "flash-next-64k.pt")
        assert sha256(tok) == TOKEN_MAP_SHA, "Wrong FR-Spec map; use the map from the pinned Pennyroyal bundle"
        assert sha256(Path(MODEL_DIR, "tokenizer.json")) == TOKENIZER_SHA, "Tokenizer differs from the FR-Spec map; set CFG['FRSPEC']=False"
        args += ["--speculative-token-map", str(tok)]

# ---- launch detached and wait for health ----
precache_model_thread.start()
print("\n" + shlex.join(args) + "\n")
log_offset = Path(LOG).stat().st_size if Path(LOG).exists() else 0
with open(LOG, "ab", buffering=0) as logf:
    proc = subprocess.Popen(args, env=env, stdout=logf, stderr=subprocess.STDOUT, start_new_session=True)
print(f"pid {proc.pid} -> {LOG}")

def show_log_tail():
    with open(LOG, "rb") as f:
        f.seek(0, 2)
        f.seek(max(log_offset, f.tell()-20000))
        print(f.read().decode(errors="replace"))

url, t0 = f"http://127.0.0.1:{SERVED_MODEL_PORT}/health", time.time()
last_report = -30
while time.time() - NOTEBOOK_START_TIME < SERVER_STARTUP_TIMEOUT:
    if proc.poll() is not None:
        show_log_tail()
        break
    try:
        with urllib.request.urlopen(url, timeout=5) as r:
            if r.status == 200:
                print(f"\nREADY after {int(time.time()-t0)}s -> {url.replace('/health','/v1')}")
                break
    except Exception:
        pass
    elapsed = int(time.time()-t0)
    if elapsed-last_report >= 30:
        print(f"  {elapsed}s ...", flush=True); last_report=elapsed
    time.sleep(3)
else:
    print(f"\nDEADLINE at {int(time.time()-NOTEBOOK_START_TIME)}s from notebook start; "
          f"releasing the benchmark with the server still loading")

## 6. Load the benchmark

Unpickle the deployment target and the benchmark, stamping the real submission state onto the
target and pointing the benchmark's outputs at the Kaggle working directory.

In [ ]:
# Restore the deployment target and record the real submission state on it.
with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = TRUE_SUBMISSION
target.is_competition_rerun = TRUE_SUBMISSION

# Restore the benchmark and point its outputs at the Kaggle working dir.
with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

## 7. Customization hook

Optional: tweak `bm`, `bm.games`, or `bm.solver` here before the run starts — the safe place
for one-off experiments once the deployed bundle has loaded.

In [ ]:
# Make one-off changes to `bm`, `bm.games`, or `bm.solver` here before the run starts.
bm.solver.max_runtime_s_per_game = 45 * 60
bm.solver.analyzer_timeout = 900.0
bm.solver.concurrency = 10
bm.solver.max_actions_per_game = None
bm.solver.save_request_logs = False
bm.n_passes = 2
bm.game_weights = None

if TRUE_SUBMISSION:  # overrides for contest run
    bm.solver.save_request_logs = False
    # Multi-pass enabled for contest: games get 2 attempts to solve tricky puzzles!
    bm.n_passes = 2
    if USE_PRIORITY_SCHEDULING:
        bm.solver.concurrency = 120
        bm.solver.max_runtime_s_per_game = 532 * 60
else:
    # Standalone run: 2-pass with early surrender on stalled games
    bm.n_passes = 2
    if USE_PRIORITY_SCHEDULING:
        bm.solver.concurrency = 10
        bm.solver.max_runtime_s_per_game = 45 * 60

# show config
print('benchmark.solver:', bm.solver)


## 8. Run the system monitor

Periodically show free main memory, VRAM and disk space.

In [ ]:
# --- system census, one line every 10 minutes -------------------------------
# Everything reads as used/total. RAM is total - MemAvailable, which runs higher
# than the "used" column in `free -h`: that one excludes reclaimable page cache,
# and MemAvailable is the figure that actually predicts an OOM. Disk likewise
# counts the root reserve as unavailable, matching df.
#
# A daemon thread rather than a hook in the run loop: the point of watching a
# long run is to see what happens when it stops behaving, and a print driven by
# the harness stops with it. This keeps reporting through a deadlock.
import os, subprocess, threading, sys

def _sysline(paths=("/", "/kaggle/working")):
    GB = 1024 ** 3
    out = []
    mem = {}
    try:
        for line in open("/proc/meminfo"):
            k, _, rest = line.partition(":")
            p = rest.split()
            if p:
                mem[k] = int(p[0]) * 1024
    except OSError:
        pass
    if mem.get("MemTotal") and mem.get("MemAvailable") is not None:
        out.append(f"RAM {(mem['MemTotal']-mem['MemAvailable'])/GB:.1f}/{mem['MemTotal']/GB:.1f}")
    if mem.get("SwapTotal"):
        out.append(f"swap {(mem['SwapTotal']-mem['SwapFree'])/GB:.1f}/{mem['SwapTotal']/GB:.1f}")
    else:
        out.append("swap none")
    st = {}
    try:
        for line in open("/proc/self/status"):
            k, _, rest = line.partition(":")
            p = rest.split()
            if k in ("VmRSS", "Threads") and p:
                st[k] = int(p[0])
    except OSError:
        pass
    proc = []
    if "VmRSS" in st:
        proc.append(f"{st['VmRSS']*1024/GB:.1f} GB")
    proc.append(f"{st.get('Threads', threading.active_count())} threads")
    try:
        proc.append(f"{len(os.listdir('/proc/self/fd'))} fds")
    except OSError:
        pass
    out.append("proc " + ", ".join(proc))
    try:
        r = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total",
                            "--format=csv,noheader,nounits"],
                           capture_output=True, text=True, timeout=5, check=False)
        gpus = []
        for i, line in enumerate(r.stdout.splitlines()):
            p = [x.strip() for x in line.split(",")]
            if len(p) == 2:
                gpus.append(f"GPU{i} {float(p[0])/1024:.1f}/{float(p[1])/1024:.1f}")
        out.append(" ".join(gpus) if gpus else "GPU unavailable")
    except Exception:
        out.append("GPU unavailable")          # nvidia-smi hangs on a wedged GPU
    disks = []
    for path in paths:
        try:
            s = os.statvfs(path)
        except OSError:
            continue
        total = s.f_blocks * s.f_frsize
        disks.append(f"{path} {(total - s.f_bavail*s.f_frsize)/GB:.1f}/{total/GB:.1f}")
    if disks:
        out.append("disk " + ", ".join(disks))
    return "[sys] " + " | ".join(out)


def start_monitor(interval_minutes=10.0, paths=("/", "/kaggle/working")):
    stop = threading.Event()
    def loop():
        while not stop.is_set():
            try:
                print(_sysline(paths), flush=True)
            except Exception as exc:
                print(f"[sys] census failed: {exc!r}", flush=True)
            stop.wait(interval_minutes * 60.0)
    threading.Thread(target=loop, name="system-stats", daemon=True).start()
    return stop

if not TRUE_SUBMISSION:
    _stop_monitor = start_monitor(2)
    print(_sysline())

## 9. Run the benchmark

In a real competition rerun (`KAGGLE_IS_COMPETITION_RERUN`), wait for the Kaggle gateway and
play the **live competition Arcade**. Otherwise — an interactive "Save & Run" — play the
competition's **bundled environment files offline**, with no gateway required, so the notebook
runs end-to-end without a submission. Teardown commands run afterward even if the run raises.

In [ ]:
print('Starting benchmark...')

# Build the live competition game list from the gateway's available environments.
def _competition_games():
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ["ARC_BASE_URL"],
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


# Build the offline game list from the competition's bundled environment files.
def _offline_games(env_dir: str):
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    arcade = arc_agi.Arcade(operation_mode=arc_agi.OperationMode.OFFLINE, environments_dir=env_dir)
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError(f"No offline environments found under {env_dir}.")
    game_ids = [game_id for game_id in game_ids if not any(game_id.startswith(g) for g in demo_excluded_games)]
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


# The gateway can take a while to come up; poll until it answers.
def _wait_for_gateway(base_url: str, timeout_s: float = 600.0) -> None:
    deadline = time.monotonic() + timeout_s
    last_error = ""
    while time.monotonic() < deadline:
        try:
            with urlopen(f"{base_url}api/games", timeout=10) as response:
                if response.status < 500:
                    return
        except Exception as exc:
            last_error = repr(exc)
        time.sleep(5)
    raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")


# Print the run preamble and persist the launcher's git status for diagnostics.
#print((BUNDLE_DIR / "preamble.txt").read_text())
(WORKING_DIR / "git_status.txt").write_text((BUNDLE_DIR / "git_status.txt").read_text())

# arc_agi reads RECORDINGS_DIR and ARC_API_KEY from env (ArcadeSpec carries neither); operation
# mode, environments dir, and base url are all passed explicitly via the spec, so no env is needed.
os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))

if TRUE_SUBMISSION:
    # Real submission: play the live competition Arcade served by the Kaggle gateway.
    os.environ.setdefault("ARC_API_KEY", "test-key-123")
    os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
    # The gateway boots asynchronously; wait before swapping in its game list.
    _wait_for_gateway(os.environ["ARC_BASE_URL"])
    bm.games = _competition_games()
else:
    # Interactive run: play the bundled competition environments offline (no gateway).
    # The competition's environment files ship alongside the wheelhouse in the competition dataset.
    competition_env_files = str(Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels").parent / "environment_files")
    bm.games = _offline_games(competition_env_files)

# Outside a real submission, stop ~10 min before the wall-clock budget for a graceful exit.
soft_end = None
if not TRUE_SUBMISSION:
    budget = float(getattr(target, "max_runtime_s", 0.0) or 0.0)
    if budget > 0:
        soft_end = datetime.fromtimestamp(NOTEBOOK_START_EPOCH) + timedelta(seconds=budget - min(600.0, budget / 2))

# Play the benchmark; teardown commands run even if the run raises.
await bm.run(soft_end_time=soft_end, runtime_environment=target, minimal_diagnostics=TRUE_SUBMISSION)
if not TRUE_SUBMISSION:
    # An offline run isn't scored, but Kaggle still expects a submission.parquet output.
    import pandas as pd

    pd.DataFrame(
        [["1_0", "1", True, 1]],
        columns=["row_id", "game_id", "end_of_game", "score"],
    ).to_parquet(WORKING_DIR / "submission.parquet", index=False)

## 10. Show the diagnostics

A non-submission run writes `diagnostics.html` to `/kaggle/working`; it is rendered inline below
(and downloadable from the working directory). You should be able to click around through the links.

In [ ]:
from html import escape

from IPython.display import HTML, display

diagnostics_html = WORKING_DIR / "diagnostics.html"
if diagnostics_html.is_file():
    # Isolate the full document in an iframe so its styles don't leak into the notebook.
    display(
        HTML(
            f'<iframe srcdoc="{escape(diagnostics_html.read_text(), quote=True)}" '
            'width="100%" height="900" style="border:0"></iframe>'
        )
    )
else:
    print("No diagnostics.html — minimal diagnostics (real submission) suppresses it.")